In [ ]:
import matplotlib
import pandas as pd 
from scipy.spatial import distance
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
from itertools import combinations
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
import matplotlib    
from scipy.stats import fisher_exact
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#3D5A80","#C03830"])
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
from sklearn.cluster import HDBSCAN
import ripleyk
import matplotlib.patches as patchest
# from pointpats import (
#     distance_statistics,
#     QStatistic,
#     random,
#     PointPattern,
# )
#from pointpats import PointPattern, PoissonPointProcess
#import pointpats
from rdfpy import rdf
from matplotlib.patches import Rectangle
from sklearn.cluster import OPTICS
from scipy.spatial import Voronoi, voronoi_plot_2d
from shapely.geometry import Polygon,Point,LineString
from scipy.spatial import ConvexHull
from intersect import intersection
from scipy.spatial.distance import pdist

import matplotlib.colors as mcolors
from random import randrange
import math 
import matplotlib as mpl
import matplotlib.cm as cmס;

In [ ]:
ww = matplotlib.colors.LinearSegmentedColormap.from_list(" ", ["#B9B6B6","#18827E","#B34F91"])
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#B9B6B6',"#D55273","#3D8BD5"])


In [ ]:
def flatten(l):
    return [item for sublist in l for item in sublist]
def df2clustersize(df):
    r = []
    for i in list(set(df[(df.db!=-1)&(~pd.isna(df.db))].db)):
        D=pdist(df.loc[df[df.db==i].index,['x','y']].to_numpy())
        r.append(D.max())#D.max()/len(df[df.db==i]))#D.max()
    return r #r.append({'val':D.max(),'ele':'WT','norm':D.max()/len(df[df.db==i])})
def area2clusters(df,element,state='bobby',squre_number=0):
    #print(df[~pd.isna(df.db)].db.value_counts().reset_index())
    x = df[~pd.isna(df.db)].db.value_counts().reset_index().sort_values(by='db').iloc[1:].copy()#index
    x['element'] = element
    x['state'] = state
    x['squre_number'] = squre_number
    x['cluster_size'] = x['db']
    return x.loc[:,['cluster_size','state','element','squre_number']].reset_index(drop=True)
def cluster_fraction(df):
    x = df[~pd.isna(df.db)].db.value_counts().reset_index().sort_values(by='db').iloc[1:].copy()#index
    val = []
    for i in x.index:
        #print(x.loc[i])
        free_spots = len(df[(df.db!=x.loc[i]['db'])&(pd.isna(df.db)|(df.db==-1))])#index
        val.append((x.loc[i].db)/free_spots)
    return val
def df2clustersize(df):
    r = []
    for i in list(set(df[(df.db!=-1)&(~pd.isna(df.db))].db)):
        D=pdist(df.loc[df[df.db==i].index,['x','y']].to_numpy())
        r.append(D.max())#D.max()/len(df[df.db==i]))#D.max()
    return r #r.append({'val':D.max(),'ele':'WT','norm':D.max()/len(df[df.db==i])})
def voronoi_volumes(points):
    v = Voronoi(points)
    vol = np.zeros(v.npoints)
    for i, reg_num in enumerate(v.point_region):
        indices = v.regions[reg_num]
        if -1 in indices: # some regions can be opened
            vol[i] = np.inf
        else:
            vol[i] = ConvexHull(v.vertices[indices]).volume
    return vol    

def ThnderSTORM2df(file,squre_size=50000,x1=0,y1=0,parent='APP',element='WT',cell=1,rectangle=1,replicate=1):
    s= pd.read_csv('csv/v2/%s.csv'%file)#5000 pixes = 1micron
    x2 = x1+squre_size
    y2 = y1+squre_size
    sub_df = s[(s['uncertainty [nm]']<20)&(s.frame>=200)].copy()#.sample(10000)
    #sub_df = s[(s['uncertainty [nm]']<20)&(s.frame.between(s.frame.max()-10000,s.frame.max()))].copy()
    fig = plt.figure(figsize = (12,6),constrained_layout=False)
    axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.3,'wspace':.2})
    
    sns.scatterplot(x='x [nm]',y='y [nm]',data=sub_df,s=.15,edgecolor=None,c='k',ax=axs[0],rasterized=False)#18827E
    axs[0].add_patch(Rectangle((x1,y1),squre_size,squre_size,linewidth=2,edgecolor='#C03830',facecolor='none'))
    ss = sub_df[(sub_df['x [nm]'].between(x1,x2))&
                    (sub_df['y [nm]'].between(y1,y2))].copy()

    sns.scatterplot(x='x [nm]',y='y [nm]',data=ss,edgecolor=None,c='k',alpha=1,s=4,ax=axs[1])
    axs[1].set_xlim([x1,x1+squre_size])
    axs[1].set_ylim([y1,y1+squre_size])
    axs[1].legend([],frameon=False)
    axs[1].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
    #axs[1].axis('off')
    axs[0].axis('off')
    axs[0].set_title('%s'%file)
    axs[0].plot([40000,50000],[10000,10000],color='k')
    plt.savefig('plots/%s_%d_%d.svg'%(file,x1,x2),bbox_inches='tight',transparent=False)
    plt.savefig('plots/pngs/%s_%d_%d.png'%(file,x1,x2),bbox_inches='tight',transparent=False,dpi=600)
    ss['x'] = ss['x [nm]']
    ss['y'] = ss['y [nm]']
    ss['t'] =ss.frame.astype(int)
    ss['parent'] =parent
    ss['element'] =element
    ss['cell'] =cell
    ss['rectangle'] =rectangle
    ss['replicate'] =replicate
    ss['name'] =file
    ss['intensity'] = ss['intensity [photon]']
    return ss.loc[:,['x','y','intensity','parent','element','cell','rectangle','replicate','name']]


In [ ]:
dAPP_1_2_1 = ThnderSTORM2df('dAPP_1_2',squre_size=4000,x1=24000,y1=28000,parent='APP',element='MUT',cell=2,rectangle=1,replicate=1)
APP_2_2_3 = ThnderSTORM2df('APP_2_2',squre_size=4000,x1=25000,y1=18000,parent='APP',element='WT',cell=2,rectangle=1,replicate=2)
APP_1_3_3 = ThnderSTORM2df('APP_1_3',squre_size=4000,x1=23000,y1=16000,parent='APP',element='WT',cell=3,rectangle=3,replicate=1)
APP_1_6_2 = ThnderSTORM2df('APP_1_6',squre_size=4000,x1=21000,y1=21000,parent='APP',element='WT',cell=6,rectangle=2,replicate=1)
APP_1_6_2#squre_size=4000,x1=25000,y1=18000

In [ ]:
#x = pd.read_csv('csv/v2/APP_1_2.csv')
#APP_1_6_1 = ThnderSTORM2df('APP_1_6',squre_size=4000,x1=17000,y1=21000,parent='APP',element='WT',cell=6,rectangle=1,replicate=1)
#dAPP_1_2_2 = voro2cluster('dAPP_1_2',squre_size=4000,x1=23000,y1=24000,parent='APP',element='MUT',cell=2,rectangle=2,replicate=1)

import matplotlib.colors as colors


In [ ]:
a_df = pd.read_csv('csv/v2/APP_1_6.csv')
x_APP = a_df[(a_df['x [nm]'].between(17000,17000+4000))&
            (a_df['y [nm]'].between(21000,21000+4000))].sample(2053)
x_A = x_APP['x [nm]']
y_A = x_APP['y [nm]']
fig, ax = plt.subplots(figsize=(3,3))
sns.histplot(x=x_A,y=y_A,bins=30,cbar=1,cmap='inferno',vmax=70,vmin=1)
#sns.scatterplot(x=x_A,y=y_A,s=1)

ax.set_facecolor('xkcd:black')


In [ ]:
len(x_APP)

In [ ]:
d_df = pd.read_csv('csv/v2/dAPP_1_2.csv')
x_dAPP = d_df[(d_df['x [nm]'].between(23000,23000+4000))&
            (d_df['y [nm]'].between(24000,24000+4000))].sample(2053)
x_D = x_dAPP['x [nm]']
y_D = x_dAPP['y [nm]']
norm = colors.Normalize(vmin=-500,vmax=4000)
fig, ax = plt.subplots(figsize=(3,3))
sns.histplot(x=x_D,y=y_D,bins=30,cbar=1,cmap='inferno',vmax=70,vmin=1)
ax.set_facecolor('xkcd:black')


In [ ]:
def voro2cluster(file,squre_size=50000,x1=0,y1=0,parent='APP',element='WT',cell=1,rectangle=1,replicate=1):
    s= pd.read_csv('csv/v2/%s.csv'%file)#5000 pixes = 1micron
    x2 = x1+squre_size
    y2 = y1+squre_size
    #xx[(ss.frame.between(ss.frame.max()-400,ss.frame.max()))]
    sub_df = s[(s['uncertainty [nm]']<20)&(s.frame>=200)].copy()#.sample(10000)
    #sub_df = s[(s['uncertainty [nm]']<20)&(ss.frame.between(ss.frame.max()-400,ss.frame.max()))].copy()
    fig = plt.figure(figsize = (3,9),constrained_layout=False)
    axs = fig.subplots(nrows=3, ncols=1,gridspec_kw = {'hspace':.01,'wspace':.1})
    
    sns.scatterplot(x='x [nm]',y='y [nm]',data=sub_df[(sub_df['x [nm]'].between(x1-5000,x2+5000))&
                    (sub_df['y [nm]'].between(y1-5000,y2+5000))],s=.05,edgecolor=None,c='k',ax=axs[0],rasterized=True)
    axs[0].add_patch(Rectangle((x1,y1),squre_size,squre_size,linewidth=1,edgecolor='#18827E',facecolor='none'))
    #axs[0].plot([x1,x1-5000],[y1-9500,y1-9500],color='k')
    axs[0].set_facecolor('xkcd:white')
    ss = sub_df[(sub_df['x [nm]'].between(x1,x2))&
                    (sub_df['y [nm]'].between(y1,y2))].copy()
    axs[0].tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    axs[0].set_xlabel('')
    axs[0].set_ylabel('')
    ss['x'] = ss['x [nm]']
    ss['y'] = ss['y [nm]']
    ss['t'] =ss.frame.astype(int)
    ss['parent'] =parent
    ss['element'] =element
    ss['cell'] =cell
    ss['rectangle'] =rectangle
    ss['replicate'] =replicate
    ss['name'] =file
    df,x,thershold,_,cluster_probabilty = voronoi_thershold_v2(ss.loc[:,['x','y']].to_numpy(),n=1,size_of_grid=4000,min_cluster_size=5)
    
    sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=5,edgecolor=None,hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=axs[1],rasterized=True)
    axs[1].legend([],frameon=False)
    # norm = plt.Normalize(vmin=-6, vmax=-2)
    # sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
    # sm.set_array([])
    # #ax.get_legend().remove()
    # axs[1].figure.colorbar(sm,fraction=0.03, pad=0.01)
    points = ss.loc[:,['x','y']].to_numpy()
    vor = Voronoi(points)
    ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=axs[1],rasterized=False)
    axs[1].set_xlim([df.x.min(),df.x.max()])
    axs[1].set_ylim([df.y.min(),df.y.max()])
    axs[1].tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    axs[1].set_xlabel('')
    axs[1].set_ylabel('')
    samp = len(set(df[~pd.isna(df.db)].db))-1
    cmap_test = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#F55536",'#0C8346','#B8B8B8','#8D5A97','#274C77'],samp)
    color1 = [mcolors.rgb2hex(cmap_test(i)) for i in range(samp)]
    color11 = pd.Series(color1).sample((samp)).to_list()
    sns.scatterplot(x='x',y='y',data=df[(df.db!=-1)&(~pd.isna(df.db))],s=10,edgecolor=None,hue='db', 
                    hue_norm=(0,len(set(df[(df.db!=-1)&(~pd.isna(df.db))].db))),palette=cmap_test,ax=axs[2],rasterized=True)
    sns.scatterplot(x='x',y='y',data=df[((df.db==-1)|(pd.isna(df.db)))],s=1,edgecolor=None,c='k',alpha=.1,ax=axs[2],rasterized=True)
    ax.legend([],frameon=False)
    plt.xlim([df.x.min(),df.x.max()])
    plt.ylim([df.y.min(),df.y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    axs[2].legend([],frameon=False)
    
    plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/figures/%s_%d_%d.png'%(file,cell,rectangle),bbox_inches='tight',transparent=True,dpi=600)#cell=1,rectangle=1,replicate=1
    #plt.savefig('plots/pngs/%s_%d_%d.png'%(file,x1,x2),bbox_inches='tight',transparent=False)
    return ss.loc[:,['x','y','parent','element','cell','rectangle','replicate','name']]


In [ ]:
dAPP_1_2_1 = voro2cluster('dAPP_1_2',squre_size=4000,x1=24000,y1=28000,parent='APP',element='MUT',cell=2,rectangle=1,replicate=1)


In [ ]:
dAPP_1_2_1

In [ ]:
def voronoi_thershold_ploting(points,n=10,size_of_grid=4000):
    l = []
    thershold = []
    nine_th  = np.percentile(voronoi_volumes(points),90)
    b = (int((nine_th)/100))
    counts_GT, vol_GT = np.histogram([i for i in voronoi_volumes(points) if i<nine_th],bins=[int(i) for i in range(0,int(nine_th),b)])
    for i in enumerate(vol_GT[:-1]):
            l.append({'element':'Ground_truth','x':i[1],'y':counts_GT[i[0]]})
    for j in range(n):
        rand_vor = voronoi_volumes(np.random.uniform(0, size_of_grid, size=(len(points),2)))
        counts_Random, vol_Random  =  np.histogram([i for i in rand_vor if i<nine_th],
                                                   bins=[int(i) for i in range(0,int(nine_th),b)]) 
        thershold.append(np.percentile(rand_vor,1))
        for i in enumerate(vol_Random[:-1]): 
            l.append({'element':'random','x':i[1],'y':counts_Random[i[0]]})
    voronoi_thershold_df =  pd.DataFrame(l)    
    
    # xx = voronoi_thershold_df.groupby(['element','x']).mean().reset_index()
    # line1= [xx[xx.element=='random'].x.to_list(),xx[xx.element=='random'].y.to_list()]
    # line2= [xx[xx.element!='random'].x.to_list(),xx[xx.element!='random'].y.to_list()]
    # x, y = intersection(line1[0], line1[1], line2[0], line2[1])
    # thershold = x[0]
    ll = []
    vor = Voronoi(points)
    for i,j,h in zip(vor.points,voronoi_volumes(points),vor.point_region):
        ll.append({'x':i[0],'y':i[1],'vol':j,'point_region':h})
    return voronoi_thershold_df,np.mean(thershold),pd.DataFrame(ll)

In [ ]:
p = np.random.uniform(0, 4000, size=(1000,2))
v = Voronoi(p)
ll =[]    
for i,j,h in zip(v.points,voronoi_volumes(p),vor.point_region):
        ll.append({'x':i[0],'y':i[1],'vol':j,'point_region':h})
sns.scatterplot(x='x',y='y',data=pd.DataFrame(ll))

In [ ]:
def voronoi_thershold_v2(points,n=10,size_of_grid=10000,min_cluster_size=5):

    five = []
    l = []
    l3 = []
    for i in range(n):
        r_dist = np.random.uniform(0, size_of_grid, size=(len(points),2))
        random_dist = [i for i in voronoi_volumes(r_dist) if i<1e9]
        for j in random_dist:
            l.append({'vol':j,'trial':i})
        five.append(np.percentile(random_dist, 1))   
        v = Voronoi(r_dist)
        for y,j,h in zip(v.points,voronoi_volumes(r_dist),v.point_region):
            l3.append({'x':y[0],'y':y[1],'vol':j,'point_region':h,'trial':i})
    random_thershold_df =  pd.DataFrame(l)       

    ll =[]
    vor = Voronoi(points)
    for i,j,h in zip(vor.points,voronoi_volumes(points),vor.point_region):
        ll.append({'x':i[0],'y':i[1],'vol':j,'point_region':h})
    thershold = np.mean(five)
    df = pd.DataFrame(ll)
    if len(df)<min_cluster_size:
        min_cluster_size = len(df)
    df['vol_invers'] =np.log10(1/df['vol'])
    hdb = HDBSCAN(min_cluster_size=min_cluster_size)
    hdb.fit(df[df.vol<thershold].loc[:,['x','y']])
    #hdb = DBSCAN(eps=50, min_samples=3).fit(df[df.vol<thershold].loc[:,['x','y']])
    df.loc[df[df.vol<thershold].index,['db']] = hdb.labels_
    z = area2clusters(df,element='Bobby',squre_number=0)
    return df,random_thershold_df, np.mean(five),z['cluster_size']/len(df),pd.DataFrame(l3)




In [ ]:
dAPP_1_2_1 = voro2cluster('dAPP_1_2',squre_size=4000,x1=24000,y1=28000,parent='APP',element='MUT',cell=2,rectangle=1,replicate=1)
dAPP_1_2_2 = voro2cluster('dAPP_1_2',squre_size=4000,x1=23000,y1=24000,parent='APP',element='MUT',cell=2,rectangle=2,replicate=1)
dAPP_1_2_3 = voro2cluster('dAPP_1_2',squre_size=4000,x1=19500,y1=22000,parent='APP',element='MUT',cell=2,rectangle=3,replicate=1)
dAPP_1_2_4 = voro2cluster('dAPP_1_2',squre_size=4000,x1=19500,y1=18000,parent='APP',element='MUT',cell=2,rectangle=4,replicate=1)

dAPP_1_3_1 = voro2cluster('dAPP_1_3',squre_size=4000,x1=18000,y1=30000,parent='APP',element='MUT',cell=3,rectangle=1,replicate=1)
dAPP_1_3_2 = voro2cluster('dAPP_1_3',squre_size=4000,x1=22000,y1=30000,parent='APP',element='MUT',cell=3,rectangle=2,replicate=1)
dAPP_1_3_3 = voro2cluster('dAPP_1_3',squre_size=4000,x1=21000,y1=26000,parent='APP',element='MUT',cell=3,rectangle=3,replicate=1)

dAPP_1_4_1 = voro2cluster('dAPP_1_4',squre_size=4000,x1=20000,y1=15000,parent='APP',element='MUT',cell=4,rectangle=1,replicate=1)
dAPP_1_4_2 = voro2cluster('dAPP_1_4',squre_size=4000,x1=28000,y1=15000,parent='APP',element='MUT',cell=4,rectangle=2,replicate=1)
dAPP_1_4_3 = voro2cluster('dAPP_1_4',squre_size=4000,x1=32000,y1=15000,parent='APP',element='MUT',cell=4,rectangle=3,replicate=1)
dAPP_1_4_4 = voro2cluster('dAPP_1_4',squre_size=4000,x1=25000,y1=23000,parent='APP',element='MUT',cell=4,rectangle=4,replicate=1)

dAPP_1_5_1 = voro2cluster('dAPP_1_5',squre_size=4000,x1=20000,y1=28000,parent='APP',element='MUT',cell=5,rectangle=1,replicate=1)
dAPP_1_5_2 = voro2cluster('dAPP_1_5',squre_size=4000,x1=4000,y1=32000,parent='APP',element='MUT',cell=5,rectangle=2,replicate=1)
dAPP_1_5_3 = voro2cluster('dAPP_1_5',squre_size=4000,x1=31500,y1=27500,parent='APP',element='MUT',cell=5,rectangle=3,replicate=1)


dAPP_2_2_1 = voro2cluster('dAPP_2_2',squre_size=4000,x1=31000,y1=15000,parent='APP',element='MUT',cell=2,rectangle=1,replicate=2)
dAPP_2_2_2 = voro2cluster('dAPP_2_2',squre_size=4000,x1=31000,y1=19000,parent='APP',element='MUT',cell=2,rectangle=2,replicate=2)
dAPP_2_2_3 = voro2cluster('dAPP_2_2',squre_size=4000,x1=18000,y1=25000,parent='APP',element='MUT',cell=2,rectangle=3,replicate=2)

dAPP_2_1_1 = voro2cluster('dAPP_2_1',squre_size=4000,x1=26000,y1=25000,parent='APP',element='MUT',cell=1,rectangle=1,replicate=2)
dAPP_2_1_2 = voro2cluster('dAPP_2_1',squre_size=4000,x1=22000,y1=22000,parent='APP',element='MUT',cell=1,rectangle=2,replicate=2)

dAPP_2_3_1 = voro2cluster('dAPP_2_3',squre_size=4000,x1=14000,y1=33000,parent='APP',element='MUT',cell=3,rectangle=1,replicate=2)
dAPP_2_3_2 = voro2cluster('dAPP_2_3',squre_size=4000,x1=14000,y1=29000,parent='APP',element='MUT',cell=3,rectangle=2,replicate=2)
dAPP_2_3_3 = voro2cluster('dAPP_2_3',squre_size=4000,x1=34000,y1=21000,parent='APP',element='MUT',cell=3,rectangle=3,replicate=2)
dAPP_2_3_4 = voro2cluster('dAPP_2_3',squre_size=4000,x1=30000,y1=21000,parent='APP',element='MUT',cell=3,rectangle=4,replicate=2)

dAPP_2_6_1 = voro2cluster('dAPP_2_6',squre_size=4000,x1=30000,y1=25000,parent='APP',element='MUT',cell=6,rectangle=1,replicate=2)
dAPP_2_6_2 = voro2cluster('dAPP_2_6',squre_size=4000,x1=34000,y1=25000,parent='APP',element='MUT',cell=6,rectangle=2,replicate=2)
dAPP_2_6_3 = voro2cluster('dAPP_2_6',squre_size=4000,x1=35000,y1=32000,parent='APP',element='MUT',cell=6,rectangle=3,replicate=2)



APP_1_6_1 = voro2cluster('APP_1_6',squre_size=4000,x1=17000,y1=21000,parent='APP',element='WT',cell=6,rectangle=1,replicate=1)
APP_1_6_2 = voro2cluster('APP_1_6',squre_size=4000,x1=21000,y1=21000,parent='APP',element='WT',cell=6,rectangle=2,replicate=1)

APP_1_5_1 = voro2cluster('APP_1_5',squre_size=4000,x1=23000,y1=25000,parent='APP',element='WT',cell=5,rectangle=1,replicate=1)
APP_1_5_2 = voro2cluster('APP_1_5',squre_size=4000,x1=27000,y1=25000,parent='APP',element='WT',cell=5,rectangle=2,replicate=1)

APP_1_4_1 = voro2cluster('APP_1_4',squre_size=4000,x1=20000,y1=23000,parent='APP',element='WT',cell=4,rectangle=1,replicate=1)
APP_1_4_2 = voro2cluster('APP_1_4',squre_size=4000,x1=16000,y1=23000,parent='APP',element='WT',cell=4,rectangle=2,replicate=1)
APP_1_4_3 = voro2cluster('APP_1_4',squre_size=4000,x1=16000,y1=19000,parent='APP',element='WT',cell=4,rectangle=3,replicate=1)


APP_1_2_1 = voro2cluster('APP_1_2',squre_size=4000,x1=25000,y1=27000,parent='APP',element='WT',cell=2,rectangle=1,replicate=1)
APP_1_2_2 = voro2cluster('APP_1_2',squre_size=4000,x1=25000,y1=17000,parent='APP',element='WT',cell=2,rectangle=2,replicate=1)
APP_1_2_3 = voro2cluster('APP_1_2',squre_size=4000,x1=25000,y1=13000,parent='APP',element='WT',cell=2,rectangle=3,replicate=1)
APP_1_2_4 = voro2cluster('APP_1_2',squre_size=4000,x1=33000,y1=13000,parent='APP',element='WT',cell=2,rectangle=4,replicate=1)

APP_1_3_1 = voro2cluster('APP_1_3',squre_size=4000,x1=29000,y1=23000,parent='APP',element='WT',cell=3,rectangle=1,replicate=1)
APP_1_3_2 = voro2cluster('APP_1_3',squre_size=4000,x1=29000,y1=19000,parent='APP',element='WT',cell=3,rectangle=2,replicate=1)
APP_1_3_3 = voro2cluster('APP_1_3',squre_size=4000,x1=23000,y1=16000,parent='APP',element='WT',cell=3,rectangle=3,replicate=1)
APP_1_3_4 = voro2cluster('APP_1_3',squre_size=4000,x1=21000,y1=21500,parent='APP',element='WT',cell=3,rectangle=4,replicate=1)


APP_2_2_1 = voro2cluster('APP_2_2',squre_size=4000,x1=21000,y1=21000,parent='APP',element='WT',cell=2,rectangle=1,replicate=2)
APP_2_2_2 = voro2cluster('APP_2_2',squre_size=4000,x1=20000,y1=24000,parent='APP',element='WT',cell=2,rectangle=2,replicate=2)
APP_2_2_3 = voro2cluster('APP_2_2',squre_size=4000,x1=25000,y1=18000,parent='APP',element='WT',cell=2,rectangle=3,replicate=2)
APP_2_2_4 = voro2cluster('APP_2_2',squre_size=4000,x1=15000,y1=30000,parent='APP',element='WT',cell=2,rectangle=4,replicate=2)

APP_2_3_1 = voro2cluster('APP_2_3',squre_size=4000,x1=23000,y1=30000,parent='APP',element='WT',cell=3,rectangle=1,replicate=2)
APP_2_3_2 = voro2cluster('APP_2_3',squre_size=4000,x1=23000,y1=26000,parent='APP',element='WT',cell=3,rectangle=2,replicate=2)
APP_2_3_3 = voro2cluster('APP_2_3',squre_size=4000,x1=25000,y1=22000,parent='APP',element='WT',cell=3,rectangle=3,replicate=2)
APP_2_3_4 = voro2cluster('APP_2_3',squre_size=4000,x1=19000,y1=22000,parent='APP',element='WT',cell=3,rectangle=4,replicate=2)

APP_2_4_1 = voro2cluster('APP_2_4',squre_size=4000,x1=35000,y1=32000,parent='APP',element='WT',cell=4,rectangle=1,replicate=2)
APP_2_4_2 = voro2cluster('APP_2_4',squre_size=4000,x1=10000,y1=36000,parent='APP',element='WT',cell=4,rectangle=2,replicate=2)
APP_2_4_3 = voro2cluster('APP_2_4',squre_size=4000,x1=10000,y1=40000,parent='APP',element='WT',cell=4,rectangle=3,replicate=2)
APP_2_4_4 = voro2cluster('APP_2_4',squre_size=4000,x1=8000,y1=44000,parent='APP',element='WT',cell=4,rectangle=3,replicate=2)

APP_2_5_1 = voro2cluster('APP_2_5',squre_size=4000,x1=19000,y1=22000,parent='APP',element='WT',cell=5,rectangle=1,replicate=2)
APP_2_5_2 = voro2cluster('APP_2_5',squre_size=4000,x1=19000,y1=26000,parent='APP',element='WT',cell=5,rectangle=2,replicate=2)
APP_2_5_3 = voro2cluster('APP_2_5',squre_size=4000,x1=23000,y1=26000,parent='APP',element='WT',cell=5,rectangle=3,replicate=2)
APP_2_5_4 = voro2cluster('APP_2_5',squre_size=4000,x1=21000,y1=29000,parent='APP',element='WT',cell=5,rectangle=4,replicate=2)

APP_2_6_1 = voro2cluster('APP_2_6',squre_size=4000,x1=19000,y1=29000,parent='APP',element='WT',cell=6,rectangle=1,replicate=2)
APP_2_6_2 = voro2cluster('APP_2_6',squre_size=4000,x1=26000,y1=34000,parent='APP',element='WT',cell=6,rectangle=2,replicate=2)
APP_2_6_3 = voro2cluster('APP_2_6',squre_size=4000,x1=29000,y1=37000,parent='APP',element='WT',cell=6,rectangle=3,replicate=2)
APP_2_6_4 = voro2cluster('APP_2_6',squre_size=4000,x1=26000,y1=10000,parent='APP',element='WT',cell=6,rectangle=4,replicate=2)


APP_2_7_1 = voro2cluster('APP_2_7',squre_size=4000,x1=26000,y1=16000,parent='APP',element='WT',cell=7,rectangle=1,replicate=2)
APP_2_7_2 = voro2cluster('APP_2_7',squre_size=4000,x1=26000,y1=20000,parent='APP',element='WT',cell=7,rectangle=2,replicate=2)
APP_2_7_3 = voro2cluster('APP_2_7',squre_size=4000,x1=26000,y1=24000,parent='APP',element='WT',cell=7,rectangle=3,replicate=2)
APP_2_7_4 = voro2cluster('APP_2_7',squre_size=4000,x1=22000,y1=24000,parent='APP',element='WT',cell=7,rectangle=4,replicate=2)
APP_2_7_5 = voro2cluster('APP_2_7',squre_size=4000,x1=22000,y1=20000,parent='APP',element='WT',cell=7,rectangle=5,replicate=2)

APP_2_8_1 = voro2cluster('APP_2_8',squre_size=4000,x1=16000,y1=29000,parent='APP',element='WT',cell=8,rectangle=1,replicate=2)
APP_2_8_2 = voro2cluster('APP_2_8',squre_size=4000,x1=16000,y1=25000,parent='APP',element='WT',cell=8,rectangle=2,replicate=2)
APP_2_8_3 = voro2cluster('APP_2_8',squre_size=4000,x1=34000,y1=29000,parent='APP',element='WT',cell=8,rectangle=3,replicate=2)
APP_2_8_4 = voro2cluster('APP_2_8',squre_size=4000,x1=26000,y1=7000,parent='APP',element='WT',cell=8,rectangle=4,replicate=2)

In [ ]:
dAPP_1_2_1 = ThnderSTORM2df('dAPP_1_2',squre_size=4000,x1=24000,y1=28000,parent='APP',element='MUT',cell=2,rectangle=1,replicate=1)
dAPP_1_2_2 = ThnderSTORM2df('dAPP_1_2',squre_size=4000,x1=23000,y1=24000,parent='APP',element='MUT',cell=2,rectangle=2,replicate=1)
dAPP_1_2_3 = ThnderSTORM2df('dAPP_1_2',squre_size=4000,x1=19500,y1=22000,parent='APP',element='MUT',cell=2,rectangle=3,replicate=1)
dAPP_1_2_4 = ThnderSTORM2df('dAPP_1_2',squre_size=4000,x1=19500,y1=18000,parent='APP',element='MUT',cell=2,rectangle=4,replicate=1)

dAPP_1_3_1 = ThnderSTORM2df('dAPP_1_3',squre_size=4000,x1=18000,y1=30000,parent='APP',element='MUT',cell=3,rectangle=1,replicate=1)
dAPP_1_3_2 = ThnderSTORM2df('dAPP_1_3',squre_size=4000,x1=22000,y1=30000,parent='APP',element='MUT',cell=3,rectangle=2,replicate=1)
dAPP_1_3_3 = ThnderSTORM2df('dAPP_1_3',squre_size=4000,x1=21000,y1=26000,parent='APP',element='MUT',cell=3,rectangle=3,replicate=1)

dAPP_1_4_1 = ThnderSTORM2df('dAPP_1_4',squre_size=4000,x1=20000,y1=15000,parent='APP',element='MUT',cell=4,rectangle=1,replicate=1)
dAPP_1_4_2 = ThnderSTORM2df('dAPP_1_4',squre_size=4000,x1=28000,y1=15000,parent='APP',element='MUT',cell=4,rectangle=2,replicate=1)
dAPP_1_4_3 = ThnderSTORM2df('dAPP_1_4',squre_size=4000,x1=32000,y1=15000,parent='APP',element='MUT',cell=4,rectangle=3,replicate=1)
dAPP_1_4_4 = ThnderSTORM2df('dAPP_1_4',squre_size=4000,x1=25000,y1=23000,parent='APP',element='MUT',cell=4,rectangle=4,replicate=1)

dAPP_1_5_1 = ThnderSTORM2df('dAPP_1_5',squre_size=4000,x1=20000,y1=28000,parent='APP',element='MUT',cell=5,rectangle=1,replicate=1)
dAPP_1_5_2 = ThnderSTORM2df('dAPP_1_5',squre_size=4000,x1=4000,y1=32000,parent='APP',element='MUT',cell=5,rectangle=2,replicate=1)
dAPP_1_5_3 = ThnderSTORM2df('dAPP_1_5',squre_size=4000,x1=31500,y1=27500,parent='APP',element='MUT',cell=5,rectangle=3,replicate=1)


dAPP_2_2_1 = ThnderSTORM2df('dAPP_2_2',squre_size=4000,x1=31000,y1=15000,parent='APP',element='MUT',cell=2,rectangle=1,replicate=2)
dAPP_2_2_2 = ThnderSTORM2df('dAPP_2_2',squre_size=4000,x1=31000,y1=19000,parent='APP',element='MUT',cell=2,rectangle=2,replicate=2)
dAPP_2_2_3 = ThnderSTORM2df('dAPP_2_2',squre_size=4000,x1=18000,y1=25000,parent='APP',element='MUT',cell=2,rectangle=3,replicate=2)

dAPP_2_1_1 = ThnderSTORM2df('dAPP_2_1',squre_size=4000,x1=26000,y1=25000,parent='APP',element='MUT',cell=1,rectangle=1,replicate=2)
dAPP_2_1_2 = ThnderSTORM2df('dAPP_2_1',squre_size=4000,x1=22000,y1=22000,parent='APP',element='MUT',cell=1,rectangle=2,replicate=2)

dAPP_2_3_1 = ThnderSTORM2df('dAPP_2_3',squre_size=4000,x1=14000,y1=33000,parent='APP',element='MUT',cell=3,rectangle=1,replicate=2)
dAPP_2_3_2 = ThnderSTORM2df('dAPP_2_3',squre_size=4000,x1=14000,y1=29000,parent='APP',element='MUT',cell=3,rectangle=2,replicate=2)
dAPP_2_3_3 = ThnderSTORM2df('dAPP_2_3',squre_size=4000,x1=34000,y1=21000,parent='APP',element='MUT',cell=3,rectangle=3,replicate=2)
dAPP_2_3_4 = ThnderSTORM2df('dAPP_2_3',squre_size=4000,x1=30000,y1=21000,parent='APP',element='MUT',cell=3,rectangle=4,replicate=2)

dAPP_2_6_1 = ThnderSTORM2df('dAPP_2_6',squre_size=4000,x1=30000,y1=25000,parent='APP',element='MUT',cell=6,rectangle=1,replicate=2)
dAPP_2_6_2 = ThnderSTORM2df('dAPP_2_6',squre_size=4000,x1=34000,y1=25000,parent='APP',element='MUT',cell=6,rectangle=2,replicate=2)
dAPP_2_6_3 = ThnderSTORM2df('dAPP_2_6',squre_size=4000,x1=35000,y1=32000,parent='APP',element='MUT',cell=6,rectangle=3,replicate=2)



APP_1_6_1 = ThnderSTORM2df('APP_1_6',squre_size=4000,x1=17000,y1=21000,parent='APP',element='WT',cell=6,rectangle=1,replicate=1)
APP_1_6_2 = ThnderSTORM2df('APP_1_6',squre_size=4000,x1=21000,y1=21000,parent='APP',element='WT',cell=6,rectangle=2,replicate=1)

APP_1_5_1 = ThnderSTORM2df('APP_1_5',squre_size=4000,x1=23000,y1=25000,parent='APP',element='WT',cell=5,rectangle=1,replicate=1)
APP_1_5_2 = ThnderSTORM2df('APP_1_5',squre_size=4000,x1=27000,y1=25000,parent='APP',element='WT',cell=5,rectangle=2,replicate=1)

APP_1_4_1 = ThnderSTORM2df('APP_1_4',squre_size=4000,x1=20000,y1=23000,parent='APP',element='WT',cell=4,rectangle=1,replicate=1)
APP_1_4_2 = ThnderSTORM2df('APP_1_4',squre_size=4000,x1=16000,y1=23000,parent='APP',element='WT',cell=4,rectangle=2,replicate=1)
APP_1_4_3 = ThnderSTORM2df('APP_1_4',squre_size=4000,x1=16000,y1=19000,parent='APP',element='WT',cell=4,rectangle=3,replicate=1)


APP_1_2_1 = ThnderSTORM2df('APP_1_2',squre_size=4000,x1=25000,y1=27000,parent='APP',element='WT',cell=2,rectangle=1,replicate=1)
APP_1_2_2 = ThnderSTORM2df('APP_1_2',squre_size=4000,x1=25000,y1=17000,parent='APP',element='WT',cell=2,rectangle=2,replicate=1)
APP_1_2_3 = ThnderSTORM2df('APP_1_2',squre_size=4000,x1=25000,y1=13000,parent='APP',element='WT',cell=2,rectangle=3,replicate=1)
APP_1_2_4 = ThnderSTORM2df('APP_1_2',squre_size=4000,x1=33000,y1=13000,parent='APP',element='WT',cell=2,rectangle=4,replicate=1)

APP_1_3_1 = ThnderSTORM2df('APP_1_3',squre_size=4000,x1=29000,y1=23000,parent='APP',element='WT',cell=3,rectangle=1,replicate=1)
APP_1_3_2 = ThnderSTORM2df('APP_1_3',squre_size=4000,x1=29000,y1=19000,parent='APP',element='WT',cell=3,rectangle=2,replicate=1)
APP_1_3_3 = ThnderSTORM2df('APP_1_3',squre_size=4000,x1=23000,y1=16000,parent='APP',element='WT',cell=3,rectangle=3,replicate=1)
APP_1_3_4 = ThnderSTORM2df('APP_1_3',squre_size=4000,x1=21000,y1=21500,parent='APP',element='WT',cell=3,rectangle=4,replicate=1)


APP_2_2_1 = ThnderSTORM2df('APP_2_2',squre_size=4000,x1=21000,y1=21000,parent='APP',element='WT',cell=2,rectangle=1,replicate=2)
APP_2_2_2 = ThnderSTORM2df('APP_2_2',squre_size=4000,x1=20000,y1=24000,parent='APP',element='WT',cell=2,rectangle=2,replicate=2)
APP_2_2_3 = ThnderSTORM2df('APP_2_2',squre_size=4000,x1=25000,y1=18000,parent='APP',element='WT',cell=2,rectangle=3,replicate=2)
APP_2_2_4 = ThnderSTORM2df('APP_2_2',squre_size=4000,x1=15000,y1=30000,parent='APP',element='WT',cell=2,rectangle=4,replicate=2)

APP_2_3_1 = ThnderSTORM2df('APP_2_3',squre_size=4000,x1=23000,y1=30000,parent='APP',element='WT',cell=3,rectangle=1,replicate=2)
APP_2_3_2 = ThnderSTORM2df('APP_2_3',squre_size=4000,x1=23000,y1=26000,parent='APP',element='WT',cell=3,rectangle=2,replicate=2)
APP_2_3_3 = ThnderSTORM2df('APP_2_3',squre_size=4000,x1=25000,y1=22000,parent='APP',element='WT',cell=3,rectangle=3,replicate=2)
APP_2_3_4 = ThnderSTORM2df('APP_2_3',squre_size=4000,x1=19000,y1=22000,parent='APP',element='WT',cell=3,rectangle=4,replicate=2)

APP_2_4_1 = ThnderSTORM2df('APP_2_4',squre_size=4000,x1=35000,y1=32000,parent='APP',element='WT',cell=4,rectangle=1,replicate=2)
APP_2_4_2 = ThnderSTORM2df('APP_2_4',squre_size=4000,x1=10000,y1=36000,parent='APP',element='WT',cell=4,rectangle=2,replicate=2)
APP_2_4_3 = ThnderSTORM2df('APP_2_4',squre_size=4000,x1=10000,y1=40000,parent='APP',element='WT',cell=4,rectangle=3,replicate=2)
APP_2_4_4 = ThnderSTORM2df('APP_2_4',squre_size=4000,x1=8000,y1=44000,parent='APP',element='WT',cell=4,rectangle=3,replicate=2)

APP_2_5_1 = ThnderSTORM2df('APP_2_5',squre_size=4000,x1=19000,y1=22000,parent='APP',element='WT',cell=5,rectangle=1,replicate=2)
APP_2_5_2 = ThnderSTORM2df('APP_2_5',squre_size=4000,x1=19000,y1=26000,parent='APP',element='WT',cell=5,rectangle=2,replicate=2)
APP_2_5_3 = ThnderSTORM2df('APP_2_5',squre_size=4000,x1=23000,y1=26000,parent='APP',element='WT',cell=5,rectangle=3,replicate=2)
APP_2_5_4 = ThnderSTORM2df('APP_2_5',squre_size=4000,x1=21000,y1=29000,parent='APP',element='WT',cell=5,rectangle=4,replicate=2)

APP_2_6_1 = ThnderSTORM2df('APP_2_6',squre_size=4000,x1=19000,y1=29000,parent='APP',element='WT',cell=6,rectangle=1,replicate=2)
APP_2_6_2 = ThnderSTORM2df('APP_2_6',squre_size=4000,x1=26000,y1=34000,parent='APP',element='WT',cell=6,rectangle=2,replicate=2)
APP_2_6_3 = ThnderSTORM2df('APP_2_6',squre_size=4000,x1=29000,y1=37000,parent='APP',element='WT',cell=6,rectangle=3,replicate=2)
APP_2_6_4 = ThnderSTORM2df('APP_2_6',squre_size=4000,x1=26000,y1=10000,parent='APP',element='WT',cell=6,rectangle=4,replicate=2)


APP_2_7_1 = ThnderSTORM2df('APP_2_7',squre_size=4000,x1=26000,y1=16000,parent='APP',element='WT',cell=7,rectangle=1,replicate=2)
APP_2_7_2 = ThnderSTORM2df('APP_2_7',squre_size=4000,x1=26000,y1=20000,parent='APP',element='WT',cell=7,rectangle=2,replicate=2)
APP_2_7_3 = ThnderSTORM2df('APP_2_7',squre_size=4000,x1=26000,y1=24000,parent='APP',element='WT',cell=7,rectangle=3,replicate=2)
APP_2_7_4 = ThnderSTORM2df('APP_2_7',squre_size=4000,x1=22000,y1=24000,parent='APP',element='WT',cell=7,rectangle=4,replicate=2)
APP_2_7_5 = ThnderSTORM2df('APP_2_7',squre_size=4000,x1=22000,y1=20000,parent='APP',element='WT',cell=7,rectangle=5,replicate=2)

APP_2_8_1 = ThnderSTORM2df('APP_2_8',squre_size=4000,x1=16000,y1=29000,parent='APP',element='WT',cell=8,rectangle=1,replicate=2)
APP_2_8_2 = ThnderSTORM2df('APP_2_8',squre_size=4000,x1=16000,y1=25000,parent='APP',element='WT',cell=8,rectangle=2,replicate=2)
APP_2_8_3 = ThnderSTORM2df('APP_2_8',squre_size=4000,x1=34000,y1=29000,parent='APP',element='WT',cell=8,rectangle=3,replicate=2)
APP_2_8_4 = ThnderSTORM2df('APP_2_8',squre_size=4000,x1=26000,y1=7000,parent='APP',element='WT',cell=8,rectangle=4,replicate=2)


In [ ]:
apps = [APP_1_6_1,APP_1_6_2,APP_1_5_1,APP_1_5_2,APP_1_4_1,APP_1_4_2,APP_1_4_3,APP_1_2_1,APP_1_2_2,APP_1_2_3,APP_1_2_4,APP_1_3_1,APP_1_3_2,APP_1_3_3,APP_1_3_4,APP_2_2_1,APP_2_2_2,APP_2_2_3,APP_2_2_4,APP_2_3_1,APP_2_3_2,APP_2_3_3,APP_2_3_4,APP_2_4_1,APP_2_4_2,
        APP_2_4_3,APP_2_4_4,APP_2_5_1,APP_2_5_2,APP_2_5_3,APP_2_5_4,APP_2_6_1,APP_2_6_2,APP_2_6_3,APP_2_6_4,APP_2_7_1,APP_2_7_2,APP_2_7_3,APP_2_7_4,APP_2_7_5,APP_2_8_1,APP_2_8_2,APP_2_8_3,APP_2_8_4]
dapps = [dAPP_1_2_1,dAPP_1_2_2,dAPP_1_2_3,dAPP_1_2_4,dAPP_1_3_1,dAPP_1_3_2,dAPP_1_3_3,dAPP_1_4_1,dAPP_1_4_2,dAPP_1_4_3,dAPP_1_4_4,dAPP_1_5_1,dAPP_1_5_2,dAPP_1_5_3
         ,dAPP_2_6_1,dAPP_2_6_2,dAPP_2_6_3]#dAPP_2_3_1,dAPP_2_3_2,dAPP_2_3_3,dAPP_2_3_4,dAPP_2_2_1,dAPP_2_2_2,dAPP_2_2_3,dAPP_2_1_1,dAPP_2_1_2

In [ ]:
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#B9B6B6',"#B4B5B4","#C03830"])


In [ ]:
voronoi_thershold_df,t,df = voronoi_thershold_ploting(APP_1_3_3.loc[:,['x','y']].to_numpy(),n=30)


In [ ]:
dAPP_1_2_1,dAPP_1_2_2,dAPP_1_2_3,APP_1_6_1,APP_1_6_2,APP_1_5_1

In [ ]:
voronoi_thershold_df,t,df = voronoi_thershold_ploting(APP_2_2_3.loc[:,['x','y']].to_numpy(),n=30)
df,x,thershold,cluster_probabilty,l3 = voronoi_thershold_v2(APP_2_2_3.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
l3['vol_invers'] = np.log10(1/l3['vol'])
thres =[]
for i in range(30):
    thres.append(np.percentile(l3[(l3.trial==i)&(l3.vol<1e9)].vol_invers,99))
z =[]
for i in l3[l3.vol<1e6].vol_invers:
    z.append({'val':i,'ele':'s'})
for i in df[df.vol<1e6].vol_invers:
    z.append({'val':i,'ele':'r'})
fig, ax = plt.subplots(figsize=(4,3)) 
sns.violinplot(x='val',data=pd.DataFrame(z),hue='ele',split=1,fill=0,hue_order=['r','s'],palette=['k','gray'])
plt.xlim([-6,-1])
plt.axvline(np.mean(thres),color='r')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)

plt.ylabel('')
plt.xlabel('')
plt.savefig('APP_2_2_3_voronoi_random_vs_real.svg',bbox_inches='tight',transparent=True,dpi=600)
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<1e6)],s=10,edgecolor=None,color='k')
plt.xlim([l3[(l3.trial==1)&(l3.vol<1e6)].x.min(),l3[(l3.trial==1)&(l3.vol<1e6)].x.max()])
plt.ylim([l3[(l3.trial==1)&(l3.vol<1e6)].y.min(),l3[(l3.trial==1)&(l3.vol<1e6)].y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_scatter_random.png',bbox_inches='tight',transparent=True,dpi=600)
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<1e6)],s=10,edgecolor=None,color='k')
sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<t)],s=10,edgecolor=None,color='r')

plt.xlim([l3[(l3.trial==1)&(l3.vol<1e6)].x.min(),l3[(l3.trial==1)&(l3.vol<1e6)].x.max()])
plt.ylim([l3[(l3.trial==1)&(l3.vol<1e6)].y.min(),l3[(l3.trial==1)&(l3.vol<1e6)].y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_voronoi_random_post_t.png',bbox_inches='tight',transparent=True,dpi=600)


fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<1e6)],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-1),palette='RdGy_r',ax=ax)
#sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor='r',hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
ax.legend([],frameon=False)
norm = plt.Normalize(vmin=-6, vmax=-1)
sm = plt.cm.ScalarMappable(cmap='RdGy_r', norm=norm)
sm.set_array([])
# #ax.get_legend().remove()
ax.figure.colorbar(sm,fraction=0.03,shrink=.4, pad=0.01)
# #ax.figure.colorbar(sm,fraction=0.03, pad=0.01,ticks=[-6,-1*np.log10(t), -2],ax=ax)
rand_df = l3[(l3.trial==1)]
points = l3[(l3.trial==1)].loc[:,['x','y']].to_numpy()
vor = Voronoi(points)
# ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
plt.xlim([l3[(l3.trial==1)&(l3.vol<1e6)].x.min(),l3[(l3.trial==1)&(l3.vol<1e6)].x.max()])
plt.ylim([l3[(l3.trial==1)&(l3.vol<1e6)].y.min(),l3[(l3.trial==1)&(l3.vol<1e6)].y.max()])
norm = plt.Normalize(vmin=-6, vmax=-1)#mpl.colors.Normalize(vmin=df[df.vol<1e6].vol_invers.min(), vmax=df[df.vol<1e6].vol_invers.max(), clip=True)
mapper = cm.ScalarMappable(norm=norm, cmap=cm.RdGy_r)


for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    #point= vor.regions[vor.point_region[r]]
    
    area = rand_df[rand_df.x==vor.points[r][0]].vol_invers.values[0]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color=mapper.to_rgba(area))
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_voronoi_random.png',bbox_inches='tight',transparent=True,dpi=600)

sns.set(font_scale=1,style='ticks')
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-1),palette='RdGy_r',ax=ax)
#sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor='r',hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
ax.legend([],frameon=False)
norm = plt.Normalize(vmin=-6, vmax=-1)
sm = plt.cm.ScalarMappable(cmap='RdGy_r', norm=norm)
sm.set_array([])
#ax.get_legend().remove()
ax.figure.colorbar(sm,fraction=0.03,shrink=.4, pad=0.01)
#ax.figure.colorbar(sm,fraction=0.03, pad=0.01,ticks=[-6,-1*np.log10(t), -2],ax=ax)
points = APP_2_2_3.loc[:,['x','y']].to_numpy()
vor = Voronoi(points)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
plt.xlim([df.x.min(),df.x.max()])
plt.ylim([df.y.min(),df.y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')

# find min/max values for normalization
minima = 0
maxima = 5


# normalize chosen colormap
norm = plt.Normalize(vmin=-6, vmax=-1)#mpl.colors.Normalize(vmin=df[df.vol<1e6].vol_invers.min(), vmax=df[df.vol<1e6].vol_invers.max(), clip=True)
mapper = cm.ScalarMappable(norm=norm, cmap=plt.cm.RdGy_r)


for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    #point= vor.regions[vor.point_region[r]]
    
    area = df[df.x==vor.points[r][0]].vol_invers.values[0]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color=mapper.to_rgba(area))
plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
#sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=2,edgecolor=None,color='k')
plt.savefig('APP_2_2_3_voronoi.png',bbox_inches='tight',transparent=True,dpi=600)
sns.set(font_scale=1,style='white')
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-1),palette='RdGy_r',ax=ax)
#sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor='r',hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
ax.legend([],frameon=False)
norm = plt.Normalize(vmin=-6, vmax=-1)
sm = plt.cm.ScalarMappable(cmap='RdGy_r', norm=norm)
sm.set_array([])
#ax.get_legend().remove()
ax.figure.colorbar(sm,fraction=0.03,shrink=.4, pad=0.01)
#ax.figure.colorbar(sm,fraction=0.03, pad=0.01,ticks=[-6,-1*np.log10(t), -2],ax=ax)
points = APP_2_2_3.loc[:,['x','y']].to_numpy()
vor = Voronoi(points)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
plt.xlim([df.x.min(),df.x.max()])
plt.ylim([df.y.min(),df.y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')

# find min/max values for normalization
minima = 0
maxima = 5


# normalize chosen colormap
norm = plt.Normalize(vmin=-6, vmax=-1)#mpl.colors.Normalize(vmin=df[df.vol<1e6].vol_invers.min(), vmax=df[df.vol<1e6].vol_invers.max(), clip=True)
mapper = cm.ScalarMappable(norm=norm, cmap=plt.cm.RdGy_r)


for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    #point= vor.regions[vor.point_region[r]]
    
    area = df[df.x==vor.points[r][0]].vol_invers.values[0]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color=mapper.to_rgba(area))
plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
#sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=2,edgecolor=None,color='k')
plt.savefig('APP_2_2_3_voronoi.png',bbox_inches='tight',transparent=True,dpi=600)
c = df.db.value_counts().reset_index()
for ind in df[(df.db!=-1)&(~pd.isna(df.db))].index:
    
    df.loc[ind,'cluster_size'] = c[c['index']==df.loc[ind,'db']].db.values[0]
df.loc[df[df.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
df.loc[df[df.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
df.loc[df[df.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
df.loc[df[df.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=df[(df.vol<1e6)&(df.db!=-1)],s=10,edgecolor=None,hue='cluster_range',
                zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'])
plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.legend([],frameon=False)
plt.savefig('APP_2_2_3_voronoi_cluster_db.png',bbox_inches='tight',transparent=True,dpi=600)

fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor=None,color='r')
sns.scatterplot(x='x',y='y',data=df[df.vol>t],s=10,edgecolor=None,color='k')
plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_voronoi_cluster.png',bbox_inches='tight',transparent=True,dpi=600)
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,color='k')
plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_voronoi_real.png',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
def supplement_figure(file,name='APP_1_2_3'):
    voronoi_thershold_df,t,df = voronoi_thershold_ploting(file.loc[:,['x','y']].to_numpy(),n=30)
    df,x,thershold,cluster_probabilty,l3 = voronoi_thershold_v2(file.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
    l3['vol_invers'] = np.log10(1/l3['vol'])
    thres =[]
    for i in range(30):
        thres.append(np.percentile(l3[(l3.trial==i)&(l3.vol<1e9)].vol_invers,99))
    z =[]
    for i in l3[l3.vol<1e6].vol_invers:
        z.append({'val':i,'ele':'s'})
    for i in df[df.vol<1e6].vol_invers:
        z.append({'val':i,'ele':'r'})
    fig, ax = plt.subplots(figsize=(4,3)) 
    sns.violinplot(x='val',data=pd.DataFrame(z),hue='ele',split=1,fill=0,hue_order=['r','s'],palette=['k','gray'])
    plt.xlim([-6,-1])
    plt.axvline(np.mean(thres),color='r')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
    
    plt.ylabel('')
    plt.xlabel('')
    plt.savefig('%s_voronoi_random_vs_real.svg'%name,bbox_inches='tight',transparent=True,dpi=600)
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<1e6)],s=10,edgecolor=None,color='k')
    plt.xlim([l3[(l3.trial==1)&(l3.vol<1e6)].x.min(),l3[(l3.trial==1)&(l3.vol<1e6)].x.max()])
    plt.ylim([l3[(l3.trial==1)&(l3.vol<1e6)].y.min(),l3[(l3.trial==1)&(l3.vol<1e6)].y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    plt.savefig('%s_scatter_random.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,color='k')
    plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
    plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    plt.savefig('%s_voronoi_real.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    
    
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<1e6)],s=10,edgecolor=None,color='#F8F8F8')
    sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<t)],s=10,edgecolor=None,color='#D82E8A')
    
    plt.xlim([l3[(l3.trial==1)&(l3.vol<1e6)].x.min(),l3[(l3.trial==1)&(l3.vol<1e6)].x.max()])
    plt.ylim([l3[(l3.trial==1)&(l3.vol<1e6)].y.min(),l3[(l3.trial==1)&(l3.vol<1e6)].y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    plt.savefig('%s_scatter_random_post_t.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    
    
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=l3[(l3.trial==1)&(l3.vol<1e6)],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-1),palette='RdGy_r',ax=ax)
    #sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor='r',hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
    ax.legend([],frameon=False)
    norm = plt.Normalize(vmin=-6, vmax=-1)
    sm = plt.cm.ScalarMappable(cmap='RdGy_r', norm=norm)
    sm.set_array([])
    # #ax.get_legend().remove()
    #ax.figure.colorbar(sm,fraction=0.03,shrink=.4, pad=0.01)
    # #ax.figure.colorbar(sm,fraction=0.03, pad=0.01,ticks=[-6,-1*np.log10(t), -2],ax=ax)
    rand_df = l3[(l3.trial==1)]
    points = l3[(l3.trial==1)].loc[:,['x','y']].to_numpy()
    vor = Voronoi(points)
    # ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
    plt.xlim([l3[(l3.trial==1)&(l3.vol<1e6)].x.min(),l3[(l3.trial==1)&(l3.vol<1e6)].x.max()])
    plt.ylim([l3[(l3.trial==1)&(l3.vol<1e6)].y.min(),l3[(l3.trial==1)&(l3.vol<1e6)].y.max()])
    norm = plt.Normalize(vmin=-6, vmax=-1)#mpl.colors.Normalize(vmin=df[df.vol<1e6].vol_invers.min(), vmax=df[df.vol<1e6].vol_invers.max(), clip=True)
    mapper = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.RdGy_r)
    
    
    for r in range(len(vor.point_region)):
        region = vor.regions[vor.point_region[r]]
        #point= vor.regions[vor.point_region[r]]
        
        area = rand_df[rand_df.x==vor.points[r][0]].vol_invers.values[0]
        if not -1 in region:
            polygon = [vor.vertices[i] for i in region]
            plt.fill(*zip(*polygon), color=mapper.to_rgba(area))
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    plt.savefig('%s_voronoi_random.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    
    sns.set(font_scale=1,style='white')
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-1),palette='RdGy_r',ax=ax)
    #sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor='r',hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
    ax.legend([],frameon=False)
    norm = plt.Normalize(vmin=-6, vmax=-1)
    sm = plt.cm.ScalarMappable(cmap='RdGy_r', norm=norm)
    sm.set_array([])
    #ax.get_legend().remove()
    #ax.figure.colorbar(sm,fraction=0.03,shrink=.4, pad=0.01)
    #ax.figure.colorbar(sm,fraction=0.03, pad=0.01,ticks=[-6,-1*np.log10(t), -2],ax=ax)
    points = file.loc[:,['x','y']].to_numpy()
    vor = Voronoi(points)
    ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
    plt.xlim([df.x.min(),df.x.max()])
    plt.ylim([df.y.min(),df.y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    
    # find min/max values for normalization
    minima = 0
    maxima = 5
    
    
    # normalize chosen colormap
    norm = plt.Normalize(vmin=-6, vmax=-1)#mpl.colors.Normalize(vmin=df[df.vol<1e6].vol_invers.min(), vmax=df[df.vol<1e6].vol_invers.max(), clip=True)
    mapper = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.RdGy_r)
    
    
    for r in range(len(vor.point_region)):
        region = vor.regions[vor.point_region[r]]
        #point= vor.regions[vor.point_region[r]]
        
        area = df[df.x==vor.points[r][0]].vol_invers.values[0]
        if not -1 in region:
            polygon = [vor.vertices[i] for i in region]
            plt.fill(*zip(*polygon), color=mapper.to_rgba(area))
    plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
    plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
    #sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=2,edgecolor=None,color='k')
    plt.savefig('%s_voronoi.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    sns.set(font_scale=1,style='white')
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-1),palette='RdGy_r',ax=ax)
    #sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor='r',hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
    ax.legend([],frameon=False)
    norm = plt.Normalize(vmin=-6, vmax=-1)
    sm = plt.cm.ScalarMappable(cmap='RdGy_r', norm=norm)
    sm.set_array([])
    #ax.get_legend().remove()
    #ax.figure.colorbar(sm,fraction=0.03,shrink=.4, pad=0.01)
    #ax.figure.colorbar(sm,fraction=0.03, pad=0.01,ticks=[-6,-1*np.log10(t), -2],ax=ax)
    points = file.loc[:,['x','y']].to_numpy()
    vor = Voronoi(points)
    ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
    plt.xlim([df.x.min(),df.x.max()])
    plt.ylim([df.y.min(),df.y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    
    # find min/max values for normalization
    minima = 0
    maxima = 5
    
    
    # normalize chosen colormap
    norm = plt.Normalize(vmin=-6, vmax=-1)#mpl.colors.Normalize(vmin=df[df.vol<1e6].vol_invers.min(), vmax=df[df.vol<1e6].vol_invers.max(), clip=True)
    mapper = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.RdGy_r)
    
    
    for r in range(len(vor.point_region)):
        region = vor.regions[vor.point_region[r]]
        #point= vor.regions[vor.point_region[r]]
        
        area = df[df.x==vor.points[r][0]].vol_invers.values[0]
        if not -1 in region:
            polygon = [vor.vertices[i] for i in region]
            plt.fill(*zip(*polygon), color=mapper.to_rgba(area))
    plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
    plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
    #sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=2,edgecolor=None,color='k')
    plt.savefig('%s_voronoi.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    plt.savefig('%s_voronoi.svg'%name,bbox_inches='tight',transparent=True,dpi=600)
    c = df.db.value_counts().reset_index()
    for ind in df[(df.db!=-1)&(~pd.isna(df.db))].index:
        
        df.loc[ind,'cluster_size'] = c[c['index']==df.loc[ind,'db']].db.values[0]
    df.loc[df[df.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
    df.loc[df[df.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
    df.loc[df[df.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
    df.loc[df[df.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=df[(df.vol<1e6)&(df.db!=-1)],s=10,edgecolor=None,hue='cluster_range',
                    zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'])
    plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
    plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    plt.legend([],frameon=False)
    plt.savefig('%s_voronoi_cluster_db.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    
    fig, ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor=None,color='#D82E8A')
    sns.scatterplot(x='x',y='y',data=df[df.vol>t],s=10,edgecolor=None,color='#E6E7E8')
    plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
    plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
    plt.tick_params(left = False, right = False , labelleft = False ,
                    labelbottom = False, bottom = False)
    plt.xlabel('')
    plt.ylabel('')
    
    plt.savefig('%s_voronoi_cluster.png'%name,bbox_inches='tight',transparent=True,dpi=600)
    
    return (len(df[(df.vol<1e6)])),100*(len(df[(df.vol<t)])/len(df[(df.vol<1e6)]))


In [ ]:
#supplement_figure(APP_1_2_3,name='APP_1_2_3')
#supplement_figure(APP_2_2_3,name='APP_2_2_3')
supplement_figure(APP_1_6_2,name='APP_1_6_2')

In [ ]:
df

In [ ]:
#supplement_figure(APP_1_3_1,name='APP_1_3_1')
#supplement_figure(APP_1_4_1,name='APP_1_4_1')
#supplement_figure(dAPP_1_5_1,name='dAPP_1_5_1')
supplement_figure(dAPP_1_3_1,name='dAPP_1_3_1')


In [ ]:
c = df.db.value_counts().reset_index()
for ind in df[(df.db!=-1)&(~pd.isna(df.db))].index:
    
    df.loc[ind,'cluster_size'] = c[c['index']==df.loc[ind,'db']].db.values[0]
df.loc[df[df.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
df.loc[df[df.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
df.loc[df[df.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
df.loc[df[df.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
ax.grid()

In [ ]:
sns.scatterplot(x='x',y='y',data=df[(df.vol<1e6)&(df.db==-1)],s=1)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=df[(df.vol<1e6)&(df.db!=-1)],s=10,edgecolor=None,hue='cluster_range',
                zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'])
plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.legend([],frameon=False)
#plt.savefig('APP_2_2_3_voronoi_cluster_db.png',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=5,edgecolor=None,color='#D82E8A')
sns.scatterplot(x='x',y='y',data=df[df.vol>t],s=5,edgecolor=None,color='#E6E7E8')
plt.xlim([df[df.vol<1e6].x.min(),df[df.vol<1e6].x.max()])
plt.ylim([df[df.vol<1e6].y.min(),df[df.vol<1e6].y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_voronoi_cluster.png',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
sns.set(font_scale=1,style='white')
fig, ax = plt.subplots(figsize=(6,6))
sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
#sns.scatterplot(x='x',y='y',data=df[df.vol<t],s=10,edgecolor='r',hue='vol_invers', hue_norm=(-6,-2),palette='viridis',ax=ax)
ax.legend([],frameon=False)
norm = plt.Normalize(vmin=-6, vmax=-2)
sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
#ax.get_legend().remove()
ax.figure.colorbar(sm,fraction=0.03, pad=0.01,ticks=[-6,-1*np.log10(t), -2],ax=ax)
points = APP_2_2_3.loc[:,['x','y']].to_numpy()
vor = Voronoi(points)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.4,ax=ax)
plt.xlim([df.x.min(),df.x.max()])
plt.ylim([df.y.min(),df.y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_voronoi.png',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
fig, ax = plt.subplots(figsize=(4,4)) 
sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=5,edgecolor=None,c='k')
#ax.axis('off')
#plt.savefig('APP_2_2_3_templet.png',bbox_inches='tight',transparent=False,dpi=600)
plt.xlim([df.x.min(),df.x.max()])
plt.ylim([df.y.min(),df.y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('APP_2_2_3_templet.png',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
voronoi_thershold_df,t,df = voronoi_thershold_ploting(APP_2_2_3.loc[:,['x','y']].to_numpy(),n=30)
voronoi_thershold_df['x_log'] =np.log10(voronoi_thershold_df['x'])
plot_data = voronoi_thershold_df[(voronoi_thershold_df.x_log>0)&(voronoi_thershold_df.element!='Ground_truth')]
plot_data_GT = voronoi_thershold_df[(voronoi_thershold_df.x_log>0)&(voronoi_thershold_df.element=='Ground_truth')]

sns.set(font_scale=1.5,style='white')
fig = plt.figure(figsize = (8,4),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.01,'wspace':.02})
plot_data_group_GT  =plot_data_GT.groupby('x').mean().reset_index()
axs[0].vlines(x=plot_data_group_GT.x, ymin=0, ymax=plot_data_group_GT.y, color='#317EC2',linewidth=3, alpha=1)
sns.barplot(x=plot_data_GT.x,y=plot_data_GT.y, color='#317EC2',native_scale=True,ax=axs[0])
axs[0].axvline(t,ls='-',color='r',lw=4,zorder=2)
axs[0].grid()
#lt.axvline(np.log10(t),ls='-',color='r',lw=4,zorder=1)
#axs[0].legend(['Data','','Random trails'],title='',fontsize=14)
axs[0].set_ylabel('Count',fontsize=16)
axs[0].set_xlabel('Voronoi polygon area (nm$^{2}$)',fontsize=16)
#sns.barplot(x=plot_data_GT.x,y=plot_data_GT.y, color='#317EC2',native_scale=True)
axs[0].set_ylim([0,230])
axs[1].grid()
axs[1].set_yticklabels([])


plot_data_group  =plot_data.groupby('x').mean().reset_index()
plt.vlines(x=plot_data_group.x, ymin=0, ymax=plot_data_group.y, color='#F2B342',linewidth=3, alpha =1)
sns.barplot(x=plot_data.x,y=plot_data.y, color='#F2B342',native_scale=True)

axs[1].vlines(x=plot_data_group.x, ymin=0, ymax=plot_data_group.y, color='#F2B342',linewidth=3, alpha=1)
sns.barplot(x=plot_data.x,y=plot_data.y, color='#F2B342',native_scale=True,ax=axs[1])
axs[1].set_ylim([0,230])
axs[1].set_xlim([-100,1.7e4])
axs[0].set_xlim([-100,1.7e4])
axs[1].set_ylabel('')
axs[1].set_xlabel('')
axs[1].axvline(t,ls='-',color='r',lw=4,zorder=2)
plt.savefig('APP_2_2_3_coutoff.svg',bbox_inches='tight',transparent=True)

In [ ]:
sns.violinplot(y=''df

In [ ]:
plot_data_group  =plot_data_GT.groupby('x_log').mean().reset_index()
plt.vlines(x=plot_data_group.x, ymin=0, ymax=plot_data_group.y, color='#F2B342',linewidth=3, alpha =1)
#sns.barplot(x=plot_data_GT.x_log,y=plot_data_GT.y, color='#F2B342',native_scale=True)

In [ ]:
plot_data_group

In [ ]:
df,x,thershold,cluster_probabilty,_ = voronoi_thershold_v2(APP_1_6_2.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
tem = df[df.vol<1e6]
cluster2size = tem[tem.db!=-1].value_counts('db').reset_index()
for ind in tem[(tem.db!=-1)&(~pd.isna(tem.db))].index:
    tem.loc[ind,'cluster_size'] = cluster2size[cluster2size.db==tem.loc[ind,'db']].values[0][-1]
tem.loc[tem[tem.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
tem.loc[tem[tem.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
tem.loc[tem[tem.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
tem.loc[tem[tem.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
#tem = df[df.vol<1e6]
points = APP_1_6_2.loc[:,['x','y']].to_numpy()

vor = Voronoi(points)
tem.loc[tem[tem.vol<thershold].index,'binary_theshold'] = '0'
tem.loc[tem[tem.vol>=thershold].index,'binary_theshold'] = '0'

fig = plt.figure(figsize = (9,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.01,'wspace':.02})
# fig, ax = plt.subplots(figsize=(10,10)) 
sns.scatterplot(x='x',y='y',data=tem,s=1,edgecolor=None,hue='binary_theshold'
                ,palette=["k","k"],hue_order=['0','1'],ax=axs[0],zorder=2)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=0.1,ax=axs[1],zorder=2)
sns.scatterplot(x='x',y='y',data=tem,s=1,edgecolor=None,hue='binary_theshold',
                palette=["k","#C03830"],hue_order=['0','1'],ax=axs[1],zorder=2,rasterized=0)
axs[1].legend([],frameon=False)

samp = len(set(tem[~pd.isna(tem.db)].db))-1
cmap_test = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#F55536",'#0C8346','#B8B8B8','#8D5A97','#274C77'],samp)
color1 = [mcolors.rgb2hex(cmap_test(i)) for i in range(samp)]
color11 = pd.Series(color1).sample((samp)).to_list()
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],s=10,edgecolor=None,
                hue='cluster_range',ax=axs[2],zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'],rasterized=0)
sns.scatterplot(x='x',y='y',data=tem[((tem.db==-1)|(pd.isna(tem.db)))],
                s=3,edgecolor=None,c='k',alpha=.1,ax=axs[2],rasterized=0)
axs[0].legend([],frameon=False)
for i in [0,1,2]:
    axs[i].set_xlim([df.x.min(),df.x.max()])
    axs[i].set_ylim([df.y.min(),df.y.max()])
    axs[i].set_xlabel('')
    axs[i].set_ylabel('')
    #axs[i].grid(zorder=0)
    axs[i].locator_params(nbins=10)
    #axs[i].legend([],frameon=False)
    axs[i].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel.svg',bbox_inches='tight',transparent=False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel.png',dpi=600,bbox_inches='tight',transparent=False)


In [ ]:
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],s=10,edgecolor=None,
                hue='cluster_range',ax=ax,zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'],rasterized=0)

In [ ]:
fig = plt.figure(figsize = (3,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=1,gridspec_kw = {'hspace':.01,'wspace':.02})
# # fig, ax = plt.subplots(figsize=(10,10)) 
# sns.scatterplot(x='x',y='y',data=tem,s=.1,edgecolor=None,hue='binary_theshold'
#                 ,palette=["k","k"],hue_order=['0','1'],ax=axs,zorder=2)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=0.25,ax=axs,zorder=2)
axs.set_xlim([df.x.min(),df.x.max()])
axs.set_ylim([df.y.min(),df.y.max()])
axs.set_xlabel('')
axs.set_ylabel('')
#axs[i].grid(zorder=0)
axs.locator_params(nbins=10)
#axs[i].legend([],frameon=False)
axs.tick_params(left = False, right = False , labelleft = False ,
            labelbottom = False, bottom = False)
plt.savefig('voronori_thershold_APP_1_6_2.svg',bbox_inches='tight',transparent=False)

In [ ]:
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],s=1,edgecolor=None,
                hue='cluster_range',zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'],rasterized=0)

In [ ]:

fig = plt.figure(figsize = (3,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=1,gridspec_kw = {'hspace':.01,'wspace':.02})
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],s=1,edgecolor=None,
                hue='cluster_range',ax=axs,zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'],rasterized=0)
axs.set_xlim([tem.x.min(),tem.x.max()])
axs.set_ylim([tem.y.min(),tem.y.max()])
axs.set_xlabel('')
axs.set_ylabel('')
#axs[i].grid(zorder=0)
axs.locator_params(nbins=10)
#axs[i].legend([],frameon=False)
axs.tick_params(left = False, right = False , labelleft = False ,
            labelbottom = False, bottom = False)
plt.savefig('voronori_thershold_APP_1_6_2.svg',bbox_inches='tight',transparent=False)

In [ ]:

fig = plt.figure(figsize = (3,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=1,gridspec_kw = {'hspace':.01,'wspace':.02})
sns.scatterplot(x='x',y='y',data=tem[(tem.db==-1)|(pd.isna(tem.db))],c='#E7E8E9',s=1,edgecolor=None)
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],c='#D53196',s=1,edgecolor=None)
axs.set_xlim([tem.x.min(),tem.x.max()])
axs.set_ylim([tem.y.min(),tem.y.max()])
axs.set_xlabel('')
axs.set_ylabel('')
#axs[i].grid(zorder=0)
axs.locator_params(nbins=10)
#axs[i].legend([],frameon=False)
axs.tick_params(left = False, right = False , labelleft = False ,
            labelbottom = False, bottom = False)
plt.savefig('voronori_thershold_APP_1_6_2.svg',bbox_inches='tight',transparent=False)

In [ ]:
df,x,thershold,cluster_probabilty,_ = voronoi_thershold_v2(APP_1_6_2.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
tem = df[df.vol<1e6]
cluster2size = tem[tem.db!=-1].value_counts('db').reset_index()
for ind in tem[(tem.db!=-1)&(~pd.isna(tem.db))].index:
    tem.loc[ind,'cluster_size'] = cluster2size[cluster2size.db==tem.loc[ind,'db']].values[0][-1]
tem.loc[tem[tem.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
tem.loc[tem[tem.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
tem.loc[tem[tem.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
tem.loc[tem[tem.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
#tem = df[df.vol<1e6]
points = APP_1_6_2.loc[:,['x','y']].to_numpy()

vor = Voronoi(points)
tem.loc[tem[tem.vol<thershold].index,'binary_theshold'] = '1'
tem.loc[tem[tem.vol>=thershold].index,'binary_theshold'] = '0'

fig = plt.figure(figsize = (15,5),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.01,'wspace':.02})
# fig, ax = plt.subplots(figsize=(10,10)) 
sns.scatterplot(x='x',y='y',data=tem,s=3,edgecolor=None,hue='binary_theshold'
                ,palette=["k","k"],hue_order=['0','1'],ax=axs[0],zorder=2)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=.3,ax=axs[1],zorder=2)
sns.scatterplot(x='x',y='y',data=tem,s=3,edgecolor=None,hue='binary_theshold',
                palette=["k","#C03830"],hue_order=['0','1'],ax=axs[1],zorder=2)

samp = len(set(tem[~pd.isna(tem.db)].db))-1
cmap_test = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#F55536",'#0C8346','#B8B8B8','#8D5A97','#274C77'],samp)
color1 = [mcolors.rgb2hex(cmap_test(i)) for i in range(samp)]
color11 = pd.Series(color1).sample((samp)).to_list()
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],s=10,edgecolor=None,
                hue='cluster_range',ax=axs[2],zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'])
sns.scatterplot(x='x',y='y',data=tem[((tem.db==-1)|(pd.isna(tem.db)))],
                s=3,edgecolor=None,c='k',alpha=.1,rasterized=True, ax=axs[2])

for i in [0,1,2]:
    axs[i].set_xlim([df.x.min(),df.x.max()])
    axs[i].set_ylim([df.y.min(),df.y.max()])
    axs[i].set_xlabel('')
    axs[i].set_ylabel('')
    axs[i].grid(zorder=0)
    axs[i].locator_params(nbins=10)
    #axs[i].legend([],frameon=False)
    axs[i].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel.svg',bbox_inches='tight',transparent=False)


In [ ]:
def roi2df(roi=APP_1_6_2):
    df,x,thershold,cluster_probabilty,l3 = voronoi_thershold_v2(roi.loc[:,['x','y']].to_numpy(),
                                                             n=30,size_of_grid=4000,min_cluster_size=5)
    
    tem = df[df.vol<1e6]
    cluster2size = tem[tem.db!=-1].value_counts('db').reset_index()
    for ind in tem[(tem.db!=-1)&(~pd.isna(tem.db))].index:
        tem.loc[ind,'cluster_size'] = cluster2size[cluster2size.db==tem.loc[ind,'db']].values[0][-1]
    tem.loc[tem[tem.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
    tem.loc[tem[tem.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
    tem.loc[tem[tem.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
    tem.loc[tem[tem.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
    #tem = df[df.vol<1e6]
    points = tem.loc[:,['x','y']].to_numpy()
    vor = Voronoi(points)#
    tem.loc[tem[tem.vol<thershold].index,'binary_theshold'] = '1'
    tem.loc[tem[tem.vol>=thershold].index,'binary_theshold'] = '0'
    tem.loc[:,'name'] = roi.name.values[0]
    tem.loc[:,'cell'] = roi.cell.values[0]
    tem.loc[:,'rectangle'] = roi.rectangle.values[0]
    tem.loc[:,'replicate'] = roi.replicate.values[0]
    return tem

In [ ]:
roi2df()

In [ ]:
t

In [ ]:
def precent_clustered(roi=APP_1_6_2):
    df,x,thershold,cluster_probabilty,l3 = voronoi_thershold_v2(roi.loc[:,['x','y']].to_numpy(),
                                                             n=30,size_of_grid=4000,min_cluster_size=5)
    tem = df[df.vol<1e6]
    
    # cluster2size = tem[tem.db!=-1].value_counts('db').reset_index()
    # for ind in tem[(tem.db!=-1)&(~pd.isna(tem.db))].index:
    #     tem.loc[ind,'cluster_size'] = cluster2size[cluster2size.db==tem.loc[ind,'db']].values[0][-1]
    # tem.loc[tem[tem.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
    # tem.loc[tem[tem.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
    # tem.loc[tem[tem.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
    # tem.loc[tem[tem.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
    # #tem = df[df.vol<1e6]
    # vor = Voronoi(points)
    # tem.loc[tem[tem.vol<thershold].index,'binary_theshold'] = '1'
    # tem.loc[tem[tem.vol>=thershold].index,'binary_theshold'] = '0'
    # tem.loc[:,'name'] = roi.name.values[0]
    # tem.loc[:,'cell'] = roi.cell.values[0]
    # tem.loc[:,'rectangle'] = roi.rectangle.values[0]
    # tem.loc[:,'replicate'] = roi.replicate.values[0]
    return 100*(len(tem[tem.vol<thershold])/len(tem[tem.vol<1e6])),len(tem)

In [ ]:
tem = pd.read_pickle('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/temp.pkl')
tem[tem.db==4]


In [ ]:
tem

In [ ]:
precent_clusterd_list = []
for i in enumerate(apps_v2):
    pre,len_df = precent_clustered(apps_v2[i[0]])
    precent_clusterd_list.append({'ele':'APP','val_pre':pre,'loc_number':len_df})
for i in enumerate(dapp_v2):
    pre,len_df = precent_clustered(dapp_v2[i[0]])
    precent_clusterd_list.append({'ele':'dAPP','val_pre':pre,'loc_number':len_df})

In [ ]:
# fig, ax = plt.subplots(figsize=(3,3))
# sns.scatterplot(x='loc_number',y='val_pre',data=pd.DataFrame(precent_clusterd_list),hue='ele'
#                 ,hue_order=['dAPP','APP'],palette=['gray','r'],edgecolor=None)
# plt.xscale('log')

g= sns.JointGrid(x='loc_number',y='val_pre',data=pd.DataFrame(precent_clusterd_list),height=4)#,edgecolor=None,s=10
g.ax_joint.grid()
g.plot_joint(sns.scatterplot,hue='ele',data=pd.DataFrame(precent_clusterd_list),edgecolor=None,s=25,hue_order=['dAPP','APP'],palette=['gray','r'])
g.plot_marginals(sns.violinplot,hue='ele',data=pd.DataFrame(precent_clusterd_list),split=True,saturation=1,fill=0,hue_order=['dAPP','APP'],palette=['gray','r'])
g.ax_joint.locator_params(nbins=4)
g.ax_joint.set_xscale('log')
g.ax_joint.set_yticklabels(g.ax_joint.get_yticklabels(), rotation=0,fontsize=16)
g.ax_joint.set_xticklabels(g.ax_joint.get_xticklabels(), rotation=0,fontsize=16)
g.ax_joint.set_xlabel('')
g.ax_joint.set_ylabel('')
g.ax_joint.legend([],frameon=False)
plt.tight_layout()
plt.savefig('precent_pass_thershold.svg',bbox_inches='tight',transparent=False)


In [ ]:
pass_thers = pd.DataFrame(precent_clusterd_list)
stats.ks_2samp(pass_thers[pass_thers.ele=='APP'].val_pre,
pass_thers[pass_thers.ele!='APP'].val_pre)

In [ ]:
pass_thers

In [ ]:
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(2,3))
sns.boxplot(y='val_pre',data=pass_thers,hue='ele',fill=False,width=0.6,
            
              hue_order=['dAPP','APP'],palette=['gray','#C03830'],showcaps=False)
plt.locator_params(axis='y',nbins=4)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.legend([],frameon=False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('precent_pass_thershold.svg',bbox_inches='tight',transparent=False)


In [ ]:
x = tem[tem.db==6]['x'].to_list()
y = tem[tem.db==6]['y'].to_list()
cx = np.dot(x, y) / np.sum(y)
cy = np.dot(y, x) / np.sum(x)
print(cx,cy)



In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
sns.scatterplot(x='x',y='y',data=tem[tem.db==6],rasterized=True,color='#C03830',s=5,edgecolor=None,ax=ax)
plt.scatter(x=cx,y=cy,s=5,c='k')

# ax.axis('off')
ax.set_xlim(cx-400,cx+400)
ax.set_ylim(cy-400,cy+400)
ax.plot([cx-50,cx+50],[cy-400,cy-400],color='k')

In [ ]:
def cluster2png(tem,cluster=1,ele='APP',sizes='5-10'):
    #tem.to_pickle('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/temp.pkl')
    name = tem.name.drop_duplicates().values[0]
    cell = tem.cell.drop_duplicates().values[0]
    rectangle = tem.rectangle.drop_duplicates().values[0] 
    replicate = tem.replicate.drop_duplicates().values[0] 
    x = tem[tem.db==cluster]['x'].to_list()
    y = tem[tem.db==cluster]['y'].to_list()
    cx = np.dot(x, y) / np.sum(y)
    cy = np.dot(y, x) / np.sum(x)
    fig, ax = plt.subplots(figsize=(3,3)) 
    if ele =='APP':
        sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=True,color='#C03830',s=3,edgecolor=None,ax=ax)
    else:
        sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=True,color='#B4B5B4',s=3,edgecolor=None,ax=ax)
    ax.set_xlim(cx-400,cx+400)
    ax.set_ylim(cy-400,cy+400)
    ax.axis('off')
    if sizes =='>100':
        plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,'above_100',cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    else:
        plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,sizes,cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    
    plt.close()

In [ ]:
def cluster2png_smooth(tem,cluster=1,ele='APP'):
    #tem.to_pickle('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/temp.pkl')
    name = tem.name.drop_duplicates().values[0]
    cell = tem.cell.drop_duplicates().values[0]
    rectangle = tem.rectangle.drop_duplicates().values[0] 
    replicate = tem.replicate.drop_duplicates().values[0] 
    x = tem[tem.db==cluster]['x'].to_list()
    y = tem[tem.db==cluster]['y'].to_list()
    cx = np.dot(x, y) / np.sum(y)
    cy = np.dot(y, x) / np.sum(x)
    fig, ax = plt.subplots(figsize=(3,3)) 
    if ele =='APP':
        sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=True,color='#C03830',s=3,edgecolor=None,ax=ax)
    else:
        sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=True,color='#B4B5B4',s=3,edgecolor=None,ax=ax)
    ax.set_xlim(cx-400,cx+400)
    ax.set_ylim(cy-400,cy+400)
    ax.axis('off')
    if sizes =='>100':
        plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,'above_100',cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    else:
        plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,sizes,cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    
    plt.close()

In [ ]:
ele = 'APP'
for roi in tqdm.tqdm(apps):
    tem  = roi2df(roi=roi)
    for size in ['5-10','11-50','51-100','>100']:   
        for i in tem[tem['cluster_range']==size].db.drop_duplicates():
            cluster2png(tem,cluster=i,ele=ele,sizes=size)

ele = 'dAPP'
for roi in tqdm.tqdm(dapps):
    tem  = roi2df(roi=roi)
    for size in ['5-10','11-50','51-100','>100']:
        for i in tem[tem['cluster_range']==size].db.drop_duplicates():
            cluster2png(tem,cluster=i,ele=ele,sizes=size)
  

In [ ]:
ele='dAPP'
for size in ['5-10','11-50','51-100','>100']:   
    if size =='>100':
        files = glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/above_100/*.png'%ele)
    else:
        files = glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/*.png'%(ele,size))
    N = len(files)
    avg=Image.open(files[0])
    for i in tqdm.tqdm(range(1,N)):
        img=Image.open(files[i])
        avg=Image.blend(avg,img,1.0/float(i+1))
    if size =='>100':
        avg.save("Blend_%s_above_100.png"%(ele))
    else:
        avg.save("Blend_%s_%s.png"%(ele,size))


In [ ]:
ele = 'APP'
files = glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/above_100/*.png'%ele)
N = len(files)
avg=Image.open(files[0])
for i in tqdm.tqdm(range(1,N)):
    img=Image.open(files[i])
    avg=Image.blend(avg,img,1.0/float(i+1))
avg.save("Blend_%s_101.png"%ele)
avg.show()

In [ ]:
import glob, numpy, PIL
from PIL import Image

In [ ]:
len(glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/51-100/*.png'%'APP'))

In [ ]:
# Alternative method using PIL blend function


In [ ]:
# Generate, save and preview final image
out=Image.fromarray(arr,mode="RGB")
#out.save("Average.png")
out.show()

In [ ]:
voronoi_thershold_df,t,df = voronoi_thershold_ploting(APP_1_3_3.loc[:,['x','y']].to_numpy(),n=30)

df,x,thershold,cluster_probabilty = voronoi_thershold_v2(APP_1_3_3.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
fig, ax = plt.subplots(figsize=(3,3)) 
sns.scatterplot(x='x',y='y',data=df[df.vol<1e6],s=10,edgecolor=None,hue='vol_invers', hue_norm=(-6,-2),palette=ww,ax=ax)
ax.legend([],frameon=False)
norm = plt.Normalize(vmin=-6, vmax=-2)
sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
sm.set_array([])
#ax.get_legend().remove()
ax.figure.colorbar(sm,fraction=0.03, pad=0.01)
points = APP_1_3_3.loc[:,['x','y']].to_numpy()
vor = Voronoi(points)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
plt.xlim([df.x.min(),df.x.max()])
plt.ylim([df.y.min(),df.y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('  .svg',bbox_inches='tight',transparent=False)
#voronoi_thershold_df,t,df = voronoi_thershold_ploting(APP_2_2_3.loc[:,['x','y']].to_numpy(),n=30)
voronoi_thershold_df['x_log'] =np.log10(voronoi_thershold_df['x'])
fig, ax = plt.subplots(figsize=(2.5,2.5)) 
sns.lineplot(x='x_log',y='y',data=voronoi_thershold_df[voronoi_thershold_df.x_log>0],hue='element',errorbar=('sd',1),lw=3,
             err_style='band',palette=["#1D3461","gray"])
#voronoi_thershold_df[voronoi_thershold_df.x_log>0]
plt.axvline(np.log10(t),ls='--',color='k')
ax.grid()
plt.ylabel('$Count$')
plt.xlabel('$Voronoi\;polygon\;area\;(nm^{2})$')
#plt.ylim([-10,700])
#plt.xlim([-500,12000])
#plt.text(8000,40,'$n=%d$'%(len(df)))
# ax.fill_between(range(0,int(t)), -10,750 
#                ,color='#B34F91', alpha=.9)

plt.legend(['$Data$','','$Random\;trails$'],title='',fontsize=10)
plt.locator_params(nbins=6)
plt.title('$Thresholding$')
plt.savefig('voronori_thershold_APP_1_3_3.svg',bbox_inches='tight',transparent=False)


In [ ]:
voronoi_thershold_df[voronoi_thershold_df.x_log>0].min()

In [ ]:
def cluster_sizes_full(dbs_all,dbs_sub,df):
    full_cluster_size = []
    full_cluster_size_nm = []
    if dbs_sub.empty ==True:
        return np.NaN,np.NaN
    else:
        for i in dbs_sub['db']:
            if dbs_sub[dbs_sub['db']==i].db.values[0] == dbs_all[dbs_all['db']==i].db.values[0]:
                full_cluster_size.append(dbs_sub[dbs_sub['db']==i].db.values[0])
                full_cluster_size_nm.append(len(df[df.db==i].loc[:,['x','y']]))#/ConvexHull(df[df.db==i].loc[:,['x','y']].to_numpy()).volume)#pdist(df.loc[df[df.db==i].index,['x','y']].to_numpy()).max())
        return full_cluster_size,full_cluster_size_nm

In [ ]:
df,x,thershold,cluster_probabilty = voronoi_thershold_v2(APP_1_6_2.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
tem = df[df.vol<1e6]
cluster2size = tem[tem.db!=-1].value_counts('db').reset_index()
for ind in tem[(tem.db!=-1)&(~pd.isna(tem.db))].index:
    tem.loc[ind,'cluster_size'] = cluster2size[cluster2size.db==tem.loc[ind,'db']].values[0][-1]
tem.loc[tem[tem.cluster_size.between(5,10)].index,'cluster_range'] = '5-10'
tem.loc[tem[tem.cluster_size.between(11,50)].index,'cluster_range'] = '11-50'
tem.loc[tem[tem.cluster_size.between(51,100)].index,'cluster_range'] = '51-100'
tem.loc[tem[tem.cluster_size.between(101,1000)].index,'cluster_range'] = '>100'
#tem = df[df.vol<1e6]
points = APP_1_6_2.loc[:,['x','y']].to_numpy()

vor = Voronoi(points)
tem.loc[tem[tem.vol<thershold].index,'binary_theshold'] = '1'
tem.loc[tem[tem.vol>=thershold].index,'binary_theshold'] = '0'

fig = plt.figure(figsize = (6,2),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.01,'wspace':.02})
# fig, ax = plt.subplots(figsize=(10,10)) 
sns.scatterplot(x='x',y='y',data=tem,s=1,edgecolor=None,hue='binary_theshold'
                ,palette=["k","k"],hue_order=['0','1'],ax=axs[0],zorder=2)
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=0.05,ax=axs[1],zorder=2)
sns.scatterplot(x='x',y='y',data=tem,s=1,edgecolor=None,hue='binary_theshold',
                palette=["k","#C03830"],hue_order=['0','1'],ax=axs[1],zorder=2,rasterized=0)
axs[1].legend([],frameon=False)

samp = len(set(tem[~pd.isna(tem.db)].db))-1
cmap_test = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#F55536",'#0C8346','#B8B8B8','#8D5A97','#274C77'],samp)
color1 = [mcolors.rgb2hex(cmap_test(i)) for i in range(samp)]
color11 = pd.Series(color1).sample((samp)).to_list()
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],s=5,edgecolor=None,
                hue='cluster_range',ax=axs[2],zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'],rasterized=0)
sns.scatterplot(x='x',y='y',data=tem[((tem.db==-1)|(pd.isna(tem.db)))],
                s=1,edgecolor=None,c='k',alpha=.1,ax=axs[2],rasterized=0)
axs[0].legend([],frameon=False)
axs[2].legend([],frameon=False)
for i in [0,1,2]:
    axs[i].set_xlim([df.x.min(),df.x.max()])
    axs[i].set_ylim([df.y.min(),df.y.max()])
    axs[i].set_xlabel('')
    axs[i].set_ylabel('')
    axs[i].grid(zorder=0,linewidth=.5)
    axs[i].locator_params(nbins=10)
    #axs[i].legend([],frameon=False)
    axs[i].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel.svg',bbox_inches='tight',transparent=False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel.png',dpi=600,bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
sns.scatterplot(x='x',y='y',data=tem,s=1,edgecolor=None,hue='binary_theshold'
            ,palette=["k","k"],hue_order=['0','1'],zorder=2)
ax.set_xlim([df.x.min(),df.x.max()])
ax.set_ylim([df.y.min(),df.y.max()])
ax.set_xlabel('')
ax.set_ylabel('')
ax.grid(zorder=0,linewidth=.5)
ax.locator_params(nbins=10)
ax.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
ax.legend([],frameon=False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel_1.png',dpi=300,bbox_inches='tight',transparent=0)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=0.5,ax=ax,zorder=2)
sns.scatterplot(x='x',y='y',data=tem,s=1,edgecolor=None,hue='binary_theshold',
                 palette=["k","#C03830"],hue_order=['0','1'],zorder=2,rasterized=0)
plt.xlim([df.x.min(),df.x.max()])
plt.ylim([df.y.min(),df.y.max()])
plt.xlabel('')
plt.ylabel('')
plt.grid(zorder=0,linewidth=.5)
plt.locator_params(nbins=10)
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.legend([],frameon=False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel_2.png',dpi=300,bbox_inches='tight',transparent=0)


In [ ]:
samp = len(set(tem[~pd.isna(tem.db)].db))-1
cmap_test = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#F55536",'#0C8346','#B8B8B8','#8D5A97','#274C77'],samp)
color1 = [mcolors.rgb2hex(cmap_test(i)) for i in range(samp)]
color11 = pd.Series(color1).sample((samp)).to_list()


fig, ax = plt.subplots(figsize=(3,3)) 
sns.scatterplot(x='x',y='y',data=tem[(tem.db!=-1)&(~pd.isna(tem.db))],s=5,edgecolor=None,
                hue='cluster_range',ax=ax,zorder=2,palette='inferno_r',hue_order=['5-10','11-50','51-100','>100'],rasterized=0)
sns.scatterplot(x='x',y='y',data=tem[((tem.db==-1)|(pd.isna(tem.db)))],
                s=1,edgecolor=None,c='k',alpha=.1,ax=ax,rasterized=0)
plt.xlim([df.x.min(),df.x.max()])
plt.ylim([df.y.min(),df.y.max()])
plt.xlabel('')
plt.ylabel('')
plt.grid(zorder=0,linewidth=.5)
plt.locator_params(nbins=10)
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.legend([],frameon=False)
plt.savefig('voronori_thershold_APP_1_6_2_pannel_3.png',dpi=300,bbox_inches='tight',transparent=0)


In [ ]:
tem['color'] = tem.cluster_range.map({'5-10':'#EDA843','11-50':'#CE5D43','51-100':'#893066','>100':'#3F2762'})
import matplotlib

In [ ]:
tem.groupby('cluster_range').sample(1,random_state=42)

In [ ]:
def cluster2rad_for_fig(tem):
    samp = tem.groupby('cluster_range').sample(1,random_state=1)
    fig, ax = plt.subplots(figsize=(3,3)) 
    plt.xlim([tem.x.min(),tem.x.max()])
    plt.ylim([tem.y.min(),tem.y.max()])
    plt.xlabel('')
    plt.ylabel('')
    u = 0
    #tem.to_pickle('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/temp.pkl')
    for cluster in set(tem[((tem.db!=-1)&(~pd.isna(tem.db)))].db):
        u = u+1
        x = tem[tem.db==cluster]['x'].to_list()
        y = tem[tem.db==cluster]['y'].to_list()
        cx = np.dot(x, y) / np.sum(y)
        cy = np.dot(y, x) / np.sum(x)
        #ax.scatter(x=cx,y=cy,s=1,c='r')
        ax.scatter(x=x,y=y,s=1,c=tem[tem.db==cluster]['color'].values[0])
        pdis  =[]
        for ind in tem[tem.db==cluster].index:
            x = tem.loc[ind,'x']
            y = tem.loc[ind,'y']
            dist = math.hypot(x - cx, y - cy)
            pdis.append({'ind':ind,'dist':dist})
            pdis_df = pd.DataFrame(pdis)
    #print(pdis_df)
        ind = pdis_df.loc[pdis_df['dist'].idxmax(),'ind']
        r = pdis_df.loc[pdis_df['dist'].idxmax(),'dist']
        ax.add_patch(plt.Circle((cx, cy), r, color=tem[tem.db==cluster]['color'].values[0],alpha=.4))
        
        if cluster in [0,1,5,6,106,47]:
            ax.text(cx,cy,'R=%.0fnm'%r+'\n'+' n=%d'%len(tem[tem.db==cluster]),fontsize=6)
#        ax.text(cx,cy,'%d'%cluster,fontsize=6)
        
    plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
    plt.legend([],frameon=False)
    #sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=False,color='#C03830',s=20,edgecolor=None,ax=ax,zorder=5)
    #sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=False,color='#B4B5B4',s=20,edgecolor=None,ax=ax,zorder=5)
    # ax.set_xlim(cx-200,cx+200)
    # ax.set_ylim(cy-200,cy+200)
    # ax.axis('off')
    # ax.scatter(x=cx,y=cy,s=1,c=)
    
    # pdis  =[]
    # for ind in tem[tem.db==cluster].index:
    #     x = tem.loc[ind,'x']
    #     y = tem.loc[ind,'y']
    #     dist = math.hypot(x - cx, y - cy)
    #     pdis.append({'ind':ind,'dist':dist})
    # pdis_df = pd.DataFrame(pdis)
    # #print(pdis_df)
    # ind = pdis_df.loc[pdis_df['dist'].idxmax(),'ind']
    # r = pdis_df.loc[pdis_df['dist'].idxmax(),'dist']
    # #ax.scatter(x=tem.loc[ind].x,y=tem.loc[ind].y,color='#C03830',s=3,edgecolor='k')
    # ax.plot([cx,tem.loc[ind].x],[cy,tem.loc[ind].y],ls='-',lw=3,color='k')
    # ax.add_patch(plt.Circle((cx, cy), r, color='#E5E5E5',alpha=1))
    # ax.set_title('R=%.0f nm'%r+'\n'+'n=%d'%len(tem[tem.db==cluster]),fontsize=12)
    # #ax.plot([cx+10,cx+60],[cy -47,cy -47],ls='-',lw=1,color='k')
    # print(r)
    plt.savefig('/Users/aelazar/notebooks_for_paper/radi_all.svg',bbox_inches='tight',transparent=False)
    # if sizes =='>100':
    #     plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,'above_100',cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    # else:
    #     plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,sizes,cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    
    # plt.close()

In [ ]:
cluster2rad_for_fig(tem)

In [ ]:


dff = pd.DataFrame()
precent_pass_thershold = []
cluser_prob = []
cluser_size_nm = []
cluster_fraction_l = []
no_cluster_vs_number_locazations = []
cluster_size_vs_number_locazations = []
density = []
no_cluster_vs_number_locazations_sub = []
c = []
apps_v2 = [APP_1_2_1,APP_1_2_2,APP_1_2_3,APP_1_2_4,APP_1_3_1,APP_1_3_2,APP_1_3_3,APP_1_3_4,APP_1_4_1,APP_1_4_2,APP_1_4_3,APP_1_5_1,APP_1_5_2,APP_1_6_1,APP_1_6_2,APP_2_2_1,APP_2_2_2,APP_2_2_3,APP_2_2_4,APP_2_3_1,APP_2_3_2,APP_2_3_3,APP_2_3_4,APP_2_4_1,APP_2_4_2,APP_2_4_3,APP_2_5_1,APP_2_5_2,APP_2_5_3,APP_2_5_4,APP_2_6_1,APP_2_6_2,APP_2_6_3,APP_2_6_4,APP_2_7_1,APP_2_7_2,APP_2_7_3,APP_2_7_4,APP_2_7_5,APP_2_8_1,APP_2_8_2,APP_2_8_3]
for i in tqdm.tqdm(enumerate(apps_v2)):#apps
    original_df = i[1]
    rec = i[1]['rectangle'].to_list()[0]
    rep = i[1]['replicate'].to_list()[0]#i[1].iloc[0,'replicate']
    cell  = i[1]['cell'].to_list()[0]#i[1].iloc[0,'cell']
    df = roi2df(i[1])
    points = i[1].loc[:,['x','y']].to_numpy()
    vor = Voronoi(points)
#    df,x,thershold,cluster_probabilty = voronoi_thershold_v2(i[1].loc[:,['x','y']].to_numpy(),n=1,size_of_grid=4000,min_cluster_size=5)
    q,x,thershold,cluster_probabilty,y = voronoi_thershold_v2(i[1].loc[:,['x','y']].to_numpy(),n=1,size_of_grid=4000,min_cluster_size=5)

    df = roi2df(df)
    
    dff = pd.concat([area2clusters(df,element='APP',squre_number=i[0]),dff])
    precent_pass_thershold.append({'val':len(df[(~pd.isna(df.db))])/len(df),'ele':'APP'})
    dbs_all = df.db.value_counts().reset_index()
    density.append({'cell':cell,'rectangle':rec,'replicate':rep,'ele':'APP','dens':len(df)})
    #print(dbs_all.db.min())
    for j in cluster_probabilty:
        cluser_prob.append({'val':j,'ele':'APP'})
    for h in df2clustersize(df):
        cluser_size_nm.append({'val':h,'ele':'APP'})
    for k in cluster_fraction(df):
        cluster_fraction_l.append({'val':k,'ele':'APP'})  

    no_cluster_vs_number_locazations.append({'val':len(area2clusters(df,element='APP',squre_number=i[0]))/len(df),'ele':'APP','state':'bobby',
                                             'num_clusters':len(area2clusters(df,element='APP',squre_number=i[0])),'number_points':len(df)})
    xs = list(range(int(df.x.min().round()),int(df.x.max().round())+10,500))
    ys = list(range(int(df.y.min().round()),int(df.y.max().round()+10),500))
    for i in range(len(xs)-1):
        for j in range(len(ys)-1):
            dbs_sub = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().reset_index()
            db_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))])
            vor_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(~pd.isna(df.db))])
            all_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))])
            ori_data = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))]
            #intensity= original_df[(original_df.x.between(xs[i],xs[i+1]))&(original_df.y.between(ys[j],ys[j+1]))].intensity
            cluster_size_all = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().to_list()
            cluster_size_medain = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().median()
            cluster_sizes_full_vals,cluster_sizes_full_nm_vals = cluster_sizes_full(dbs_all,dbs_sub,df)
            clusters_in_roi = len(area2clusters(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))],element='APP'))
            if (cluster_sizes_full_vals is not np.NaN):
                if  (len(cluster_sizes_full_vals)>0):
                    cluster_sizes_full_vals_mean= (cluster_sizes_full_vals)
                    cluster_sizes_full_vals_nm_mean= (cluster_sizes_full_nm_vals)
                else:
                    cluster_sizes_full_vals_mean = np.NaN
                    cluster_sizes_full_vals_nm_mean = np.NaN
            else:
                cluster_sizes_full_vals_mean = np.NaN
                cluster_sizes_full_vals_nm_mean = np.NaN
            c.append({'all_points':all_points,'db_points':db_points,'vor_points':vor_points,'original_df':ori_data,'vor':vor,'clusters_in_roi':clusters_in_roi,
                      'cluster_sizes_full':cluster_sizes_full_vals,'cluster_sizes_all':cluster_size_all,'cluster_size_medain':cluster_size_medain,
                      'ele':'APP','cell':cell,'rep':rep,'rec':rec,'sub_x':i,'sub_y':j,'cluster_sizes_full_mean':cluster_sizes_full_vals_mean,
                      'cluster_sizes_full_vals_nm_mean':cluster_sizes_full_vals_nm_mean,'db_ids':dbs_sub.db.to_list()})#'mean_intensety':intensity})
            # if len(cluster_size_all)>0:
            #     print(c[-1])
dapp_v2 = [dAPP_1_2_1,dAPP_1_2_2,dAPP_1_2_3,dAPP_1_2_4,dAPP_1_3_1,dAPP_1_3_2,dAPP_1_3_3,dAPP_1_4_1,dAPP_1_4_2,dAPP_1_4_3,dAPP_1_4_4,dAPP_1_5_1,dAPP_1_5_2,dAPP_1_5_3,dAPP_2_6_1,dAPP_2_6_2,dAPP_2_6_3]
for i in tqdm.tqdm(enumerate(dapp_v2)):#dapps
    original_df = i[1]
    rec = i[1]['rectangle'].to_list()[0]
    rep = i[1]['replicate'].to_list()[0]#i[1].iloc[0,'replicate']
    cell  = i[1]['cell'].to_list()[0]#i[1].iloc[0,'cell']
    points = i[1].loc[:,['x','y']].to_numpy()
    vor = Voronoi(points)
    df= roi2df(i[1])
#    df,x,thershold,cluster_probabilty= voronoi_thershold_v2(i[1].loc[:,['x','y']].to_numpy(),n=1,size_of_grid=4000,min_cluster_size=5)
    q,x,thershold,cluster_probabilty,y = voronoi_thershold_v2(i[1].loc[:,['x','y']].to_numpy(),n=1,size_of_grid=4000,min_cluster_size=5)

    precent_pass_thershold.append({'val':len(df[(~pd.isna(df.db))])/len(df),'ele':'dAPP'})
    dff = pd.concat([area2clusters(df,element='dAPP',squre_number=i[0]),dff])
    dbs_all = df.db.value_counts().reset_index()
    density.append({'cell':cell,'rectangle':rec,'replicate':rep,'ele':'dAPP','dens':len(df)})
    for j in cluster_probabilty:
        cluser_prob.append({'val':j,'ele':'dAPP'})
    for h in df2clustersize(df):
        cluser_size_nm.append({'val':h,'ele':'dAPP'})
    for k in cluster_fraction(df):
        cluster_fraction_l.append({'val':k,'ele':'dAPP'}) 
    no_cluster_vs_number_locazations.append({'val':len(area2clusters(df,element='dAPP',squre_number=i[0]))/len(df),'ele':'dAPP','state':'bobby',
                                             'num_clusters':len(area2clusters(df,element='dAPP',squre_number=i[0])),'number_points':len(df)})
    xs = list(range(int(df.x.min().round()),int(df.x.max().round()+40),500))
    ys = list(range(int(df.y.min().round()),int(df.y.max().round()+40),500))
    for i in range(len(xs)-1):
        for j in range(len(ys)-1):
            dbs_sub = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().reset_index()
            db_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))])
            vor_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(~pd.isna(df.db))])

            all_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))])
            ori_data = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))]

            #intensity= original_df[(original_df.x.between(xs[i],xs[i+1]))&(original_df.y.between(ys[j],ys[j+1]))].intensity
            cluster_size_medain = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().median()
            cluster_sizes_full_vals,cluster_sizes_full_nm_vals = cluster_sizes_full(dbs_all,dbs_sub,df)
            #no_cluster_vs_number_locazations_sub.append({'val':len(area2clusters(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))],element='dAPP',squre_number=1))/all_points,'ele':'dAPP','state':'bobby'})
            clusters_in_roi = len(area2clusters(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))],element='dAPP'))
            if (cluster_sizes_full_vals is not np.NaN):
                if  (len(cluster_sizes_full_vals)>0):
                    cluster_sizes_full_vals_mean= (cluster_sizes_full_vals)
                    cluster_sizes_full_vals_nm_mean= (cluster_sizes_full_nm_vals)
                else:
                    cluster_sizes_full_vals_mean = np.NaN
                    cluster_sizes_full_vals_nm_mean = np.NaN
            else:
                cluster_sizes_full_vals_mean = np.NaN
                cluster_sizes_full_vals_nm_mean = np.NaN
            c.append({'all_points':all_points,'db_points':db_points,'vor_points':vor_points,'original_df':ori_data,'vor':vor,'clusters_in_roi':clusters_in_roi,
                      'cluster_sizes_full':cluster_sizes_full_vals,'cluster_sizes_all':cluster_size_all,'cluster_size_medain':cluster_size_medain,
                      'ele':'dAPP','cell':cell,'rep':rep,'rec':rec,'sub_x':i,'sub_y':j,'cluster_sizes_full_mean':cluster_sizes_full_vals_mean,
                      'cluster_sizes_full_vals_nm_mean':cluster_sizes_full_vals_nm_mean,'db_ids':dbs_sub.db.to_list()})#,'mean_intensety':intensity})
            

In [ ]:
pd.DataFrame(c)

In [ ]:
no_cluster_vs_number_locazations_df

In [ ]:
no_cluster_vs_number_locazations_df = pd.DataFrame(no_cluster_vs_number_locazations)
no_cluster_vs_number_locazations_df['box'] ='A'
no_cluster_vs_number_locazations_df['vl_area'] = no_cluster_vs_number_locazations_df['val']/(4**2)

In [ ]:
no_cluster_vs_number_locazations_df['log_v'] = np.log(no_cluster_vs_number_locazations_df['val'])

In [ ]:

fig, ax = plt.subplots(figsize=(3,4))
sns.violinplot(x='box',y='val',data=no_cluster_vs_number_locazations_df,hue='ele',fill=False,split=True,inner='box',width=0.8,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'])
plt.yticks(fontsize=12)
plt.locator_params(axis='y',nbins=3)
#ax.ticklabel_format(axis='y', style='scientific',scilimits=(0,0))
plt.legend([],frameon=False)
plt.ylabel('')
plt.xlabel('')
plt.savefig('cluster_number_vs_dens.svg',bbox_inches='tight',transparent=False)

In [ ]:
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(2,3))
sns.boxplot(x='box',y='val',data=no_cluster_vs_number_locazations_df,hue='ele',fill=False,width=0.6,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'],showcaps=False)
# sns.stripplot(x='box',y='val',data=no_cluster_vs_number_locazations_df,hue='ele',dodge=True,
#               hue_order=['dAPP','APP'],palette=['gray','#C03830'])
plt.locator_params(axis='y',nbins=4)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.legend([],frameon=False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('cluster_number_vs_dens.svg',bbox_inches='tight',transparent=False)

In [ ]:
sns.scatterplot(x='number_points',y='log_v',data=no_cluster_vs_number_locazations_df,hue='ele')
plt.xscale('log')

In [ ]:
from scipy import stats
print(len(no_cluster_vs_number_locazations_df[no_cluster_vs_number_locazations_df.ele=='APP']))
print(len(no_cluster_vs_number_locazations_df[no_cluster_vs_number_locazations_df.ele=='dAPP']))

stats.ttest_ind(no_cluster_vs_number_locazations_df[no_cluster_vs_number_locazations_df.ele=='APP'].vl_area,
               no_cluster_vs_number_locazations_df[no_cluster_vs_number_locazations_df.ele=='dAPP'].vl_area)

In [ ]:
no_cluster_vs_number_locazations_df

In [ ]:

fig, ax = plt.subplots(figsize=(3,4))
sns.violinplot(x='box',y='vl_area',data=no_cluster_vs_number_locazations_df,hue='ele',fill=False,split=True,inner='box',width=0.8,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'])

plt.yticks(fontsize=16)
plt.locator_params(axis='y',nbins=5)
ax.ticklabel_format(axis='y', style='scientific',scilimits=(0,0))
plt.legend([],frameon=False)
plt.ylabel('')
plt.xlabel('')
plt.savefig('cluster_number_vs_dens.svg',bbox_inches='tight',transparent=False)

In [ ]:
local_numbers  =[] 
for i in dapp_v2:
    local_numbers.append({'box':'A','ele':'dAPP','val':len(i)/(4**2),'n':np.log10(len(i))})
for i in apps_v2:
    local_numbers.append({'box':'A','ele':'APP','val':len(i)/(4**2),'n':np.log10(len(i))})

fig, ax = plt.subplots(figsize=(3,4))
local_dens = pd.DataFrame(local_numbers)
sns.violinplot(x='box',y='val',data=local_dens,hue='ele',fill=False,split=True,inner='box',width=0.8,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'])
plt.locator_params(axis='y',nbins=4)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.legend([],frameon=False)
plt.xlabel('')
plt.ylabel('')

plt.savefig('loc_vs_dens.svg',bbox_inches='tight',transparent=False)


In [ ]:
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(2,3))
sns.boxplot(x='box',y='val',data=local_dens,hue='ele',fill=False,width=0.6,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'],showcaps=False)
plt.locator_params(axis='y',nbins=4)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.legend([],frameon=False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('loc_vs_dens.svg',bbox_inches='tight',transparent=False)


In [ ]:
stats.ttest_ind(local_dens[local_dens.ele=='APP'].val,local_dens[local_dens.ele=='dAPP'].val)

In [ ]:
size_loc = []
for i in enumerate(dapp_v2):
    ten = roi2df(dapp_v2[i[0]])
    for cluster in set(ten[(ten.db!=-1)&(~pd.isna(ten.db))].db):
        v = ten[ten.db==cluster].cluster_size.values[0]
        size_loc.append({'ele':'dAPP','number_of_loc':len(ten),'cluster_size':v,'log_V':np.log10(v)})
for i in enumerate(apps_v2):
    ten = roi2df(apps_v2[i[0]])
    for cluster in set(ten[(ten.db!=-1)&(~pd.isna(ten.db))].db):
        v = ten[ten.db==cluster].cluster_size.values[0]
        size_loc.append({'ele':'APP','number_of_loc':len(ten),'cluster_size':v,'log_V':np.log10(v)})   

In [ ]:
pd.Series(list(set(pd.DataFrame(size_loc).number_of_loc))).hist()

In [ ]:
size_loc_df = pd.DataFrame(size_loc)

In [ ]:
set(size_loc_df[(size_loc_df.number_of_loc.between(0,1000))].number_of_loc)
size_loc_df

In [ ]:
size_loc_df.loc[size_loc_df[(size_loc_df.number_of_loc.between(300,1000))].index,'loc_range'] = '300-1000'
#size_loc_df.loc[size_loc_df[(size_loc_df.number_of_loc.between(1001,2000))].index,'loc_range'] = '1001-2000'
size_loc_df.loc[size_loc_df[(size_loc_df.number_of_loc.between(1001,6000))].index,'loc_range'] = '1001-6000'
size_loc_df.loc[size_loc_df[(size_loc_df.number_of_loc.between(6001,12000))].index,'loc_range'] = '6001-12000'

In [ ]:
fig, ax = plt.subplots(figsize=(3,4)) 
sns.violinplot(fill=0,y='cluster_size',x='loc_range',data=size_loc_df,log_scale=True,hue='ele',
               order=['300-1000','1001-6000','6001-12000'],split=True,hue_order=['dAPP','APP'],palette=['gray','r'])
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30,fontsize=16)
plt.ylabel('')
plt.xlabel('')
plt.savefig('size_loc__size_clusters.svg',bbox_inches='tight',transparent=True)

In [ ]:
stats.ttest_ind(local_dens[local_dens.ele=='APP'].val,
               local_dens[local_dens.ele=='dAPP'].val)

In [ ]:
l =[]
for i in apps_v2:
    df= roi2df(i)
    s = area2clusters(df,element='APP',squre_number=1)['cluster_size']
    for j in s:
        l.append({'val':j,'ele':'APP'})
for i in dapp_v2:
    df= roi2df(i)
    s = area2clusters(df,element='dAPP',squre_number=1)['cluster_size']
    for j in s:
        l.append({'val':j,'ele':'dAPP'})

In [ ]:
ll = pd.DataFrame(l)
ll['box']='A'

In [ ]:
ll['log_val']=np.log10(ll['val'])


In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.violinplot(x='box',y='log_val',data=ll,hue='ele',fill=False,split=True,inner='box',width=0.8,log_scale=False,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'])
plt.yticks(fontsize=16)
plt.locator_params(axis='y',nbins=4)
# ax.ticklabel_format(axis='y', style='scientific',scilimits=(0,0))
plt.legend([],frameon=False)
plt.ylabel('')
plt.xlabel('')
plt.savefig('loc_in_cluster.svg',bbox_inches='tight',transparent=False)


In [ ]:
print(len(ll[ll.ele=='APP'].val))
print(len(ll[ll.ele=='dAPP'].val))

stats.ttest_ind(ll[ll.ele=='APP'].val,
               ll[ll.ele=='dAPP'].val)

In [ ]:

good_roi = []
for i in tqdm.tqdm(grid_df[(grid_df.des<1500)].index):
    if grid_df.loc[i,'original_df'].empty==False:
        good_roi.append('%s_%s_%s_%s'%(grid_df.loc[i,'ele'],grid_df.loc[i,'rep'],grid_df.loc[i,'cell'],grid_df.loc[i,'rec']))
#grid_df[grid_df.des<1500].iloc[0].original_df.name

In [ ]:
grid_df = pd.DataFrame(c)
grid_df.sort_values(by='all_points')
scale_factor = .5**2
scale_factor_nm = 500**2
grid_df['des'] = grid_df['all_points']/scale_factor
grid_df['des_nm'] = grid_df['all_points']/scale_factor_nm
grid_df['box']='A'
grid_df['precent_clusterd']  = 100*(grid_df['db_points']/grid_df['all_points'])

In [ ]:
#grid_df.to_pickle('grid_df.pkl')
grid_df2 = pd.read_pickle('grid_df.pkl')
grid_df = pd.read_pickle('grid_df.pkl')

In [ ]:
pd.Series([x for xs in grid_df2[grid_df2.ele=='APP'].cluster_sizes_full.dropna().values for x in xs]).max()

In [ ]:
grid_df.columns

In [ ]:
sns.scatterplot(x='all_points',y='vor_points',data=grid_df[grid_df.des<1500],hue='ele')
#plt.xscale('log')
#plt.yscale('log')

In [ ]:
r = list(range(0,2000,500))

In [ ]:
color_app = ['b','g','orange','k']

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
for i in range(0,len(r)-1):
    sns.kdeplot(flatten(grid_df[grid_df.des.between(r[i],r[i+1])&(grid_df.ele=='APP')].mean_intensety),color=color_app[i],log_scale=True)
    sns.kdeplot(flatten(grid_df[grid_df.des.between(r[i],r[i+1])&(grid_df.ele!='APP')].mean_intensety),color='gray',log_scale=True)
#plt.xscale('log')

In [ ]:
grid_df

In [ ]:
cc = grid_df[grid_df.des<1000].groupby(['all_points','ele']).mean().reset_index()
cc['fq'] = 100*(cc['db_points']/ cc['all_points'])
cc.loc[cc[cc.ele=='APP'].index,'smooth'] = cc[cc.ele=='APP'].db_points.rolling(window=1).mean()
cc.loc[cc[cc.ele=='dAPP'].index,'smooth'] = cc[cc.ele=='dAPP'].db_points.rolling(window=1).mean()
cc.loc[cc[cc.ele=='APP'].index,'smooth_fq'] = cc[cc.ele=='APP'].fq.rolling(window=10,min_periods=1).mean()
cc.loc[cc[cc.ele=='dAPP'].index,'smooth_fq'] = cc[cc.ele=='dAPP'].fq.rolling(window=10,min_periods=1).mean()
#sns.lineplot(x='all_points',y='smooth_fq',data=cc,hue='ele',palette=['k','k'])
#plt.xlim([2,1000])
#plt.xscale('log')
#sns.scatterplot(x='all_points',y='precent_clusterd',data=grid_df[grid_df.des<1000],hue='ele',alpha=1)
sns.scatterplot(x='all_points',y='fq',data=cc,hue='ele',alpha=1)

In [ ]:
#grid_df[(grid_df.des<1500)
#2 reps, 6 cells 20 ROIs dAPP 18 
#2 reps, 12 cells 43 ROIs APP
len(grid_df[(grid_df.des<1500)&(grid_df.ele!='APP')].drop_duplicates(subset=['cell','rep','rec']))

In [ ]:
import scipy.optimize as opt 
from sklearn.metrics import r2_score
def KD(x,BMAX,KD):#Y = Bmax*X/(Kd + X)
    nominator = BMAX*x
    denominator = KD+x
    return nominator/denominator
    KD

In [ ]:
sns.scatterplot(x='des',y='smooth_fq',data=cc[cc.des<1000],hue='ele')
#plt.xscale('log')

In [ ]:
sns.boxplot(cc[(cc.des<1000)&(cc.ele=='dAPP')&(cc.des.between(0,300))].fq)

In [ ]:
r_app = len(cc[(cc.ele=='APP')&(cc.des.between(0,1500))])
fitCoefs_APP, covMatri_APP = opt.curve_fit(KD,cc[cc.ele=='APP'].iloc[1:r_app].des,cc[cc.ele=='APP'].iloc[1:r_app].smooth_fq ,method='lm',p0=[.1,200])
y_pred_APP = KD(cc[cc.ele=='APP'].des[1:r_app], *fitCoefs_APP)
r2  =r2_score(cc[cc.ele=='APP'].smooth_fq[1:r_app], y_pred_APP)
print(fitCoefs_APP,r2)

r_dapp = len(cc[(cc.ele=='dAPP')&(cc.des.between(0,1500))])
fitCoefs_dAPP, covMatri_APP = opt.curve_fit(KD,cc[cc.ele!='APP'].iloc[1:r_dapp].des,cc[cc.ele!='APP'].iloc[1:r_dapp].smooth_fq ,method='lm',p0=[.1,200])
y_pred_dAPP = KD(cc[cc.ele=='dAPP'].des[1:r_dapp], *fitCoefs_dAPP)
r2  =r2_score(cc[cc.ele=='dAPP'].smooth_fq[1:r_dapp], y_pred_dAPP)
print(fitCoefs_dAPP,r2)
refDose = list(range(0,1500))
fig, ax = plt.subplots(figsize=(4,4)) 
ax.grid()
sns.scatterplot(x='des',y='fq',data=cc[cc.des<1500],hue='ele',alpha=.3,s=10,hue_order=['APP','dAPP'],palette=['#C03830','#B4B5B4'],)
sns.scatterplot(x='des',y='smooth_fq',data=cc[cc.des<1500],hue='ele',alpha=1,s=10,hue_order=['APP','dAPP'],palette=['#C03830','#B4B5B4'],edgecolor='k',zorder=1)
plt.plot(refDose,[KD(i,*list(fitCoefs_APP)) for i in refDose],color='#C03830',lw=4)
plt.plot(refDose,[KD(i,*list(fitCoefs_dAPP)) for i in refDose],color='#B4B5B4',lw=4)
#plt.plot(cc[cc.ele=='APP'].des[1:175],y_pred_APP)
#plt.xscale('log')
plt.xlim([5,1500])
plt.ylim([-0,1])
plt.axhline(fitCoefs_APP[0],color='k',ls='--',lw=.75)
ax.plot([fitCoefs_APP[1],fitCoefs_APP[1]],[0,KD(fitCoefs_APP[1], *fitCoefs_APP)],color='k',ls='--',lw=.75)
plt.legend([],frameon=False)
plt.xlabel('$\\rho\;(No.localizations/\\mu m^{2}}$)',fontsize=10)
ax.fill_between([0,300], 0,1 
               ,color='#2336A5', alpha=.15)
ax.fill_between([301,900], 0,1 
               ,color='#B9BABC', alpha=.15)
ax.fill_between([901,1000], 0,250 
               ,color='#C2523A', alpha=.15)
plt.ylabel('$\\frac{Localizations\;in\;clusters}{\Sigma Localizations}$',fontsize=12)
#plt.xscale('')
#plt.savefig('fq_smooth_fit.svg',bbox_inches='tight',transparent=False)


In [ ]:
bins_l = []
for j in flatten(grid_df[(grid_df.des.between(0,300))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(0,300))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
spec = dict(x='densitiy',y='cluster_sizes_full_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)
fig, ax = plt.subplots(figsize=(3,3)) 
spec = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'],
            )
sns.stripplot(**spec,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=['0-300','301-900','901-1500'])
sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=pd.DataFrame(bins_l).groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['APP','dAPP'],
    join=False, dodge=.8 - .8 / 2,order=['0-300','301-900','901-1500'],
    markers="_", scale=1.2, errorbar=None,palette=['k','k'])
plt.yscale('log')
plt.ylim([4,350])
ax.grid()
plt.xticks(rotation=0,fontsize=11)
plt.ylabel('$Localizations\;per\;cluster$',fontsize=14)
plt.xlabel('$\\rho\;(No.localizations/\\mu m^{2}}$)',fontsize=14)
plt.legend([],frameon=False)
ax.fill_between([-.5,.45], 0,350 
               ,color='#2336A5', alpha=.2)
ax.fill_between([.5,1.45], 0,350 
               ,color='#B9BABC', alpha=.2)
ax.fill_between([1.5,2.5], 0,350 
               ,color='#C2523A', alpha=.2)
#plt.savefig('Localizations_per_clsuter_SNAPtm.svg',bbox_inches='tight',transparent=False)


In [ ]:

from sklearn.linear_model import LinearRegression

In [ ]:
x = np.array(grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<1500)].all_points.to_list()).reshape((-1, 1))
y= grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<1500)].db_points.to_list()
model_APP = LinearRegression().fit(x,y)
r_sq_APP = model_APP.score(x_APP,y_APP_valus)
print(f"coefficient of determination: {r_sq_APP}")
print(f"intercept: {model_APP.intercept_}")
print(f"slope: {model_APP.coef_}")
y_pred_APP = model_APP.predict(x_APP)

In [ ]:
stats.spearmanr(grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].all_points,
              grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].db_points)

In [ ]:
sns.scatterplot(x='all_points',y='vor_points',data=grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)],
                edgecolor=None,zorder=2,alpha=.5,color=['#C03830'],s=30)
#plt.plot(x_APP,y_pred_APP,color='k',zorder=2)

In [ ]:
dens_top = 1500
x_APP = np.array(grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].all_points.to_list()).reshape((-1, 1))
y_APP_valus= grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].db_points.to_list()
model_APP = LinearRegression().fit(x_APP,y_APP_valus)
r_sq_APP = model_APP.score(x_APP,y_APP_valus)
print(f"coefficient of determination: {r_sq_APP}")
print(f"intercept: {model_APP.intercept_}")
print(f"slope: {model_APP.coef_}")
y_pred_APP = model_APP.predict(x_APP)

x_dAPP = np.array(grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].all_points.to_list()).reshape((-1, 1))
y_dAPP_valus= grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].db_points.to_list()
model_dAPP = LinearRegression().fit(x_dAPP,y_dAPP_valus)
r_sq_dAPP = model_dAPP.score(x_dAPP,y_dAPP_valus)
print(f"coefficient of determination: {r_sq_dAPP}")
print(f"intercept: {model_dAPP.intercept_}")
print(f"slope: {model_dAPP.coef_}")
y_pred_dAPP = model_dAPP.predict(x_dAPP)
reg_app  = grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].groupby(['all_points']).median().reset_index()
reg_dapp  = grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].groupby(['all_points']).median().reset_index()

fig, ax = plt.subplots(figsize=(3,3)) 
ax.grid()
sns.scatterplot(x='all_points',y='db_points',data=grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)],
                edgecolor=None,zorder=2,alpha=.5,color=['#C03830'],s=30)
#plt.plot(x_APP,y_pred_APP,color='k',zorder=2)

sns.scatterplot(x='all_points',y='db_points',data=grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)],
                edgecolor=None,zorder=2,alpha=.5,color='#B4B5B4',s=30)
#plt.plot(x_dAPP,y_pred_dAPP,zorder=3,c='#B4B5B4',lw=3)
plt.plot(x_dAPP,y_pred_dAPP,zorder=2,c='k',lw=1)
plt.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1)
plt.xlabel('Total No. points',fontsize=12)
plt.ylabel('No. points in Clusters',fontsize=12)
plt.plot([365,0],[365,0],ls='--',lw=1,color='gray')
plt.locator_params(nbins=8)
plt.xlim([0,365])
plt.ylim([0,365])
#sns.scatterplot(x='all_points',y='db_points',data=reg_app,color='r',zorder=3)
#sns.scatterplot(x='all_points',y='db_points',data=reg_dapp,color='k',zorder=3)

In [ ]:
grid_df['box']='A'
grid_df[(pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)]

In [ ]:
grid_df.loc[10,'original_df']

In [ ]:
for i in grid_df[~(pd.isna(grid_df.cluster_sizes_all))&(grid_df.des<dens_top)].index:
    grid_df.loc[i,'number_clusters'] = len(grid_df.loc[i,'cluster_sizes_all'])
    

In [ ]:
grid_df[~(pd.isna(grid_df.cluster_sizes_all))&(grid_df.des<dens_top)].sample(40).iloc[0:20]

In [ ]:
grid_df['db_vspoints'] = grid_df['db_points']/grid_df['all_points']
grid_df

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.violinplot(x='box',y='des_nm',data=grid_df[(grid_df.des<dens_top)],hue='ele',fill=False,split=True,inner='box',width=0.8,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'])
plt.yticks(fontsize=16)
plt.locator_params(axis='y',nbins=6)
ax.ticklabel_format(axis='y', style='scientific',scilimits=(0,0))
plt.legend([],frameon=False)
plt.ylabel('')
plt.xlabel('')
plt.savefig('loc_vs_dens.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.violinplot(x='box',y='db_vspoints',data=grid_df[(grid_df.des<dens_top)],hue='ele',fill=False,split=True,inner='box',width=0.8,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'])
plt.yticks(fontsize=16)
plt.locator_params(axis='y',nbins=6)
ax.ticklabel_format(axis='y', style='scientific',scilimits=(0,0))
plt.legend([],frameon=False)
plt.ylabel('')
plt.xlabel('')
plt.axhline(.3)

In [ ]:
#sns.violinplot(x='box',y='des',data=grid_df[(grid_df.des<dens_top)],hue='ele',fill=False,split=True)
sns.scatterplot(x='all_points',y='db_points',data=grid_df[(grid_df.des<dens_top)],hue='ele')

In [ ]:
dens_top = 1500
y_APP_valus = grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].vor_points.to_list()
x_APP= np.array(grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].all_points.to_list()).reshape(-1, 1)
model_APP = LinearRegression(fit_intercept=False).fit(x_APP,y_APP_valus)
r_sq_APP = model_APP.score(x_APP,y_APP_valus)
print(f"coefficient of determination: {r_sq_APP}")
print(f"intercept: {model_APP.intercept_}")
print(f"slope: {model_APP.coef_}")
y_pred_APP = model_APP.predict(x_APP)

y_dAPP_valus = grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].vor_points.to_list()
x_dAPP= np.array(grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].all_points.to_list()).reshape(-1, 1)
model_dAPP = LinearRegression(fit_intercept=False).fit(x_dAPP,y_dAPP_valus)
r_sq_dAPP = model_dAPP.score(x_dAPP,y_dAPP_valus)
print(f"coefficient of determination: {r_sq_dAPP}")
print(f"intercept: {model_dAPP.intercept_}")
print(f"slope: {model_dAPP.coef_}")
y_pred_dAPP = model_dAPP.predict(x_dAPP)
# reg_app  = grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].groupby(['all_points']).median().reset_index()
# reg_dapp  = grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].groupby(['all_points']).median().reset_index()


In [ ]:
from scipy import stats

def get_conf_int(alpha, lr, X=x_dAPP, y=y_dAPP_valus):
    """
    Returns (1-alpha) 2-sided confidence intervals
    for sklearn.LinearRegression coefficients
    as a pandas DataFrame
    """
    
    coefs = np.r_[[lr.intercept_], lr.coef_]
    
    # Check if X is a DataFrame or ndarray and handle accordingly
    if isinstance(X, pd.DataFrame):
        X_aux = X.copy()
        X_aux.insert(0, 'const', 1)
    else:  # Assume it's a numpy array
        X_aux = np.column_stack((np.ones(X.shape[0]), X))
    
    dof = X_aux.shape[0] - X_aux.shape[1]
    mse = np.sum((y - lr.predict(X)) ** 2) / dof
    var_params = np.diag(np.linalg.inv(X_aux.T.dot(X_aux)))
    t_val = stats.t.isf(alpha/2, dof)
    gap = t_val * np.sqrt(mse * var_params)
    
    column_names = ['const'] + (X.columns.tolist() if isinstance(X, pd.DataFrame) else [f'X{i}' for i in range(X.shape[1])])
    
    return pd.DataFrame({
        'lower': coefs - gap, 'upper': coefs + gap
    }, index=column_names)

In [ ]:
get_conf_int(0.05,model_dAPP, X=x_dAPP, y=y_dAPP_valus)

In [ ]:
(0.507186+ 0.557226)
0.2560559-0.241890
# 0.246- 0.226312

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
sns.scatterplot(y='db_points',x='all_points',edgecolor=None,s=25,data=grid_df[grid_df.all_points<400],hue='ele',hue_order=['dAPP','APP'],palette=['gray','#C03830'],alpha=1,zorder=2)

ax.set_ylim([-10,400])
ax.set_xlim([-10,400])
ax.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1)

In [ ]:
grid_df

In [ ]:
g= sns.JointGrid(y='all_points',x='db_points',data=grid_df[grid_df.all_points<400],hue='ele',height=4)#,edgecolor=None,s=10
g.ax_joint.grid()
g.ax_joint.set_ylim([-10,385])
g.ax_joint.set_xlim([-10,210])
g.plot_joint(sns.scatterplot,edgecolor=None,s=25,hue_order=['dAPP','APP'],palette=['gray','#C03830'])

sns.scatterplot(data=grid_df.loc[[3148,2983,3001]], x='all_points', y='db_points',s=1000)

g.plot_marginals(sns.violinplot,split=True,saturation=1,fill=0,hue_order=['dAPP','APP'],palette=['gray','#C03830'])
g.ax_joint.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1,ls='-.')
g.ax_joint.plot(x_dAPP,y_pred_dAPP,color='k',zorder=2,lw=1,ls='-')
g.ax_joint.set_xlabel('')
g.ax_joint.set_ylabel('')
g.ax_joint.legend([],frameon=False)
plt.savefig('cl_vs_loca_SNAPtm.svg',bbox_inches='tight',transparent=False)


In [ ]:
grid_df[grid_df.all_points<400].ele.value_counts()

In [ ]:
grid_df['dens_loc'] = grid_df.all_points/250
grid_df['dens_cluster'] = grid_df.db_points/250
grid_df['dens_vor_all'] = grid_df.vor_points/250

In [ ]:
g= sns.JointGrid(x='all_points',y='db_points',data=grid_df[grid_df.all_points<400],hue='ele',height=4,space=0)#,edgecolor=None,s=10
g.ax_joint.grid(zorder=0)
g.ax_joint.set_ylim([-10,300])
g.ax_joint.set_xlim([-10,400])
g.plot_joint(sns.scatterplot,edgecolor=None,s=25,hue_order=['dAPP','APP'],palette=['gray','#C03830'],alpha=1,zorder=2)#,scatter_kws={'color':'#C03830','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25
#g.plot_joint(sns.scatterplot,edgecolor=None,s=500,hue_order=['dAPP','APP'],palette=['gray','#C03830'],data=grid_df.loc[[3148,2983,3001]])#,scatter_kws={'color':'#C03830','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25
# sns.scatterplot(data=grid_df.loc[[3148,2983,2995]], x='all_points', y='db_points',s=70,ax=g.ax_joint,color='gray',edgecolor='k')
# sns.scatterplot(data=grid_df.loc[[198,272,494]], x='all_points', y='db_points',s=70,ax=g.ax_joint,color='#C03830',edgecolor='k')


g.plot_marginals(sns.violinplot,split=True,saturation=1,fill=0,hue_order=['dAPP','APP'],palette=['gray','#C03830'])
g.ax_joint.plot(x_dAPP,y_pred_dAPP,color='k',zorder=2,lw=1,ls='-.')
g.ax_joint.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1)
#g.ax_joint.plot([385,0],[385,0],ls='--',lw=1,color='gray')
g.ax_joint.set_xlabel('')
g.ax_joint.set_ylabel('')
g.ax_joint.locator_params(nbins=6)
g.ax_joint.legend([],frameon=False)
plt.savefig('cl_vs_loca_SNAPtm.svg',bbox_inches='tight',transparent=False)


In [ ]:
dens_top = 1500
y_APP_valus = grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].dens_vor_all.to_list()
x_APP= np.array(grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].dens_loc.to_list()).reshape(-1, 1)
model_APP = LinearRegression(fit_intercept=False).fit(x_APP,y_APP_valus)
r_sq_APP = model_APP.score(x_APP,y_APP_valus)
print(f"coefficient of determination: {r_sq_APP}")
print(f"intercept: {model_APP.intercept_}")
print(f"slope: {model_APP.coef_}")
y_pred_APP = model_APP.predict(x_APP)

y_dAPP_valus = grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].dens_vor_all.to_list()
x_dAPP= np.array(grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].dens_loc.to_list()).reshape(-1, 1)
model_dAPP = LinearRegression(fit_intercept=False).fit(x_dAPP,y_dAPP_valus)
r_sq_dAPP = model_dAPP.score(x_dAPP,y_dAPP_valus)
print(f"coefficient of determination: {r_sq_dAPP}")
print(f"intercept: {model_dAPP.intercept_}")
print(f"slope: {model_dAPP.coef_}")
y_pred_dAPP = model_dAPP.predict(x_dAPP)

In [ ]:
model_dAPP.intercept_

In [ ]:
g= sns.JointGrid(x='dens_loc',y='dens_vor_all',data=grid_df[grid_df.all_points<400],hue='ele',height=4,space=0)#,edgecolor=None,s=10
g.ax_joint.grid(zorder=0)
# g.ax_joint.set_ylim([-10,300])
# g.ax_joint.set_xlim([-10,400])
g.plot_joint(sns.scatterplot,edgecolor=None,s=25,hue_order=['dAPP','APP'],palette=['gray','#C03830'],alpha=1,zorder=2)#,scatter_kws={'color':'#C03830','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25
#g.plot_joint(sns.scatterplot,edgecolor=None,s=500,hue_order=['dAPP','APP'],palette=['gray','#C03830'],data=grid_df.loc[[3148,2983,3001]])#,scatter_kws={'color':'#C03830','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25
# sns.scatterplot(data=grid_df.loc[[3148,2983,2995]], x='all_points', y='db_points',s=70,ax=g.ax_joint,color='gray',edgecolor='k')
# sns.scatterplot(data=grid_df.loc[[198,272,494]], x='all_points', y='db_points',s=70,ax=g.ax_joint,color='#C03830',edgecolor='k')


g.plot_marginals(sns.boxplot,saturation=1,fill=0,hue_order=['dAPP','APP'],palette=['gray','#C03830'])
g.ax_joint.plot(x_dAPP,y_pred_dAPP,color='k',zorder=2,lw=1,ls='-.')
g.ax_joint.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1)
#g.ax_joint.plot([385,0],[385,0],ls='--',lw=1,color='gray')
g.ax_joint.set_xlabel('')
g.ax_joint.set_ylabel('')
g.ax_joint.locator_params(nbins=4)
g.ax_joint.legend([],frameon=False)
plt.savefig('cl_vs_loca_SNAPtm.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3))

sns.scatterplot(x='dens_loc',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='dAPP')],edgecolor=None,c='gray',s=10,alpha=1)
sns.scatterplot(x='dens_loc',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')],edgecolor=None,c='#C03830',s=10,alpha=1)
ax.plot(x_dAPP,y_pred_dAPP,color='k',zorder=2,lw=1,ls='-.')
ax.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1,ls='-')
plt.ylim([-.1,1.2])
plt.xlim([-.1,1.7])
plt.locator_params(nbins=4)
ax.grid()
plt.savefig('cl_vs_loca_SNAPtm.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
sns.histplot(x='dens_loc',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')],
             bins=18,hue='ele',palette=['#C03830'],cbar=True)
#sns.scatterplot(x='dens_loc',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')],edgecolor=None,c='#C03830',s=10,alpha=1)

# plt.ylim([-.1,1.2])
# plt.xlim([-.1,1.7])
plt.locator_params(nbins=6)
ax.grid()
plt.legend([])

In [ ]:
print(grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')&(grid_df.dens_loc.between(0,.25))].dens_vor_all.mean())
print(grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')&(grid_df.dens_loc.between(0.25,.5))].dens_vor_all.mean())
print(grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')&(grid_df.dens_loc.between(0.5,.75))].dens_vor_all.mean())
print(grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')&(grid_df.dens_loc.between(1,1.25))].dens_vor_all.mean())
print(grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')&(grid_df.dens_loc.between(1.25,1.7))].dens_vor_all.mean())

In [ ]:
grid_df['dens_loc_range'] = np.NaN
grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.dens_loc.between(0,.25))].index,'dens_loc_range'] = '0.00-.25'
grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.dens_loc.between(0.25,.5))].index,'dens_loc_range'] = '0.25-.50'
grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.dens_loc.between(0.5,.75))].index,'dens_loc_range'] = '0.50-.75'
grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.dens_loc.between(0.75,1))].index,'dens_loc_range'] = '.75-1.00'
grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.dens_loc.between(1,1.5))].index,'dens_loc_range'] = '1.00-1.50'
# grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.dens_loc.between(1.25,1.5))].index,'dens_loc_range'] = '1.25'

In [ ]:
from numpy import std, mean, sqrt

#correct if the population S.D. is expected to be equal for the two groups.
def cohen_d(x,y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    return (mean(x) - mean(y)) / sqrt(((nx-1)*std(x, ddof=1) ** 2 + (ny-1)*std(y, ddof=1) ** 2) / dof)


In [ ]:
for i in ['0.00-.25',
 '0.25-.50',
 '0.50-.75',
    '.75-1.00',
 '1.00-1.50']:
    #print(i)
    app_ = grid_df[(grid_df.all_points<400)&(grid_df.dens_loc_range==i)&(grid_df.ele=='APP')].dens_vor_all
    dapp_ = grid_df[(grid_df.all_points<400)&(grid_df.dens_loc_range==i)&(grid_df.ele=='dAPP')].dens_vor_all
    print(i,len(grid_df[(grid_df.all_points<400)&(grid_df.dens_loc_range==i)&(grid_df.ele=='APP')]),len(grid_df[(grid_df.all_points<400)&(grid_df.dens_loc_range==i)&(grid_df.ele=='dAPP')]))
    print(cohen_d(app_,dapp_))
    print(stats.ttest_ind(app_,dapp_))
    # print(grid_df[(grid_df.all_points<400)&(grid_df.dens_loc_range==i)]['ele'].value_counts()  )

In [ ]:
grid_df

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
sns.boxplot(x='dens_loc_range',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)]
             ,palette=['gray','#C03830'],hue='ele',hue_order=['dAPP','APP'],fill=False,showfliers=False,showcaps=False)
sns.stripplot(x='dens_loc_range',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)]
             ,palette=['gray','#C03830'],hue_order=['dAPP','APP'],hue='ele',dodge=True,s=2)
plt.legend([],frameon=False)
plt.ylim([-.1,1.5])
plt.locator_params(axis='y',nbins=4)
plt.xlabel('')
plt.ylabel('')
plt.xticks(rotation=90)

# ax.grid()
plt.savefig('cl_vs_loca_SNAPtm_v2.svg',bbox_inches='tight',transparent=False)

In [ ]:
grid_df_plot  = grid_df[(grid_df.all_points<400)]

In [ ]:
grid_df_plot['dens_loc_range_'],_ = pd.qcut(grid_df_plot.dens_loc, q=4, labels=False, retbins=True)
for i in enumerate(grid_df_plot.sort_values(by='dens_loc').index):
    grid_df_plot.loc[i[1],'rank'] = i[0]


In [ ]:
grid_df_plot[grid_df_plot.dens_loc_range_==0]['ele'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
sns.boxplot(x='dens_loc_range_',y='dens_vor_all',data=grid_df_plot[(grid_df_plot.all_points<400)]
             ,palette=['gray','#C03830'],hue='ele',hue_order=['dAPP','APP'],fill=False,showfliers=False,showcaps=False)
sns.stripplot(x='dens_loc_range_',y='dens_vor_all',data=grid_df_plot[(grid_df_plot.all_points<400)]
             ,palette=['gray','#C03830'],hue_order=['dAPP','APP'],hue='ele',dodge=True,s=2)
plt.legend([],frameon=False)
plt.locator_params(axis='y',nbins=4)
#plt.ylim([-.1,1.5])
#plt.yscale('log')
#plt.locator_params(axis='y',nbins=4)
# plt.xlabel('')
# plt.ylabel('')
# plt.xticks(rotation=90)

# # ax.grid()
# plt.savefig('cl_vs_loca_SNAPtm_v2.svg',bbox_inches='tight',transparent=False)

In [ ]:
grid_df[(grid_df.all_points<400)&(grid_df.dens_vor_all.between(1,1.25))]

In [ ]:
vstack_app = grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')].index,['dens_loc','dens_vor_all']]
from scipy.stats import gaussian_kde
xy_app = np.vstack([vstack_app.dens_loc,vstack_app.dens_vor_all])
z_app = gaussian_kde(xy_app)(xy_app)

vstack_dapp = grid_df.loc[grid_df[(grid_df.all_points<400)&(grid_df.ele=='dAPP')].index,['dens_loc','dens_vor_all']]
xy_dAPP = np.vstack([vstack_dapp.dens_loc,vstack_dapp.dens_vor_all])
z_dapp = gaussian_kde(xy_dAPP)(xy_dAPP)

fig, ax = plt.subplots(figsize=(4,3))
scatter = ax.scatter(vstack_dapp.dens_loc, vstack_dapp.dens_vor_all, c=z_dapp, s=3,cmap='viridis',rasterized=False)
#cax = fig.add_axes([0.18, 0.6, 0.025, 0.25])
# plt.colorbar(sm,cax=cax)
#scatter = ax.scatter(vstack_dapp.log_number, vstack_dapp.log_r, c=z_dapp, s=3,cmap='viridis',rasterized=True)
colorbar = plt.colorbar(scatter, ax=cax,shrink=1)  # Add colorbar
colorbar.set_label('Density')
colorbar = plt.colorbar(scatter, ax=ax,shrink=.5)
#ax.plot(x_dAPP,fy_pred_dAPP,color='k',zorder=2,lw=1,ls='-.')
plt.plot([-.1,1.6],[-.1,1.6],ls='--',color='gray')
plt.ylim([-.1,1.6])
plt.xlim([-.1,1.6])
plt.locator_params(nbins=6)
#ax.grid()
sns.lineplot(x=np.linspace(0,2,100),y=power_model(np.linspace(0,2,100), *fit_dAPP))


plt.savefig('dapp_reg_.svg',bbox_inches='tight',transparent=False)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
scatter = ax.scatter(vstack_app.dens_loc, vstack_app.dens_vor_all, c=z_app, s=3,cmap='viridis',rasterized=False)
#scatter = ax.scatter(vstack_dapp.log_number, vstack_dapp.log_r, c=z_dapp, s=3,cmap='viridis',rasterized=True)
colorbar = plt.colorbar(scatter, ax=ax,shrink=.5)  # Add colorbar
colorbar.set_label('Density')
sns.lineplot(x=np.linspace(0,2,100),y=power_model(np.linspace(0,2,100), *fit_APP))

#ax.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1,ls='-.')
plt.plot([-.1,1.6],[-.1,1.6],ls='--',color='gray')
plt.ylim([-.1,1.6])
plt.xlim([-.1,1.6])
plt.locator_params(nbins=6)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
# plt.xscale('log')
#ax.grid()
plt.savefig('app_reg_.svg',bbox_inches='tight',transparent=False)

In [ ]:
import scipy.optimize as opt 
from sklearn.metrics import r2_score

In [ ]:
!pwd

In [ ]:
import scipy.optimize as opt 
from sklearn.metrics import r2_score
def power_model(X, a, b):
    return a * X**b
def power_law_with_plateau(X, a, b, max_response):
    """
    Power-law model with a hard upper plateau.
    
    X : array-like or float
    a : float, scaling factor
    b : float, exponent
    max_response : float, upper plateau value
    """
    return np.minimum(a * X**b, max_response)


In [ ]:
# Calculate confidence intervals (95% confidence)
confidence_level = 0.95
z_score = 1.96  # For 95% confidence 
fit_APP, covar_APP = opt.curve_fit(power_model,vstack_app.dens_loc, vstack_app.dens_vor_all,maxfev=100000)
y_pred_APP= power_model(vstack_app.dens_loc, *fit_APP)
print(fit_APP,r2_score(vstack_app.dens_vor_all, y_pred_APP))
param_errors_APP = np.sqrt(np.diag(covar_APP))
ci_a_APP = (fit_APP[0] - z_score * param_errors_APP[0], fit_APP[0] + z_score * param_errors_APP[0])
ci_b_APP = (fit_APP[1] - z_score * param_errors_APP[1], fit_APP[1] + z_score * param_errors_APP[1])
print(f"Confidence intervals for parameters a and b: \n a: {ci_a_APP} \n b: {ci_b_APP}")


fit_dAPP, covar_dAPP = opt.curve_fit(power_model,vstack_dapp.dens_loc, vstack_dapp.dens_vor_all,maxfev=100000)
y_pred_dAPP = power_model(vstack_dapp.dens_loc, *fit_dAPP)
print(fit_dAPP,r2_score(vstack_dapp.dens_vor_all, y_pred_dAPP))
param_errors_dAPP = np.sqrt(np.diag(covar_dAPP))
ci_a_APP = (fit_dAPP[0] - z_score * param_errors_dAPP[0], fit_dAPP[0] + z_score * param_errors_dAPP[0])
ci_b_APP = (fit_dAPP[1] - z_score * param_errors_dAPP[1], fit_dAPP[1] + z_score * param_errors_dAPP[1])
print(f"Confidence intervals for parameters a and b: \n a: {ci_a_APP} \n b: {ci_b_APP}")


In [ ]:
# Calculate confidence intervals (95% confidence)
confidence_level = 0.95
z_score = 1.96  # For 95% confidence 
fit_APP, covar_APP = opt.curve_fit(power_law_with_plateau,vstack_app.dens_loc, vstack_app.dens_vor_all,maxfev=100000)
y_pred_APP= power_law_with_plateau(vstack_app.dens_loc, *fit_APP)
print(fit_APP,r2_score(vstack_app.dens_vor_all, y_pred_APP))
param_errors_APP = np.sqrt(np.diag(covar_APP))
ci_a_APP = (fit_APP[0] - z_score * param_errors_APP[0], fit_APP[0] + z_score * param_errors_APP[0])
ci_b_APP = (fit_APP[1] - z_score * param_errors_APP[1], fit_APP[1] + z_score * param_errors_APP[1])
print(f"Confidence intervals for parameters a and b: \n a: {ci_a_APP} \n b: {ci_b_APP}")


fit_dAPP, covar_dAPP = opt.curve_fit(power_law_with_plateau,vstack_dapp.dens_loc, vstack_dapp.dens_vor_all,maxfev=100000)
y_pred_dAPP = power_law_with_plateau(vstack_dapp.dens_loc, *fit_dAPP)
print(fit_dAPP,r2_score(vstack_dapp.dens_vor_all, y_pred_dAPP))
param_errors_dAPP = np.sqrt(np.diag(covar_dAPP))
ci_a_APP = (fit_dAPP[0] - z_score * param_errors_dAPP[0], fit_dAPP[0] + z_score * param_errors_dAPP[0])
ci_b_APP = (fit_dAPP[1] - z_score * param_errors_dAPP[1], fit_dAPP[1] + z_score * param_errors_dAPP[1])
print(f"Confidence intervals for parameters a and b: \n a: {ci_a_APP} \n b: {ci_b_APP}")


In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
sns.lineplot(x=np.linspace(0,2,100),y=power_law_with_plateau(np.linspace(0,2,100), *fit_APP),color='#C5413A',zorder=2,lw=3)
sns.lineplot(x=np.linspace(0,2,100),y=power_law_with_plateau(np.linspace(0,2,100), *fit_dAPP),color='#A7A9AC',zorder=2,lw=3)
plt.plot([-.1,1.6],[-.1,1.6],ls='--',color='gray')
plt.ylim([-.1,1.6])
plt.xlim([-.1,1.6])
plt.locator_params(nbins=6)
plt.savefig('regg_power_model_dAPP_APP.svg',bbox_inches='tight',transparent=False)


In [ ]:
def exp_decay(x, A, b):
    return A * np.exp(-b * x)
    
# def fit_prisim(mdSAM='-'):
#     x_data = df[df.mdSAM==mdSAM].loc[:,['norm_full','time_post_bleach']].groupby(['time_post_bleach']).mean().index
#     y_data = df[df.mdSAM==mdSAM].loc[:,['norm_full','time_post_bleach']].groupby(['time_post_bleach']).mean().norm_full
#     y_std = df[df.mdSAM==mdSAM].loc[:,['norm_full','time_post_bleach']].groupby(['time_post_bleach']).std().norm_full
    fit_, covar_ = opt.curve_fit(prisim, x_data[1:], y_data[1:],
                               sigma = y_std[1:],
                               absolute_sigma = True,maxfev=100000)
#print(best_fit_ab)
    sigma_abs = np.sqrt(np.diagonal(covar_))
    y_pred = prisim(x_data, *fit_)
    return fit_,sigma_abs,r2_score(y_data, y_pred)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))

# sns.scatterplot(x='dens_loc',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='dAPP')],edgecolor=None,c='gray',s=10,alpha=1)
# sns.scatterplot(x='dens_loc',y='dens_vor_all',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')],edgecolor=None,c='#C03830',s=10,alpha=1)
#plt.plot([-.1,1.6],[-.1,1.6],ls='--',color='gray')
ax.plot(x_dAPP,y_pred_dAPP,color='#A7A9AC',zorder=2,lw=1,ls='-.')
ax.plot(x_APP,y_pred_APP,color='#C5413A',zorder=2,lw=1,ls='-')
plt.plot([-.1,1.6],[-.1,1.6],ls='--',color='gray')
plt.ylim([-.1,1.6])
plt.xlim([-.1,1.6])
plt.locator_params(nbins=6)
plt.savefig('regg_dAPP_APP.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
ax.plot(x_APP,y_pred_APP,color='#C03830',zorder=2,lw=2,ls='-')
sns.scatterplot(x='dens_loc',y='dens_cluster',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='APP')],edgecolor=None,c='#C03830',s=10,alpha=.4)#),,s=25,hue_order=['dAPP','APP'],palette=['gray','#C03830'],alpha=1,zorder=2)
plt.ylim([-.1,1.2])
plt.xlim([-.1,1.7])
plt.locator_params(nbins=4)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
ax.plot(x_dAPP,y_pred_dAPP,color='k',zorder=2,lw=1,ls='-.')
sns.scatterplot(x='dens_loc',y='dens_cluster',data=grid_df[(grid_df.all_points<400)&(grid_df.ele=='dAPP')],edgecolor=None,c='gray')#),,s=25,hue_order=['dAPP','APP'],palette=['gray','#C03830'],alpha=1,zorder=2)

In [ ]:
grid_df.loc[3148]

In [ ]:
fig = plt.figure(figsize = (9,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.05,'wspace':.1})
kwargs = {'rasterize': True}
c = 0 
for ind,number_of_points in zip([3148,2983,2995],[69,196,293]):
    sca = grid_df.loc[ind,'original_df']
    vor  = grid_df.loc[ind,'vor']
    sns.scatterplot(x='x',y='y',data=sca,s=20,edgecolor=None,hue='binary_theshold',rasterized=True
                ,palette=["k","#C03830"],hue_order=['0','1'],zorder=2,ax=axs[c])
    ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=.5,zorder=2,ax=axs[c],**kwargs)
    axs[c].set_xlim([sca.x.min(),sca.x.max()])
    axs[c].set_ylim([sca.y.min(),sca.y.max()])
    axs[c].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
    #axs[c].set_title('n=%d'%number_of_points)
    axs[c].set_xlabel('')
    axs[c].set_ylabel('')
    #axs[c].set_title('n=%d'%number_of_points,fontsize=12)
    axs[c].legend([],frameon=False)
    c = c+1 
plt.savefig('Localizations_vs_clusters_SNAPtm_dAPP_v2.png',bbox_inches='tight',transparent=False,dpi=900)
fig = plt.figure(figsize = (9,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.05,'wspace':.1})
c = 0 

for ind,number_of_points in zip([198,272,494],[82,140,295]):
    sca = grid_df.loc[ind,'original_df']
    vor  = grid_df.loc[ind,'vor']
    sns.scatterplot(x='x',y='y',data=sca,s=30,edgecolor=None,hue='binary_theshold'
                ,palette=["k","#C03830"],hue_order=['0','1'],zorder=2,ax=axs[c],rasterized=True)
    ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=.3,zorder=2,ax=axs[c] ,**kwargs)
    axs[c].set_xlim([sca.x.min(),sca.x.max()])
    axs[c].set_ylim([sca.y.min(),sca.y.max()])
    axs[c].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
    #axs[c].set_title('n=%d'%number_of_points)
    axs[c].set_xlabel('')
    axs[c].set_ylabel('')
    #axs[c].set_ylabel('n=%d'%number_of_points,fontsize=12)
    #axs[c].set_title('n=%d'%number_of_points,fontsize=12)
    axs[c].legend([],frameon=False)
    c = c+1 
plt.savefig('Localizations_vs_clusters_SNAPtm_APP_v2.png',bbox_inches='tight',transparent=False,dpi=600)


In [ ]:
fig = plt.figure(figsize = (3,6),constrained_layout=False)
axs = fig.subplots(nrows=2, ncols=1,gridspec_kw = {'hspace':.01,'wspace':.02})

sns.scatterplot(x='all_points',y='db_points',data=grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)],
                edgecolor=None,zorder=2,alpha=.75,color=['#C03830'],s=20,ax=axs[0])
axs[0].plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1)
sns.scatterplot(x='all_points',y='db_points',data=grid_df[(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)],
                edgecolor=None,zorder=2,alpha=.75,color='#B4B5B4',s=20,ax=axs[1])
axs[1].plot(x_dAPP,y_pred_dAPP,color='k',zorder=2,lw=1)
for i in range(2):
    axs[i].grid()
    axs[i].plot([385,0],[385,0],ls='--',lw=1,color='gray')
    axs[i].locator_params(nbins=6)
    axs[i].set_xlim([0,385])
    axs[i].set_ylim([0,385])

axs[1].set_xlabel('Total No. points',fontsize=12)
axs[1].set_ylabel('No. points in Clusters',fontsize=12)
axs[0].set_xlabel('',fontsize=12)
axs[0].set_ylabel('',fontsize=12)
axs[0].set_xticklabels([])
#axs[0].set_yticklabels([])
axs[0].text(10,275,'$r^{2}=%.2f$'%r_sq_APP)
axs[0].text(10,325,'$a=%.2f$'%model_APP.coef_)
print(f"coefficient of determination: {r_sq_APP}")
print(f"slope: {model_APP.coef_}")
print('----')
print(f"coefficient of determination: {r_sq_dAPP}")
print(f"slope: {model_dAPP.coef_}")
axs[1].text(10,275,'$r^{2}=%.2f$'%r_sq_dAPP)
axs[1].text(10,325,'$a=%.2f$'%model_dAPP.coef_)
plt.savefig('Localizations_vs_cl_SNAPtm.svg',bbox_inches='tight',transparent=False)


In [ ]:
g= sns.JointGrid(x='all_points',y='db_points',data=grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)],height=4)#,edgecolor=None,s=10
g.ax_joint.grid()
#g.plot_joint(sns.scatterplot,c='#C03830',edgecolor=None,s=25)
g.plot_marginals(sns.boxplot,color='#C03830')
g.plot_joint(sns.regplot,scatter_kws={'color':'#C03830','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25

g.ax_joint.plot([385,0],[385,0],ls='--',lw=1,color='gray')
g.ax_joint.locator_params(nbins=6)
g.ax_joint.set_xlim([0,385])
g.ax_joint.set_ylim([0,385])

g.ax_joint.plot(x_APP,y_pred_APP,color='k',zorder=2,lw=1)
g.ax_joint.grid(zorder=3)
g.ax_joint.set_xlabel('Total No. points',fontsize=16)
g.ax_joint.text(250,350,'n=%d'%len(grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)]))
g.ax_joint.set_ylabel('No. points in Clusters',fontsize=16)
# g.ax_joint.text(10,275,'$r^{2}=%.2f$'%r_sq_APP,fontsize=12)
# g.ax_joint.text(10,325,'$a=%.2f$'%model_APP.coef_,fontsize=12)
plt.savefig('Localizations_vs_cluster_SNAPtm_APP.svg',bbox_inches='tight',transparent=False)


In [ ]:
g= sns.JointGrid(x='all_points',y='db_points',data=grid_df[(grid_df.ele!='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)],height=4)#,edgecolor=None,s=10
g.ax_joint.grid()
g.plot_joint(sns.regplot,scatter_kws={'color':'#B4B5B4','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25
g.plot_marginals(sns.boxplot,color='#B4B5B4')

g.ax_joint.plot([385,0],[385,0],ls='--',lw=1,color='gray')
g.ax_joint.locator_params(nbins=6)
g.ax_joint.set_xlim([0,385])
g.ax_joint.set_ylim([0,385])

g.ax_joint.plot(x_dAPP,y_pred_dAPP,color='k',zorder=2,lw=1)
g.ax_joint.grid(zorder=3)
g.ax_joint.text(250,350,'n=%d'%len(grid_df[(grid_df.ele!='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)]))
g.ax_joint.set_xlabel('Total No. points',fontsize=16)
g.ax_joint.set_ylabel('No. points in Clusters',fontsize=16)
plt.savefig('Localizations_vs_cluster_SNAPtm_dAPP.svg',bbox_inches='tight',transparent=False)

In [ ]:
e = []
for ind in tqdm.tqdm(grid_df[(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].index):
    ele = grid_df.loc[ind,'ele']
    all_points = grid_df.loc[ind,'all_points']
    for cluster in grid_df.loc[ind]['cluster_sizes_full']:
        e.append({'ele':ele,'all_points':all_points,'cluster_size':cluster})


In [ ]:
ee= pd.DataFrame(e)

In [ ]:
g= sns.JointGrid(x='all_points',y='cluster_size',data=ee[ee.ele=='APP'],height=4)#,edgecolor=None,s=10
g.ax_joint.grid()
g.plot_joint(sns.regplot,scatter_kws={'color':'#C03830','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25
g.plot_marginals(sns.boxplot,color='#C03830')
g.ax_joint.set_ylim([0,250])

#g.ax_joint.set_yscale('symlog')

In [ ]:
g= sns.JointGrid(x='all_points',y='cluster_size',data=ee[ee.ele!='APP'],height=4)#,edgecolor=None,s=10
g.ax_joint.grid()
g.plot_joint(sns.regplot,scatter_kws={'color':'#B4B5B4','edgecolor':None,'s':25},line_kws={'color':'k'})#,c='#B4B5B4',edgecolor=None,s=25
g.plot_marginals(sns.boxplot,color='#B4B5B4')

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) #'#C03830','#B4B5B4'
sns.scatterplot(x='all_points',y='cluster_size',data=ee[ee.ele=='APP'],edgecolor=None,s=10,color='#C03830')
#plt.plot([400,1],[400,1],ls='--',lw=1,color='orange')
#plt.yscale('log')
plt.ylim([0,270])
plt.xlim([0,365])
plt.fill_between(range(5,350), 2, 10
               ,color='k', alpha=.5,edgecolor=None)
# plt.fill_between(range(5,400), 2, 15
#                ,color='k', alpha=.5,edgecolor=None)

fig, ax = plt.subplots(figsize=(3,3)) 
sns.scatterplot(x='all_points',y='cluster_size',data=ee[ee.ele=='dAPP'],edgecolor=None,s=10,color='#B4B5B4')
#plt.xscale('log')
#plt.yscale('log')
# plt.axhline(5)
# plt.axhline(10)
plt.ylim([0,270])
plt.xlim([0,365])

plt.fill_between(range(5,350), 2, 10
               ,color='k', alpha=.5,edgecolor=None)

In [ ]:
def cluster2rad(tem,cluster=1,ele='APP'):
    #tem.to_pickle('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/temp.pkl')
    name = tem.name.drop_duplicates().values[0]
    cell = tem.cell.drop_duplicates().values[0]
    rectangle = tem.rectangle.drop_duplicates().values[0] 
    replicate = tem.replicate.drop_duplicates().values[0] 
    x = tem[tem.db==cluster]['x'].to_list()
    y = tem[tem.db==cluster]['y'].to_list()
    cx = np.dot(x, y) / np.sum(y)
    cy = np.dot(y, x) / np.sum(x)
    fig, ax = plt.subplots(figsize=(3,3)) 
    if ele =='APP':
        sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=False,color='#C03830',s=20,edgecolor=None,ax=ax,zorder=5)
    else:
        sns.scatterplot(x='x',y='y',data=tem[tem.db==cluster],rasterized=False,color='#B4B5B4',s=20,edgecolor=None,ax=ax,zorder=5)
    ax.set_xlim(cx-200,cx+200)
    ax.set_ylim(cy-200,cy+200)
    ax.axis('off')
    ax.scatter(x=cx,y=cy,s=5,c='k')
    
    pdis  =[]
    for ind in tem[tem.db==cluster].index:
        x = tem.loc[ind,'x']
        y = tem.loc[ind,'y']
        dist = math.hypot(x - cx, y - cy)
        pdis.append({'ind':ind,'dist':dist})
    pdis_df = pd.DataFrame(pdis)
    #print(pdis_df)
    ind = pdis_df.loc[pdis_df['dist'].idxmax(),'ind']
    r = pdis_df.loc[pdis_df['dist'].idxmax(),'dist']
    #ax.scatter(x=tem.loc[ind].x,y=tem.loc[ind].y,color='#C03830',s=3,edgecolor='k')
    ax.plot([cx,tem.loc[ind].x],[cy,tem.loc[ind].y],ls='-',lw=3,color='k')
    ax.add_patch(plt.Circle((cx, cy), r, color='#E5E5E5',alpha=1))
    ax.set_title('R=%.0f nm'%r+'\n'+'n=%d'%len(tem[tem.db==cluster]),fontsize=12)
    #ax.plot([cx+10,cx+60],[cy -47,cy -47],ls='-',lw=1,color='k')
    print(r)
    plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/rad_example.svg',bbox_inches='tight',transparent=False)
    # if sizes =='>100':
    #     plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,'above_100',cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    # else:
    #     plt.savefig('/Users/aelazar/notebooks_for_paper/dSTORM/cluster_reps/%s/%s/%s_%s_%s_%d.png'%(ele,sizes,cell,replicate,rectangle,cluster),dpi=300,bbox_inches='tight',transparent=False)
    
    # plt.close()

In [ ]:
ax.plot([cx,tem.loc[ind].x],[cy,tem.loc[ind].y],ls='-',lw=3,color='k')
ax.add_patch(plt.Circle((cx, cy), r, color='k',alpha=.1))

In [ ]:
def cluster2rad_v2(tem,cluster=1):
    x = tem[tem.db==cluster]['x'].to_list()
    y = tem[tem.db==cluster]['y'].to_list()
    cx = np.dot(x, y) / np.sum(y)
    cy = np.dot(y, x) / np.sum(x)
    pdis  =[]
    for ind in tem[tem.db==cluster].index:
        x = tem.loc[ind,'x']
        y = tem.loc[ind,'y']
        dist = math.hypot(x - cx, y - cy)
        pdis.append({'ind':ind,'dist':dist})
    pdis_df = pd.DataFrame(pdis)
    #ind = pdis_df.loc[pdis_df['dist'].idxmax(),'ind']
    r = pdis_df.loc[pdis_df['dist'].idxmax(),'dist']
    return r


In [ ]:
tt = roi2df(t)

In [ ]:
cluster2rad(roi2df(apps_v2[0]))

In [ ]:
tt.loc[0,'name']

In [ ]:
grid_df.columns

In [ ]:
def cluster2rad_v3(t,ele='APP',counter=0):
    name = t.name.values[0]
    rep = t.replicate.values[0]
    rec = t.rectangle.values[0]
    radi =[]
    for cluster in set(t[(t.db>=0)].db):
        av = []
        for i in grid_df[(grid_df.ele==ele)&(grid_df.rep==rep)&(grid_df.cell==2)&(grid_df.rec==rec)].index:
            if cluster in grid_df.loc[i,'db_ids']:
                #print(grid_df.loc[i,'all_points'])
                av.append(grid_df.loc[i,'all_points'])
        x = t[t.db==cluster]['x'].to_list()
        y = t[t.db==cluster]['y'].to_list()
        cx = np.dot(x, y) / np.sum(y)
        cy = np.dot(y, x) / np.sum(x)
        pdis  =[]
        for ind in t[t.db==cluster].index:
            x = t.loc[ind,'x']
            y = t.loc[ind,'y']
            dist = math.hypot(x - cx, y - cy)
            pdis.append({'ind':ind,'dist':dist})
        pdis_df = pd.DataFrame(pdis)
    #ind = pdis_df.loc[pdis_df['dist'].idxmax(),'ind']
        r = pdis_df.loc[pdis_df['dist'].idxmax(),'dist']
        radi.append({'cluster_size':len(t[t.db==cluster]),'r':r,'ele':ele,'counter':counter,'cluster':cluster,'total_loc':len(t),'av_sub_loc':np.mean(av)})
    return radi


In [ ]:
tt= roi2df(apps_v2[0])
#cluster2rad_v3(tt)
tt

In [ ]:
tt

In [ ]:

av = []
for i in grid_df[(grid_df.ele=='dAPP')&(grid_df.rep==1)&(grid_df.cell==2)&(grid_df.rep==1)].index:

    if 20 in grid_df.loc[i,'db_ids']:
        #print(grid_df.loc[i,'all_points'])
        av.append(grid_df.loc[i,'all_points'])
print(np.mean(av))

In [ ]:
clust = []
for i in enumerate(apps_v2):
    for j in cluster2rad_v3(roi2df(i[1]),ele='APP',counter=i[0]):
        clust.append(j)
for i in enumerate(dapp_v2):
    for j in cluster2rad_v3(roi2df(i[1]),ele='dAPP',counter=i[0]):
        clust.append(j)
cluster_df = pd.DataFrame(clust)




In [ ]:
cluster_df['dens'] = 1*np.log10(cluster_df['cluster_size']/(np.pi*cluster_df['r']**2))
cluster_df['log_number'] = np.log10(cluster_df['cluster_size'])
cluster_df['log_r'] = np.log10(cluster_df['r'])
cluster_df['total_loc_log'] = np.log10(cluster_df['total_loc'])
cluster_df['total_loc_dens'] = cluster_df['total_loc']

In [ ]:
cluster_df.loc[cluster_df[cluster_df.total_loc.between(300,1000)].index,'total_bin'] = '300-500'
cluster_df.loc[cluster_df[cluster_df.total_loc.between(1001,6000)].index,'total_bin'] = '500,700'
cluster_df.loc[cluster_df[cluster_df.total_loc.between(6001,12000)].index,'total_bin'] = '700,900'
# cluster_df.loc[cluster_df[cluster_df.total_loc.between(900,1100)].index,'total_bin'] = '900,1100'
# cluster_df.loc[cluster_df[cluster_df.total_loc.between(1100,1300)].index,'total_bin'] = '1100,1300'

In [ ]:
cluster_df.total_loc.max()

In [ ]:
sns.violinplot(x='total_bin',y='log_number',data=cluster_df
              ,hue='ele')

In [ ]:
cluster_df[(cluster_df.ele=='dAPP')&(cluster_df.total_loc_log.between(bins[i],bins[i+1]))].cluster_size

In [ ]:
bins = np.linspace(2.8,4.1,6)
l = []
for i in range(len(bins)-1):
    v =cluster_df[(cluster_df.ele=='dAPP')&(cluster_df.total_loc_log.between(bins[i],bins[i+1]))].cluster_size.mean()
    sem =cluster_df[(cluster_df.ele=='dAPP')&(cluster_df.total_loc_log.between(bins[i],bins[i+1]))].cluster_size.sem()
    l.append({'step':i,'val':v,'ele':'dAPP','se':sem})
    v =cluster_df[(cluster_df.ele=='APP')&(cluster_df.total_loc_log.between(bins[i],bins[i+1]))].cluster_size.mean()
    sem =cluster_df[(cluster_df.ele=='APP')&(cluster_df.total_loc_log.between(bins[i],bins[i+1]))].cluster_size.sem()
    l.append({'step':i,'val':v,'ele':'APP','se':sem})

In [ ]:
np.linspace(2.8,4.1,6)

In [ ]:
ll = pd.DataFrame(l)
fig, ax = plt.subplots(figsize=(3,3))
ax.errorbar(x = ll[ll.ele=='dAPP'].step, y=ll[ll.ele=='dAPP'].val, yerr=ll[ll.ele=='dAPP'].se,color='gray')
ax.errorbar(x = ll[ll.ele=='APP'].step, y=ll[ll.ele=='APP'].val, yerr=ll[ll.ele=='APP'].se,color='r')

In [ ]:
bins = np.linspace(2.8,4.1,6)
bins_ = [np.round((10**i)/16)  for i in np.linspace(2.8,4.1,6)]
l = []
for i in range(len(bins)-1):
    
    v =cluster_df[(cluster_df.ele=='dAPP')&(cluster_df.total_loc_log.between(bins[i],bins[i+1]))].cluster_size
    for s in v:
        l.append({'step':i,'val':s,'ele':'dAPP','range_':'%d-%d'%(bins_[i],bins_[i+1])})
    v =cluster_df[(cluster_df.ele=='APP')&(cluster_df.total_loc_log.between(bins[i],bins[i+1]))].cluster_size
    for s in v:
        l.append({'step':i,'val':s,'ele':'APP','range_':'%d-%d'%(bins_[i],bins_[i+1])})

In [ ]:
pd.DataFrame(l).iloc[:,:-1].groupby(['step','ele']).describe().reset_index()

In [ ]:

fig, ax = plt.subplots(figsize=(3,3))
sns.pointplot(x='range_',y='val',data=pd.DataFrame(l),hue='ele',log_scale=(False,False),errorbar='se',hue_order=['APP','dAPP'],palette=['r','gray'])
plt.locator_params(axis='y',nbins=4)
plt.ylabel('')
plt.xlabel('')
plt.xticks(rotation=30)
plt.savefig('Localizations_vs_cluster_size.svg',bbox_inches='tight',transparent=False)

In [ ]:
sns.scatterplot(x='dens',y='log_number',data=cluster_df,hue='ele',alpha=.3)
#plt.yscale('log')

In [ ]:
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(2,3))
sns.boxplot(y='log_number',data=cluster_df,hue='ele',fill=False,width=0.8,
              hue_order=['dAPP','APP'],palette=['gray','#C03830'],showcaps=False)
plt.locator_params(axis='y',nbins=4)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.legend([],frameon=False)
plt.xlabel('')
plt.ylabel('')
plt.savefig('log_number_box.svg',bbox_inches='tight',transparent=False)

In [ ]:
cluster_df.loc[cluster_df[cluster_df.dens.between(-3,0)].index,['ele','cluster_size']].groupby('ele').median()

In [ ]:
sns.violinplot(x='log_number',data=cluster_df[cluster_df.dens.between(-6,0)],hue='ele',split=True,fill=False)

In [ ]:
print(vstack_app[vstack_app.cluster_size.between(5,10)].r.mean())
print(vstack_dapp[vstack_dapp.cluster_size.between(5,10)].r.mean())
print(vstack_app[vstack_app.cluster_size.between(11,20)].r.mean())
print(vstack_dapp[vstack_dapp.cluster_size.between(11,20)].r.mean())
print(vstack_app[vstack_app.cluster_size.between(21,30)].r.mean())
print(vstack_dapp[vstack_dapp.cluster_size.between(11,30)].r.mean())

In [ ]:
vstack_app = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','cluster_size','log_r','dens','r']]
from scipy.stats import gaussian_kde
xy_app = np.vstack([vstack_app.log_number,vstack_app.log_r])
z_app = gaussian_kde(xy_app)(xy_app)
vstack_dapp = cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','cluster_size','log_r','dens','r']]

xy_dAPP = np.vstack([vstack_dapp.cluster_size,vstack_dapp.r])
z_dapp = gaussian_kde(xy_dAPP)(xy_dAPP)

fig, ax = plt.subplots(figsize=(3,3))
scatter = ax.scatter(vstack_dapp.log_number, vstack_dapp.dens, c=z_dapp, s=3,cmap='magma',rasterized=True)
#scatter = ax.scatter(vstack_dapp.log_number, vstack_dapp.log_r, c=z_dapp, s=3,cmap='viridis',rasterized=True)
colorbar = plt.colorbar(scatter, ax=ax,shrink=.5)  # Add colorbar
colorbar.set_label('Density') 
#plt.ylim([0,500])
#plt.xlim([4,100])
# plt.xlim([np.log10(4),1.4])
# plt.ylabel('GroEL')
# plt.xlabel('hamm')
# #plt.xscale('log')
# #plt.yscale('log')
# plt.savefig('hamming_GroEL_mDHFR.svg',bbox_inches='tight',transparent=True,dpi=900)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
scatter = ax.scatter(vstack_app.log_number, vstack_app.dens, c=z_app, s=3,cmap='magma',rasterized=True)
#scatter = ax.scatter(vstack_dapp.log_number, vstack_dapp.log_r, c=z_dapp, s=3,cmap='viridis',rasterized=True)
colorbar = plt.colorbar(scatter, ax=ax,shrink=.5)  # Add colorbar
colorbar.set_label('Density') 
# plt.xlim([4,100])
# plt.ylim([0,500])

In [ ]:
sns.scatterplot(x='log_number',y='dens',data=cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d

# Generate random points
np.random.seed(13)
num_points = 11  # Adjust number of points as needed
points = np.random.rand(num_points, 2)  # Generate random points in 2D space

# Compute Voronoi diagram
vor = Voronoi(points)

# Plot the Voronoi diagram
fig, ax = plt.subplots(figsize=(2, 2))
voronoi_plot_2d(vor, ax=ax, show_vertices=False, line_colors='gray', line_width=4, point_size=10)

# Plot the original points
ax.plot(points[:, 0], points[:, 1], 'ko', markersize=5)  # Red points for original sites

# Set plot limits
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.xticks([])
plt.yticks([])


plt.savefig('Voronoi_toy.svg',bbox_inches='tight',transparent=False)



In [ ]:
cluster_df

In [ ]:
sns.kdeplot(x='cluster_size',y='r',data=cluster_df.loc[cluster_df[cluster_df.ele!='APP'].index,['cluster_size','r']])

In [ ]:
fig = plt.figure(figsize = (8,4),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.05})
_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#C03830"])
n_levels = 6
rvs = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','log_r']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
y_flat = np.r_[.5:3.25:64j]
x_flat = np.r_[0.5:2:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
#fig,ax=plt.subplots(figsize=(3,3)) 
#axs[1] = fig.gca()
axs[1].grid(zorder=2)
cfset = axs[1].contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf,zorder=1)
cset = axs[1].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[1].clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')
axs[1].set_xlabel('')
axs[1].set_ylabel('')
axs[1].set_xticklabels([])
axs[1].set_yticklabels([])
grays_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#464546"])

rvs2 = cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','log_r']].to_numpy()
kde2 = stats.gaussian_kde(rvs2.T)
z = kde2(grid_coords.T)
z = z.reshape(64,64)
# #fig,ax=plt.subplots(figsize=(3,3)) 
# axs[0] = fig.gca()
axs[0].grid(zorder=3)
#axs[0].axhline(np.log10(5))
cfset = axs[0].contourf(x_flat, y_flat, z,levels=n_levels, cmap=grays_assaf,zorder=1)
cset = axs[0].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[0].clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')
axs[0].locator_params(nbins=5)
axs[0].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
axs[0].set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
axs[0].set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
#axs[0].set_ylabel('Cluster density (log$_{10}$)',fontsize=16)
#axs[0].set_xlabel('No. points in Clusters (log$_{10}$)',fontsize=16)
#axs[0].text(1.5,4,'n=%d'%len(cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','dens']].to_numpy()),fontsize=16)
#axs[1].text(1.5,4,'n=%d'%len(cluster_df.loc[cluster_df[cluster_df.ele!='dAPP'].index,['log_number','dens']].to_numpy()),fontsize=16)
#plt.savefig('Clusters_vs_density.svg',bbox_inches='tight',transparent=False)


In [ ]:
cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','log_r']]

In [ ]:
n_levels = 6
rvs = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','log_r']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
y_flat = np.r_[.5:3:64j]
x_flat = np.r_[0.5:2:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
g= sns.JointGrid(x='log_number',y='log_r',data=cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','log_r']],height=4,space=0)#,edgecolor=None,s=10
#g.plot_joint(sns.scatterplot,c='#317EC2',edgecolor=None,s=25)
ax=g.ax_joint
ax.contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf)
ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=0)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=1)
ax.clabel(cset, inline=.05, fontsize=14,inline_spacing=6,fmt='%1.1f')
#g.ax_joint.contourf(x_flat, y_flat, z,levels=n_levels,cmap=grays_assaf)
# g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
ax.locator_params(nbins=4,axis='both')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
# cset = g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# g.ax_joint.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')

g.ax_joint.set_xlim([0.5,2])
g.ax_joint.set_ylim([.5,3])
g.plot_marginals(sns.violinplot,color='#C03830',split=True,saturation=1,fill=0)
plt.savefig('Clusters_vs_radii_APP.svg',bbox_inches='tight',transparent=False)




# n_levels = 6
# rvs = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']].to_numpy()
# kde = stats.gaussian_kde(rvs.T)
# # Regular grid to evaluate kde upon
# y_flat = np.r_[-5:-1.25:64j]
# x_flat = np.r_[0.5:2:64j]
# x,y = np.meshgrid(x_flat,y_flat)
# grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
# z = kde(grid_coords.T)
# z = z.reshape(64,64)
# g= sns.JointGrid(x='log_number',y='dens',data=cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']],height=4)#,edgecolor=None,s=10
# #g.plot_joint(sns.scatterplot,c='#317EC2',edgecolor=None,s=25)
# g.ax_joint.contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf)
# g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# cset = g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# g.ax_joint.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')

# g.ax_joint.set_xlim([0.5,2])
# g.ax_joint.set_ylim([-5,-1.25])
# g.plot_marginals(sns.violinplot,color='#C03830',split=True,saturation=1,fill=0)
# plt.savefig('Clusters_vs_density_APP.svg',bbox_inches='tight',transparent=False)

In [ ]:
n_levels = 6
rvs = cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','log_r']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
y_flat = np.r_[.5:3:64j]
x_flat = np.r_[0.5:2:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
g= sns.JointGrid(x='log_number',y='log_r',data=cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','log_r']],height=4,space=0)#,edgecolor=None,s=10
#g.plot_joint(sns.scatterplot,c='#317EC2',edgecolor=None,s=25)
ax=g.ax_joint
ax.contourf(x_flat, y_flat, z,levels=n_levels,cmap=grays_assaf)
ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=0)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=1)
ax.clabel(cset, inline=.05, fontsize=14,inline_spacing=6,fmt='%1.1f')
#g.ax_joint.contourf(x_flat, y_flat, z,levels=n_levels,cmap=grays_assaf)
# g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
ax.locator_params(nbins=4,axis='both')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
# cset = g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# g.ax_joint.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')

g.ax_joint.set_xlim([0.5,2])
g.ax_joint.set_ylim([.5,3])
g.plot_marginals(sns.violinplot,color='gray',split=True,saturation=1,fill=0)
plt.savefig('Clusters_vs_radii_dAPP.svg',bbox_inches='tight',transparent=False)

In [ ]:

divnorm = colors.TwoSlopeNorm(vmin=-0.55, vcenter=-.0, vmax=0.25)
z = kde(grid_coords.T) - kde2(grid_coords.T)
z = z.reshape(64,64)
fig,ax=plt.subplots(figsize=(4,4)) 
# ax.tick_params(which='both', width=2)
# ax.tick_params(which='major', length=7)
ax = fig.gca()

ax.grid(zorder=0)
levels =[-0.25,-0.1,.08,0.3]
#regions = [-0.3,-0.2,-0.15,-0.1,.075,0.3]#3#[-.15,0,.07]
regions = [-.53,-.35,-.25,-.1,0.1,0.15,0.2,.24]

cfset = ax.contourf(x_flat, y_flat,z, cmap='RdGy_r',zorder=1,levels=regions,norm=divnorm)
cset = ax.contour(x_flat, y_flat, z, colors='k',linewidths=.75,levels=regions)
ax.locator_params(nbins=4)
# ax.set_xlim([0.5,2])
# ax.set_ylim([-5,-1.25])
# ax.axhline(-3,lw=1,ls='--',color='k')
# ax.axvline(1,lw=1,ls='--',color='k')

#ax.axvline(2,lw=1,ls='--',color='k')
# ax.axvline(4,lw=1,ls='--',color='k')
ax.axvline(1,lw=1,ls='--',color='k')
ax.axhline(2,lw=1,ls='--',color='k')
cbar_ax = fig.add_axes([0.64, .33, 0.025, 0.5])#[left, bottom, width, height]
#im = ax.imshow(data, cmap='bone')
cbar = fig.colorbar(cfset, cax=cbar_ax, orientation='vertical')

# ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
# ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)

cbar.ax.locator_params(nbins=5)

# ax.fill_between([.9,1.9], -3, -2
#                ,color='w', alpha=1,edgecolor='#427BB1',lw=3)
ax.set_xlim([0.5,2])
ax.set_ylim([.5,3])
ax.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.2f')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.savefig('Clusters_vs_radii_sub.svg',bbox_inches='tight',transparent=False)

In [ ]:
def G_hypergeom_radii(q=1,ele='APP'):
    total_runners = len(cluster_df)
    number_team_A = len(cluster_df[cluster_df.ele==ele])  
    if q == 4:
        number_winners = len(cluster_df[(cluster_df.log_r>=2)&(cluster_df.log_number>=1)])
        number_win_from_A = len(cluster_df[(cluster_df.log_r>=2)&(cluster_df.log_number>=1)&(cluster_df.ele==ele)])

    if q == 3:
        number_winners = len(cluster_df[(cluster_df.log_r>2)&(cluster_df.log_number<1)])
        number_win_from_A = len(cluster_df[(cluster_df.log_r>2)&(cluster_df.log_number<1)&(cluster_df.ele==ele)])
    if q == 2:
        number_winners = len(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number<1)])
        number_win_from_A = len(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number<1)&(cluster_df.ele==ele)])
    if q == 1:
        number_winners = len(cluster_df[(cluster_df.log_r<2)&(cluster_df.log_number>=1)])
        number_win_from_A = len(cluster_df[(cluster_df.log_r<2)&(cluster_df.log_number>=1)&(cluster_df.ele==ele)])
    expected = (number_team_A*number_winners)/total_runners
    #print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr

In [ ]:
cluster_df[(cluster_df.log_r>=2)&(cluster_df.log_number>=1)].r.max()

In [ ]:
cluster_df[]

In [ ]:
gtest = []
for i in ['APP','dAPP']:
    for j in [1,2,3,4]:
        number_win_from_A,expected,p = G_hypergeom_radii(q=j,ele=i)
        gtest.append({'q':j,'ele':i,'number_win_from_A':number_win_from_A,'expected':expected,'pval':p})
gtest_df = pd.DataFrame(gtest)
gtest_df['fdr_log'] = [-1*np.log10(i) for i in fdr(gtest_df['pval'])]
gtest_df['fdr'] = [(i) for i in fdr(gtest_df['pval'])]
gtest_df['enrich'] = np.log2(gtest_df['number_win_from_A']/gtest_df['expected'])


sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(4,3))
sns.barplot(x='q',y='enrich',data=gtest_df,hue='ele',hue_order=['APP','dAPP'],palette=['#B24438','#818181'])
plt.locator_params(axis='y',nbins=5)
plt.axhline(0,color='k',ls='--')
plt.yticks(fontsize=12)
plt.savefig('mdsam_log_number_radii_barplot.svg',bbox_inches='tight',transparent=False)

In [ ]:
cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')].cluster_size.median()

In [ ]:
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(3,4))
sns.swarmplot(y='log_r',data=cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')],s=3,palette=['#C03830']
              ,zorder=0,edgecolor=None,log_scale=False)
sns.violinplot(split=0,y='log_r',data=cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')],
               fill=0,palette=['#C03830'],width=.8,zorder=3,log_scale=False)
plt.axhline(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')].log_r.median(),zorder=0,ls='--',c='k')
plt.locator_params(axis='y',nbins=4)
plt.legend([],frameon=False)
plt.savefig('mdsam_radi_compat_and_hig.svg',bbox_inches='tight',transparent=False)

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.swarmplot(y='log_number',data=cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')],s=3,
              palette=['#C03830'],zorder=0,edgecolor=None,log_scale=False)
sns.violinplot(split=0,y='log_number',data=cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')],
               fill=0,palette=['#C03830'],width=.8,zorder=3,log_scale=False)
plt.locator_params(axis='y',nbins=4)
plt.legend([],frameon=False)
plt.axhline(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')].log_number.median(),zorder=0,ls='--',c='k')
plt.savefig('mdsam_log_number_compat_and_hig.svg',bbox_inches='tight',transparent=False)

In [ ]:

def calculate_upper_bound(data):
    """Calculates Q3 + 1.5 * IQR for a list.

    Args:
      data: A list of numerical data.

    Returns:
      The calculated upper bound (Q3 + 1.5 * IQR).
    """
    q1 = np.percentile(data, 25) # Calculate the 25th percentile (Q1).
    q3 = np.percentile(data, 75) # Calculate the 75th percentile (Q3).
    iqr = q3 - q1 # Calculate the IQR.
    upper_bound = q3 + 1.5 * iqr # Calculate Q3 + 1.5 * IQR.
    return upper_bound


In [ ]:
cluster_df.r.max()

In [ ]:
cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')].cluster_size.min()

In [ ]:
print(calculate_upper_bound(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')].r))
print(calculate_upper_bound(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>=1)&(cluster_df.ele=='APP')].cluster_size))

In [ ]:
# print(calculate_upper_bound(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>1)&(cluster_df.ele!='APP')].r))
# print(calculate_upper_bound(cluster_df[(cluster_df.log_r<=2)&(cluster_df.log_number>1)&(cluster_df.ele!='APP')].cluster_size))

In [ ]:
fig = plt.figure(figsize = (6,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.05})
reds_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#C03830"])
n_levels = 6
rvs = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
y_flat = np.r_[-5:-1.25:64j]
x_flat = np.r_[0.5:2:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
#fig,ax=plt.subplots(figsize=(3,3)) 
#axs[1] = fig.gca()
axs[1].grid(zorder=2)
cfset = axs[1].contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf,zorder=1)
cset = axs[1].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[1].clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')
axs[1].set_xlabel('')
axs[1].set_ylabel('')
axs[1].set_xticklabels([])
axs[1].set_yticklabels([])
grays_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#464546"])

rvs2 = cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','dens']].to_numpy()
kde2 = stats.gaussian_kde(rvs2.T)
z = kde2(grid_coords.T)
z = z.reshape(64,64)
# #fig,ax=plt.subplots(figsize=(3,3)) 
# axs[0] = fig.gca()
axs[0].grid(zorder=3)
#axs[0].axhline(np.log10(5))
cfset = axs[0].contourf(x_flat, y_flat, z,levels=n_levels, cmap=grays_assaf,zorder=1)
cset = axs[0].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[0].clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')
axs[0].locator_params(nbins=5)
axs[0].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
axs[0].set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
axs[0].set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
#axs[0].set_ylabel('Cluster density (log$_{10}$)',fontsize=16)
#axs[0].set_xlabel('No. points in Clusters (log$_{10}$)',fontsize=16)
#axs[0].text(1.5,4,'n=%d'%len(cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','dens']].to_numpy()),fontsize=16)
#axs[1].text(1.5,4,'n=%d'%len(cluster_df.loc[cluster_df[cluster_df.ele!='dAPP'].index,['log_number','dens']].to_numpy()),fontsize=16)
#plt.savefig('Clusters_vs_density.svg',bbox_inches='tight',transparent=False)


In [ ]:
cluster_df['cluster_size']/(np.pi*cluster_df['r']**2)
cluster_df

In [ ]:
cluster_df[(cluster_df.dens<-3)&(cluster_df.cluster_size>10)].dens.min()

In [ ]:
divnorm = colors.TwoSlopeNorm(vmin=-.4, vcenter=-.0, vmax=.2)
z = kde(grid_coords.T) - kde2(grid_coords.T)
z = z.reshape(64,64)
fig,ax=plt.subplots(figsize=(4,4)) 
ax = fig.gca()
ax.grid(zorder=0)
levels =4#[-0.24,-0.18,-0.06,0.05]
cfset = ax.contourf(x_flat, y_flat, z,levels=levels, cmap='RdGy_r',zorder=1,norm=divnorm)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=levels,linewidths=.75)
# ax.axvline(3.35,lw=1,ls='--',color='k')
# ax.axhline(-2.5,lw=1,ls='--',color='k')
# ax.fill_between([.9,1.9], -3, -2
#                ,color='w', alpha=1,edgecolor='#427BB1',lw=3)
ax.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.2f')
# # cbar_ax = fig.add_axes([0.93, .25, 0.025, 0.5])#[left, bottom, width, height]
# cbar = fig.colorbar(cfset, cax=cbar_ax, orientation='vertical')
# cbar.ax.locator_params(nbins=4)
# cbar.ax.tick_params(labelsize=10)
# ax.set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
# ax.set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
# ax.set_ylabel('Cluster density (log$_{10}$)',fontsize=16)
# ax.set_xlabel('No. points in Clusters (log$_{10}$)',fontsize=16)
# ax.locator_params(nbins=5)
# ax.axhline(-3,ls='--',color='k')
# ax.axvline(1,ls='--',color='k')
#plt.savefig('Clusters_vs_density_kde_subtract.svg',bbox_inches='tight',transparent=False)

In [ ]:
n_levels = 6
rvs = cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','dens']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
y_flat = np.r_[-5:-1.25:64j]
x_flat = np.r_[0.5:2:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
g= sns.JointGrid(x='log_number',y='dens',data=cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_number','dens']],height=4)#,edgecolor=None,s=10
#g.plot_joint(sns.scatterplot,c='#317EC2',edgecolor=None,s=25)
ax=g.ax_joint
ax.contourf(x_flat, y_flat, z,levels=n_levels,cmap=grays_assaf)
ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=0)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=1)
ax.clabel(cset, inline=.05, fontsize=14,inline_spacing=6,fmt='%1.1f')
#g.ax_joint.contourf(x_flat, y_flat, z,levels=n_levels,cmap=grays_assaf)
# g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
ax.locator_params(nbins=4,axis='both')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
# cset = g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# g.ax_joint.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')

g.ax_joint.set_xlim([0.5,2])
g.ax_joint.set_ylim([-5,-1.25])
g.plot_marginals(sns.violinplot,color='gray',split=True,saturation=1,fill=0)
plt.savefig('Clusters_vs_density_dAPP.svg',bbox_inches='tight',transparent=False)

In [ ]:
n_levels = 6
rvs = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
y_flat = np.r_[-5:-1.25:64j]
x_flat = np.r_[0.5:2:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
g= sns.JointGrid(x='log_number',y='dens',data=cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']],height=4)#,edgecolor=None,s=10
#g.plot_joint(sns.scatterplot,c='#317EC2',edgecolor=None,s=25)
ax=g.ax_joint
ax.contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf)
ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=0)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=1)
ax.clabel(cset, inline=.05, fontsize=14,inline_spacing=6,fmt='%1.1f')
#g.ax_joint.contourf(x_flat, y_flat, z,levels=n_levels,cmap=grays_assaf)
# g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
ax.locator_params(nbins=4,axis='both')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
# cset = g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# g.ax_joint.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')

g.ax_joint.set_xlim([0.5,2])
g.ax_joint.set_ylim([-5,-1.25])
g.plot_marginals(sns.violinplot,color='#C03830',split=True,saturation=1,fill=0)
plt.savefig('Clusters_vs_density_APP.svg',bbox_inches='tight',transparent=False)




# n_levels = 6
# rvs = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']].to_numpy()
# kde = stats.gaussian_kde(rvs.T)
# # Regular grid to evaluate kde upon
# y_flat = np.r_[-5:-1.25:64j]
# x_flat = np.r_[0.5:2:64j]
# x,y = np.meshgrid(x_flat,y_flat)
# grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
# z = kde(grid_coords.T)
# z = z.reshape(64,64)
# g= sns.JointGrid(x='log_number',y='dens',data=cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_number','dens']],height=4)#,edgecolor=None,s=10
# #g.plot_joint(sns.scatterplot,c='#317EC2',edgecolor=None,s=25)
# g.ax_joint.contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf)
# g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# cset = g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
# g.ax_joint.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.1f')

# g.ax_joint.set_xlim([0.5,2])
# g.ax_joint.set_ylim([-5,-1.25])
# g.plot_marginals(sns.violinplot,color='#C03830',split=True,saturation=1,fill=0)
# plt.savefig('Clusters_vs_density_APP.svg',bbox_inches='tight',transparent=False)

In [ ]:

divnorm = colors.TwoSlopeNorm(vmin=-0.3, vcenter=-.0, vmax=0.3)
z = kde(grid_coords.T) - kde2(grid_coords.T)
z = z.reshape(64,64)
fig,ax=plt.subplots(figsize=(4,4)) 
# ax.tick_params(which='both', width=2)
# ax.tick_params(which='major', length=7)
ax = fig.gca()

ax.grid(zorder=0)
levels =[-0.25,-0.1,.08,0.3]
regions = [-0.3,-0.2,-0.15,-0.1,.075,0.3]#3#[-.15,0,.07]

cfset = ax.contourf(x_flat, y_flat, z,levels=regions, cmap='RdGy_r',zorder=1)#,norm=divnorm)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=regions,linewidths=.75)
ax.locator_params(nbins=4)
ax.set_xlim([0.5,2])
ax.set_ylim([-5,-1.25])
ax.axhline(-3,lw=1,ls='--',color='k')
ax.axvline(1,lw=1,ls='--',color='k')

#ax.axvline(2,lw=1,ls='--',color='k')
# ax.axvline(4,lw=1,ls='--',color='k')
# ax.axhline(1,lw=1,ls='--',color='k')
# ax.axhline(1.5,lw=1,ls='--',color='k')
cbar_ax = fig.add_axes([0.64, .33, 0.025, 0.5])#[left, bottom, width, height]
#im = ax.imshow(data, cmap='bone')
cbar = fig.colorbar(cfset, cax=cbar_ax, orientation='vertical')

# ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
# ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)

cbar.ax.locator_params(nbins=5)

# ax.fill_between([.9,1.9], -3, -2
#                ,color='w', alpha=1,edgecolor='#427BB1',lw=3)
ax.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.2f')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.savefig('Clusters_vs_density_sub_v2.svg',bbox_inches='tight',transparent=False)

In [ ]:
divnorm = colors.TwoSlopeNorm(vmin=-.2, vcenter=-.0, vmax=.1)
fig,ax=plt.subplots(figsize=(3,3)) 
ax = fig.gca()
# Contourf plot
cfset = ax.contourf(x_flat, y_flat, z,levels=[-0.15,-0.1,-0.05],cmap='RdGy_r',norm=divnorm)

#cfset = ax.contourf(x_flat, y_flat, z,levels=[-0.04,-0.03,-0.02,-0.01,.01,0.02,0.03,.04], cmap='RdGy_r')
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=[-0.2,-0.1,.01],linewidths=.75)
ax.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.2f')
ax.set_ylim([0.5,2])
ax.set_xlim([1,5])

In [ ]:
levels=[-0.04,-0.03,-0.02,-0.01,.01,0.02,0.03,.04]

In [ ]:
divnorm = colors.TwoSlopeNorm(vmin=-.25, vcenter=-0, vmax=.2)
z = kde(grid_coords.T) - kde2(grid_coords.T)
z = z.reshape(64,64)
n_levels =6#[-0.24,-0.18,-0.06,0.04]
g= sns.JointGrid(x='dens',y='log_number',data=cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['dens','log_number']],height=4)#,edgecolor=None,s=10
#g.plot_joint(sns.scatterplot,c='#317EC2',edgecolor=None,s=25)
g.ax_joint.contourf(x_flat, y_flat, z,levels=n_levels,cmap='RdGy_r',norm=divnorm)
g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
cset = g.ax_joint.contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
g.ax_joint.clabel(cset, inline=.05, fontsize=12,inline_spacing=5,fmt='%1.2f')

g.ax_joint.set_ylim([0.5,2])
g.ax_joint.set_xlim([1,5])
g.ax_joint.axvline(3,color='k',ls='--')
g.ax_joint.axhline(1,color='k',ls='--')
g.plot_marginals(sns.violinplot,color='#C03830',split=True,saturation=1,fill=0)
plt.savefig('Clusters_vs_density_sub.svg',bbox_inches='tight',transparent=False)

In [ ]:
import scipy
from scipy.spatial import distance
distance.jensenshannon(kde(grid_coords.T),kde2(grid_coords.T))

In [ ]:
cluster_df

In [ ]:
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(3,4))
sns.swarmplot(y='r',data=cluster_df[(cluster_df.dens>-3)&(cluster_df.log_number>1)&(cluster_df.ele=='APP')],s=3,palette=['#C03830']
              ,zorder=0,edgecolor=None,log_scale=True)
sns.violinplot(split=0,y='r',data=cluster_df[(cluster_df.dens>-3)&(cluster_df.log_number>1)&(cluster_df.ele=='APP')],
               fill=0,palette=['#C03830'],width=.8,zorder=3,log_scale=True)
plt.axhline(cluster_df[(cluster_df.dens>=-3)&(cluster_df.log_number>1)&(cluster_df.ele=='APP')].r.median(),zorder=0,ls='--',c='k')

plt.legend([],frameon=False)
plt.savefig('mdsam_radi_dens_and_big.svg',bbox_inches='tight',transparent=False)

In [ ]:
cluster_df[cluster_df.cluster_size.between(5,11)].ele.value_counts()

In [ ]:
cluster_df.loc[cluster_df[cluster_df.cluster_size.between(5,10)].index,'cluster_size_range'] = '5-10  '
cluster_df.loc[cluster_df[cluster_df.cluster_size.between(11,20)].index,'cluster_size_range'] = '11-20  '
cluster_df.loc[cluster_df[cluster_df.cluster_size.between(21,40)].index,'cluster_size_range'] = '21-40  '
cluster_df.loc[cluster_df[cluster_df.cluster_size.between(41,100)].index,'cluster_size_range'] = '41-100  '

cluster_df.loc[cluster_df[cluster_df.r.between(0,100)].index,'r_range'] = '5-100  '
cluster_df.loc[cluster_df[cluster_df.r.between(101,200)].index,'r_range'] = '101-200  '
cluster_df.loc[cluster_df[cluster_df.r.between(201,400)].index,'r_range'] = '201-400  '
cluster_df.loc[cluster_df[cluster_df.r.between(401,1000)].index,'r_range'] = '401-1000  '

In [ ]:
cluster_df.loc[cluster_df[cluster_df.r.between(401,1000)].index,'r_range']

In [ ]:
fig, ax = plt.subplots(figsize=(3,4)) 
sns.violinplot(fill=0,y='cluster_size_range',x='r',data=cluster_df,log_scale=True,hue='ele',split=1,
               hue_order=['dAPP','APP'],order=['5-10  ','11-20  ','21-40  ','41-100  '],palette=['gray','r'])
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)

plt.ylabel('')
plt.xlabel('')
plt.savefig('mdsam_cluster_size_r_vs_loc.svg',bbox_inches='tight',transparent=False)

In [ ]:
cluster_df['r_range'],_ = pd.qcut(cluster_df.log_r, q=4, labels=False, retbins=True)
for i in enumerate(cluster_df.sort_values(by='log_r').index):
    cluster_df.loc[i[1],'rank'] = i[0]
stats.ttest_ind(cluster_df[(cluster_df.r_range==3)&(cluster_df.ele=='APP')].cluster_size,
cluster_df[(cluster_df.r_range==3)&(cluster_df.ele!='APP')].cluster_size,alternative='greater')

In [ ]:
_


In [ ]:
fig, ax = plt.subplots(figsize=(2,3)) 
sns.scatterplot(x='rank',y='log_r',data=cluster_df,hue='r_range',edgecolor=None,palette='viridis')
plt.locator_params(axis='both',nbins=3)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.savefig('log_r_Quantile.svg',bbox_inches='tight',transparent=False)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
sns.boxplot(fill=0,y='cluster_size',x='r_range',data=cluster_df,log_scale=True,hue='ele',
               hue_order=['dAPP','APP'],palette=['#808080','#C44E52'])
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.ylim([4,3e3])
# plt.ylabel('')
# plt.xlabel('')
plt.savefig('mdsam_cluster_size_r_vs_radii.svg',bbox_inches='tight',transparent=False)

In [ ]:
cluster_df[cluster_df.r_range==3]['ele'].value_counts()

In [ ]:
cluster_df['total_loc_log_range'],_ = pd.qcut(cluster_df.total_loc_log, q=5, labels=False, retbins=True)
for i in enumerate(cluster_df.sort_values(by='total_loc_log').index):
    cluster_df.loc[i[1],'rank_'] = i[0]

fig, ax = plt.subplots(figsize=(2,3)) 
sns.scatterplot(x='rank_',y='total_loc_log',data=cluster_df,hue='total_loc_log_range',edgecolor=None,palette='viridis')
plt.locator_params(axis='both',nbins=3)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.savefig('total_loc_log_Quantile.svg',bbox_inches='tight',transparent=False)

In [ ]:
cluster_df[cluster_df.r_range==3]['ele'].value_counts()

In [ ]:

fig, ax = plt.subplots(figsize=(3,3)) 
sns.boxplot(fill=0,y='cluster_size',x='total_loc_log_range',data=cluster_df,hue='ele',
               hue_order=['dAPP','APP'],palette=['#808080','#C44E52'],log_scale=True)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=12)
plt.ylim([4,3e3])
# plt.ylabel('')
# plt.xlabel('')
plt.savefig('mdsam_cluster_size_r_vs_total_loc_log_range.svg',bbox_inches='tight',transparent=False)

In [ ]:
stats.ttest_ind(cluster_df[(cluster_df.total_loc_log_range==3)&(cluster_df.ele=='APP')].cluster_size,
cluster_df[(cluster_df.total_loc_log_range==3)&(cluster_df.ele!='APP')].cluster_size,alternative='greater')

In [ ]:
vstack_app = cluster_df.loc[cluster_df[cluster_df.ele=='APP'].index,['log_r','log_number']]
from scipy.stats import gaussian_kde
xy_app = np.vstack([vstack_app.log_r,vstack_app.log_number])
z_app = gaussian_kde(xy_app)(xy_app)
vstack_dapp = cluster_df.loc[cluster_df[cluster_df.ele=='dAPP'].index,['log_r','log_number']]

xy_dAPP = np.vstack([vstack_dapp.log_r,vstack_dapp.log_number])
z_dapp = gaussian_kde(xy_dAPP)(xy_dAPP)

fig, ax = plt.subplots(figsize=(3,3))
scatter = ax.scatter(vstack_dapp.log_r, vstack_dapp.log_number, c=z_dapp, s=5,cmap='viridis',rasterized=True)
#scatter = ax.scatter(vstack_dapp.log_number, vstack_dapp.log_r, c=z_dapp, s=3,cmap='viridis',rasterized=True)
colorbar = plt.colorbar(scatter, ax=ax,shrink=.5)  # Add colorbar
colorbar.set_label('Density') 
plt.ylim([0.6,3])
plt.xlim([0,3])
plt.locator_params(axis='both',nbins=4)
plt.title('dAPP')
# plt.xlim([np.log10(4),1.4])
# plt.ylabel('GroEL')
# plt.xlabel('hamm')
# #plt.xscale('log')
# #plt.yscale('log')
plt.savefig('cluster_r_vs_stoc_dAPP.svg',bbox_inches='tight',transparent=True,dpi=900)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
scatter = ax.scatter(vstack_app.log_r, vstack_app.log_number, c=z_app, s=5,cmap='viridis',rasterized=True)
#scatter = ax.scatter(vstack_dapp.log_number, vstack_dapp.log_r, c=z_dapp, s=3,cmap='viridis',rasterized=True)
colorbar = plt.colorbar(scatter, ax=ax,shrink=.5)  # Add colorbar
colorbar.set_label('Density') 
plt.ylim([0.6,3])
plt.xlim([0,3])
plt.locator_params(axis='both',nbins=4)
plt.title('APP')

plt.savefig('cluster_r_vs_stoc_APP.svg',bbox_inches='tight',transparent=True,dpi=900)

In [ ]:
len(vstack_dapp)

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.swarmplot(y='cluster_size',data=cluster_df[(cluster_df.dens>=-3)&(cluster_df.log_number>1)&(cluster_df.ele=='APP')],s=3,
              palette=['#C03830'],zorder=0,edgecolor=None,log_scale=True)
sns.violinplot(split=0,y='cluster_size',data=cluster_df[(cluster_df.dens>=-3)&(cluster_df.log_number>1)&(cluster_df.ele=='APP')],
               fill=0,palette=['#C03830'],width=.8,zorder=3,log_scale=True)
plt.legend([],frameon=False)
plt.axhline(cluster_df[(cluster_df.dens>=-3)&(cluster_df.log_number>1)&(cluster_df.ele=='APP')].cluster_size.median(),zorder=0,ls='--',c='k')
plt.savefig('mdsam_log_number_dens_and_big.svg',bbox_inches='tight',transparent=False)

In [ ]:
import statsmodels.api as sm
stats.normaltest(cluster_df[(cluster_df.dens>=-3)&(cluster_df.log_number>1)&(cluster_df.ele=='APP')].r)

In [ ]:
-1*np.log10(14/(np.pi*55**2))
cluster_df

In [ ]:
def G_hypergeom(q=1,ele='APP'):
    total_runners = len(cluster_df)
    number_team_A = len(cluster_df[cluster_df.ele==ele])  
    if q == 1:
        number_winners = len(cluster_df[(cluster_df.dens>=-3)&(cluster_df.log_number>1)])
        number_win_from_A = len(cluster_df[(cluster_df.dens>=-3)&(cluster_df.log_number>1)&(cluster_df.ele==ele)])

    if q == 2:
        number_winners = len(cluster_df[(cluster_df.dens>-3)&(cluster_df.log_number<1)])
        number_win_from_A = len(cluster_df[(cluster_df.dens>-3)&(cluster_df.log_number<1)&(cluster_df.ele==ele)])
    if q == 3:
        number_winners = len(cluster_df[(cluster_df.dens<=-3)&(cluster_df.log_number<1)])
        number_win_from_A = len(cluster_df[(cluster_df.dens<=-3)&(cluster_df.log_number<1)&(cluster_df.ele==ele)])
    if q == 4:
        number_winners = len(cluster_df[(cluster_df.dens<-3)&(cluster_df.log_number>1)])
        number_win_from_A = len(cluster_df[(cluster_df.dens<-3)&(cluster_df.log_number>1)&(cluster_df.ele==ele)])
    expected = (number_team_A*number_winners)/total_runners
    #print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr

In [ ]:
gtest = []
for i in ['APP','dAPP']:
    for j in [1,2,3,4]:
        number_win_from_A,expected,p = G_hypergeom(q=j,ele=i)
        gtest.append({'q':j,'ele':i,'number_win_from_A':number_win_from_A,'expected':expected,'pval':p})
gtest_df = pd.DataFrame(gtest)
gtest_df['fdr_log'] = [-1*np.log10(i) for i in fdr(gtest_df['pval'])]
gtest_df['fdr'] = [(i) for i in fdr(gtest_df['pval'])]
gtest_df['enrich'] = np.log2(gtest_df['number_win_from_A']/gtest_df['expected'])
fig, ax = plt.subplots(figsize=(3,3),subplot_kw=dict(projection='polar'))
#fig, ax = plt.subplots(figsize=(4,3)) 
# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, 4, endpoint=False).tolist()

# Close the circle
values = np.concatenate((values, [values[0]]))
angles += angles[:1]
sns.barplot(x='q',y='enrich',data=gtest_df,hue='ele',hue_order=['dAPP','APP'],palette=['gray','#C03830'],order=[4,3,2,1])
plt.locator_params(nbins=3,axis='y')
plt.setp(ax.patches, linewidth=0)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
plt.xlabel('')
plt.ylabel('')
plt.legend([],frameon=False)
ax.grid()
plt.savefig('mdsam_log_number_dens_barplot.svg',bbox_inches='tight',transparent=False)

In [ ]:
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(4,3))
sns.barplot(x='q',y='enrich',data=gtest_df,hue='ele',hue_order=['APP','dAPP'],palette=['#B24438','#818181'])
plt.locator_params(axis='y',nbins=4)
plt.axhline(0,color='k',ls='--')
plt.yticks(fontsize=12)
plt.savefig('mdsam_log_number_dens_barplot.svg',bbox_inches='tight',transparent=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sample data
labels = np.array(['A', 'B',  'D', 'E'])
values = gtest_df[gtest_df.ele=='APP'].sort_values(by='q').enrich.to_list()
values_d = gtest_df[gtest_df.ele=='dAPP'].sort_values(by='q').enrich.to_list()
num_vars = len(labels)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

# Close the circle
values = np.concatenate((values, [values[0]]))
values_d = np.concatenate((values_d, [values_d[0]]))
angles += angles[:1]

# Create the figure
fig, ax = plt.subplots(figsize=(3, 3), subplot_kw=dict(polar=True))

# Plot data
#ax.plot(angles, values_d, color='blue', alpha=0.25)

ax.fill(angles, values, color='r', alpha=.5)
ax.fill(angles, values_d, color='gray', alpha=.5)
#ax.plot(angles, values_d, color='blue', linewidth=2)

# Add labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

plt.show()


In [ ]:
gtest_df

In [ ]:
ele = 'dAPP'
dens = 3
size = 1
total_runners =  len(cluster_df) # No. of hTMs Nodes in the Graph 
number_team_A = len(cluster_df[cluster_df.ele==ele]) # number mdSAM nodes in the Graph  
number_winners = len(cluster_df[(cluster_df.dens<dens)&(cluster_df.log_number>1)])## No. of PPIs i.e Edges 
number_win_from_A = len(cluster_df[(cluster_df.dens<dens)&(cluster_df.log_number>1)&(cluster_df.ele==ele)]) ## PPIs that are mdSAM
#print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
if  number_win_from_A-expected >0:
    p = 1-p 
print(number_win_from_A,expected,p)

In [ ]:
def G_hypergeom(G,acc,mdSAMs=mdSAM_PM):

    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 

In [ ]:
l = []
for ind in tqdm.tqdm(grid_df[grid_df.des<1500].index):
    ele=grid_df.loc[ind,'ele']
    all_points=grid_df.loc[ind,'all_points']
    tem = grid_df.loc[ind,'original_df']
    for cluster in tem[(tem.db!=-1)&(~pd.isna(tem.db))].db.drop_duplicates():
        r= cluster2rad_v2(tem,cluster=cluster)
        size = len(tem[(tem.db==cluster)])
        l.append({'ele':ele,'all_points':all_points,'cluster_size':size,'cluster_rad':r,'ind':ind,'cluster_id':cluster})

In [ ]:
c = pd.DataFrame(l)

In [ ]:
cluster_anataomy = pd.DataFrame(l)
cluster_anataomy = cluster_anataomy[cluster_anataomy.cluster_size>=5]
# cluster_anataomy['dens'] = cluster_anataomy['cluster_size']/cluster_anataomy['cluster_rad']
# cluster_anataomy['dens_log'] =np.log10(cluster_anataomy['dens'] )
cluster_anataomy['box']='A'
cluster_anataomy['cluster_area']=np.pi*(cluster_anataomy['cluster_rad']**2)
cluster_anataomy['cluster_circumference']=2*np.pi*(cluster_anataomy['cluster_rad'])

cluster_anataomy['cluster_area_log']=np.log10(cluster_anataomy['cluster_area'])
cluster_anataomy['dens'] = cluster_anataomy['cluster_size']/cluster_anataomy['cluster_area']
cluster_anataomy['dens_log']=np.log10(cluster_anataomy['dens'])
cluster_anataomy['cluster_size_log']=np.log10(cluster_anataomy['cluster_size'])
cluster_anataomy['cluster_rad_log']=np.log10(cluster_anataomy['cluster_rad'])
cluster_anataomy['cluster_circumference_log']=np.log10(cluster_anataomy['cluster_rad'])

In [ ]:
len(cluster_anataomy[cluster_anataomy.ele=='APP'])

In [ ]:

fig, ax = plt.subplots(figsize=(3.5,3.5)) #*1.618
ax.grid()
sns.ecdfplot(x='cluster_rad',data=z[z.ele=='APP'],log_scale=(False,False),
             stat='percent',lw=4,color='#C03830')
#plt.xlabel('No. points in Cluster',fontsize=16)
plt.xlabel('Cluster Radius(nm)',fontsize=16)

plt.ylabel('Percent',fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.savefig('cluster_rad_ecdfplot.svg',bbox_inches='tight',transparent=False)
print(len(z[z.ele=='APP']))

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
data = StandardScaler().fit_transform(cluster_anataomy.loc[:,['cluster_rad','dens','cluster_size']])
data_not_norm = cluster_anataomy.loc[:,['cluster_rad','dens','cluster_size']]

In [ ]:
sns.kdeplot(x='cluster_size_log',y='dens_log',data=cluster_anataomy[cluster_anataomy.ele=='dAPP'])


In [ ]:
sns.scatterplot(x='cluster_rad_log',y='dens_log',data=cluster_anataomy[cluster_anataomy.ele=='APP'],alpha=.1,color='#C03830')

In [ ]:
sns.scatterplot(x='cluster_rad_log',y='dens_log',data=cluster_anataomy[cluster_anataomy.ele=='dAPP'],alpha=.1,color='#B4B5B4',edgecolor=None)
sns.kdeplot(x='cluster_rad_log',y='dens_log',data=cluster_anataomy[cluster_anataomy.ele=='dAPP'],alpha=1,color='k')

In [ ]:
fig = plt.figure(figsize = (8,4),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.05})
reds_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#C03830"])
n_levels = 8
rvs = cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='APP'].index,['cluster_rad_log','dens_log']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
x_flat = np.r_[.5:2.75:64j]
y_flat = np.r_[-5:-1:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
#fig,ax=plt.subplots(figsize=(3,3)) 
#axs[1] = fig.gca()
axs[1].grid(zorder=2)
cfset = axs[1].contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf,zorder=1)
cset = axs[1].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[1].clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
axs[1].set_xlabel('')
axs[1].set_ylabel('')
axs[1].set_xticklabels([])
axs[1].set_yticklabels([])
grays_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#464546"])

rvs2 = cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='dAPP'].index,['cluster_rad_log','dens_log']].to_numpy()
kde2 = stats.gaussian_kde(rvs2.T)
z = kde2(grid_coords.T)
z = z.reshape(64,64)
# #fig,ax=plt.subplots(figsize=(3,3)) 
# axs[0] = fig.gca()
axs[0].grid(zorder=3)

cfset = axs[0].contourf(x_flat, y_flat, z,levels=n_levels, cmap=grays_assaf,zorder=1)
cset = axs[0].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[0].clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
axs[0].locator_params(nbins=5)
axs[0].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
# sns.scatterplot(x='cluster_rad_log',y='dens_log',data=cluster_anataomy[cluster_anataomy.ele=='dAPP'],alpha=1,color='#B4B5B4',edgecolor=None,ax=axs[0])
# sns.scatterplot(x='cluster_rad_log',y='dens_log',data=cluster_anataomy[cluster_anataomy.ele=='APP'],alpha=1,color='#C03830',edgecolor=None,ax=axs[1])

axs[0].set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
axs[0].set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
axs[0].set_xlabel('Cluster Radius nm (log$_{10})$',fontsize=16)
axs[0].set_ylabel('Cluster density (log$_{10}$)',fontsize=16)
#axs[0].text(1.5,.6,'n=%d'%len(cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='dAPP'].index,['cluster_size_log','dens_log']].to_numpy()),fontsize=16)
#axs[1].text(1.5,.6,'n=%d'%len(cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele!='dAPP'].index,['cluster_size_log','dens_log']].to_numpy()),fontsize=16)
#plt.savefig('Clusters_vs_density.svg',bbox_inches='tight',transparent=False)
divnorm = colors.TwoSlopeNorm(vmin=-.4, vcenter=-.0, vmax=.4)
z = kde(grid_coords.T) - kde2(grid_coords.T)
z = z.reshape(64,64)
fig,ax=plt.subplots(figsize=(4,4)) 
ax = fig.gca()
ax.grid(zorder=0)
# Contourf plot
cfset = ax.contourf(x_flat, y_flat, z,levels=10, cmap='RdGy_r',zorder=1,norm=divnorm)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=10,linewidths=.75)
# ax.axvline(3.35,lw=1,ls='--',color='k')
# ax.axhline(-2.5,lw=1,ls='--',color='k')
# ax.fill_between([.9,1.9], -3, -2
#                ,color='w', alpha=1,edgecolor='#427BB1',lw=3)
# ax.clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
# cbar_ax = fig.add_axes([0.93, .25, 0.025, 0.5])#[left, bottom, width, height]
cbar = fig.colorbar(cfset, cax=cbar_ax, orientation='vertical')
cbar.ax.locator_params(nbins=4)
cbar.ax.tick_params(labelsize=10)
ax.set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
ax.set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
#ax.set_ylabel('Cluster density (log$_{10}$)',fontsize=16)
#ax.set_xlabel('No. points in Clusters (log$_{10}$)',fontsize=16)
ax.locator_params(nbins=5)
ax.axhline(-3)
ax.axvline(1.5)
#plt.savefig('Clusters_vs_density_kde_subtract.svg',bbox_inches='tight',transparent=False)


In [ ]:
l= []
for ele in ['APP','dAPP']:
    for ran in [(0,25),(26,100),(100,400)]:
        for i in cluster_anataomy[(cluster_anataomy.cluster_rad.between(ran[0],ran[1]))&(cluster_anataomy.ele==ele)].dens:
            l.append({'ele':ele,'nm':'%d-%d'%(ran[0],ran[1]),'dens':i})

In [ ]:
total_runners =  len(cluster_anataomy) # No. of hTMs Nodes in the Graph 
number_team_A = len(cluster_anataomy_APP) # number mdSAM nodes in the Graph  
number_winners = len(cluster_anataomy[(cluster_anataomy.cluster_rad_log>1.)&(cluster_anataomy.dens_log>-3)])## No. of PPIs i.e Edges 
number_win_from_A =len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_rad_log>1)&(cluster_anataomy_APP.dens_log>-3)]) ## PPIs that are mdSAM
print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
print(expected)
p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
print(1-p)

In [ ]:
cluster_anataomy_APP = cluster_anataomy[cluster_anataomy.ele=='APP']
cluster_anataomy_dAPP = cluster_anataomy[cluster_anataomy.ele=='dAPP']
cutt_off = 1.5
from scipy.stats.contingency import expected_freq
from scipy.stats import fisher_exact
a = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_rad_log>cutt_off)&(cluster_anataomy_APP.dens_log>-3)])
b = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_rad_log<cutt_off)&(cluster_anataomy_APP.dens_log>-3)])
c = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_rad_log>cutt_off)&(cluster_anataomy_APP.dens_log<-3)])
d = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_rad_log<cutt_off)&(cluster_anataomy_APP.dens_log<-3)])
print('--------APP-----')
print([[a,b],[c,d]])
print(expected_freq([[a,b],[c,d]]))
print([[a,b],[c,d]]-expected_freq([[a,b],[c,d]]))
print(fisher_exact([[a,b],[c,d]]))
a = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_rad_log>cutt_off)&(cluster_anataomy_dAPP.dens_log>-3)])
b = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_rad_log<cutt_off)&(cluster_anataomy_dAPP.dens_log>-3)])
c = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_rad_log>cutt_off)&(cluster_anataomy_dAPP.dens_log<-3)])
d = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_rad_log<cutt_off)&(cluster_anataomy_dAPP.dens_log<-3)])
print('--------')
print('--------dAPP-----')
print([[a,b],[c,d]])
print(expected_freq([[a,b],[c,d]]))
print([[a,b],[c,d]]-expected_freq([[a,b],[c,d]]))

fisher_exact([[a,b],[c,d]])
print(fisher_exact([[a,b],[c,d]]))

In [ ]:
fig = plt.figure(figsize = (8,4),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.05})
reds_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#C03830"])
n_levels = 10
rvs = cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='APP'].index,['cluster_size_log','dens_log']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
x_flat = np.r_[.5:2:64j]
y_flat = np.r_[-5:-1:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
#fig,ax=plt.subplots(figsize=(3,3)) 
#axs[1] = fig.gca()
axs[1].grid(zorder=2)
cfset = axs[1].contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf,zorder=1)
cset = axs[1].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[1].clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
axs[1].set_xlabel('')
axs[1].set_ylabel('')
axs[1].set_xticklabels([])
axs[1].set_yticklabels([])
grays_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#464546"])

rvs2 = cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='dAPP'].index,['cluster_size_log','dens_log']].to_numpy()
kde2 = stats.gaussian_kde(rvs2.T)
z = kde2(grid_coords.T)
z = z.reshape(64,64)
# #fig,ax=plt.subplots(figsize=(3,3)) 
# axs[0] = fig.gca()
axs[0].grid(zorder=3)

cfset = axs[0].contourf(x_flat, y_flat, z,levels=n_levels, cmap=grays_assaf,zorder=1)
cset = axs[0].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[0].clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
axs[0].locator_params(nbins=5)
axs[0].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
axs[1].locator_params(nbins=5)
axs[0].set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
axs[0].set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
axs[0].set_ylabel('Cluster density (log$_{10}$)',fontsize=16)
axs[0].set_xlabel('No. points in Clusters (log$_{10}$)',fontsize=16)
axs[0].text(1.5,-4,'n=%d'%len(cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='dAPP'].index,['cluster_size_log','dens_log']].to_numpy()),fontsize=16)
axs[1].text(1.5,-4,'n=%d'%len(cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele!='dAPP'].index,['cluster_size_log','dens_log']].to_numpy()),fontsize=16)
plt.savefig('Clusters_vs_density.svg',bbox_inches='tight',transparent=False)


In [ ]:
rvs2.T

In [ ]:
divnorm = colors.TwoSlopeNorm(vmin=-.4, vcenter=-.0, vmax=.2)
z = kde(grid_coords.T) - kde2(grid_coords.T)
z = z.reshape(64,64)
fig,ax=plt.subplots(figsize=(4,4)) 
ax = fig.gca()
ax.grid(zorder=0)
# Contourf plot
cfset = ax.contourf(x_flat, y_flat, z,levels=10, cmap='RdGy_r',zorder=1,norm=divnorm)
cset = ax.contour(x_flat, y_flat, z, colors='k',levels=10,linewidths=.75)
# ax.axvline(3.35,lw=1,ls='--',color='k')
# ax.axhline(-2.5,lw=1,ls='--',color='k')
# ax.fill_between([.9,1.9], -3, -2
#                ,color='w', alpha=1,edgecolor='#427BB1',lw=3)
ax.clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
# cbar_ax = fig.add_axes([0.93, .25, 0.025, 0.5])#[left, bottom, width, height]
cbar = fig.colorbar(cfset, cax=cbar_ax, orientation='vertical')
cbar.ax.locator_params(nbins=4)
cbar.ax.tick_params(labelsize=10)
ax.set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
ax.set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
ax.set_ylabel('Cluster density (log$_{10}$)',fontsize=16)
ax.set_xlabel('No. points in Clusters (log$_{10}$)',fontsize=16)
ax.locator_params(nbins=5)
ax.axhline(-3,ls='--',color='k')
ax.axvline(1,ls='--',color='k')
plt.savefig('Clusters_vs_density_kde_subtract.svg',bbox_inches='tight',transparent=False)


In [ ]:
import scipy
from scipy.spatial import distance
distance.jensenshannon(kde(grid_coords.T),kde2(grid_coords.T))

In [ ]:
len(cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele!='APP'].index,['cluster_size_log','dens_log']])

In [ ]:
np.array(lis).T

In [ ]:
from sklearn import preprocessing

In [ ]:
x_flat = np.r_[.5:2:64j]
y_flat = np.r_[-5:-1:64j]
e = []
for trial in range(1000):
    min_max_scaler = preprocessing.MinMaxScaler(feature_range=(.5, 2))
    x = min_max_scaler.fit_transform(np.random.random((1448,1)).reshape(-1, 1))
    min_max_scaler = preprocessing.MinMaxScaler(feature_range=(-5, -1))
    y = min_max_scaler.fit_transform(np.random.random((1448,1)).reshape(-1, 1))
    lis = []
    for i,j in zip(x,y):
        lis.append([i[0],j[0]])
    kde_rand_1 = stats.gaussian_kde(np.array(lis).T)
    
    min_max_scaler = preprocessing.MinMaxScaler(feature_range=(.5, 2))
    x = min_max_scaler.fit_transform(np.random.random((1210,1)).reshape(-1, 1))
    min_max_scaler = preprocessing.MinMaxScaler(feature_range=(-5, -1))
    y = min_max_scaler.fit_transform(np.random.random((1210,1)).reshape(-1, 1))
    lis_2 = []
    for i,j in zip(x,y):
        lis_2.append([i[0],j[0]])
    kde_rand_2 = stats.gaussian_kde(np.array(lis_2).T)
    #e.append(scipy.special.kl_div(kde_rand_2(grid_coords.T),kde_rand_1(grid_coords.T)).sum())
    e.append(distance.jensenshannon(kde_rand_2(grid_coords.T),kde_rand_1(grid_coords.T)))

In [ ]:
pd.Series(e).hist()
#plt.axvline(51.34630811648252)

In [ ]:
e.append(0.1443303820913752)
stats.zscore(e)[-1]

In [ ]:
scipy.special.kl_div(kde2(grid_coords.T),kde(grid_coords.T)).sum()

In [ ]:

rvs = cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='APP'].index,['cluster_size_log','dens_log']].to_numpy()
kde = stats.gaussian_kde(rvs.T

In [ ]:

total_runners =  len(cluster_anataomy) # No. of hTMs Nodes in the Graph 
number_team_A = len(cluster_anataomy_dAPP) # number mdSAM nodes in the Graph  
number_winners = len(cluster_anataomy[(cluster_anataomy.cluster_size_log>=1)&(cluster_anataomy.dens_log>-3)])## No. of PPIs i.e Edges 
number_win_from_A =len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_size_log>=1)&(cluster_anataomy_dAPP.dens_log>-3)]) ## PPIs that are mdSAM
print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
print(expected)
p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
print((number_win_from_A-expected)/number_win_from_A)
print(p)

In [ ]:
cluster_anataomy_APP = cluster_anataomy[cluster_anataomy.ele=='APP']
cluster_anataomy_dAPP = cluster_anataomy[cluster_anataomy.ele=='dAPP']
cutt_off = 1
from scipy.stats.contingency import expected_freq
from scipy.stats import fisher_exact
a = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_size_log>=cutt_off)&(cluster_anataomy_APP.dens_log>=-3)])
b = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_size_log<cutt_off)&(cluster_anataomy_APP.dens_log>=-3)])
c = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_size_log>=cutt_off)&(cluster_anataomy_APP.dens_log<-3)])
d = len(cluster_anataomy_APP[(cluster_anataomy_APP.cluster_size_log<cutt_off)&(cluster_anataomy_APP.dens_log<-3)])
print('--------APP-----')
print([[a,b],[c,d]])
print(expected_freq([[a,b],[c,d]]))
print([[a,b],[c,d]]-expected_freq([[a,b],[c,d]])[0])
print(fisher_exact([[a,b],[c,d]]))
a = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_size_log>=cutt_off)&(cluster_anataomy_dAPP.dens_log>=-3)])
b = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_size_log<cutt_off)&(cluster_anataomy_dAPP.dens_log>=-3)])
c = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_size_log>=cutt_off)&(cluster_anataomy_dAPP.dens_log<-3)])
d = len(cluster_anataomy_dAPP[(cluster_anataomy_dAPP.cluster_size_log<cutt_off)&(cluster_anataomy_dAPP.dens_log<-3)])
print('--------')
print('--------dAPP-----')
print([[a,b],[c,d]])
print(expected_freq([[a,b],[c,d]]))
print([[a,b],[c,d]]-expected_freq([[a,b],[c,d]])[0])

fisher_exact([[a,b],[c,d]])
print(fisher_exact([[a,b],[c,d]]))

In [ ]:
c,d]])

In [ ]:
z = cluster_anataomy#[(cluster_anataomy.cluster_size_log.between(.9,1.9))&(cluster_anataomy.dens_log.between(-3,-2))]


In [ ]:
sns.set(font_scale=1,style='white')

fig, ax = plt.subplots(figsize=(3,3)) #'#C03830','#B4B5B4'
sns.scatterplot(x='all_points',y='cluster_area',data=z[z.ele=='APP'],edgecolor=None,s=10,color='#C03830')
#plt.plot([400,1],[400,1],ls='--',lw=1,color='orange')
#plt.yscale('log')
#plt.xscale('log')
#plt.ylim([0,100])
plt.xlim([0,400])
#plt.ylim([0,100])
# plt.fill_between(range(5,350), 5, 20
#                ,color='k', alpha=.5,edgecolor=None)

fig, ax = plt.subplots(figsize=(3,3)) 
sns.scatterplot(x='all_points',y='cluster_area',data=z[z.ele=='dAPP'],edgecolor=None,s=10,color='#B4B5B4')
#plt.xscale('log')
#plt.yscale('log')
# plt.axhline(5)
# plt.axhline(10)
#plt.ylim([0,100])
plt.xlim([0,400])

# plt.fill_between(range(5,350), 5, 20
#                ,color='k', alpha=.5,edgecolor=None)

In [ ]:
cluster_anataomy

In [ ]:
bins_l = []
ranges = [(0,75),(76,150),(150,400)]
ranges_strings = ['%d-%d'%(ranges[0][0],ranges[0][1]),'%d-%d'%(ranges[1][0],ranges[1][1]),'%d-%d'%(ranges[2][0],ranges[2][1])]#,'%d-%d'%(ranges[3][0],ranges[3][1])
for ele in ['dAPP','APP']:
    for r in ranges:
        for j in z[(z.ele==ele)&(z.all_points.between(r[0],r[1]))].cluster_size:
            bins_l.append({'densitiy':'%d-%d'%(r[0],r[1]),'cluster_size':j,'ele':ele})
spec = dict(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)
fig, ax = plt.subplots(figsize=(3,3)) 
spec = dict(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'],
            )
sns.stripplot(**spec,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=ranges_strings)
sns.pointplot(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l).groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['APP','dAPP'],
    join=False, dodge=.8 - .8 / 2,order=ranges_strings,
    markers="_", scale=1.2, errorbar=None,palette=['k','k'])
#plt.yscale('log')
#plt.ylim([4,350])
ax.grid()
plt.xticks(rotation=0,fontsize=11)
#plt.ylabel('No. ',fontsize=12)
plt.xlabel('No.localizations',fontsize=12)
plt.ylabel('No. points in Clusters',fontsize=12)

In [ ]:
x_APP = np.array(zz.cluster_size.to_list()).reshape((-1, 1))
y_APP_valus= zz.cluster_rad.to_list()
model_APP = LinearRegression().fit(x_APP,y_APP_valus)
r_sq_APP = model_APP.score(x_APP,y_APP_valus)
print(f"coefficient of determination: {r_sq_APP}")
print(f"intercept: {model_APP.intercept_}")
print(f"slope: {model_APP.coef_}")
y_pred_APP = model_APP.predict(x_APP)


In [ ]:
cluster2rad(grid_df.loc[405,'original_df'],cluster=4.0)

In [ ]:
samp_df = z.value_counts('cluster_size').reset_index()
samp_df[samp_df[0]>3].cluster_size

In [ ]:
zz = z[z.ele=='APP'].groupby(['cluster_size']).mean().reset_index()
zz['cluster_rad_rolling'] = zz['cluster_rad'].rolling(window=1,min_periods=1,center=True).mean()

In [ ]:
x_APP = np.array(zz.cluster_size.to_list()).reshape((-1, 1))
y_APP_valus= zz.cluster_rad_rolling.to_list()
model_APP = LinearRegression().fit(x_APP,y_APP_valus)
r_sq_APP = model_APP.score(x_APP,y_APP_valus)
print(f"coefficient of determination: {r_sq_APP}")
print(f"intercept: {model_APP.intercept_}")
print(f"slope: {model_APP.coef_}")
y_pred_APP = model_APP.predict(x_APP)


In [ ]:

fitCoefs_APP, covMatri_APP = opt.curve_fit(KD,zz.cluster_size,zz.cluster_rad_rolling,method='lm',p0=[30,80])
y_pred_APP = KD(zz.cluster_size, *fitCoefs_APP)
r2  =r2_score(zz.cluster_size, y_pred_APP)
print(fitCoefs_APP,r2)


In [ ]:
len(cluster_anataomy[cluster_anataomy.ele!='APP'])

In [ ]:
z = cluster_anataomy[(cluster_anataomy.cluster_size_log.between(.9,1.9))&(cluster_anataomy.dens_log.between(-3,-2))]

fig = plt.figure(figsize = (3,6),constrained_layout=False)
axs = fig.subplots(nrows=2, ncols=1,gridspec_kw = {'hspace':.05,'wspace':.02})
reds_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#C03830"])
n_levels = 8
rvs = cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele=='APP'].index,['all_points','cluster_size']].to_numpy()
kde = stats.gaussian_kde(rvs.T)
# Regular grid to evaluate kde upon
x_flat = np.r_[-20:300:64j]
y_flat = np.r_[0:40:64j]
x,y = np.meshgrid(x_flat,y_flat)
grid_coords = np.append(x.reshape(-1,1),y.reshape(-1,1),axis=1)
z = kde(grid_coords.T)
z = z.reshape(64,64)
#fig,ax=plt.subplots(figsize=(3,3)) 
#axs[1] = fig.gca()
axs[0].grid(zorder=0)
cfset = axs[0].contourf(x_flat, y_flat, z,levels=n_levels,cmap=reds_assaf,zorder=1)
cset = axs[0].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[0].clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
axs[0].set_xlabel('')
axs[0].set_ylabel('')
axs[0].set_xticklabels([])
axs[0].set_yticklabels([])
grays_assaf = matplotlib.colors.LinearSegmentedColormap.from_list("", ["w","#464546"])

rvs2 = cluster_anataomy.loc[cluster_anataomy[cluster_anataomy.ele!='APP'].index,['all_points','cluster_size']].to_numpy()
kde2 = stats.gaussian_kde(rvs2.T)
z = kde2(grid_coords.T)
z = z.reshape(64,64)
# #fig,ax=plt.subplots(figsize=(3,3)) 
# axs[0] = fig.gca()
axs[1].grid()

cfset = axs[1].contourf(x_flat, y_flat, z,levels=n_levels, cmap=grays_assaf,zorder=1)
cset = axs[1].contour(x_flat, y_flat, z, colors='k',levels=n_levels,linewidths=.75)
axs[1].clabel(cset, inline=.05, fontsize=8,inline_spacing=5,fmt='%1.2f')
axs[1].locator_params(nbins=5)
axs[0].locator_params(nbins=5)
axs[1].set_ylabel('cluster_size',fontsize=12)
axs[1].set_xlabel('all_points',fontsize=12)
#plt.savefig('Clusters_vs_density.svg',bbox_inches='tight',transparent=False)



In [ ]:
z = cluster_anataomy[(cluster_anataomy.cluster_size_log.between(.9,1.9))&(cluster_anataomy.dens_log.between(-3,-2))]


In [ ]:

fig, ax = plt.subplots(figsize=(3.5,3.5)) #*1.618
ax.grid()
sns.ecdfplot(x='cluster_rad',data=z[z.ele=='APP'],log_scale=(False,False),
             stat='percent',lw=4,color='#C03830')
#plt.xlabel('No. points in Cluster',fontsize=16)
plt.xlabel('Cluster Radius(nm)',fontsize=16)

plt.ylabel('Percent',fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.savefig('cluster_rad_ecdfplot.svg',bbox_inches='tight',transparent=False)
print(len(z[z.ele=='APP']))

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
ax.grid()
sns.scatterplot(x='all_points',y='cluster_size',data=z[z.ele=='APP'],edgecolor=None,alpha=1,color='#C03830',s=5,zorder=2)
# sns.scatterplot(x='cluster_size',y='cluster_rad_rolling',data=zz,
#                 color='k',edgecolor=None,s=10,zorder=3)
#plt.plot(list(range(5,100)),[KD(i,*list(fitCoefs_APP)) for i in list(range(5,100))],color='#B4B5B4',lw=4)
#plt.xscale('log')
# plt.yscale('log')refDose = list(range(0,1500))


In [ ]:
len(z[z.ele=='APP'])/len(z[z.ele=='dAPP'])

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3.5)) #*1.618
ax.grid()
sns.ecdfplot(x='cluster_rad',data=z,log_scale=(False,False),
            hue='ele')
#plt.xlabel('No. points in Cluster',fontsize=16)
plt.xlabel('Cluster Radius(nm)',fontsize=16)

plt.ylabel('Percent',fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
# plt.savefig('cluster_rad_ecdfplot.svg',bbox_inches='tight',transparent=False)
# print(len(z[z.ele=='APP']))

In [ ]:
sns.scatterplot(x='cluster_size',y='cluster_rad',data=z,
            hue='ele')

In [ ]:
bins_l = []
ranges = [(0,100),(101,200),(201,500)]
for ele in ['dAPP','APP']:
    for r in ranges:
        for j in flatten(grid_df[(grid_df.all_points.between(r[0],r[1]))&(grid_df.ele==ele)&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.index.isin(z.ind))].cluster_sizes_full_mean):
            bins_l.append({'densitiy':'%d-%d'%(r[0],r[1]),'cluster_sizes_full_vals_nm_mean':j,'ele':ele})
spec = dict(x='densitiy',y='cluster_sizes_full_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)
fig, ax = plt.subplots(figsize=(3,3)) 
spec = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'],
            )
sns.stripplot(**spec,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=['%d-%d'%(ranges[0][0],ranges[0][1]),'%d-%d'%(ranges[1][0],ranges[1][1]),'%d-%d'%(ranges[2][0],ranges[2][1])])
sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=pd.DataFrame(bins_l).groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['APP','dAPP'],
    join=False, dodge=.8 - .8 / 2,order=['%d-%d'%(ranges[0][0],ranges[0][1]),'%d-%d'%(ranges[1][0],ranges[1][1]),'%d-%d'%(ranges[2][0],ranges[2][1])],
    markers="_", scale=1.2, errorbar=None,palette=['k','k'])
#plt.yscale('log')
plt.ylim([4,50])
ax.grid()
plt.xticks(rotation=0,fontsize=11)
plt.ylabel('Localizations per cluster',fontsize=12)
plt.xlabel('No.localizations',fontsize=12)
plt.legend([],frameon=False)
# ax.fill_between([-.5,.45], 0,350 
#                ,color='#2336A5', alpha=.2)
# ax.fill_between([.5,1.45], 0,350 
#                ,color='#B9BABC', alpha=.2)
# ax.fill_between([1.5,2.5], 0,350 
#                ,color='#C2523A', alpha=.2)
#plt.savefig('Localizations_per_clsuter_SNAPtm.svg',bbox_inches='tight',transparent=False)


In [ ]:
ll = pd.DataFrame(bins_l)
ord = ['%d-%d'%(ranges[0][0],ranges[0][1]),'%d-%d'%(ranges[1][0],ranges[1][1]),'%d-%d'%(ranges[2][0],ranges[2][1])]

In [ ]:
fig, ax = plt.subplots(figsize=(5,5)) 

sns.boxplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele!='APP']
                  ,showfliers=False,zorder=2,ax=ax,
                 fliersize=0, width=0.75, medianprops=dict(color="k", alpha=1,zorder=3,linewidth=2), boxprops=dict(linewidth=1, facecolor='w', edgecolor='k', alpha=1), 
                 whiskerprops=dict(linewidth=1, color='k', alpha=1,zorder=3), capprops=dict(linewidth=1, color='k', alpha=1,zorder=3),native_scale=True)
sns.swarmplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele!='APP'],s=5,zorder=2,color='#B4B5B4',ax=ax)
sns.scatterplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele!='APP'].groupby(['densitiy']).mean().reset_index(),edgecolor=None,zorder=4,
               s=50,color='k',ax=ax)

ax.grid()
ax.set_ylim([0,100])
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
ax.set_xlabel('Total No. Points',fontsize=16)
ax.set_ylabel('Cluster size (No. points)',fontsize=16)
plt.savefig('Localizations_per_clsuter_SNAPtm_dAPP.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(5,5)) 

sns.boxplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele=='APP']
                  ,showfliers=False,zorder=2,ax=ax,
                 fliersize=0, width=0.75, medianprops=dict(color="k", alpha=1,zorder=3,linewidth=2), boxprops=dict(linewidth=1, facecolor='w', edgecolor='k', alpha=1), 
                 whiskerprops=dict(linewidth=1, color='k', alpha=1,zorder=3), capprops=dict(linewidth=1, color='k', alpha=1,zorder=3),native_scale=True)
sns.swarmplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele=='APP'],s=5,zorder=2,color='#C03830',ax=ax)
sns.scatterplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele=='APP'].groupby(['densitiy']).mean().reset_index(),edgecolor=None,zorder=4,
               s=50,color='k',ax=ax)

ax.grid()
ax.set_ylim([0,100])
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
ax.set_xlabel('Total No. Points',fontsize=16)
ax.set_ylabel('Cluster size (No. points)',fontsize=16)
plt.savefig('Localizations_per_clsuter_SNAPtm_APP.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(5,10)) 
sns.boxplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele!='APP']
                  ,showfliers=False,zorder=2,
                 fliersize=0, width=0.9, medianprops=dict(color="k", alpha=1,zorder=2,linewidth=1), boxprops=dict(linewidth=1, facecolor='w', edgecolor='k', alpha=.61), 
                 whiskerprops=dict(linewidth=1, color='k', alpha=.6,zorder=3.75), capprops=dict(linewidth=1, color='k', alpha=.6,zorder=3),native_scale=True)
sns.swarmplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=ll[ll.ele!='APP'],s=3,zorder=1)
plt.ylim([0,40])

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3.5)) #*1.618
sns.boxplot(x='ele',y='cluster_rad',data=z,log_scale=(False,False))

In [ ]:
z

In [ ]:
# sns.swarmplot(x='cluster_rad',data=z[z.ele=='APP']
#               ,color='#C03830',alpha=1)
# sns.boxplot(x='cluster_rad',data=z[z.ele=='APP']
#               ,color='#C03830')

#fig, ax = plt.subplots(figsize=(6,2)) #*1.618

fig = plt.figure(figsize = (5,4),constrained_layout=False)
axs = fig.subplots(nrows=2, ncols=1,gridspec_kw = {'hspace':.9,'wspace':.02})

sns.swarmplot(x='cluster_rad',data=z[z.ele=='APP'],ax=axs[0],dodge=False,palette=['#C03830']
                  ,alpha=1,s=3)
sns.boxplot(x='cluster_rad',data=z[z.ele=='APP'],ax=axs[0]
                  ,showfliers=False,zorder=2,
           palette = ['w'],
                 fliersize=0, width=0.5, medianprops=dict(color="k", alpha=1,zorder=3,linewidth=3), boxprops=dict(linewidth=1, facecolor='w', edgecolor='k', alpha=1), 
                 whiskerprops=dict(linewidth=1, color='k', alpha=1,zorder=3), capprops=dict(linewidth=1, color='k', alpha=.61,zorder=3))

axs[0].set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
axs[0].set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
axs[0].grid()

# fig, ax = plt.subplots(figsize=(6,2)) #*1.618
sns.swarmplot(x='cluster_size',data=z[z.ele=='APP'],ax=axs[1],dodge=False,palette=['#C03830']
                  ,alpha=1,s=3)
sns.boxplot(x='cluster_size',data=z[z.ele=='APP'],ax=axs[1]
                  ,showfliers=False,zorder=2,
           palette = ['w'],
                 fliersize=0, width=0.5, medianprops=dict(color="k", alpha=1,zorder=3,linewidth=3), boxprops=dict(linewidth=1, facecolor='w', edgecolor='k', alpha=1), 
                 whiskerprops=dict(linewidth=1, color='k', alpha=1,zorder=3), capprops=dict(linewidth=1, color='k', alpha=.61,zorder=3))

axs[1].set_yticklabels(axs[1].get_yticklabels(),fontsize=16)
axs[1].set_xticklabels(axs[1].get_xticklabels(),fontsize=16)
axs[1].grid()
axs[1].set_title('Stoichiometry',fontsize=16
                )
axs[0].set_title('Size',fontsize=16
                )
axs[0].set_xlabel('Cluster radius(nm)',fontsize=16)
axs[1].set_xlabel('Cluster size(No. points)',fontsize=16)
plt.savefig('SNAPtm_APP_dSTROM_metrics.svg',bbox_inches='tight',transparent=False)


In [ ]:
z[z.ele=='APP'].cluster_rad.median()

In [ ]:
# z[z.cluster_rad.between(60,75)].sample(10)
cluster2rad(grid_df.loc[2084,'original_df'],cluster=19)# n = 54 

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3.5)) #*1.618

sns.violinplot(y='cluster_size',x='cluster_rad',data=bins_df,split=True,common_norm=False,width=.9,
               inner='point',zorder=2,color='#C03830',inner_kws={'s':3},linewidth=0)

sns.violinplot(y='cluster_size',x='cluster_rad',data=bins_df,split=True,common_norm=False,width=.9,
               zorder=2,color='#C03830',inner_kws=dict(box_width=10, whis_width=2, color="0.5"),linewidth=0)
ax.locator_params(axis='x', nbins=5)
ax.grid()
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.ylabel('No. points in Cluster',fontsize=16)
plt.xlabel('Cluster Radius(nm)',fontsize=16)

plt.savefig('Radius_vs_size.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(6,3)) 
ax.grid()
sns.histplot(y='cluster_size',x='cluster_rad',data=bins_df,cbar=True,bins=20,zorder=2)

In [ ]:
z[z.cluster_size.between(50,55)].sample()

In [ ]:
bins_l = []
ranges = [(0,100),(101,200),(201,500)]
ranges_strings = ['%d-%d'%(ranges[0][0],ranges[0][1]),'%d-%d'%(ranges[1][0],ranges[1][1]),'%d-%d'%(ranges[2][0],ranges[2][1])]#,'%d-%d'%(ranges[3][0],ranges[3][1])
for ele in ['dAPP','APP']:
    for r in ranges:
        for j in z[(z.ele==ele)&(z.cluster_size.between(r[0],r[1]))].cluster_rad:
            bins_l.append({'densitiy':'%d-%d'%(r[0],r[1]),'cluster_size':j,'ele':ele})
spec = dict(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)
fig, ax = plt.subplots(figsize=(3,3)) 
spec = dict(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'],
            )
sns.stripplot(**spec,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=ranges_strings)
sns.pointplot(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l).groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['APP','dAPP'],
    join=False, dodge=.8 - .8 / 2,order=ranges_strings,
    markers="_", scale=1.2, errorbar=None,palette=['k','k'])
#plt.yscale('log')
#plt.ylim([4,350])
ax.grid()
plt.xticks(rotation=0,fontsize=11)
#plt.ylabel('No. ',fontsize=12)
plt.xlabel('No.localizations',fontsize=12)
plt.ylabel('No. points in Clusters',fontsize=12)

In [ ]:
bins_l = []
ranges = [(8,20),(20,40),(40,80)]
ranges_strings = ['%d-%d'%(ranges[0][0],ranges[0][1]),'%d-%d'%(ranges[1][0],ranges[1][1]),'%d-%d'%(ranges[2][0],ranges[2][1])]#,'%d-%d'%(ranges[3][0],ranges[3][1])
for ele in ['dAPP','APP']:
    for r in ranges:
        for j in z[(z.ele==ele)&(z.cluster_size.between(r[0],r[1]))].cluster_rad:
            bins_l.append({'densitiy':'%d-%d'%(r[0],r[1]),'cluster_size':j,'ele':ele})
spec = dict(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)
fig, ax = plt.subplots(figsize=(3,3)) 
spec = dict(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'],
            )
sns.stripplot(**spec,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=ranges_strings)
sns.pointplot(x='densitiy',y='cluster_size',data=pd.DataFrame(bins_l).groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['APP','dAPP'],
    join=False, dodge=.8 - .8 / 2,order=ranges_strings,
    markers="_", scale=1.2, errorbar=None,palette=['k','k'])
#plt.yscale('log')
#plt.ylim([4,350])
ax.grid()
plt.xticks(rotation=0,fontsize=11)
#plt.ylabel('No. ',fontsize=12)
plt.xlabel('No.localizations',fontsize=12)
plt.ylabel('No. points in Clusters',fontsize=12)

In [ ]:
sns.boxplot(hue='ele',x='all_points',data=z[z.all_points.between(0,100)],log_scale=True)#,stat='count')
# plt.axvline(10)


In [ ]:
B

In [ ]:
z[z.ele=='APP']

In [ ]:

x_APP = np.array(e[(e.ele=='APP')].all_points.to_list()).reshape((-1, 1))
y_APP_valus= grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<dens_top)].db_points.to_list()
model_APP = LinearRegression().fit(x_APP,y_APP_valus)
r_sq_APP = model_APP.score(x_APP,y_APP_valus)
print(f"coefficient of determination: {r_sq_APP}")
print(f"intercept: {model_APP.intercept_}")
print(f"slope: {model_APP.coef_}")
y_pred_APP = model_APP.predict(x_APP)

In [ ]:


x = np.array(grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<1500)].all_points.to_list()).reshape((-1, 1))
y_valus = [np.log10(i) for i in x[x.ele=='APP'].groupby(['dox']).median().reset_index()['no.puncta_by_area'].values]
model = LinearRegression().fit(x_dox,y_valus)
r_sq = model.score(x_dox, y_valus)
print(f"coefficient of determination: {r_sq}")
print(f"intercept: {model.intercept_}")
print(f"slope: {model.coef_}")
y_pred = model.predict(x_dox)
print(y_pred)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4)) 
ax.grid()
sns.scatterplot(x='all_points',y='db_points',data=grid_df[(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<1500)],edgecolor=None)
sns.scatterplot(x='all_points',y='db_points',data=grid_df[(grid_df.ele!='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))&(grid_df.des<1500)],edgecolor=None)
plt.xlabel('No. points')

In [ ]:
bins_l = []
for j in flatten(grid_df[(grid_df.all_points.between(0,300))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(0,300))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
spec = dict(x='densitiy',y='cluster_sizes_full_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)

In [ ]:
bins_l = []
for j in flatten(grid_df[(grid_df.des.between(0,300))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(0,300))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
spec = dict(x='densitiy',y='cluster_sizes_full_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)

In [ ]:
bins_df[bins_df.ele!='APP'].groupby(['densitiy','ele']).mean().reset_index()

In [ ]:

fig, ax = plt.subplots(figsize=(2,3)) 
spec = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.ele!='APP'],hue='ele',hue_order=['APP','dAPP'],
            )
sns.stripplot(**spec,
    dodge=False, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=['0-300','301-900','901-1500'])
sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.ele!='APP'].groupby(['densitiy','ele']).mean().reset_index(),
              linestyles='none',order=['0-300','301-900','901-1500'],
    markers="_", scale=1.2, errorbar=None,color='k')
# sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.ele!='APP'].groupby(['densitiy','ele']).mean().reset_index(),
#     join=True, dodge=.8 - .8 / 2,order=['0-300','301-900','901-1500'],
#     markers="_", scale=1.2, errorbar=None,color='k')
plt.yscale('log')
plt.ylim([4,350])
ax.grid(zorder=0)
plt.xticks(rotation=30,fontsize=8)
plt.ylabel('$Localizations\;per\;cluster$',fontsize=10)
plt.xlabel('$\\rho\;(No.localizations/\\mu m^{2}}$)',fontsize=10)
plt.legend([],frameon=False)
ax.fill_between([-.5,.45], 0,350 
               ,color='#2336A5', alpha=.1)
ax.fill_between([.5,1.45], 0,350 
               ,color='#B9BABC', alpha=.1)
ax.fill_between([1.5,2.5], 0,350 
               ,color='#C2523A', alpha=.1)
plt.tight_layout()
plt.savefig('Localizations_per_clsuter_SNAPtm_v2_dAPP.svg',bbox_inches='tight',transparent=False)


In [ ]:
grid_df[(grid_df.ele=='dAPP')&(grid_df.all_points.between(250,350))].sample(5)

In [ ]:
#APP 
#1993 --> 53 points
#577 --> 140 points
#284 -->295 points
#dAPP
#3148 --> 69 points
#2983 --> 196 points
#3195 -->282 points


In [ ]:
fig = plt.figure(figsize = (9,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.05,'wspace':.1})
c = 0 
for ind,number_of_points in zip([1993,577,284],[53,140,295]):
    sca = grid_df.loc[ind,'original_df']
    vor  = grid_df.loc[ind,'vor']
    sns.scatterplot(x='x',y='y',data=sca,s=30,edgecolor=None,hue='binary_theshold'
                ,palette=["k","#C03830"],hue_order=['0','1'],zorder=2,ax=axs[c])
    ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=.3,zorder=2,ax=axs[c])
    axs[c].set_xlim([sca.x.min(),sca.x.max()])
    axs[c].set_ylim([sca.y.min(),sca.y.max()])
    axs[c].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
    #axs[c].set_title('n=%d'%number_of_points)
    axs[c].set_xlabel('')
    axs[c].set_ylabel('')
    #axs[c].set_ylabel('n=%d'%number_of_points,fontsize=12)
    axs[c].set_title('n=%d'%number_of_points,fontsize=12)
    axs[c].legend([],frameon=False)
    c = c+1 
plt.savefig('Localizations_vs_clusters_SNAPtm_APP.png',bbox_inches='tight',transparent=False,dpi=600)


In [ ]:
fig = plt.figure(figsize = (9,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.05,'wspace':.1})
c = 0 
for ind,number_of_points in zip([3148,2983,3195],[69,196,282]):
    sca = grid_df.loc[ind,'original_df']
    vor  = grid_df.loc[ind,'vor']
    sns.scatterplot(x='x',y='y',data=sca,s=30,edgecolor=None,hue='binary_theshold'
                ,palette=["k","#C03830"],hue_order=['0','1'],zorder=2,ax=axs[c])
    ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=.3,zorder=2,ax=axs[c])
    axs[c].set_xlim([sca.x.min(),sca.x.max()])
    axs[c].set_ylim([sca.y.min(),sca.y.max()])
    axs[c].tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
    #axs[c].set_title('n=%d'%number_of_points)
    axs[c].set_xlabel('')
    axs[c].set_ylabel('')
    axs[c].set_title('n=%d'%number_of_points,fontsize=12)
    axs[c].legend([],frameon=False)
    c = c+1 
plt.savefig('Localizations_vs_clusters_SNAPtm_dAPP.png',bbox_inches='tight',transparent=False,dpi=600)


In [ ]:
#grid_df[(grid_df.ele=='dAPP')&(grid_df.all_points.between(75,100))]#['original_df']#.values[0]--> loc 3235

sca = grid_df.loc[3195,'original_df']
vor  = grid_df.loc[3195,'vor']
fig, ax = plt.subplots(figsize=(3,3))
sns.scatterplot(x='x',y='y',data=sca,s=30,edgecolor=None,hue='binary_theshold'
                ,palette=["k","#C03830"],hue_order=['0','1'],zorder=2,ax=ax)

# sns.scatterplot(x='x',y='y',data=sca[sca.db!=-1],s=10,edgecolor=None,hue='cluster_range'
#                 ,zorder=2,ax=ax)

ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=0,line_width=.3,zorder=2,ax=ax)
plt.xlim([sca.x.min(),sca.x.max()])
plt.ylim([sca.y.min(),sca.y.max()])
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
# sns.scatterplot(x='x',y='y',data=tem,s=3,edgecolor=None,hue='binary_theshold',
#                 palette=["k","#C03830"],hue_order=['0','1'],ax=axs[1],zorder=2)

# ax = voronoi_plot_2d(vor,show_points=False,show_vertices=False,point_size=1,line_width=.1,ax=ax)
# plt.xlim([df.x.min(),df.x.max()])
# plt.ylim([df.y.min(),df.y.max()])
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
# plt.xlabel('')
# plt.ylabel('')

In [ ]:
bins_l = []
for j in flatten(grid_df[(grid_df.des.between(0,300))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_vals_nm_mean))].cluster_sizes_full_vals_nm_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(0,300))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_vals_nm_mean))].cluster_sizes_full_vals_nm_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,300),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_vals_nm_mean))].cluster_sizes_full_vals_nm_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(301,900))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_vals_nm_mean))].cluster_sizes_full_vals_nm_mean):
    bins_l.append({'densitiy':'%d-%d'%(301,900),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='dAPP')&(~pd.isna(grid_df.cluster_sizes_full_vals_nm_mean))].cluster_sizes_full_vals_nm_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'dAPP'})
for j in flatten(grid_df[(grid_df.des.between(901,1500))&(grid_df.ele=='APP')&(~pd.isna(grid_df.cluster_sizes_full_vals_nm_mean))].cluster_sizes_full_vals_nm_mean):
    bins_l.append({'densitiy':'%d-%d'%(901,1500),'cluster_sizes_full_vals_nm_mean':j,'ele':'APP'})
spec = dict(x='densitiy',y='cluster_sizes_full_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'])
bins_df = pd.DataFrame(bins_l)
fig, ax = plt.subplots(figsize=(3,3)) 
spec = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=pd.DataFrame(bins_l),hue='ele',hue_order=['APP','dAPP'],
            )
sns.stripplot(**spec,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=['0-300','301-900','901-1500'])
sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=pd.DataFrame(bins_l).groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['APP','dAPP'],
    join=False, dodge=.8 - .8 / 2,order=['0-300','301-900','901-1500'],
    markers="_", scale=1.2, errorbar=None,palette=['k','k'])
plt.yscale('log')
#plt.ylim([1e-4,0])
ax.grid()
plt.xticks(rotation=0,fontsize=10)
plt.ylabel('$Cluster\;density$',fontsize=10)
plt.xlabel('$\\rho\;(No.localizations/\\mu m^{2}}$)',fontsize=10)
plt.legend([],frameon=False)
ax.fill_between([-.5,.45], 0,5 
               ,color='#2336A5', alpha=.2)
ax.fill_between([.5,1.45], 0,5 
               ,color='#B9BABC', alpha=.2)
ax.fill_between([1.5,2.5], 0,5 
               ,color='#C2523A', alpha=.2)

In [ ]:
from scipy.spatial import ConvexHull,convex_hull_plot_2d
from shapely import MultiPoint, Polygon
import shapely
cluster = 15
hull = ConvexHull(df[df.db==cluster].loc[:,['x','y']].to_numpy())
points = df[df.db==cluster].loc[:,['x','y']].to_numpy()
plt.plot(points[:,0], points[:,1], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')
hull.volume

In [ ]:
for i in cluster = 12
points = df[df.db==cluster].loc[:,['x','y']].to_numpy()
ploy = shapely.concave_hull(MultiPoint(points), ratio=1)


In [ ]:
ploy = shapely.concave_hull(MultiPoint(points), ratio=1)
ploy_hull = shapely.concave_hull(MultiPoint(points), ratio=.5)

In [ ]:

from matplotlib.patches import Polygon

In [ ]:
# voronoi_thershold_df,t,df = voronoi_thershold_ploting(APP_1_3_3.loc[:,['x','y']].to_numpy(),n=30)

# voronoi_thershold_df,t,df = voronoi_thershold_ploting(dAPP_1_2_1.loc[:,['x','y']].to_numpy(),n=30)

#df,x,thershold,cluster_probabilty = voronoi_thershold_v2(APP_1_3_3.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
df,x,thershold,cluster_probabilty = voronoi_thershold_v2(dAPP_1_2_1.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)


In [ ]:
sns.scatterplot(x='x',y='y',data=df,hue='db')
sns.scatterplot(x='x',y='y',data=df[df.db==0],c='red')

In [ ]:
for i in 
df[df.db==0].vol.sum()

In [ ]:
apps = [APP_1_6_1,APP_1_6_2,APP_1_5_1,APP_1_5_2,APP_1_4_1,APP_1_4_2,APP_1_4_3,APP_1_2_1,APP_1_2_2,APP_1_2_3,APP_1_2_4,APP_1_3_1,APP_1_3_2,APP_1_3_3,APP_1_3_4,APP_2_2_1,APP_2_2_2,APP_2_2_3,APP_2_2_4,APP_2_3_1,APP_2_3_2,APP_2_3_3,APP_2_3_4,APP_2_4_1,APP_2_4_2,
        APP_2_4_3,APP_2_4_4,APP_2_5_1,APP_2_5_2,APP_2_5_3,APP_2_5_4,APP_2_6_1,APP_2_6_2,APP_2_6_3,APP_2_6_4,APP_2_7_1,APP_2_7_2,APP_2_7_3,APP_2_7_4,APP_2_7_5,APP_2_8_1,APP_2_8_2,APP_2_8_3,APP_2_8_4]


dapps = [dAPP_1_2_1,dAPP_1_2_2,dAPP_1_2_3,dAPP_1_2_4,dAPP_1_3_1,dAPP_1_3_2,dAPP_1_3_3,dAPP_1_4_1,dAPP_1_4_2,dAPP_1_4_3,dAPP_1_4_4,dAPP_1_5_1,dAPP_1_5_2,dAPP_1_5_3,dAPP_2_2_1,dAPP_2_2_2,dAPP_2_2_3,dAPP_2_1_1,dAPP_2_1_2,
         dAPP_2_3_1,dAPP_2_3_2,dAPP_2_3_3,dAPP_2_3_4,dAPP_2_6_1,dAPP_2_6_2,dAPP_2_6_3]

In [ ]:
#df,x,thershold,cluster_probabilty = voronoi_thershold_v2(APP_1_3_3.loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
aa = []
for h in [APP_1_6_1,APP_1_6_2,APP_1_5_1,APP_1_5_2,APP_1_4_1,APP_1_4_2,APP_1_4_3,APP_1_2_1,APP_1_2_2,APP_1_2_3,APP_1_2_4,APP_1_3_1,APP_1_3_2,APP_1_3_3,APP_1_3_4,APP_2_2_1,APP_2_2_2,APP_2_2_3,APP_2_2_4,APP_2_3_1,APP_2_3_2,APP_2_3_3,APP_2_3_4,APP_2_4_1,APP_2_4_2,
        APP_2_4_3,APP_2_4_4,APP_2_5_1,APP_2_5_2,APP_2_5_3,APP_2_5_4,APP_2_6_1,APP_2_6_2,APP_2_6_3,APP_2_6_4,APP_2_7_1,APP_2_7_2,APP_2_7_3,APP_2_7_4,APP_2_7_5,APP_2_8_1,APP_2_8_2,APP_2_8_3,APP_2_8_4]:
    df,x,thershold,cluster_probabilty = voronoi_thershold_v2(h.loc[:,['x','y']].to_numpy(),n=1,size_of_grid=4000,min_cluster_size=5)

    for j in [1]:
        for i in set(df[(df.db!=-1)&(~pd.isna(df.db))].db):
            points = df[df.db==i].loc[:,['x','y']].to_numpy()
            ploy = shapely.concave_hull(MultiPoint(points), ratio=j)
            aa.append({'cluster_size':len(df[df.db==i]),'area':ploy.area,'ratio':j,'ele':'APP','id':i,'vor_vol_sum':df[df.db==i].vol.sum()})


for h in [dAPP_1_2_1,dAPP_1_2_2,dAPP_1_2_3,dAPP_1_2_4,dAPP_1_3_1,dAPP_1_3_2,dAPP_1_3_3,dAPP_1_4_1,dAPP_1_4_2,dAPP_1_4_3,dAPP_1_4_4,dAPP_1_5_1,dAPP_1_5_2,dAPP_1_5_3,dAPP_2_3_3,dAPP_2_3_4,dAPP_2_6_1,dAPP_2_6_2,dAPP_2_6_3]:
    df,x,thershold,cluster_probabilty = voronoi_thershold_v2(h.loc[:,['x','y']].to_numpy(),n=1,size_of_grid=4000,min_cluster_size=5)

    for j in [1]:
        for i in set(df[(df.db!=-1)&(~pd.isna(df.db))].db):
            points = df[df.db==i].loc[:,['x','y']].to_numpy()
            ploy = shapely.concave_hull(MultiPoint(points), ratio=j)
            aa.append({'cluster_size':len(df[df.db==i]),'area':ploy.area,'ratio':j,'ele':'dAPP','id':i,'vor_vol_sum':df[df.db==i].vol.sum()})

In [ ]:
area_df = pd.DataFrame(aa)
area_df['dens'] = area_df['cluster_size']/area_df['vor_vol_sum']
area_df

In [ ]:
sns.ecdfplot(x='dens',data=area_df,log_scale=True,hue='ele')

In [ ]:
#sns.scatterplot(x='cluster_size',y='area',data=pd.DataFrame(a),c='b')
sns.pointplot(x='cluster_size',y='vor_vol_sum',data=area_df,hue='ele',native_scale=True,linestyles='none',errorbar=('se'),
             estimator=np.median,hue_order=['APP','dAPP'],palette=['red','gray'])

#sns.scatterplot(x='cluster_size',y='area',data=pd.DataFrame(a).groupby(['cluster_size']).mean())

plt.xscale('log')
plt.yscale('log')


In [ ]:
sns.scatterplot(x='v',y='vor_vol_sum',data=area_df,hue='ele')

#sns.scatterplot(x='cluster_size',y='area',data=pd.DataFrame(a).groupby(['cluster_size']).mean())

plt.xscale('log')
plt.yscale('log')


In [ ]:
stats.spearmanr(area_df[area_df.ele=='dAPP']['vor_vol_sum'],area_df[area_df.ele=='dAPP']['area'])

In [ ]:
# fig, ax = plt.subplots(figsize=(3,3)) 
# x,y = ploy.exterior.xy
# ax.plot(x,y)
# ax.scatter([i[0] for i in points],[i[1] for i in points])
points = df[df.db==43].loc[:,['x','y']].to_numpy()
ploy = shapely.concave_hull(MultiPoint(points), ratio=j)
ax = gpd.GeoSeries(ploy)
ax.plot(figsize=(10,5))
plt.scatter([i[0] for i in points],[i[1] for i in points],zorder=1,color='k')
print(ploy.area)

In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import geopandas as gpd

polygon1 = Polygon([(0,5),
                (1,1),
                (3,0),
                ])

p = gpd.GeoSeries(polygon1)
p.plot()
plt.show()

In [ ]:
HTR7_B_1_2_1 = ThnderSTORM2df('HTR7_Basal_1_2',squre_size=4000,x1=27000,y1=20000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=1)
HTR7_B_1_2_2 = ThnderSTORM2df('HTR7_Basal_1_2',squre_size=4000,x1=26000,y1=24000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=1)
HTR7_B_1_2_3 = ThnderSTORM2df('HTR7_Basal_1_2',squre_size=4000,x1=5000,y1=27000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=1)

HTR7_B_1_3_1 = ThnderSTORM2df('HTR7_Basal_1_3',squre_size=4000,x1=25000,y1=28500,parent='HTR7',element='WT',cell=3,rectangle=1,replicate=1)
HTR7_B_1_3_2 = ThnderSTORM2df('HTR7_Basal_1_3',squre_size=4000,x1=29000,y1=28500,parent='HTR7',element='WT',cell=3,rectangle=2,replicate=1)
HTR7_B_1_3_3 = ThnderSTORM2df('HTR7_Basal_1_3',squre_size=4000,x1=33000,y1=26500,parent='HTR7',element='WT',cell=3,rectangle=3,replicate=1)

HTR7_B_1_4_1 = ThnderSTORM2df('HTR7_Basal_1_4',squre_size=4000,x1=8000,y1=27000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=1)
HTR7_B_1_4_2 = ThnderSTORM2df('HTR7_Basal_1_4',squre_size=4000,x1=31000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=1)
HTR7_B_1_4_3 = ThnderSTORM2df('HTR7_Basal_1_4',squre_size=4000,x1=41000,y1=33500,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=1)

HTR7_B_1_5_1 = ThnderSTORM2df('HTR7_Basal_1_5',squre_size=4000,x1=17000,y1=33500,parent='HTR7',element='WT',cell=5,rectangle=1,replicate=1)
HTR7_B_1_5_2 = ThnderSTORM2df('HTR7_Basal_1_5',squre_size=4000,x1=27000,y1=14500,parent='HTR7',element='WT',cell=5,rectangle=2,replicate=1)
HTR7_B_1_5_3 = ThnderSTORM2df('HTR7_Basal_1_5',squre_size=4000,x1=31000,y1=14500,parent='HTR7',element='WT',cell=5,rectangle=3,replicate=1)
HTR7_B_1_5_4 = ThnderSTORM2df('HTR7_Basal_1_5',squre_size=4000,x1=41000,y1=3000,parent='HTR7',element='WT',cell=5,rectangle=4,replicate=1)

HTR7_B_1_6_1 = ThnderSTORM2df('HTR7_Basal_1_6',squre_size=4000,x1=15000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=1)
HTR7_B_1_6_2 = ThnderSTORM2df('HTR7_Basal_1_6',squre_size=4000,x1=19000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=1)
HTR7_B_1_6_3 = ThnderSTORM2df('HTR7_Basal_1_6',squre_size=4000,x1=24000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=1)

HTR7_B_1_7_1 = ThnderSTORM2df('HTR7_Basal_1_7',squre_size=4000,x1=34000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=1,replicate=1)
HTR7_B_1_7_2 = ThnderSTORM2df('HTR7_Basal_1_7',squre_size=4000,x1=38000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=2,replicate=1)
HTR7_B_1_7_3 = ThnderSTORM2df('HTR7_Basal_1_7',squre_size=4000,x1=42000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=3,replicate=1)


HTR7_B_1_8_1 = ThnderSTORM2df('HTR7_Basal_1_8',squre_size=4000,x1=35000,y1=30000,parent='HTR7',element='WT',cell=8,rectangle=1,replicate=1)
HTR7_B_1_8_2 = ThnderSTORM2df('HTR7_Basal_1_8',squre_size=4000,x1=30000,y1=27000,parent='HTR7',element='WT',cell=8,rectangle=2,replicate=1)
HTR7_B_1_8_3 = ThnderSTORM2df('HTR7_Basal_1_8',squre_size=4000,x1=26000,y1=27000,parent='HTR7',element='WT',cell=8,rectangle=3,replicate=1)
HTR7_B_1_8_4 = ThnderSTORM2df('HTR7_Basal_1_8',squre_size=4000,x1=20000,y1=34000,parent='HTR7',element='WT',cell=8,rectangle=4,replicate=1)

HTR7_B_1_9_1 = ThnderSTORM2df('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=31000,parent='HTR7',element='WT',cell=9,rectangle=1,replicate=1)
HTR7_B_1_9_2 = ThnderSTORM2df('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=27000,parent='HTR7',element='WT',cell=9,rectangle=2,replicate=1)
HTR7_B_1_9_3 = ThnderSTORM2df('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=23000,parent='HTR7',element='WT',cell=9,rectangle=3,replicate=1)
HTR7_B_1_9_4 = ThnderSTORM2df('HTR7_Basal_1_9',squre_size=4000,x1=29000,y1=20000,parent='HTR7',element='WT',cell=9,rectangle=4,replicate=1)

HTR7_B_1_10_1 = ThnderSTORM2df('HTR7_Basal_1_10',squre_size=4000,x1=17000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=1,replicate=1)
HTR7_B_1_10_2 = ThnderSTORM2df('HTR7_Basal_1_10',squre_size=4000,x1=21000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=2,replicate=1)
HTR7_B_1_10_3 = ThnderSTORM2df('HTR7_Basal_1_10',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='WT',cell=10,rectangle=3,replicate=1)
HTR7_B_1_10_4 = ThnderSTORM2df('HTR7_Basal_1_10',squre_size=4000,x1=29000,y1=16500,parent='HTR7',element='WT',cell=10,rectangle=4,replicate=1)


HTR7_B_2_1_1 = ThnderSTORM2df('HTR7_B_2_1',squre_size=4000,x1=27000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=1,replicate=2)
HTR7_B_2_1_2 = ThnderSTORM2df('HTR7_B_2_1',squre_size=4000,x1=23000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=2,replicate=2)
HTR7_B_2_1_3 = ThnderSTORM2df('HTR7_B_2_1',squre_size=4000,x1=19000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=3,replicate=2)
HTR7_B_2_1_4 = ThnderSTORM2df('HTR7_B_2_1',squre_size=4000,x1=15000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=4,replicate=2)
HTR7_B_2_1_5 = ThnderSTORM2df('HTR7_B_2_1',squre_size=4000,x1=18000,y1=20000,parent='HTR7',element='WT',cell=1,rectangle=5,replicate=2)

HTR7_B_2_2_1 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=26000,y1=10000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=2)
HTR7_B_2_2_2 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=26000,y1=14000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=2)
HTR7_B_2_2_3 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=26000,y1=18000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=2)
HTR7_B_2_2_4 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=26000,y1=22000,parent='HTR7',element='WT',cell=2,rectangle=4,replicate=2)
HTR7_B_2_2_5 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='WT',cell=2,rectangle=5,replicate=2)
HTR7_B_2_2_6 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='WT',cell=2,rectangle=6,replicate=2)
HTR7_B_2_2_7 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=40000,y1=10000,parent='HTR7',element='WT',cell=2,rectangle=7,replicate=2)
HTR7_B_2_2_8 = ThnderSTORM2df('HTR7_B_2_2',squre_size=4000,x1=40000,y1=14000,parent='HTR7',element='WT',cell=2,rectangle=8,replicate=2)



HTR7_B_2_3_1 = ThnderSTORM2df('HTR7_B_2_3',squre_size=4000,x1=30000,y1=14000,parent='HTR7',element='WT',cell=3,rectangle=1,replicate=2)
HTR7_B_2_3_2 = ThnderSTORM2df('HTR7_B_2_3',squre_size=4000,x1=26000,y1=14000,parent='HTR7',element='WT',cell=3,rectangle=2,replicate=2)
HTR7_B_2_3_3 = ThnderSTORM2df('HTR7_B_2_3',squre_size=4000,x1=26000,y1=18000,parent='HTR7',element='WT',cell=3,rectangle=3,replicate=2)
HTR7_B_2_3_4 = ThnderSTORM2df('HTR7_B_2_3',squre_size=4000,x1=30000,y1=18000,parent='HTR7',element='WT',cell=3,rectangle=4,replicate=2)
HTR7_B_2_3_5 = ThnderSTORM2df('HTR7_B_2_3',squre_size=4000,x1=20000,y1=20000,parent='HTR7',element='WT',cell=3,rectangle=5,replicate=2)


HTR7_B_2_4_1 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=18000,y1=21000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=2)
HTR7_B_2_4_2 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=18000,y1=25000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=2)
HTR7_B_2_4_3 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=18000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=2)
HTR7_B_2_4_4 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=19000,y1=37000,parent='HTR7',element='WT',cell=4,rectangle=4,replicate=2)
HTR7_B_2_4_5 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=24000,y1=37000,parent='HTR7',element='WT',cell=4,rectangle=5,replicate=2)
HTR7_B_2_4_6 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=24000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=6,replicate=2)
HTR7_B_2_4_7 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=35000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=7,replicate=2)
HTR7_B_2_4_8 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=31000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=8,replicate=2)
HTR7_B_2_4_9 = ThnderSTORM2df('HTR7_B_2_4',squre_size=4000,x1=35000,y1=8000,parent='HTR7',element='WT',cell=4,rectangle=9,replicate=2)


HTR7_B_2_5_1 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=30000,y1=8000,parent='HTR7',element='WT',cell=5,rectangle=1,replicate=2)
HTR7_B_2_5_2 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=26000,y1=8000,parent='HTR7',element='WT',cell=5,rectangle=2,replicate=2)
HTR7_B_2_5_3 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=26000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=3,replicate=2)
HTR7_B_2_5_4 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=37000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=4,replicate=2)
HTR7_B_2_5_5 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=42000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=5,replicate=2)
HTR7_B_2_5_6 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=42000,y1=16000,parent='HTR7',element='WT',cell=5,rectangle=6,replicate=2)
HTR7_B_2_5_7 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=36000,y1=16000,parent='HTR7',element='WT',cell=5,rectangle=7,replicate=2)
HTR7_B_2_5_8 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=10000,y1=40000,parent='HTR7',element='WT',cell=5,rectangle=8,replicate=2)
HTR7_B_2_5_9 = ThnderSTORM2df('HTR7_B_2_5',squre_size=4000,x1=2000,y1=30000,parent='HTR7',element='WT',cell=5,rectangle=9,replicate=2)


HTR7_B_2_6_1 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=2000,y1=10000,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=2)
HTR7_B_2_6_2 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=2000,y1=14000,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=2)
HTR7_B_2_6_3 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=8000,y1=14000,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=2)
HTR7_B_2_6_4 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=15000,y1=24000,parent='HTR7',element='WT',cell=6,rectangle=4,replicate=2)
HTR7_B_2_6_5 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=19000,y1=24000,parent='HTR7',element='WT',cell=6,rectangle=5,replicate=2)
HTR7_B_2_6_6 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=25000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=6,replicate=2)
HTR7_B_2_6_7 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=30000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=7,replicate=2)
HTR7_B_2_6_8 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=35000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=8,replicate=2)
HTR7_B_2_6_9 = ThnderSTORM2df('HTR7_B_2_6',squre_size=4000,x1=40000,y1=25000,parent='HTR7',element='WT',cell=6,rectangle=9,replicate=2)


HTR7_B_2_8_1 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=40500,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=1,replicate=2)
HTR7_B_2_8_2 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=38000,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=2,replicate=2)
HTR7_B_2_8_3 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=34000,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=3,replicate=2)
HTR7_B_2_8_4 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=30000,y1=40000,parent='HTR7',element='WT',cell=8,rectangle=4,replicate=2)
HTR7_B_2_8_5 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=30000,y1=36000,parent='HTR7',element='WT',cell=8,rectangle=5,replicate=2)
HTR7_B_2_8_6 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=30000,y1=32000,parent='HTR7',element='WT',cell=8,rectangle=6,replicate=2)
HTR7_B_2_8_7 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=30000,y1=28000,parent='HTR7',element='WT',cell=8,rectangle=7,replicate=2)
HTR7_B_2_8_8 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=30000,y1=23000,parent='HTR7',element='WT',cell=8,rectangle=8,replicate=2)
HTR7_B_2_8_9 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=26000,y1=23000,parent='HTR7',element='WT',cell=8,rectangle=9,replicate=2)
HTR7_B_2_8_10 = ThnderSTORM2df('HTR7_B_2_8',squre_size=4000,x1=600,y1=19000,parent='HTR7',element='WT',cell=8,rectangle=10,replicate=2)


HTR7_B_2_9_1 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=23000,y1=8000,parent='HTR7',element='WT',cell=9,rectangle=1,replicate=2)
HTR7_B_2_9_2 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=23000,y1=13000,parent='HTR7',element='WT',cell=9,rectangle=2,replicate=2)
HTR7_B_2_9_3 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=23000,y1=17000,parent='HTR7',element='WT',cell=9,rectangle=3,replicate=2)
HTR7_B_2_9_4 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=23000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=4,replicate=2)
HTR7_B_2_9_5 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=27000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=5,replicate=2)
HTR7_B_2_9_6 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=31000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=6,replicate=2)
HTR7_B_2_9_7 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=31000,y1=26000,parent='HTR7',element='WT',cell=9,rectangle=7,replicate=2)
HTR7_B_2_9_8 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=28000,y1=30000,parent='HTR7',element='WT',cell=9,rectangle=8,replicate=2)
HTR7_B_2_9_9 = ThnderSTORM2df('HTR7_B_2_9',squre_size=4000,x1=24000,y1=30000,parent='HTR7',element='WT',cell=9,rectangle=9,replicate=2)


HTR7_B_2_10_1 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=26000,y1=30000,parent='HTR7',element='WT', cell=10,rectangle=1,replicate=2)
HTR7_B_2_10_2 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=26000,y1=34000,parent='HTR7',element='WT',cell=10,rectangle=2,replicate=2)
HTR7_B_2_10_3 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=24000,y1=38000,parent='HTR7',element='WT',cell=10,rectangle=3,replicate=2)
HTR7_B_2_10_4 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=27000,y1=20000,parent='HTR7',element='WT',cell=10,rectangle=4,replicate=2)
HTR7_B_2_10_5 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=27000,y1=16000,parent='HTR7',element='WT',cell=10,rectangle=5,replicate=2)
HTR7_B_2_10_6 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=22000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=6,replicate=2)
HTR7_B_2_10_7 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=12000,y1=28000,parent='HTR7',element='WT',cell=10,rectangle=7,replicate=2)
HTR7_B_2_10_8 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=12000,y1=33000,parent='HTR7',element='WT',cell=10,rectangle=8,replicate=2)
HTR7_B_2_10_9 = ThnderSTORM2df('HTR7_B_2_10',squre_size=4000,x1=8000,y1=33000,parent='HTR7',element='WT',cell=10,rectangle=9,replicate=2)

HTR7_BM_1_2_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_2',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=1)
HTR7_BM_1_2_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_2',squre_size=4000,x1=34000,y1=23000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=1)

HTR7_BM_1_3_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=34000,y1=24000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=1)
HTR7_BM_1_3_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=30000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=1)
HTR7_BM_1_3_3 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=3,replicate=1)
HTR7_BM_1_3_4 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=22000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=4,replicate=1)
HTR7_BM_1_3_5 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=18000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=5,replicate=1)
HTR7_BM_1_3_6 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=15000,y1=30000,parent='HTR7',element='MUT',cell=3,rectangle=6,replicate=1)
HTR7_BM_1_3_7 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=15000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=7,replicate=1)
HTR7_BM_1_3_8 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=5000,y1=15000,parent='HTR7',element='MUT',cell=3,rectangle=8,replicate=1)
HTR7_BM_1_3_9 = ThnderSTORM2df('HTR7_MUT_Basal_1_3',squre_size=4000,x1=5000,y1=20000,parent='HTR7',element='MUT',cell=3,rectangle=9,replicate=1)


HTR7_BM_1_4_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=34000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=1)
HTR7_BM_1_4_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=30000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=1)
HTR7_BM_1_4_3 = ThnderSTORM2df('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=26000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=1)
HTR7_BM_1_4_4 = ThnderSTORM2df('HTR7_MUT_Basal_1_4',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=1)
HTR7_BM_1_4_5 = ThnderSTORM2df('HTR7_MUT_Basal_1_4',squre_size=4000,x1=17000,y1=29000,parent='HTR7',element='MUT',cell=4,rectangle=5,replicate=1)



HTR7_BM_1_5_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_5',squre_size=4000,x1=10000,y1=10000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=1)
HTR7_BM_1_5_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_5',squre_size=4000,x1=10000,y1=14000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=1)
HTR7_BM_1_5_3 = ThnderSTORM2df('HTR7_MUT_Basal_1_5',squre_size=4000,x1=25000,y1=14000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=1)
HTR7_BM_1_5_4 = ThnderSTORM2df('HTR7_MUT_Basal_1_5',squre_size=4000,x1=25000,y1=18000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=1)
HTR7_BM_1_5_5 = ThnderSTORM2df('HTR7_MUT_Basal_1_5',squre_size=4000,x1=28000,y1=38000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=1)
HTR7_BM_1_5_6 = ThnderSTORM2df('HTR7_MUT_Basal_1_5',squre_size=4000,x1=35000,y1=40000,parent='HTR7',element='MUT',cell=5,rectangle=6,replicate=1)


HTR7_BM_1_6_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_6',squre_size=4000,x1=17000,y1=26000,parent='HTR7',element='MUT',cell=6,rectangle=1,replicate=1)
HTR7_BM_1_6_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_6',squre_size=4000,x1=21000,y1=26000,parent='HTR7',element='MUT',cell=6,rectangle=2,replicate=1)

HTR7_BM_1_7_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_7',squre_size=4000,x1=21000,y1=26000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=1)
HTR7_BM_1_7_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_7',squre_size=4000,x1=21000,y1=30000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=1)


HTR7_BM_1_8_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=30000,parent='HTR7',element='MUT',cell=8,rectangle=1,replicate=1)
HTR7_BM_1_8_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=26000,parent='HTR7',element='MUT',cell=8,rectangle=2,replicate=1)
HTR7_BM_1_8_3 = ThnderSTORM2df('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=3,replicate=1)
HTR7_BM_1_8_4 = ThnderSTORM2df('HTR7_MUT_Basal_1_8',squre_size=4000,x1=20000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=4,replicate=1)
HTR7_BM_1_8_5 = ThnderSTORM2df('HTR7_MUT_Basal_1_8',squre_size=4000,x1=40000,y1=18000,parent='HTR7',element='MUT',cell=8,rectangle=5,replicate=1)

HTR7_BM_1_9_1 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=30000,y1=12000,parent='HTR7',element='MUT',cell=9,rectangle=1,replicate=1)
HTR7_BM_1_9_2 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=12000,parent='HTR7',element='MUT',cell=9,rectangle=2,replicate=1)
HTR7_BM_1_9_3 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=16000,parent='HTR7',element='MUT',cell=9,rectangle=3,replicate=1)
HTR7_BM_1_9_4 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=27000,y1=18000,parent='HTR7',element='MUT',cell=9,rectangle=4,replicate=1)
HTR7_BM_1_9_5 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=27000,y1=25000,parent='HTR7',element='MUT',cell=9,rectangle=5,replicate=1)
HTR7_BM_1_9_6 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=29000,parent='HTR7',element='MUT',cell=9,rectangle=6,replicate=1)
HTR7_BM_1_9_7 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=22000,y1=29000,parent='HTR7',element='MUT',cell=9,rectangle=7,replicate=1)
HTR7_BM_1_9_8 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=18000,y1=30000,parent='HTR7',element='MUT', cell=9,rectangle=8,replicate=1)
HTR7_BM_1_9_9 = ThnderSTORM2df('HTR7_MUT_Basal_1_9',squre_size=4000,x1=18000,y1=34000,parent='HTR7',element='MUT', cell=9,rectangle=9,replicate=1)


HTR7_BM_2_2_1 = ThnderSTORM2df('HTR7Mut_B_2_2',squre_size=4000,x1=24000,y1=24000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=2)
HTR7_BM_2_2_2 = ThnderSTORM2df('HTR7Mut_B_2_2',squre_size=4000,x1=24000,y1=20000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=2)
HTR7_BM_2_2_3 = ThnderSTORM2df('HTR7Mut_B_2_2',squre_size=4000,x1=22000,y1=37000,parent='HTR7',element='MUT',cell=2,rectangle=3,replicate=2)


HTR7_BM_2_3_1 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=14000,y1=32000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=2)
HTR7_BM_2_3_2 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=10000,y1=32000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=2)
HTR7_BM_2_3_3 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=25000,y1=20000,parent='HTR7',element='MUT',cell=3,rectangle=3,replicate=2)
HTR7_BM_2_3_4 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=24000,y1=16000,parent='HTR7',element='MUT',cell=3,rectangle=4,replicate=2)
HTR7_BM_2_3_5 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=24000,y1=12000,parent='HTR7',element='MUT',cell=3,rectangle=5,replicate=2)
HTR7_BM_2_3_6 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=20000,y1=8000,parent='HTR7',element='MUT',cell=3,rectangle=6,replicate=2)
HTR7_BM_2_3_7 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=16000,y1=8000,parent='HTR7',element='MUT',cell=3,rectangle=7,replicate=2)
HTR7_BM_2_3_8 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=40000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=8,replicate=2)
HTR7_BM_2_3_9 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=45000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=9,replicate=2)
HTR7_BM_2_3_10 = ThnderSTORM2df('HTR7Mut_B_2_3',squre_size=4000,x1=43000,y1=39000,parent='HTR7',element='MUT',cell=3,rectangle=10,replicate=2)


HTR7_BM_2_4_1 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=25000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=2)
HTR7_BM_2_4_2 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=29000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=2)
HTR7_BM_2_4_3 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=33000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=2)
HTR7_BM_2_4_4 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=16000,y1=33000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=2)
HTR7_BM_2_4_5 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=16000,y1=27000,parent='HTR7',element='MUT',cell=4,rectangle=5,replicate=2)
HTR7_BM_2_4_6 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=3000,parent='HTR7',element='MUT', cell=4,rectangle=6,replicate=2)
HTR7_BM_2_4_7 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=7000,parent='HTR7',element='MUT', cell=4,rectangle=7,replicate=2)
HTR7_BM_2_4_8 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=11000,parent='HTR7',element='MUT',cell=4,rectangle=8,replicate=2)
HTR7_BM_2_4_9 = ThnderSTORM2df('HTR7Mut_B_2_4',squre_size=4000,x1=36000,y1=4000,parent='HTR7',element='MUT',cell=4,rectangle=9,replicate=2)


HTR7_BM_2_5_1 = ThnderSTORM2df('HTR7Mut_B_2_5',squre_size=4000,x1=25000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=2)
HTR7_BM_2_5_2 = ThnderSTORM2df('HTR7Mut_B_2_5',squre_size=4000,x1=29000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=2)
HTR7_BM_2_5_3 = ThnderSTORM2df('HTR7Mut_B_2_5',squre_size=4000,x1=33000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=2)
HTR7_BM_2_5_4 = ThnderSTORM2df('HTR7Mut_B_2_5',squre_size=4000,x1=33000,y1=27000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=2)
HTR7_BM_2_5_5 = ThnderSTORM2df('HTR7Mut_B_2_5',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=2)


HTR7_BM_2_7_1 = ThnderSTORM2df('HTR7Mut_B_2_7',squre_size=4000,x1=20000,y1=17000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=2)
HTR7_BM_2_7_2 = ThnderSTORM2df('HTR7Mut_B_2_7',squre_size=4000,x1=16000,y1=17000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=2)
HTR7_BM_2_7_3 = ThnderSTORM2df('HTR7Mut_B_2_7',squre_size=4000,x1=20000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=3,replicate=2)
HTR7_BM_2_7_4 = ThnderSTORM2df('HTR7Mut_B_2_7',squre_size=4000,x1=25000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=4,replicate=2)
HTR7_BM_2_7_5 = ThnderSTORM2df('HTR7Mut_B_2_7',squre_size=4000,x1=30000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=5,replicate=2)


HTR7_BM_2_8_1 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=27000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=1,replicate=2)
HTR7_BM_2_8_2 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=22000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=2,replicate=2)
HTR7_BM_2_8_3 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=17000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=3,replicate=2)
HTR7_BM_2_8_4 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=13000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=4,replicate=2)
HTR7_BM_2_8_5 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=16000,y1=27000,parent='HTR7',element='MUT',cell=8,rectangle=5,replicate=2)
HTR7_BM_2_8_6 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=13000,y1=15000,parent='HTR7',element='MUT',  cell=8,rectangle=6,replicate=2)
HTR7_BM_2_8_7 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=19000,y1=15000,parent='HTR7',element='MUT',  cell=8,rectangle=7,replicate=2)
HTR7_BM_2_8_8 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=30000,y1=13000,parent='HTR7',element='MUT', cell=8,rectangle=8,replicate=2)
HTR7_BM_2_8_9 = ThnderSTORM2df('HTR7Mut_B_2_8',squre_size=4000,x1=30000,y1=9000,parent='HTR7',element='MUT', cell=8,rectangle=9,replicate=2)


HTR7_BM_2_10_1 = ThnderSTORM2df('HTR7Mut_B_2_10',squre_size=4000,x1=30000,y1=9000,parent='HTR7',element='MUT',cell=10,rectangle=1,replicate=2)
HTR7_BM_2_10_2 = ThnderSTORM2df('HTR7Mut_B_2_10',squre_size=4000,x1=26000,y1=13000,parent='HTR7',element='MUT',cell=10,rectangle=2,replicate=2)
HTR7_BM_2_10_3 = ThnderSTORM2df('HTR7Mut_B_2_10',squre_size=4000,x1=25000,y1=20000,parent='HTR7',element='MUT',cell=10,rectangle=3,replicate=2)
HTR7_BM_2_10_4 = ThnderSTORM2df('HTR7Mut_B_2_10',squre_size=4000,x1=21000,y1=25000,parent='HTR7',element='MUT',cell=10,rectangle=4,replicate=2)
HTR7_BM_2_10_5 = ThnderSTORM2df('HTR7Mut_B_2_10',squre_size=4000,x1=21000,y1=29000,parent='HTR7',element='MUT',cell=10,rectangle=5,replicate=2)
HTR7_BM_2_10_6 = ThnderSTORM2df('HTR7Mut_B_2_10',squre_size=4000,x1=20000,y1=33000,parent='HTR7',element='MUT',cell=10,rectangle=6,replicate=2)

In [ ]:
HTR7_B_1_2_1 = voro2cluster('HTR7_Basal_1_2',squre_size=4000,x1=27000,y1=20000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=1)
HTR7_B_1_2_2 = voro2cluster('HTR7_Basal_1_2',squre_size=4000,x1=26000,y1=24000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=1)
HTR7_B_1_2_3 = voro2cluster('HTR7_Basal_1_2',squre_size=4000,x1=5000,y1=27000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=1)

HTR7_B_1_3_1 = voro2cluster('HTR7_Basal_1_3',squre_size=4000,x1=25000,y1=28500,parent='HTR7',element='WT',cell=3,rectangle=1,replicate=1)
HTR7_B_1_3_2 = voro2cluster('HTR7_Basal_1_3',squre_size=4000,x1=29000,y1=28500,parent='HTR7',element='WT',cell=3,rectangle=2,replicate=1)
HTR7_B_1_3_3 = voro2cluster('HTR7_Basal_1_3',squre_size=4000,x1=33000,y1=26500,parent='HTR7',element='WT',cell=3,rectangle=3,replicate=1)

HTR7_B_1_4_1 = voro2cluster('HTR7_Basal_1_4',squre_size=4000,x1=8000,y1=27000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=1)
HTR7_B_1_4_2 = voro2cluster('HTR7_Basal_1_4',squre_size=4000,x1=31000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=1)
HTR7_B_1_4_3 = voro2cluster('HTR7_Basal_1_4',squre_size=4000,x1=41000,y1=33500,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=1)

HTR7_B_1_5_1 = voro2cluster('HTR7_Basal_1_5',squre_size=4000,x1=17000,y1=33500,parent='HTR7',element='WT',cell=5,rectangle=1,replicate=1)
HTR7_B_1_5_2 = voro2cluster('HTR7_Basal_1_5',squre_size=4000,x1=27000,y1=14500,parent='HTR7',element='WT',cell=5,rectangle=2,replicate=1)
HTR7_B_1_5_3 = voro2cluster('HTR7_Basal_1_5',squre_size=4000,x1=31000,y1=14500,parent='HTR7',element='WT',cell=5,rectangle=3,replicate=1)
HTR7_B_1_5_4 = voro2cluster('HTR7_Basal_1_5',squre_size=4000,x1=41000,y1=3000,parent='HTR7',element='WT',cell=5,rectangle=4,replicate=1)

HTR7_B_1_6_1 = voro2cluster('HTR7_Basal_1_6',squre_size=4000,x1=15000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=1)
HTR7_B_1_6_2 = voro2cluster('HTR7_Basal_1_6',squre_size=4000,x1=19000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=1)
HTR7_B_1_6_3 = voro2cluster('HTR7_Basal_1_6',squre_size=4000,x1=24000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=1)

HTR7_B_1_7_1 = voro2cluster('HTR7_Basal_1_7',squre_size=4000,x1=34000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=1,replicate=1)
HTR7_B_1_7_2 = voro2cluster('HTR7_Basal_1_7',squre_size=4000,x1=38000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=2,replicate=1)
HTR7_B_1_7_3 = voro2cluster('HTR7_Basal_1_7',squre_size=4000,x1=42000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=3,replicate=1)


HTR7_B_1_8_1 = voro2cluster('HTR7_Basal_1_8',squre_size=4000,x1=35000,y1=30000,parent='HTR7',element='WT',cell=8,rectangle=1,replicate=1)
HTR7_B_1_8_2 = voro2cluster('HTR7_Basal_1_8',squre_size=4000,x1=30000,y1=27000,parent='HTR7',element='WT',cell=8,rectangle=2,replicate=1)
HTR7_B_1_8_3 = voro2cluster('HTR7_Basal_1_8',squre_size=4000,x1=26000,y1=27000,parent='HTR7',element='WT',cell=8,rectangle=3,replicate=1)
HTR7_B_1_8_4 = voro2cluster('HTR7_Basal_1_8',squre_size=4000,x1=20000,y1=34000,parent='HTR7',element='WT',cell=8,rectangle=4,replicate=1)

HTR7_B_1_9_1 = voro2cluster('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=31000,parent='HTR7',element='WT',cell=9,rectangle=1,replicate=1)
HTR7_B_1_9_2 = voro2cluster('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=27000,parent='HTR7',element='WT',cell=9,rectangle=2,replicate=1)
HTR7_B_1_9_3 = voro2cluster('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=23000,parent='HTR7',element='WT',cell=9,rectangle=3,replicate=1)
HTR7_B_1_9_4 = voro2cluster('HTR7_Basal_1_9',squre_size=4000,x1=29000,y1=20000,parent='HTR7',element='WT',cell=9,rectangle=4,replicate=1)

HTR7_B_1_10_1 = voro2cluster('HTR7_Basal_1_10',squre_size=4000,x1=17000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=1,replicate=1)
HTR7_B_1_10_2 = voro2cluster('HTR7_Basal_1_10',squre_size=4000,x1=21000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=2,replicate=1)
HTR7_B_1_10_3 = voro2cluster('HTR7_Basal_1_10',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='WT',cell=10,rectangle=3,replicate=1)
HTR7_B_1_10_4 = voro2cluster('HTR7_Basal_1_10',squre_size=4000,x1=29000,y1=16500,parent='HTR7',element='WT',cell=10,rectangle=4,replicate=1)


HTR7_B_2_1_1 = voro2cluster('HTR7_B_2_1',squre_size=4000,x1=27000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=1,replicate=2)
HTR7_B_2_1_2 = voro2cluster('HTR7_B_2_1',squre_size=4000,x1=23000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=2,replicate=2)
HTR7_B_2_1_3 = voro2cluster('HTR7_B_2_1',squre_size=4000,x1=19000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=3,replicate=2)
HTR7_B_2_1_4 = voro2cluster('HTR7_B_2_1',squre_size=4000,x1=15000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=4,replicate=2)
HTR7_B_2_1_5 = voro2cluster('HTR7_B_2_1',squre_size=4000,x1=18000,y1=20000,parent='HTR7',element='WT',cell=1,rectangle=5,replicate=2)

HTR7_B_2_2_1 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=26000,y1=10000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=2)
HTR7_B_2_2_2 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=26000,y1=14000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=2)
HTR7_B_2_2_3 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=26000,y1=18000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=2)
HTR7_B_2_2_4 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=26000,y1=22000,parent='HTR7',element='WT',cell=2,rectangle=4,replicate=2)
HTR7_B_2_2_5 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='WT',cell=2,rectangle=5,replicate=2)
HTR7_B_2_2_6 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='WT',cell=2,rectangle=6,replicate=2)
HTR7_B_2_2_7 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=40000,y1=10000,parent='HTR7',element='WT',cell=2,rectangle=7,replicate=2)
HTR7_B_2_2_8 = voro2cluster('HTR7_B_2_2',squre_size=4000,x1=40000,y1=14000,parent='HTR7',element='WT',cell=2,rectangle=8,replicate=2)



HTR7_B_2_3_1 = voro2cluster('HTR7_B_2_3',squre_size=4000,x1=30000,y1=14000,parent='HTR7',element='WT',cell=3,rectangle=1,replicate=2)
HTR7_B_2_3_2 = voro2cluster('HTR7_B_2_3',squre_size=4000,x1=26000,y1=14000,parent='HTR7',element='WT',cell=3,rectangle=2,replicate=2)
HTR7_B_2_3_3 = voro2cluster('HTR7_B_2_3',squre_size=4000,x1=26000,y1=18000,parent='HTR7',element='WT',cell=3,rectangle=3,replicate=2)
HTR7_B_2_3_4 = voro2cluster('HTR7_B_2_3',squre_size=4000,x1=30000,y1=18000,parent='HTR7',element='WT',cell=3,rectangle=4,replicate=2)
HTR7_B_2_3_5 = voro2cluster('HTR7_B_2_3',squre_size=4000,x1=20000,y1=20000,parent='HTR7',element='WT',cell=3,rectangle=5,replicate=2)


HTR7_B_2_4_1 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=18000,y1=21000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=2)
HTR7_B_2_4_2 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=18000,y1=25000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=2)
HTR7_B_2_4_3 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=18000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=2)
HTR7_B_2_4_4 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=19000,y1=37000,parent='HTR7',element='WT',cell=4,rectangle=4,replicate=2)
HTR7_B_2_4_5 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=24000,y1=37000,parent='HTR7',element='WT',cell=4,rectangle=5,replicate=2)
HTR7_B_2_4_6 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=24000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=6,replicate=2)
HTR7_B_2_4_7 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=35000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=7,replicate=2)
HTR7_B_2_4_8 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=31000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=8,replicate=2)
HTR7_B_2_4_9 = voro2cluster('HTR7_B_2_4',squre_size=4000,x1=35000,y1=8000,parent='HTR7',element='WT',cell=4,rectangle=9,replicate=2)


HTR7_B_2_5_1 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=30000,y1=8000,parent='HTR7',element='WT',cell=5,rectangle=1,replicate=2)
HTR7_B_2_5_2 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=26000,y1=8000,parent='HTR7',element='WT',cell=5,rectangle=2,replicate=2)
HTR7_B_2_5_3 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=26000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=3,replicate=2)
HTR7_B_2_5_4 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=37000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=4,replicate=2)
HTR7_B_2_5_5 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=42000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=5,replicate=2)
HTR7_B_2_5_6 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=42000,y1=16000,parent='HTR7',element='WT',cell=5,rectangle=6,replicate=2)
HTR7_B_2_5_7 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=36000,y1=16000,parent='HTR7',element='WT',cell=5,rectangle=7,replicate=2)
HTR7_B_2_5_8 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=10000,y1=40000,parent='HTR7',element='WT',cell=5,rectangle=8,replicate=2)
HTR7_B_2_5_9 = voro2cluster('HTR7_B_2_5',squre_size=4000,x1=2000,y1=30000,parent='HTR7',element='WT',cell=5,rectangle=9,replicate=2)


HTR7_B_2_6_1 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=2000,y1=10000,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=2)
HTR7_B_2_6_2 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=2000,y1=14000,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=2)
HTR7_B_2_6_3 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=8000,y1=14000,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=2)
HTR7_B_2_6_4 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=15000,y1=24000,parent='HTR7',element='WT',cell=6,rectangle=4,replicate=2)
HTR7_B_2_6_5 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=19000,y1=24000,parent='HTR7',element='WT',cell=6,rectangle=5,replicate=2)
HTR7_B_2_6_6 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=25000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=6,replicate=2)
HTR7_B_2_6_7 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=30000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=7,replicate=2)
HTR7_B_2_6_8 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=35000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=8,replicate=2)
HTR7_B_2_6_9 = voro2cluster('HTR7_B_2_6',squre_size=4000,x1=40000,y1=25000,parent='HTR7',element='WT',cell=6,rectangle=9,replicate=2)


HTR7_B_2_8_1 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=40500,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=1,replicate=2)
HTR7_B_2_8_2 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=38000,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=2,replicate=2)
HTR7_B_2_8_3 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=34000,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=3,replicate=2)
HTR7_B_2_8_4 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=30000,y1=40000,parent='HTR7',element='WT',cell=8,rectangle=4,replicate=2)
HTR7_B_2_8_5 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=30000,y1=36000,parent='HTR7',element='WT',cell=8,rectangle=5,replicate=2)
HTR7_B_2_8_6 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=30000,y1=32000,parent='HTR7',element='WT',cell=8,rectangle=6,replicate=2)
HTR7_B_2_8_7 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=30000,y1=28000,parent='HTR7',element='WT',cell=8,rectangle=7,replicate=2)
HTR7_B_2_8_8 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=30000,y1=23000,parent='HTR7',element='WT',cell=8,rectangle=8,replicate=2)
HTR7_B_2_8_9 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=26000,y1=23000,parent='HTR7',element='WT',cell=8,rectangle=9,replicate=2)
HTR7_B_2_8_10 = voro2cluster('HTR7_B_2_8',squre_size=4000,x1=600,y1=19000,parent='HTR7',element='WT',cell=8,rectangle=10,replicate=2)


HTR7_B_2_9_1 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=23000,y1=8000,parent='HTR7',element='WT',cell=9,rectangle=1,replicate=2)
HTR7_B_2_9_2 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=23000,y1=13000,parent='HTR7',element='WT',cell=9,rectangle=2,replicate=2)
HTR7_B_2_9_3 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=23000,y1=17000,parent='HTR7',element='WT',cell=9,rectangle=3,replicate=2)
HTR7_B_2_9_4 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=23000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=4,replicate=2)
HTR7_B_2_9_5 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=27000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=5,replicate=2)
HTR7_B_2_9_6 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=31000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=6,replicate=2)
HTR7_B_2_9_7 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=31000,y1=26000,parent='HTR7',element='WT',cell=9,rectangle=7,replicate=2)
HTR7_B_2_9_8 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=28000,y1=30000,parent='HTR7',element='WT',cell=9,rectangle=8,replicate=2)
HTR7_B_2_9_9 = voro2cluster('HTR7_B_2_9',squre_size=4000,x1=24000,y1=30000,parent='HTR7',element='WT',cell=9,rectangle=9,replicate=2)


HTR7_B_2_10_1 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=26000,y1=30000,parent='HTR7',element='WT', cell=10,rectangle=1,replicate=2)
HTR7_B_2_10_2 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=26000,y1=34000,parent='HTR7',element='WT',cell=10,rectangle=2,replicate=2)
HTR7_B_2_10_3 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=24000,y1=38000,parent='HTR7',element='WT',cell=10,rectangle=3,replicate=2)
HTR7_B_2_10_4 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=27000,y1=20000,parent='HTR7',element='WT',cell=10,rectangle=4,replicate=2)
HTR7_B_2_10_5 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=27000,y1=16000,parent='HTR7',element='WT',cell=10,rectangle=5,replicate=2)
HTR7_B_2_10_6 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=22000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=6,replicate=2)
HTR7_B_2_10_7 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=12000,y1=28000,parent='HTR7',element='WT',cell=10,rectangle=7,replicate=2)
HTR7_B_2_10_8 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=12000,y1=33000,parent='HTR7',element='WT',cell=10,rectangle=8,replicate=2)
HTR7_B_2_10_9 = voro2cluster('HTR7_B_2_10',squre_size=4000,x1=8000,y1=33000,parent='HTR7',element='WT',cell=10,rectangle=9,replicate=2)

HTR7_BM_1_2_1 = voro2cluster('HTR7_MUT_Basal_1_2',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=1)
HTR7_BM_1_2_2 = voro2cluster('HTR7_MUT_Basal_1_2',squre_size=4000,x1=34000,y1=23000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=1)

HTR7_BM_1_3_1 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=34000,y1=24000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=1)
HTR7_BM_1_3_2 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=30000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=1)
HTR7_BM_1_3_3 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=3,replicate=1)
HTR7_BM_1_3_4 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=22000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=4,replicate=1)
HTR7_BM_1_3_5 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=18000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=5,replicate=1)
HTR7_BM_1_3_6 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=15000,y1=30000,parent='HTR7',element='MUT',cell=3,rectangle=6,replicate=1)
HTR7_BM_1_3_7 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=15000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=7,replicate=1)
HTR7_BM_1_3_8 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=5000,y1=15000,parent='HTR7',element='MUT',cell=3,rectangle=8,replicate=1)
HTR7_BM_1_3_9 = voro2cluster('HTR7_MUT_Basal_1_3',squre_size=4000,x1=5000,y1=20000,parent='HTR7',element='MUT',cell=3,rectangle=9,replicate=1)


HTR7_BM_1_4_1 = voro2cluster('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=34000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=1)
HTR7_BM_1_4_2 = voro2cluster('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=30000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=1)
HTR7_BM_1_4_3 = voro2cluster('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=26000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=1)
HTR7_BM_1_4_4 = voro2cluster('HTR7_MUT_Basal_1_4',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=1)
HTR7_BM_1_4_5 = voro2cluster('HTR7_MUT_Basal_1_4',squre_size=4000,x1=17000,y1=29000,parent='HTR7',element='MUT',cell=4,rectangle=5,replicate=1)



HTR7_BM_1_5_1 = voro2cluster('HTR7_MUT_Basal_1_5',squre_size=4000,x1=10000,y1=10000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=1)
HTR7_BM_1_5_2 = voro2cluster('HTR7_MUT_Basal_1_5',squre_size=4000,x1=10000,y1=14000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=1)
HTR7_BM_1_5_3 = voro2cluster('HTR7_MUT_Basal_1_5',squre_size=4000,x1=25000,y1=14000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=1)
HTR7_BM_1_5_4 = voro2cluster('HTR7_MUT_Basal_1_5',squre_size=4000,x1=25000,y1=18000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=1)
HTR7_BM_1_5_5 = voro2cluster('HTR7_MUT_Basal_1_5',squre_size=4000,x1=28000,y1=38000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=1)
HTR7_BM_1_5_6 = voro2cluster('HTR7_MUT_Basal_1_5',squre_size=4000,x1=35000,y1=40000,parent='HTR7',element='MUT',cell=5,rectangle=6,replicate=1)


HTR7_BM_1_6_1 = voro2cluster('HTR7_MUT_Basal_1_6',squre_size=4000,x1=17000,y1=26000,parent='HTR7',element='MUT',cell=6,rectangle=1,replicate=1)
HTR7_BM_1_6_2 = voro2cluster('HTR7_MUT_Basal_1_6',squre_size=4000,x1=21000,y1=26000,parent='HTR7',element='MUT',cell=6,rectangle=2,replicate=1)

HTR7_BM_1_7_1 = voro2cluster('HTR7_MUT_Basal_1_7',squre_size=4000,x1=21000,y1=26000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=1)
HTR7_BM_1_7_2 = voro2cluster('HTR7_MUT_Basal_1_7',squre_size=4000,x1=21000,y1=30000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=1)


HTR7_BM_1_8_1 = voro2cluster('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=30000,parent='HTR7',element='MUT',cell=8,rectangle=1,replicate=1)
HTR7_BM_1_8_2 = voro2cluster('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=26000,parent='HTR7',element='MUT',cell=8,rectangle=2,replicate=1)
HTR7_BM_1_8_3 = voro2cluster('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=3,replicate=1)
HTR7_BM_1_8_4 = voro2cluster('HTR7_MUT_Basal_1_8',squre_size=4000,x1=20000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=4,replicate=1)
HTR7_BM_1_8_5 = voro2cluster('HTR7_MUT_Basal_1_8',squre_size=4000,x1=40000,y1=18000,parent='HTR7',element='MUT',cell=8,rectangle=5,replicate=1)

HTR7_BM_1_9_1 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=30000,y1=12000,parent='HTR7',element='MUT',cell=9,rectangle=1,replicate=1)
HTR7_BM_1_9_2 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=12000,parent='HTR7',element='MUT',cell=9,rectangle=2,replicate=1)
HTR7_BM_1_9_3 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=16000,parent='HTR7',element='MUT',cell=9,rectangle=3,replicate=1)
HTR7_BM_1_9_4 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=27000,y1=18000,parent='HTR7',element='MUT',cell=9,rectangle=4,replicate=1)
HTR7_BM_1_9_5 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=27000,y1=25000,parent='HTR7',element='MUT',cell=9,rectangle=5,replicate=1)
HTR7_BM_1_9_6 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=29000,parent='HTR7',element='MUT',cell=9,rectangle=6,replicate=1)
HTR7_BM_1_9_7 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=22000,y1=29000,parent='HTR7',element='MUT',cell=9,rectangle=7,replicate=1)
HTR7_BM_1_9_8 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=18000,y1=30000,parent='HTR7',element='MUT', cell=9,rectangle=8,replicate=1)
HTR7_BM_1_9_9 = voro2cluster('HTR7_MUT_Basal_1_9',squre_size=4000,x1=18000,y1=34000,parent='HTR7',element='MUT', cell=9,rectangle=9,replicate=1)


HTR7_BM_2_2_1 = voro2cluster('HTR7Mut_B_2_2',squre_size=4000,x1=24000,y1=24000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=2)
HTR7_BM_2_2_2 = voro2cluster('HTR7Mut_B_2_2',squre_size=4000,x1=24000,y1=20000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=2)
HTR7_BM_2_2_3 = voro2cluster('HTR7Mut_B_2_2',squre_size=4000,x1=22000,y1=37000,parent='HTR7',element='MUT',cell=2,rectangle=3,replicate=2)


HTR7_BM_2_3_1 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=14000,y1=32000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=2)
HTR7_BM_2_3_2 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=10000,y1=32000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=2)
HTR7_BM_2_3_3 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=25000,y1=20000,parent='HTR7',element='MUT',cell=3,rectangle=3,replicate=2)
HTR7_BM_2_3_4 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=24000,y1=16000,parent='HTR7',element='MUT',cell=3,rectangle=4,replicate=2)
HTR7_BM_2_3_5 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=24000,y1=12000,parent='HTR7',element='MUT',cell=3,rectangle=5,replicate=2)
HTR7_BM_2_3_6 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=20000,y1=8000,parent='HTR7',element='MUT',cell=3,rectangle=6,replicate=2)
HTR7_BM_2_3_7 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=16000,y1=8000,parent='HTR7',element='MUT',cell=3,rectangle=7,replicate=2)
HTR7_BM_2_3_8 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=40000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=8,replicate=2)
HTR7_BM_2_3_9 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=45000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=9,replicate=2)
HTR7_BM_2_3_10 = voro2cluster('HTR7Mut_B_2_3',squre_size=4000,x1=43000,y1=39000,parent='HTR7',element='MUT',cell=3,rectangle=10,replicate=2)


HTR7_BM_2_4_1 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=25000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=2)
HTR7_BM_2_4_2 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=29000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=2)
HTR7_BM_2_4_3 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=33000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=2)
HTR7_BM_2_4_4 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=16000,y1=33000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=2)
HTR7_BM_2_4_5 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=16000,y1=27000,parent='HTR7',element='MUT',cell=4,rectangle=5,replicate=2)
HTR7_BM_2_4_6 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=3000,parent='HTR7',element='MUT', cell=4,rectangle=6,replicate=2)
HTR7_BM_2_4_7 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=7000,parent='HTR7',element='MUT', cell=4,rectangle=7,replicate=2)
HTR7_BM_2_4_8 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=11000,parent='HTR7',element='MUT',cell=4,rectangle=8,replicate=2)
HTR7_BM_2_4_9 = voro2cluster('HTR7Mut_B_2_4',squre_size=4000,x1=36000,y1=4000,parent='HTR7',element='MUT',cell=4,rectangle=9,replicate=2)


HTR7_BM_2_5_1 = voro2cluster('HTR7Mut_B_2_5',squre_size=4000,x1=25000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=2)
HTR7_BM_2_5_2 = voro2cluster('HTR7Mut_B_2_5',squre_size=4000,x1=29000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=2)
HTR7_BM_2_5_3 = voro2cluster('HTR7Mut_B_2_5',squre_size=4000,x1=33000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=2)
HTR7_BM_2_5_4 = voro2cluster('HTR7Mut_B_2_5',squre_size=4000,x1=33000,y1=27000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=2)
HTR7_BM_2_5_5 = voro2cluster('HTR7Mut_B_2_5',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=2)


HTR7_BM_2_7_1 = voro2cluster('HTR7Mut_B_2_7',squre_size=4000,x1=20000,y1=17000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=2)
HTR7_BM_2_7_2 = voro2cluster('HTR7Mut_B_2_7',squre_size=4000,x1=16000,y1=17000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=2)
HTR7_BM_2_7_3 = voro2cluster('HTR7Mut_B_2_7',squre_size=4000,x1=20000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=3,replicate=2)
HTR7_BM_2_7_4 = voro2cluster('HTR7Mut_B_2_7',squre_size=4000,x1=25000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=4,replicate=2)
HTR7_BM_2_7_5 = voro2cluster('HTR7Mut_B_2_7',squre_size=4000,x1=30000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=5,replicate=2)


HTR7_BM_2_8_1 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=27000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=1,replicate=2)
HTR7_BM_2_8_2 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=22000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=2,replicate=2)
HTR7_BM_2_8_3 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=17000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=3,replicate=2)
HTR7_BM_2_8_4 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=13000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=4,replicate=2)
HTR7_BM_2_8_5 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=16000,y1=27000,parent='HTR7',element='MUT',cell=8,rectangle=5,replicate=2)
HTR7_BM_2_8_6 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=13000,y1=15000,parent='HTR7',element='MUT',  cell=8,rectangle=6,replicate=2)
HTR7_BM_2_8_7 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=19000,y1=15000,parent='HTR7',element='MUT',  cell=8,rectangle=7,replicate=2)
HTR7_BM_2_8_8 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=30000,y1=13000,parent='HTR7',element='MUT', cell=8,rectangle=8,replicate=2)
HTR7_BM_2_8_9 = voro2cluster('HTR7Mut_B_2_8',squre_size=4000,x1=30000,y1=9000,parent='HTR7',element='MUT', cell=8,rectangle=9,replicate=2)


HTR7_BM_2_10_1 = voro2cluster('HTR7Mut_B_2_10',squre_size=4000,x1=30000,y1=9000,parent='HTR7',element='MUT',cell=10,rectangle=1,replicate=2)
HTR7_BM_2_10_2 = voro2cluster('HTR7Mut_B_2_10',squre_size=4000,x1=26000,y1=13000,parent='HTR7',element='MUT',cell=10,rectangle=2,replicate=2)
HTR7_BM_2_10_3 = voro2cluster('HTR7Mut_B_2_10',squre_size=4000,x1=25000,y1=20000,parent='HTR7',element='MUT',cell=10,rectangle=3,replicate=2)
HTR7_BM_2_10_4 = voro2cluster('HTR7Mut_B_2_10',squre_size=4000,x1=21000,y1=25000,parent='HTR7',element='MUT',cell=10,rectangle=4,replicate=2)
HTR7_BM_2_10_5 = voro2cluster('HTR7Mut_B_2_10',squre_size=4000,x1=21000,y1=29000,parent='HTR7',element='MUT',cell=10,rectangle=5,replicate=2)
HTR7_BM_2_10_6 = voro2cluster('HTR7Mut_B_2_10',squre_size=4000,x1=20000,y1=33000,parent='HTR7',element='MUT',cell=10,rectangle=6,replicate=2)

In [ ]:
HTR7_STIM_M_1_1_1 = ThnderSTORM2df('HTR7M_Sert_1_1',squre_size=4000,x1=19000,y1=15000,parent='HTR7',element='MUT',cell=1,rectangle=1,replicate=1)
HTR7_STIM_M_1_1_2 = ThnderSTORM2df('HTR7M_Sert_1_1',squre_size=4000,x1=18000,y1=11000,parent='HTR7',element='MUT',cell=1,rectangle=2,replicate=1)
HTR7_STIM_M_1_1_3 = ThnderSTORM2df('HTR7M_Sert_1_1',squre_size=4000,x1=26000,y1=11000,parent='HTR7',element='MUT',cell=1,rectangle=3,replicate=1)
HTR7_STIM_M_1_1_4 = ThnderSTORM2df('HTR7M_Sert_1_1',squre_size=4000,x1=26000,y1=31000,parent='HTR7',element='MUT',cell=1,rectangle=4,replicate=1)
HTR7_STIM_M_1_1_5 = ThnderSTORM2df('HTR7M_Sert_1_1',squre_size=4000,x1=30000,y1=33000,parent='HTR7',element='MUT',cell=1,rectangle=5,replicate=1)
HTR7_STIM_M_1_1_6 = ThnderSTORM2df('HTR7M_Sert_1_1',squre_size=4000,x1=34000,y1=40000,parent='HTR7',element='MUT',cell=1,rectangle=6,replicate=1)

HTR7_STIM_M_1_2_1 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=36000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=1)
HTR7_STIM_M_1_2_2 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=32000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=1)
HTR7_STIM_M_1_2_3 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=28000,parent='HTR7',element='MUT',cell=2,rectangle=3,replicate=1)
HTR7_STIM_M_1_2_4 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=24000,parent='HTR7',element='MUT',cell=2,rectangle=4,replicate=1)
HTR7_STIM_M_1_2_5 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=30000,y1=24000,parent='HTR7',element='MUT',cell=2,rectangle=5,replicate=1)
HTR7_STIM_M_1_2_6 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=30000,y1=28000,parent='HTR7',element='MUT',cell=2,rectangle=6,replicate=1)
HTR7_STIM_M_1_2_7 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=30000,y1=32000,parent='HTR7',element='MUT',cell=2,rectangle=7,replicate=1)
HTR7_STIM_M_1_2_8 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=10000,y1=12000,parent='HTR7',element='MUT',cell=2,rectangle=8,replicate=1)
HTR7_STIM_M_1_2_9 = ThnderSTORM2df('HTR7M_Sert_1_2',squre_size=4000,x1=6000,y1=15000,parent='HTR7',element='MUT',cell=2,rectangle=9,replicate=1)


HTR7_STIM_M_1_4_1 = ThnderSTORM2df('HTR7M_Sert_1_4',squre_size=4000,x1=20000,y1=24000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=1)
HTR7_STIM_M_1_4_2 = ThnderSTORM2df('HTR7M_Sert_1_4',squre_size=4000,x1=16000,y1=25000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=1)
HTR7_STIM_M_1_4_3 = ThnderSTORM2df('HTR7M_Sert_1_4',squre_size=4000,x1=11000,y1=18000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=1)
HTR7_STIM_M_1_4_4 = ThnderSTORM2df('HTR7M_Sert_1_4',squre_size=4000,x1=7000,y1=18000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=1)

HTR7_STIM_M_1_3_1 = ThnderSTORM2df('HTR7M_Sert_1_3',squre_size=4000,x1=19000,y1=10000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=1)
HTR7_STIM_M_1_3_2 = ThnderSTORM2df('HTR7M_Sert_1_3',squre_size=4000,x1=19000,y1=14000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=1)


HTR7_STIM_M_1_7_1 = ThnderSTORM2df('HTR7M_Sert_1_7',squre_size=4000,x1=23000,y1=37000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=1)
HTR7_STIM_M_1_7_2 = ThnderSTORM2df('HTR7M_Sert_1_7',squre_size=4000,x1=23000,y1=33000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=1)
HTR7_STIM_M_1_7_3 = ThnderSTORM2df('HTR7M_Sert_1_7',squre_size=4000,x1=23000,y1=29000,parent='HTR7',element='MUT',cell=7,rectangle=3,replicate=1)
HTR7_STIM_M_1_7_4 = ThnderSTORM2df('HTR7M_Sert_1_7',squre_size=4000,x1=10000,y1=19000,parent='HTR7',element='MUT',cell=7,rectangle=4,replicate=1)



HTR7_STIM_M_2_2_1 = ThnderSTORM2df('HTR7Mut_S_2_2',squre_size=4000,x1=23000,y1=19000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=2)
HTR7_STIM_M_2_2_2 = ThnderSTORM2df('HTR7Mut_S_2_2',squre_size=4000,x1=27000,y1=26000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=2)

HTR7_STIM_M_2_5_1 = ThnderSTORM2df('HTR7Mut_S_2_5',squre_size=4000,x1=17000,y1=12000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=2)
HTR7_STIM_M_2_5_2 = ThnderSTORM2df('HTR7Mut_S_2_5',squre_size=4000,x1=13000,y1=12000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=2)
HTR7_STIM_M_2_5_3 = ThnderSTORM2df('HTR7Mut_S_2_5',squre_size=4000,x1=13000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=2)
HTR7_STIM_M_2_5_4 = ThnderSTORM2df('HTR7Mut_S_2_5',squre_size=4000,x1=30000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=2)
HTR7_STIM_M_2_5_5 = ThnderSTORM2df('HTR7Mut_S_2_5',squre_size=4000,x1=34000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=2)
HTR7_STIM_M_2_5_6 = ThnderSTORM2df('HTR7Mut_S_2_5',squre_size=4000,x1=38000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=6,replicate=2)
HTR7_STIM_M_2_5_7 = ThnderSTORM2df('HTR7Mut_S_2_5',squre_size=4000,x1=40000,y1=20000,parent='HTR7',element='MUT',cell=5,rectangle=7,replicate=2)


HTR7_STIM_M_2_6_1 = ThnderSTORM2df('HTR7Mut_S_2_6',squre_size=4000,x1=15000,y1=25000,parent='HTR7',element='MUT',cell=6,rectangle=1,replicate=2)
HTR7_STIM_M_2_6_2 = ThnderSTORM2df('HTR7Mut_S_2_6',squre_size=4000,x1=19000,y1=25000,parent='HTR7',element='MUT',cell=6,rectangle=2,replicate=2)
HTR7_STIM_M_2_6_3 = ThnderSTORM2df('HTR7Mut_S_2_6',squre_size=4000,x1=19000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=3,replicate=2)
HTR7_STIM_M_2_6_4 = ThnderSTORM2df('HTR7Mut_S_2_6',squre_size=4000,x1=15000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=4,replicate=2)
HTR7_STIM_M_2_6_5 = ThnderSTORM2df('HTR7Mut_S_2_6',squre_size=4000,x1=35000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=5,replicate=2)
HTR7_STIM_M_2_6_6 = ThnderSTORM2df('HTR7Mut_S_2_6',squre_size=4000,x1=31000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=6,replicate=2)
HTR7_STIM_M_2_6_7 = ThnderSTORM2df('HTR7Mut_S_2_6',squre_size=4000,x1=40000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=7,replicate=2)


HTR7_STIM_M_2_7_1 = ThnderSTORM2df('HTR7Mut_S_2_7',squre_size=4000,x1=30000,y1=29000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=2)
HTR7_STIM_M_2_7_2 = ThnderSTORM2df('HTR7Mut_S_2_7',squre_size=4000,x1=27000,y1=25000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=2)
HTR7_STIM_M_2_7_3 = ThnderSTORM2df('HTR7Mut_S_2_7',squre_size=4000,x1=28000,y1=21000,parent='HTR7',element='MUT',cell=7,rectangle=3,replicate=2)
HTR7_STIM_M_2_7_4 = ThnderSTORM2df('HTR7Mut_S_2_7',squre_size=4000,x1=22000,y1=31000,parent='HTR7',element='MUT',cell=7,rectangle=4,replicate=2)


HTR7_STIM_M_2_9_1 = ThnderSTORM2df('HTR7Mut_S_2_9',squre_size=4000,x1=23000,y1=36000,parent='HTR7',element='MUT',cell=9,rectangle=1,replicate=2)
HTR7_STIM_M_2_9_2 = ThnderSTORM2df('HTR7Mut_S_2_9',squre_size=4000,x1=22000,y1=32000,parent='HTR7',element='MUT',cell=9,rectangle=2,replicate=2)

HTR7_STIM_M_2_9_3 = ThnderSTORM2df('HTR7Mut_S_2_9',squre_size=4000,x1=22000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=3,replicate=2)
HTR7_STIM_M_2_9_4 = ThnderSTORM2df('HTR7Mut_S_2_9',squre_size=4000,x1=26000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=4,replicate=2)
HTR7_STIM_M_2_9_5 = ThnderSTORM2df('HTR7Mut_S_2_9',squre_size=4000,x1=30000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=5,replicate=2)
HTR7_STIM_M_2_9_6 = ThnderSTORM2df('HTR7Mut_S_2_9',squre_size=4000,x1=34000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=6,replicate=2)
HTR7_STIM_M_2_9_7 = ThnderSTORM2df('HTR7Mut_S_2_9',squre_size=4000,x1=34000,y1=11000,parent='HTR7',element='MUT',cell=9,rectangle=7,replicate=2)


HTR7_STIM_M_2_11_3 = ThnderSTORM2df('HTR7Mut_S_2_11',squre_size=4000,x1=14000,y1=11000,parent='HTR7',element='MUT',cell=11,rectangle=1,replicate=2)
HTR7_STIM_M_2_11_4 = ThnderSTORM2df('HTR7Mut_S_2_11',squre_size=4000,x1=14000,y1=15000,parent='HTR7',element='MUT',cell=11,rectangle=2,replicate=2)
HTR7_STIM_M_2_11_5 = ThnderSTORM2df('HTR7Mut_S_2_11',squre_size=4000,x1=14000,y1=19000,parent='HTR7',element='MUT',cell=11,rectangle=3,replicate=2)
HTR7_STIM_M_2_11_6 = ThnderSTORM2df('HTR7Mut_S_2_11',squre_size=4000,x1=42000,y1=23000,parent='HTR7',element='MUT',cell=11,rectangle=4,replicate=2)
HTR7_STIM_M_2_11_7 = ThnderSTORM2df('HTR7Mut_S_2_11',squre_size=4000,x1=38000,y1=23000,parent='HTR7',element='MUT',cell=11,rectangle=5,replicate=2)

HTR7_STIM_1_1_1 = ThnderSTORM2df('HTR7_Sert_1_1',squre_size=4000,x1=35000,y1=13000,parent='HTR7',element='WT',cell=1,rectangle=1,replicate=1)
HTR7_STIM_1_1_2 = ThnderSTORM2df('HTR7_Sert_1_1',squre_size=4000,x1=27000,y1=11000,parent='HTR7',element='WT',cell=1,rectangle=2,replicate=1)
HTR7_STIM_1_1_3 = ThnderSTORM2df('HTR7_Sert_1_1',squre_size=4000,x1=22000,y1=21000,parent='HTR7',element='WT',cell=1,rectangle=3,replicate=1)



HTR7_STIM_1_2_1  = ThnderSTORM2df('HTR7_Sert_1_2',squre_size=4000,x1=40000,y1=42000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=1)
HTR7_STIM_1_2_2  = ThnderSTORM2df('HTR7_Sert_1_2',squre_size=4000,x1=36000,y1=43000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=1)
HTR7_STIM_1_2_3  = ThnderSTORM2df('HTR7_Sert_1_2',squre_size=4000,x1=20000,y1=13000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=1)

HTR7_STIM_1_4_1  = ThnderSTORM2df('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=1)
HTR7_STIM_1_4_2  = ThnderSTORM2df('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=17000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=1)
HTR7_STIM_1_4_3  = ThnderSTORM2df('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=21000,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=1)
HTR7_STIM_1_4_4  = ThnderSTORM2df('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=25000,parent='HTR7',element='WT',cell=4,rectangle=4,replicate=1)
HTR7_STIM_1_4_5  = ThnderSTORM2df('HTR7_Sert_1_4',squre_size=4000,x1=12000,y1=25000,parent='HTR7',element='WT',cell=4,rectangle=5,replicate=1)
HTR7_STIM_1_4_6  = ThnderSTORM2df('HTR7_Sert_1_4',squre_size=4000,x1=27000,y1=36000,parent='HTR7',element='WT',cell=4,rectangle=6,replicate=1)
HTR7_STIM_1_4_7  = ThnderSTORM2df('HTR7_Sert_1_4',squre_size=4000,x1=34000,y1=39000,parent='HTR7',element='WT',cell=4,rectangle=7,replicate=1)


HTR7_STIM_1_6_1  = ThnderSTORM2df('HTR7_Sert_1_6',squre_size=4000,x1=10000,y1=29600,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=1)
HTR7_STIM_1_6_2  = ThnderSTORM2df('HTR7_Sert_1_6',squre_size=4000,x1=6000,y1=29600,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=1)
HTR7_STIM_1_6_3  = ThnderSTORM2df('HTR7_Sert_1_6',squre_size=4000,x1=6000,y1=25600,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=1)
HTR7_STIM_1_6_4  = ThnderSTORM2df('HTR7_Sert_1_6',squre_size=4000,x1=12000,y1=29600,parent='HTR7',element='WT',cell=6,rectangle=4,replicate=1)
HTR7_STIM_1_6_5  = ThnderSTORM2df('HTR7_Sert_1_6',squre_size=4000,x1=36700,y1=30600,parent='HTR7',element='WT',cell=6,rectangle=5,replicate=1)
HTR7_STIM_1_6_6  = ThnderSTORM2df('HTR7_Sert_1_6',squre_size=4000,x1=41000,y1=45600,parent='HTR7',element='WT',cell=6,rectangle=6,replicate=1)

HTR7_STIM_1_7_1  = ThnderSTORM2df('HTR7_Sert_1_7',squre_size=4000,x1=10000,y1=13600,parent='HTR7',element='WT',cell=7,rectangle=1,replicate=1)
HTR7_STIM_1_7_2  = ThnderSTORM2df('HTR7_Sert_1_7',squre_size=4000,x1=14000,y1=13600,parent='HTR7',element='WT', cell=7,rectangle=2,replicate=1)
HTR7_STIM_1_7_3  = ThnderSTORM2df('HTR7_Sert_1_7',squre_size=4000,x1=18000,y1=17600,parent='HTR7',element='WT', cell=7,rectangle=3,replicate=1)
HTR7_STIM_1_7_4  = ThnderSTORM2df('HTR7_Sert_1_7',squre_size=4000,x1=22000,y1=17600,parent='HTR7',element='WT',cell=7,rectangle=4,replicate=1)
HTR7_STIM_1_7_5  = ThnderSTORM2df('HTR7_Sert_1_7',squre_size=4000,x1=26000,y1=17600,parent='HTR7',element='WT',cell=7,rectangle=5,replicate=1)
HTR7_STIM_1_7_6  = ThnderSTORM2df('HTR7_Sert_1_7',squre_size=4000,x1=28000,y1=21600,parent='HTR7',element='WT',cell=7,rectangle=6,replicate=1)
HTR7_STIM_1_7_7  = ThnderSTORM2df('HTR7_Sert_1_7',squre_size=4000,x1=32000,y1=21600,parent='HTR7',element='WT',cell=7,rectangle=7,replicate=1)


HTR7_STIM_2_1_1  = ThnderSTORM2df('HTR7_S_2_1',squre_size=4000,x1=4000,y1=26600,parent='HTR7',element='WT', cell=1,rectangle=1,replicate=2)
HTR7_STIM_2_1_2  = ThnderSTORM2df('HTR7_S_2_1',squre_size=4000,x1=4000,y1=30600,parent='HTR7',element='WT', cell=1,rectangle=2,replicate=2)
HTR7_STIM_2_1_3  = ThnderSTORM2df('HTR7_S_2_1',squre_size=4000,x1=8000,y1=30600,parent='HTR7',element='WT', cell=1,rectangle=3,replicate=2)
HTR7_STIM_2_1_4  = ThnderSTORM2df('HTR7_S_2_1',squre_size=4000,x1=8000,y1=34600,parent='HTR7',element='WT', cell=1,rectangle=4,replicate=2)
HTR7_STIM_2_1_5  = ThnderSTORM2df('HTR7_S_2_1',squre_size=4000,x1=32000,y1=38600,parent='HTR7',element='WT', cell=1,rectangle=5,replicate=2)
HTR7_STIM_2_1_6  = ThnderSTORM2df('HTR7_S_2_1',squre_size=4000,x1=36000,y1=38600,parent='HTR7',element='WT', cell=1,rectangle=6,replicate=2)
HTR7_STIM_2_1_7  = ThnderSTORM2df('HTR7_S_2_1',squre_size=4000,x1=40000,y1=38600,parent='HTR7',element='WT', cell=1,rectangle=7,replicate=2)

HTR7_STIM_2_2_1  = ThnderSTORM2df('HTR7_S_2_2',squre_size=4000,x1=21000,y1=30000,parent='HTR7',element='WT', cell=2,rectangle=1,replicate=2)


HTR7_STIM_2_3_1  = ThnderSTORM2df('HTR7_S_2_3',squre_size=4000,x1=26000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=1,replicate=2)
HTR7_STIM_2_3_2  = ThnderSTORM2df('HTR7_S_2_3',squre_size=4000,x1=30000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=2,replicate=2)
HTR7_STIM_2_3_3  = ThnderSTORM2df('HTR7_S_2_3',squre_size=4000,x1=34000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=3,replicate=2)
HTR7_STIM_2_3_4  = ThnderSTORM2df('HTR7_S_2_3',squre_size=4000,x1=38000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=4,replicate=2)
HTR7_STIM_2_3_5  = ThnderSTORM2df('HTR7_S_2_3',squre_size=4000,x1=39000,y1=21000,parent='HTR7',element='WT',cell=3,rectangle=5,replicate=2)



HTR7_STIM_2_4_1  = ThnderSTORM2df('HTR7_S_2_4',squre_size=4000,x1=23000,y1=20000,parent='HTR7',element='WT', cell=4,rectangle=1,replicate=2)
HTR7_STIM_2_4_2  = ThnderSTORM2df('HTR7_S_2_4',squre_size=4000,x1=29000,y1=14000,parent='HTR7',element='WT', cell=4,rectangle=2,replicate=2)


HTR7_STIM_2_6_1  = ThnderSTORM2df('HTR7_S_2_6',squre_size=4000,x1=22000,y1=16000,parent='HTR7',element='WT', cell=6,rectangle=1,replicate=2)
HTR7_STIM_2_6_2  = ThnderSTORM2df('HTR7_S_2_6',squre_size=4000,x1=22000,y1=20000,parent='HTR7',element='WT', cell=6,rectangle=2,replicate=2)
HTR7_STIM_2_6_3  = ThnderSTORM2df('HTR7_S_2_6',squre_size=4000,x1=22000,y1=24000,parent='HTR7',element='WT', cell=6,rectangle=3,replicate=2)
HTR7_STIM_2_6_4  = ThnderSTORM2df('HTR7_S_2_6',squre_size=4000,x1=26000,y1=24000,parent='HTR7',element='WT', cell=6,rectangle=4,replicate=2)
HTR7_STIM_2_6_5  = ThnderSTORM2df('HTR7_S_2_6',squre_size=4000,x1=30000,y1=24000,parent='HTR7',element='WT', cell=6, rectangle=5,replicate=2)
HTR7_STIM_2_6_6  = ThnderSTORM2df('HTR7_S_2_6',squre_size=4000,x1=34000,y1=24000,parent='HTR7',element='WT', cell=6, rectangle=6,replicate=2)



HTR7_STIM_2_8_1  = ThnderSTORM2df('HTR7_S_2_8',squre_size=4000,x1=30000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=1,replicate=2)
HTR7_STIM_2_8_2  = ThnderSTORM2df('HTR7_S_2_8',squre_size=4000,x1=26000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=2,replicate=2)
HTR7_STIM_2_8_3  = ThnderSTORM2df('HTR7_S_2_8',squre_size=4000,x1=22000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=3,replicate=2)
HTR7_STIM_2_8_4  = ThnderSTORM2df('HTR7_S_2_8',squre_size=4000,x1=18000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=4,replicate=2)
HTR7_STIM_2_8_5  = ThnderSTORM2df('HTR7_S_2_8',squre_size=4000,x1=20000,y1=20000,parent='HTR7',element='WT', cell=8, rectangle=5,replicate=2)
HTR7_STIM_2_8_6  = ThnderSTORM2df('HTR7_S_2_8',squre_size=4000,x1=22000,y1=16000,parent='HTR7',element='WT', cell=8, rectangle=6,replicate=2)


HTR7_STIM_2_10_1  = ThnderSTORM2df('HTR7_S_2_10',squre_size=4000,x1=32000,y1=14000,parent='HTR7',element='WT', cell=10,rectangle=1,replicate=2)
HTR7_STIM_2_10_2  = ThnderSTORM2df('HTR7_S_2_10',squre_size=4000,x1=32000,y1=18000,parent='HTR7',element='WT', cell=10,rectangle=2,replicate=2)
HTR7_STIM_2_10_3  = ThnderSTORM2df('HTR7_S_2_10',squre_size=4000,x1=32000,y1=22000,parent='HTR7',element='WT', cell=10,rectangle=3,replicate=2)
HTR7_STIM_2_10_4  = ThnderSTORM2df('HTR7_S_2_10',squre_size=4000,x1=36000,y1=22000,parent='HTR7',element='WT', cell=10,rectangle=4,replicate=2)
HTR7_STIM_2_10_5  = ThnderSTORM2df('HTR7_S_2_10',squre_size=4000,x1=24000,y1=26000,parent='HTR7',element='WT', cell=10, rectangle=5,replicate=2)
HTR7_STIM_2_10_6  = ThnderSTORM2df('HTR7_S_2_10',squre_size=4000,x1=24000,y1=6000,parent='HTR7',element='WT', cell=10, rectangle=6,replicate=2)


HTR7_STIM_2_11_1  = ThnderSTORM2df('HTR7_S_2_11',squre_size=4000,x1=24000,y1=14000,parent='HTR7',element='WT', cell=11,rectangle=1,replicate=2)
HTR7_STIM_2_11_2  = ThnderSTORM2df('HTR7_S_2_11',squre_size=4000,x1=24000,y1=18000,parent='HTR7',element='WT', cell=11,rectangle=2,replicate=2)
HTR7_STIM_2_11_3  = ThnderSTORM2df('HTR7_S_2_11',squre_size=4000,x1=23000,y1=22000,parent='HTR7',element='WT', cell=11,rectangle=3,replicate=2)
HTR7_STIM_2_11_4  = ThnderSTORM2df('HTR7_S_2_11',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='WT', cell=11,rectangle=4,replicate=2)
HTR7_STIM_2_11_5  = ThnderSTORM2df('HTR7_S_2_11',squre_size=4000,x1=19000,y1=26000,parent='HTR7',element='WT', cell=11, rectangle=5,replicate=2)
HTR7_STIM_2_11_6  = ThnderSTORM2df('HTR7_S_2_11',squre_size=4000,x1=15000,y1=26000,parent='HTR7',element='WT',  cell=11,rectangle=6,replicate=2)

HTR7_STIM_2_12_1  = ThnderSTORM2df('HTR7_S_2_12',squre_size=4000,x1=25000,y1=25000,parent='HTR7',element='WT', cell=12,rectangle=1,replicate=2)
HTR7_STIM_2_12_2  = ThnderSTORM2df('HTR7_S_2_12',squre_size=4000,x1=25000,y1=21000,parent='HTR7',element='WT', cell=12,rectangle=2,replicate=2)
HTR7_STIM_2_12_3  = ThnderSTORM2df('HTR7_S_2_12',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='WT', cell=12,rectangle=3,replicate=2)
HTR7_STIM_2_12_4  = ThnderSTORM2df('HTR7_S_2_12',squre_size=4000,x1=21000,y1=17000,parent='HTR7',element='WT', cell=12,rectangle=4,replicate=2)
HTR7_STIM_2_12_5  = ThnderSTORM2df('HTR7_S_2_12',squre_size=4000,x1=21000,y1=13000,parent='HTR7',element='WT', cell=12,rectangle=5,replicate=2)
HTR7_STIM_2_12_6  = ThnderSTORM2df('HTR7_S_2_12',squre_size=4000,x1=17000,y1=15000,parent='HTR7',element='WT',  cell=12,rectangle=6,replicate=2)



In [ ]:
HTR7_STIM_M_1_1_1 = voro2cluster('HTR7M_Sert_1_1',squre_size=4000,x1=19000,y1=15000,parent='HTR7',element='MUT',cell=1,rectangle=1,replicate=1)
HTR7_STIM_M_1_1_2 = voro2cluster('HTR7M_Sert_1_1',squre_size=4000,x1=18000,y1=11000,parent='HTR7',element='MUT',cell=1,rectangle=2,replicate=1)
HTR7_STIM_M_1_1_3 = voro2cluster('HTR7M_Sert_1_1',squre_size=4000,x1=26000,y1=11000,parent='HTR7',element='MUT',cell=1,rectangle=3,replicate=1)
HTR7_STIM_M_1_1_4 = voro2cluster('HTR7M_Sert_1_1',squre_size=4000,x1=26000,y1=31000,parent='HTR7',element='MUT',cell=1,rectangle=4,replicate=1)
HTR7_STIM_M_1_1_5 = voro2cluster('HTR7M_Sert_1_1',squre_size=4000,x1=30000,y1=33000,parent='HTR7',element='MUT',cell=1,rectangle=5,replicate=1)
HTR7_STIM_M_1_1_6 = voro2cluster('HTR7M_Sert_1_1',squre_size=4000,x1=34000,y1=40000,parent='HTR7',element='MUT',cell=1,rectangle=6,replicate=1)

HTR7_STIM_M_1_2_1 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=36000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=1)
HTR7_STIM_M_1_2_2 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=32000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=1)
HTR7_STIM_M_1_2_3 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=28000,parent='HTR7',element='MUT',cell=2,rectangle=3,replicate=1)
HTR7_STIM_M_1_2_4 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=34000,y1=24000,parent='HTR7',element='MUT',cell=2,rectangle=4,replicate=1)
HTR7_STIM_M_1_2_5 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=30000,y1=24000,parent='HTR7',element='MUT',cell=2,rectangle=5,replicate=1)
HTR7_STIM_M_1_2_6 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=30000,y1=28000,parent='HTR7',element='MUT',cell=2,rectangle=6,replicate=1)
HTR7_STIM_M_1_2_7 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=30000,y1=32000,parent='HTR7',element='MUT',cell=2,rectangle=7,replicate=1)
HTR7_STIM_M_1_2_8 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=10000,y1=12000,parent='HTR7',element='MUT',cell=2,rectangle=8,replicate=1)
HTR7_STIM_M_1_2_9 = voro2cluster('HTR7M_Sert_1_2',squre_size=4000,x1=6000,y1=15000,parent='HTR7',element='MUT',cell=2,rectangle=9,replicate=1)


HTR7_STIM_M_1_4_1 = voro2cluster('HTR7M_Sert_1_4',squre_size=4000,x1=20000,y1=24000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=1)
HTR7_STIM_M_1_4_2 = voro2cluster('HTR7M_Sert_1_4',squre_size=4000,x1=16000,y1=25000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=1)
HTR7_STIM_M_1_4_3 = voro2cluster('HTR7M_Sert_1_4',squre_size=4000,x1=11000,y1=18000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=1)
HTR7_STIM_M_1_4_4 = voro2cluster('HTR7M_Sert_1_4',squre_size=4000,x1=7000,y1=18000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=1)

HTR7_STIM_M_1_3_1 = voro2cluster('HTR7M_Sert_1_3',squre_size=4000,x1=19000,y1=10000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=1)
HTR7_STIM_M_1_3_2 = voro2cluster('HTR7M_Sert_1_3',squre_size=4000,x1=19000,y1=14000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=1)


HTR7_STIM_M_1_7_1 = voro2cluster('HTR7M_Sert_1_7',squre_size=4000,x1=23000,y1=37000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=1)
HTR7_STIM_M_1_7_2 = voro2cluster('HTR7M_Sert_1_7',squre_size=4000,x1=23000,y1=33000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=1)
HTR7_STIM_M_1_7_3 = voro2cluster('HTR7M_Sert_1_7',squre_size=4000,x1=23000,y1=29000,parent='HTR7',element='MUT',cell=7,rectangle=3,replicate=1)
HTR7_STIM_M_1_7_4 = voro2cluster('HTR7M_Sert_1_7',squre_size=4000,x1=10000,y1=19000,parent='HTR7',element='MUT',cell=7,rectangle=4,replicate=1)



HTR7_STIM_M_2_2_1 = voro2cluster('HTR7Mut_S_2_2',squre_size=4000,x1=23000,y1=19000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=2)
HTR7_STIM_M_2_2_2 = voro2cluster('HTR7Mut_S_2_2',squre_size=4000,x1=27000,y1=26000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=2)

HTR7_STIM_M_2_5_1 = voro2cluster('HTR7Mut_S_2_5',squre_size=4000,x1=17000,y1=12000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=2)
HTR7_STIM_M_2_5_2 = voro2cluster('HTR7Mut_S_2_5',squre_size=4000,x1=13000,y1=12000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=2)
HTR7_STIM_M_2_5_3 = voro2cluster('HTR7Mut_S_2_5',squre_size=4000,x1=13000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=2)
HTR7_STIM_M_2_5_4 = voro2cluster('HTR7Mut_S_2_5',squre_size=4000,x1=30000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=2)
HTR7_STIM_M_2_5_5 = voro2cluster('HTR7Mut_S_2_5',squre_size=4000,x1=34000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=2)
HTR7_STIM_M_2_5_6 = voro2cluster('HTR7Mut_S_2_5',squre_size=4000,x1=38000,y1=16000,parent='HTR7',element='MUT',cell=5,rectangle=6,replicate=2)
HTR7_STIM_M_2_5_7 = voro2cluster('HTR7Mut_S_2_5',squre_size=4000,x1=40000,y1=20000,parent='HTR7',element='MUT',cell=5,rectangle=7,replicate=2)


HTR7_STIM_M_2_6_1 = voro2cluster('HTR7Mut_S_2_6',squre_size=4000,x1=15000,y1=25000,parent='HTR7',element='MUT',cell=6,rectangle=1,replicate=2)
HTR7_STIM_M_2_6_2 = voro2cluster('HTR7Mut_S_2_6',squre_size=4000,x1=19000,y1=25000,parent='HTR7',element='MUT',cell=6,rectangle=2,replicate=2)
HTR7_STIM_M_2_6_3 = voro2cluster('HTR7Mut_S_2_6',squre_size=4000,x1=19000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=3,replicate=2)
HTR7_STIM_M_2_6_4 = voro2cluster('HTR7Mut_S_2_6',squre_size=4000,x1=15000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=4,replicate=2)
HTR7_STIM_M_2_6_5 = voro2cluster('HTR7Mut_S_2_6',squre_size=4000,x1=35000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=5,replicate=2)
HTR7_STIM_M_2_6_6 = voro2cluster('HTR7Mut_S_2_6',squre_size=4000,x1=31000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=6,replicate=2)
HTR7_STIM_M_2_6_7 = voro2cluster('HTR7Mut_S_2_6',squre_size=4000,x1=40000,y1=29000,parent='HTR7',element='MUT',cell=6,rectangle=7,replicate=2)


HTR7_STIM_M_2_7_1 = voro2cluster('HTR7Mut_S_2_7',squre_size=4000,x1=30000,y1=29000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=2)
HTR7_STIM_M_2_7_2 = voro2cluster('HTR7Mut_S_2_7',squre_size=4000,x1=27000,y1=25000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=2)
HTR7_STIM_M_2_7_3 = voro2cluster('HTR7Mut_S_2_7',squre_size=4000,x1=28000,y1=21000,parent='HTR7',element='MUT',cell=7,rectangle=3,replicate=2)
HTR7_STIM_M_2_7_4 = voro2cluster('HTR7Mut_S_2_7',squre_size=4000,x1=22000,y1=31000,parent='HTR7',element='MUT',cell=7,rectangle=4,replicate=2)


HTR7_STIM_M_2_9_1 = voro2cluster('HTR7Mut_S_2_9',squre_size=4000,x1=23000,y1=36000,parent='HTR7',element='MUT',cell=9,rectangle=1,replicate=2)
HTR7_STIM_M_2_9_2 = voro2cluster('HTR7Mut_S_2_9',squre_size=4000,x1=22000,y1=32000,parent='HTR7',element='MUT',cell=9,rectangle=2,replicate=2)

HTR7_STIM_M_2_9_3 = voro2cluster('HTR7Mut_S_2_9',squre_size=4000,x1=22000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=3,replicate=2)
HTR7_STIM_M_2_9_4 = voro2cluster('HTR7Mut_S_2_9',squre_size=4000,x1=26000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=4,replicate=2)
HTR7_STIM_M_2_9_5 = voro2cluster('HTR7Mut_S_2_9',squre_size=4000,x1=30000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=5,replicate=2)
HTR7_STIM_M_2_9_6 = voro2cluster('HTR7Mut_S_2_9',squre_size=4000,x1=34000,y1=15000,parent='HTR7',element='MUT',cell=9,rectangle=6,replicate=2)
HTR7_STIM_M_2_9_7 = voro2cluster('HTR7Mut_S_2_9',squre_size=4000,x1=34000,y1=11000,parent='HTR7',element='MUT',cell=9,rectangle=7,replicate=2)


HTR7_STIM_M_2_11_3 = voro2cluster('HTR7Mut_S_2_11',squre_size=4000,x1=14000,y1=11000,parent='HTR7',element='MUT',cell=11,rectangle=1,replicate=2)
HTR7_STIM_M_2_11_4 = voro2cluster('HTR7Mut_S_2_11',squre_size=4000,x1=14000,y1=15000,parent='HTR7',element='MUT',cell=11,rectangle=2,replicate=2)
HTR7_STIM_M_2_11_5 = voro2cluster('HTR7Mut_S_2_11',squre_size=4000,x1=14000,y1=19000,parent='HTR7',element='MUT',cell=11,rectangle=3,replicate=2)
HTR7_STIM_M_2_11_6 = voro2cluster('HTR7Mut_S_2_11',squre_size=4000,x1=42000,y1=23000,parent='HTR7',element='MUT',cell=11,rectangle=4,replicate=2)
HTR7_STIM_M_2_11_7 = voro2cluster('HTR7Mut_S_2_11',squre_size=4000,x1=38000,y1=23000,parent='HTR7',element='MUT',cell=11,rectangle=5,replicate=2)

HTR7_STIM_1_1_1 = voro2cluster('HTR7_Sert_1_1',squre_size=4000,x1=35000,y1=13000,parent='HTR7',element='WT',cell=1,rectangle=1,replicate=1)
HTR7_STIM_1_1_2 = voro2cluster('HTR7_Sert_1_1',squre_size=4000,x1=27000,y1=11000,parent='HTR7',element='WT',cell=1,rectangle=2,replicate=1)
HTR7_STIM_1_1_3 = voro2cluster('HTR7_Sert_1_1',squre_size=4000,x1=22000,y1=21000,parent='HTR7',element='WT',cell=1,rectangle=3,replicate=1)



HTR7_STIM_1_2_1  = voro2cluster('HTR7_Sert_1_2',squre_size=4000,x1=40000,y1=42000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=1)
HTR7_STIM_1_2_2  = voro2cluster('HTR7_Sert_1_2',squre_size=4000,x1=36000,y1=43000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=1)
HTR7_STIM_1_2_3  = voro2cluster('HTR7_Sert_1_2',squre_size=4000,x1=20000,y1=13000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=1)

HTR7_STIM_1_4_1  = voro2cluster('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=1)
HTR7_STIM_1_4_2  = voro2cluster('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=17000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=1)
HTR7_STIM_1_4_3  = voro2cluster('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=21000,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=1)
HTR7_STIM_1_4_4  = voro2cluster('HTR7_Sert_1_4',squre_size=4000,x1=8000,y1=25000,parent='HTR7',element='WT',cell=4,rectangle=4,replicate=1)
HTR7_STIM_1_4_5  = voro2cluster('HTR7_Sert_1_4',squre_size=4000,x1=12000,y1=25000,parent='HTR7',element='WT',cell=4,rectangle=5,replicate=1)
HTR7_STIM_1_4_6  = voro2cluster('HTR7_Sert_1_4',squre_size=4000,x1=27000,y1=36000,parent='HTR7',element='WT',cell=4,rectangle=6,replicate=1)
HTR7_STIM_1_4_7  = voro2cluster('HTR7_Sert_1_4',squre_size=4000,x1=34000,y1=39000,parent='HTR7',element='WT',cell=4,rectangle=7,replicate=1)


HTR7_STIM_1_6_1  = voro2cluster('HTR7_Sert_1_6',squre_size=4000,x1=10000,y1=29600,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=1)
HTR7_STIM_1_6_2  = voro2cluster('HTR7_Sert_1_6',squre_size=4000,x1=6000,y1=29600,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=1)
HTR7_STIM_1_6_3  = voro2cluster('HTR7_Sert_1_6',squre_size=4000,x1=6000,y1=25600,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=1)
HTR7_STIM_1_6_4  = voro2cluster('HTR7_Sert_1_6',squre_size=4000,x1=12000,y1=29600,parent='HTR7',element='WT',cell=6,rectangle=4,replicate=1)
HTR7_STIM_1_6_5  = voro2cluster('HTR7_Sert_1_6',squre_size=4000,x1=36700,y1=30600,parent='HTR7',element='WT',cell=6,rectangle=5,replicate=1)
HTR7_STIM_1_6_6  = voro2cluster('HTR7_Sert_1_6',squre_size=4000,x1=41000,y1=45600,parent='HTR7',element='WT',cell=6,rectangle=6,replicate=1)

HTR7_STIM_1_7_1  = voro2cluster('HTR7_Sert_1_7',squre_size=4000,x1=10000,y1=13600,parent='HTR7',element='WT',cell=7,rectangle=1,replicate=1)
HTR7_STIM_1_7_2  = voro2cluster('HTR7_Sert_1_7',squre_size=4000,x1=14000,y1=13600,parent='HTR7',element='WT', cell=7,rectangle=2,replicate=1)
HTR7_STIM_1_7_3  = voro2cluster('HTR7_Sert_1_7',squre_size=4000,x1=18000,y1=17600,parent='HTR7',element='WT', cell=7,rectangle=3,replicate=1)
HTR7_STIM_1_7_4  = voro2cluster('HTR7_Sert_1_7',squre_size=4000,x1=22000,y1=17600,parent='HTR7',element='WT',cell=7,rectangle=4,replicate=1)
HTR7_STIM_1_7_5  = voro2cluster('HTR7_Sert_1_7',squre_size=4000,x1=26000,y1=17600,parent='HTR7',element='WT',cell=7,rectangle=5,replicate=1)
HTR7_STIM_1_7_6  = voro2cluster('HTR7_Sert_1_7',squre_size=4000,x1=28000,y1=21600,parent='HTR7',element='WT',cell=7,rectangle=6,replicate=1)
HTR7_STIM_1_7_7  = voro2cluster('HTR7_Sert_1_7',squre_size=4000,x1=32000,y1=21600,parent='HTR7',element='WT',cell=7,rectangle=7,replicate=1)


HTR7_STIM_2_1_1  = voro2cluster('HTR7_S_2_1',squre_size=4000,x1=4000,y1=26600,parent='HTR7',element='WT', cell=1,rectangle=1,replicate=2)
HTR7_STIM_2_1_2  = voro2cluster('HTR7_S_2_1',squre_size=4000,x1=4000,y1=30600,parent='HTR7',element='WT', cell=1,rectangle=2,replicate=2)
HTR7_STIM_2_1_3  = voro2cluster('HTR7_S_2_1',squre_size=4000,x1=8000,y1=30600,parent='HTR7',element='WT', cell=1,rectangle=3,replicate=2)
HTR7_STIM_2_1_4  = voro2cluster('HTR7_S_2_1',squre_size=4000,x1=8000,y1=34600,parent='HTR7',element='WT', cell=1,rectangle=4,replicate=2)
HTR7_STIM_2_1_5  = voro2cluster('HTR7_S_2_1',squre_size=4000,x1=32000,y1=38600,parent='HTR7',element='WT', cell=1,rectangle=5,replicate=2)
HTR7_STIM_2_1_6  = voro2cluster('HTR7_S_2_1',squre_size=4000,x1=36000,y1=38600,parent='HTR7',element='WT', cell=1,rectangle=6,replicate=2)
HTR7_STIM_2_1_7  = voro2cluster('HTR7_S_2_1',squre_size=4000,x1=40000,y1=38600,parent='HTR7',element='WT', cell=1,rectangle=7,replicate=2)

HTR7_STIM_2_2_1  = voro2cluster('HTR7_S_2_2',squre_size=4000,x1=21000,y1=30000,parent='HTR7',element='WT', cell=2,rectangle=1,replicate=2)


HTR7_STIM_2_3_1  = voro2cluster('HTR7_S_2_3',squre_size=4000,x1=26000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=1,replicate=2)
HTR7_STIM_2_3_2  = voro2cluster('HTR7_S_2_3',squre_size=4000,x1=30000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=2,replicate=2)
HTR7_STIM_2_3_3  = voro2cluster('HTR7_S_2_3',squre_size=4000,x1=34000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=3,replicate=2)
HTR7_STIM_2_3_4  = voro2cluster('HTR7_S_2_3',squre_size=4000,x1=38000,y1=17000,parent='HTR7',element='WT', cell=3,rectangle=4,replicate=2)
HTR7_STIM_2_3_5  = voro2cluster('HTR7_S_2_3',squre_size=4000,x1=39000,y1=21000,parent='HTR7',element='WT',cell=3,rectangle=5,replicate=2)



HTR7_STIM_2_4_1  = voro2cluster('HTR7_S_2_4',squre_size=4000,x1=23000,y1=20000,parent='HTR7',element='WT', cell=4,rectangle=1,replicate=2)
HTR7_STIM_2_4_2  = voro2cluster('HTR7_S_2_4',squre_size=4000,x1=29000,y1=14000,parent='HTR7',element='WT', cell=4,rectangle=2,replicate=2)


HTR7_STIM_2_6_1  = voro2cluster('HTR7_S_2_6',squre_size=4000,x1=22000,y1=16000,parent='HTR7',element='WT', cell=6,rectangle=1,replicate=2)
HTR7_STIM_2_6_2  = voro2cluster('HTR7_S_2_6',squre_size=4000,x1=22000,y1=20000,parent='HTR7',element='WT', cell=6,rectangle=2,replicate=2)
HTR7_STIM_2_6_3  = voro2cluster('HTR7_S_2_6',squre_size=4000,x1=22000,y1=24000,parent='HTR7',element='WT', cell=6,rectangle=3,replicate=2)
HTR7_STIM_2_6_4  = voro2cluster('HTR7_S_2_6',squre_size=4000,x1=26000,y1=24000,parent='HTR7',element='WT', cell=6,rectangle=4,replicate=2)
HTR7_STIM_2_6_5  = voro2cluster('HTR7_S_2_6',squre_size=4000,x1=30000,y1=24000,parent='HTR7',element='WT', cell=6, rectangle=5,replicate=2)
HTR7_STIM_2_6_6  = voro2cluster('HTR7_S_2_6',squre_size=4000,x1=34000,y1=24000,parent='HTR7',element='WT', cell=6, rectangle=6,replicate=2)



HTR7_STIM_2_8_1  = voro2cluster('HTR7_S_2_8',squre_size=4000,x1=30000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=1,replicate=2)
HTR7_STIM_2_8_2  = voro2cluster('HTR7_S_2_8',squre_size=4000,x1=26000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=2,replicate=2)
HTR7_STIM_2_8_3  = voro2cluster('HTR7_S_2_8',squre_size=4000,x1=22000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=3,replicate=2)
HTR7_STIM_2_8_4  = voro2cluster('HTR7_S_2_8',squre_size=4000,x1=18000,y1=24000,parent='HTR7',element='WT', cell=8,rectangle=4,replicate=2)
HTR7_STIM_2_8_5  = voro2cluster('HTR7_S_2_8',squre_size=4000,x1=20000,y1=20000,parent='HTR7',element='WT', cell=8, rectangle=5,replicate=2)
HTR7_STIM_2_8_6  = voro2cluster('HTR7_S_2_8',squre_size=4000,x1=22000,y1=16000,parent='HTR7',element='WT', cell=8, rectangle=6,replicate=2)


HTR7_STIM_2_10_1  = voro2cluster('HTR7_S_2_10',squre_size=4000,x1=32000,y1=14000,parent='HTR7',element='WT', cell=10,rectangle=1,replicate=2)
HTR7_STIM_2_10_2  = voro2cluster('HTR7_S_2_10',squre_size=4000,x1=32000,y1=18000,parent='HTR7',element='WT', cell=10,rectangle=2,replicate=2)
HTR7_STIM_2_10_3  = voro2cluster('HTR7_S_2_10',squre_size=4000,x1=32000,y1=22000,parent='HTR7',element='WT', cell=10,rectangle=3,replicate=2)
HTR7_STIM_2_10_4  = voro2cluster('HTR7_S_2_10',squre_size=4000,x1=36000,y1=22000,parent='HTR7',element='WT', cell=10,rectangle=4,replicate=2)
HTR7_STIM_2_10_5  = voro2cluster('HTR7_S_2_10',squre_size=4000,x1=24000,y1=26000,parent='HTR7',element='WT', cell=10, rectangle=5,replicate=2)
HTR7_STIM_2_10_6  = voro2cluster('HTR7_S_2_10',squre_size=4000,x1=24000,y1=6000,parent='HTR7',element='WT', cell=10, rectangle=6,replicate=2)


HTR7_STIM_2_11_1  = voro2cluster('HTR7_S_2_11',squre_size=4000,x1=24000,y1=14000,parent='HTR7',element='WT', cell=11,rectangle=1,replicate=2)
HTR7_STIM_2_11_2  = voro2cluster('HTR7_S_2_11',squre_size=4000,x1=24000,y1=18000,parent='HTR7',element='WT', cell=11,rectangle=2,replicate=2)
HTR7_STIM_2_11_3  = voro2cluster('HTR7_S_2_11',squre_size=4000,x1=23000,y1=22000,parent='HTR7',element='WT', cell=11,rectangle=3,replicate=2)
HTR7_STIM_2_11_4  = voro2cluster('HTR7_S_2_11',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='WT', cell=11,rectangle=4,replicate=2)
HTR7_STIM_2_11_5  = voro2cluster('HTR7_S_2_11',squre_size=4000,x1=19000,y1=26000,parent='HTR7',element='WT', cell=11, rectangle=5,replicate=2)
HTR7_STIM_2_11_6  = voro2cluster('HTR7_S_2_11',squre_size=4000,x1=15000,y1=26000,parent='HTR7',element='WT',  cell=11,rectangle=6,replicate=2)

HTR7_STIM_2_12_1  = voro2cluster('HTR7_S_2_12',squre_size=4000,x1=25000,y1=25000,parent='HTR7',element='WT', cell=12,rectangle=1,replicate=2)
HTR7_STIM_2_12_2  = voro2cluster('HTR7_S_2_12',squre_size=4000,x1=25000,y1=21000,parent='HTR7',element='WT', cell=12,rectangle=2,replicate=2)
HTR7_STIM_2_12_3  = voro2cluster('HTR7_S_2_12',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='WT', cell=12,rectangle=3,replicate=2)
HTR7_STIM_2_12_4  = voro2cluster('HTR7_S_2_12',squre_size=4000,x1=21000,y1=17000,parent='HTR7',element='WT', cell=12,rectangle=4,replicate=2)
HTR7_STIM_2_12_5  = voro2cluster('HTR7_S_2_12',squre_size=4000,x1=21000,y1=13000,parent='HTR7',element='WT', cell=12,rectangle=5,replicate=2)
HTR7_STIM_2_12_6  = voro2cluster('HTR7_S_2_12',squre_size=4000,x1=17000,y1=15000,parent='HTR7',element='WT',  cell=12,rectangle=6,replicate=2)



In [ ]:
HTR7_B_WT = [HTR7_B_1_2_1 ,HTR7_B_1_2_2 ,HTR7_B_1_2_3 ,HTR7_B_1_3_1 ,HTR7_B_1_3_2 ,HTR7_B_1_3_3 ,HTR7_B_1_4_1 ,HTR7_B_1_4_2 ,HTR7_B_1_4_3 ,HTR7_B_1_5_1 ,HTR7_B_1_5_2 ,HTR7_B_1_5_3 ,HTR7_B_1_5_4 ,
             HTR7_B_1_6_1 ,HTR7_B_1_6_2 ,HTR7_B_1_6_3 ,HTR7_B_1_7_1 ,HTR7_B_1_7_2 ,HTR7_B_1_7_3 ,HTR7_B_1_8_1 ,HTR7_B_1_8_2 ,HTR7_B_1_8_3 ,HTR7_B_1_8_4 ,HTR7_B_1_9_1 ,HTR7_B_1_9_2 ,HTR7_B_1_9_3 ,
             HTR7_B_1_9_4 ,HTR7_B_1_10_1 ,HTR7_B_1_10_2 ,HTR7_B_1_10_3 ,HTR7_B_1_10_4 ,HTR7_B_2_1_1 ,HTR7_B_2_1_2 ,HTR7_B_2_1_3 ,HTR7_B_2_1_4 ,HTR7_B_2_1_5 ,HTR7_B_2_2_1 ,HTR7_B_2_2_2 ,HTR7_B_2_2_3 ,HTR7_B_2_2_4 ,
             HTR7_B_2_2_5 ,HTR7_B_2_2_6 ,HTR7_B_2_2_7 ,HTR7_B_2_2_8 ,HTR7_B_2_3_1 ,HTR7_B_2_3_2 ,HTR7_B_2_3_3 ,HTR7_B_2_3_4 ,HTR7_B_2_3_5 ,HTR7_B_2_4_1 ,HTR7_B_2_4_2 ,HTR7_B_2_4_3 ,HTR7_B_2_4_4 ,
             HTR7_B_2_4_5 ,HTR7_B_2_4_6 ,HTR7_B_2_4_7 ,HTR7_B_2_4_8 ,HTR7_B_2_4_9 ,HTR7_B_2_5_1 ,HTR7_B_2_5_2 ,HTR7_B_2_5_3 ,HTR7_B_2_5_4 ,HTR7_B_2_5_5 ,HTR7_B_2_5_6 ,HTR7_B_2_5_7 ,HTR7_B_2_5_8 ,
             HTR7_B_2_5_9 ,HTR7_B_2_6_1 ,HTR7_B_2_6_2 ,HTR7_B_2_6_3 ,HTR7_B_2_6_4 ,HTR7_B_2_6_5 ,HTR7_B_2_6_6 ,HTR7_B_2_6_7 ,HTR7_B_2_6_8 ,HTR7_B_2_6_9 ,HTR7_B_2_8_1 ,HTR7_B_2_8_2 ,HTR7_B_2_8_3 
             ,HTR7_B_2_8_4 ,HTR7_B_2_8_5 ,HTR7_B_2_8_6 ,HTR7_B_2_8_7 ,HTR7_B_2_8_8 ,HTR7_B_2_8_9 ,HTR7_B_2_9_1 ,HTR7_B_2_9_2 ,HTR7_B_2_9_3 ,HTR7_B_2_9_4 ,HTR7_B_2_9_5 
             ,HTR7_B_2_9_6 ,HTR7_B_2_9_7 ,HTR7_B_2_9_8 ,HTR7_B_2_9_9 ,HTR7_B_2_10_1 ,HTR7_B_2_10_2 ,HTR7_B_2_10_3 ,HTR7_B_2_10_4 ,HTR7_B_2_10_5 ,HTR7_B_2_10_6 ,HTR7_B_2_10_7 ,HTR7_B_2_10_8 ,HTR7_B_2_10_9 ]

HTR7_B_MUT = [HTR7_BM_1_2_1 ,HTR7_BM_1_2_2 ,HTR7_BM_1_3_1 ,HTR7_BM_1_3_2 ,HTR7_BM_1_3_3 ,HTR7_BM_1_3_4 ,HTR7_BM_1_3_5 ,HTR7_BM_1_3_6 ,HTR7_BM_1_3_7 ,HTR7_BM_1_3_8 ,HTR7_BM_1_3_9 ,HTR7_BM_1_3_1 ,HTR7_BM_1_3_2 ,
              HTR7_BM_1_3_3 ,HTR7_BM_1_3_4 ,HTR7_BM_1_3_5 ,HTR7_BM_1_3_6 ,HTR7_BM_1_3_7 ,HTR7_BM_1_3_8 ,HTR7_BM_1_3_9 ,HTR7_BM_1_4_1 ,HTR7_BM_1_4_2 ,HTR7_BM_1_4_3 ,HTR7_BM_1_4_4 ,HTR7_BM_1_4_5 ,HTR7_BM_1_5_1 ,
              HTR7_BM_1_5_2 ,HTR7_BM_1_5_3 ,HTR7_BM_1_5_4 ,HTR7_BM_1_5_5 ,HTR7_BM_1_5_6 ,HTR7_BM_1_6_1 ,HTR7_BM_1_6_2 ,HTR7_BM_1_7_1 ,HTR7_BM_1_7_2 ,HTR7_BM_1_8_1 ,HTR7_BM_1_8_2 ,HTR7_BM_1_8_3 ,HTR7_BM_1_8_4 ,
              HTR7_BM_1_8_5 ,HTR7_BM_1_9_1 ,HTR7_BM_1_9_2 ,HTR7_BM_1_9_3 ,HTR7_BM_1_9_4 ,HTR7_BM_1_9_5 ,HTR7_BM_1_9_6 ,HTR7_BM_1_9_7 ,HTR7_BM_1_9_8 ,HTR7_BM_1_9_9 ,HTR7_BM_2_2_1 ,HTR7_BM_2_2_2 ,HTR7_BM_2_2_3 ,
              HTR7_BM_2_3_1 ,HTR7_BM_2_3_2 ,HTR7_BM_2_3_3 ,HTR7_BM_2_3_4 ,HTR7_BM_2_3_5 ,HTR7_BM_2_3_6 ,HTR7_BM_2_3_7 ,HTR7_BM_2_3_8 ,HTR7_BM_2_3_9 ,HTR7_BM_2_3_10,HTR7_BM_2_4_1 ,HTR7_BM_2_4_2 ,HTR7_BM_2_4_3 ,
              HTR7_BM_2_4_4 ,HTR7_BM_2_4_5 ,HTR7_BM_2_4_6 ,HTR7_BM_2_4_7 ,HTR7_BM_2_4_8 ,HTR7_BM_2_4_9 ,HTR7_BM_2_5_1 ,HTR7_BM_2_5_2 ,HTR7_BM_2_5_3 ,HTR7_BM_2_5_4 ,HTR7_BM_2_5_5 ,HTR7_BM_2_7_1 ,HTR7_BM_2_7_2 ,
              HTR7_BM_2_7_3 ,HTR7_BM_2_7_4 ,HTR7_BM_2_7_5 ,HTR7_BM_2_8_1 ,HTR7_BM_2_8_2 ,HTR7_BM_2_8_3 ,HTR7_BM_2_8_4 ,HTR7_BM_2_8_5 ,HTR7_BM_2_8_6 ,HTR7_BM_2_8_7 ,HTR7_BM_2_8_8 ,HTR7_BM_2_8_9 ,HTR7_BM_2_10_1,
              HTR7_BM_2_10_2,HTR7_BM_2_10_3,HTR7_BM_2_10_4,HTR7_BM_2_10_5,HTR7_BM_2_10_6]



In [ ]:
HTR7_S_WT = [HTR7_STIM_1_1_1 ,HTR7_STIM_1_1_2 ,HTR7_STIM_1_1_3 ,HTR7_STIM_1_2_1 ,HTR7_STIM_1_2_2 ,HTR7_STIM_1_2_3 ,HTR7_STIM_1_4_1 ,HTR7_STIM_1_4_2 ,HTR7_STIM_1_4_3 ,HTR7_STIM_1_4_4 ,HTR7_STIM_1_4_5 ,HTR7_STIM_1_4_6 ,HTR7_STIM_1_4_7 ,HTR7_STIM_1_6_1 ,HTR7_STIM_1_6_2 ,HTR7_STIM_1_6_3 ,HTR7_STIM_1_6_4 ,HTR7_STIM_1_6_5 ,HTR7_STIM_1_6_6 ,HTR7_STIM_1_7_1 ,HTR7_STIM_1_7_2 ,HTR7_STIM_1_7_3 ,HTR7_STIM_1_7_4 ,HTR7_STIM_1_7_5 ,HTR7_STIM_1_7_6 ,HTR7_STIM_1_7_7 ,HTR7_STIM_2_1_1 ,HTR7_STIM_2_1_2 ,HTR7_STIM_2_1_3 ,HTR7_STIM_2_1_4 ,HTR7_STIM_2_1_5 ,HTR7_STIM_2_1_6 ,HTR7_STIM_2_1_7 ,HTR7_STIM_2_2_1 ,HTR7_STIM_2_3_1 ,HTR7_STIM_2_3_2 ,HTR7_STIM_2_3_3 ,HTR7_STIM_2_3_4 ,HTR7_STIM_2_3_5 ,HTR7_STIM_2_4_1 ,HTR7_STIM_2_4_2 ,HTR7_STIM_2_6_1 ,HTR7_STIM_2_6_2 ,HTR7_STIM_2_6_3 ,HTR7_STIM_2_6_4 ,HTR7_STIM_2_6_5 ,HTR7_STIM_2_6_6 ,HTR7_STIM_2_8_1 ,HTR7_STIM_2_8_2 ,HTR7_STIM_2_8_3 ,HTR7_STIM_2_8_4 ,HTR7_STIM_2_8_5 ,HTR7_STIM_2_8_6 ,HTR7_STIM_2_10_1,HTR7_STIM_2_10_2,HTR7_STIM_2_10_3,HTR7_STIM_2_10_4,HTR7_STIM_2_10_5,HTR7_STIM_2_10_6,HTR7_STIM_2_11_1,HTR7_STIM_2_11_2,HTR7_STIM_2_11_3,HTR7_STIM_2_11_4,HTR7_STIM_2_11_5,HTR7_STIM_2_11_6,HTR7_STIM_2_12_1,HTR7_STIM_2_12_2,HTR7_STIM_2_12_3,HTR7_STIM_2_12_4,HTR7_STIM_2_12_5,HTR7_STIM_2_12_6]
HTR7_S_MUT = [HTR7_STIM_M_1_1_1 ,HTR7_STIM_M_1_1_2 ,HTR7_STIM_M_1_1_3 ,HTR7_STIM_M_1_1_4 ,HTR7_STIM_M_1_1_5 ,HTR7_STIM_M_1_1_6 ,HTR7_STIM_M_1_2_1 ,HTR7_STIM_M_1_2_2 ,HTR7_STIM_M_1_2_3 ,HTR7_STIM_M_1_2_4 ,HTR7_STIM_M_1_2_5 ,HTR7_STIM_M_1_2_6 ,HTR7_STIM_M_1_2_7 ,HTR7_STIM_M_1_2_8 ,HTR7_STIM_M_1_2_9 ,HTR7_STIM_M_1_4_1 ,HTR7_STIM_M_1_4_2 ,HTR7_STIM_M_1_4_3 ,HTR7_STIM_M_1_4_4 ,HTR7_STIM_M_1_3_1 ,HTR7_STIM_M_1_3_2 ,HTR7_STIM_M_1_7_1 ,HTR7_STIM_M_1_7_2 ,HTR7_STIM_M_1_7_3 ,HTR7_STIM_M_1_7_4 ,HTR7_STIM_M_2_2_1 ,HTR7_STIM_M_2_2_2 ,HTR7_STIM_M_2_5_1 ,HTR7_STIM_M_2_5_2 ,HTR7_STIM_M_2_5_3 ,HTR7_STIM_M_2_5_4 ,HTR7_STIM_M_2_5_5 ,HTR7_STIM_M_2_5_6 ,HTR7_STIM_M_2_5_7 ,HTR7_STIM_M_2_6_1 ,HTR7_STIM_M_2_6_2 ,HTR7_STIM_M_2_6_3 ,HTR7_STIM_M_2_6_4 ,HTR7_STIM_M_2_6_5 ,HTR7_STIM_M_2_6_6 ,HTR7_STIM_M_2_6_7 ,HTR7_STIM_M_2_7_1 ,HTR7_STIM_M_2_7_2 ,HTR7_STIM_M_2_7_3 ,HTR7_STIM_M_2_7_4 ,HTR7_STIM_M_2_9_1 ,HTR7_STIM_M_2_9_2 ,HTR7_STIM_M_2_9_3 ,HTR7_STIM_M_2_9_4 ,HTR7_STIM_M_2_9_5 ,HTR7_STIM_M_2_9_6 ,HTR7_STIM_M_2_9_7 ,HTR7_STIM_M_2_11_3,HTR7_STIM_M_2_11_4,HTR7_STIM_M_2_11_5,HTR7_STIM_M_2_11_6,HTR7_STIM_M_2_11_7] 


In [ ]:
    

dff = pd.DataFrame()
precent_pass_thershold = []
cluser_prob = []
cluser_size_nm = []
cluster_fraction_l = []
no_cluster_vs_number_locazations = []
cluster_size_vs_number_locazations = []
density = []
c = []
for i in tqdm.tqdm(enumerate(HTR7_B_WT)):
    rec = i[1]['rectangle'].to_list()[0]
    rep = i[1]['replicate'].to_list()[0]#i[1].iloc[0,'replicate']
    cell  = i[1]['cell'].to_list()[0]#i[1].iloc[0,'cell']
    df,x,thershold,cluster_probabilty,l3  = voronoi_thershold_v2(i[1].loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
    dff = pd.concat([area2clusters(df,element='WT',squre_number=i[0]),dff])
    precent_pass_thershold.append({'val':len(df[(~pd.isna(df.db))])/len(df),'ele':'WT'})
    dbs_all = df.db.value_counts().reset_index()
    density.append({'cell':cell,'rectangle':rec,'replicate':rep,'ele':'WT','dens':len(df)})
    #print(dbs_all.db.min())
    for j in cluster_probabilty:
        cluser_prob.append({'val':j,'ele':'WT'})
    for h in df2clustersize(df):
        cluser_size_nm.append({'val':h,'ele':'WT'})
    for k in cluster_fraction(df):
        cluster_fraction_l.append({'val':k,'ele':'WT'})  

    no_cluster_vs_number_locazations.append({'val':len(area2clusters(df,element='WT',squre_number=i[0]))/len(df),'ele':'WT','state':'bobby'})
    xs = list(range(int(df.x.min().round()),int(df.x.max().round()+500),4000))
    ys = list(range(int(df.y.min().round()),int(df.y.max().round()+500),4000))
    # xs = list(range(int(df.x.min().round()),int(df.x.max().round()+5),500))
    # ys = list(range(int(df.y.min().round()),int(df.y.max().round()+5),500))
    for i in range(len(xs)-1):
        for j in range(len(ys)-1):
            dbs_sub = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().reset_index()
            db_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))])
            all_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))])
            cluster_size_all = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().to_list()
            cluster_size_medain = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().median()
            cluster_sizes_full_vals,cluster_sizes_full_nm_vals = cluster_sizes_full(dbs_all,dbs_sub,df)
            if (cluster_sizes_full_vals is not np.NaN):
                if  (len(cluster_sizes_full_vals)>0):
                    cluster_sizes_full_vals_mean= (cluster_sizes_full_vals)
                    cluster_sizes_full_vals_nm_mean= (cluster_sizes_full_nm_vals)
                else:
                    cluster_sizes_full_vals_mean = np.NaN
                    cluster_sizes_full_vals_nm_mean = np.NaN
            else:
                cluster_sizes_full_vals_mean = np.NaN
                cluster_sizes_full_vals_nm_mean = np.NaN
            c.append({'all_points':all_points,'db_points':db_points,
                      'cluster_sizes_full':cluster_sizes_full_vals,'cluster_sizes_all':cluster_size_all,'cluster_size_medain':cluster_size_medain,
                      'ele':'WT','cell':cell,'rep':rep,'rec':rec,'sub_x':i,'sub_y':j,'cluster_sizes_full_mean':cluster_sizes_full_vals_mean,'cluster_sizes_full_vals_nm_mean':cluster_sizes_full_vals_nm_mean})
            # if len(cluster_size_all)>0:
            #     print(c[-1])
for i in tqdm.tqdm(enumerate(HTR7_B_MUT)):
    rec = i[1]['rectangle'].to_list()[0]
    rep = i[1]['replicate'].to_list()[0]#i[1].iloc[0,'replicate']
    cell  = i[1]['cell'].to_list()[0]#i[1].iloc[0,'cell']
    #df,x,thershold,cluster_probabilty=
    df,x,thershold,cluster_probabilty,l3 = voronoi_thershold_v2(i[1].loc[:,['x','y']].to_numpy(),n=30,size_of_grid=4000,min_cluster_size=5)
    precent_pass_thershold.append({'val':len(df[(~pd.isna(df.db))])/len(df),'ele':'MUT'})
    dff = pd.concat([area2clusters(df,element='MUT',squre_number=i[0]),dff])
    dbs_all = df.db.value_counts().reset_index()
    density.append({'cell':cell,'rectangle':rec,'replicate':rep,'ele':'MUT','dens':len(df)})
    for j in cluster_probabilty:
        cluser_prob.append({'val':j,'ele':'MUT'})
    for h in df2clustersize(df):
        cluser_size_nm.append({'val':h,'ele':'MUT'})
    for k in cluster_fraction(df):
        cluster_fraction_l.append({'val':k,'ele':'MUT'}) 
    no_cluster_vs_number_locazations.append({'val':len(area2clusters(df,element='MUT',squre_number=i[0]))/len(df),'ele':'MUT','state':'bobby'})
    xs = list(range(int(df.x.min().round()),int(df.x.max().round()+500),4000))
    ys = list(range(int(df.y.min().round()),int(df.y.max().round()+500),4000))
    # xs = list(range(int(df.x.min().round()),int(df.x.max().round()+5),500))
    # ys = list(range(int(df.y.min().round()),int(df.y.max().round()+5),500))
    for i in range(len(xs)-1):
        for j in range(len(ys)-1):
            dbs_sub = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().reset_index()
            db_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))])
            all_points = len(df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))])
            cluster_size_medain = df[(df.x.between(xs[i],xs[i+1]))&(df.y.between(ys[j],ys[j+1]))&(df.db!=-1)&(~pd.isna(df.db))].db.value_counts().median()
            cluster_sizes_full_vals,cluster_sizes_full_nm_vals = cluster_sizes_full(dbs_all,dbs_sub,df)
            if (cluster_sizes_full_vals is not np.NaN):
                if  (len(cluster_sizes_full_vals)>0):
                    cluster_sizes_full_vals_mean= (cluster_sizes_full_vals)
                    cluster_sizes_full_vals_nm_mean= (cluster_sizes_full_nm_vals)
                else:
                    cluster_sizes_full_vals_mean = np.NaN
                    cluster_sizes_full_vals_nm_mean = np.NaN
            else:
                cluster_sizes_full_vals_mean = np.NaN
                cluster_sizes_full_vals_nm_mean = np.NaN
            c.append({'all_points':all_points,'db_points':db_points,
                      'cluster_sizes_full':cluster_sizes_full_vals,'cluster_sizes_all':cluster_size_all,'cluster_size_medain':cluster_size_medain,
                      'ele':'MUT','cell':cell,'rep':rep,'rec':rec,'sub_x':i,'sub_y':j,'cluster_sizes_full_mean':cluster_sizes_full_vals_mean,'cluster_sizes_full_vals_nm_mean':cluster_sizes_full_vals_nm_mean})
            

In [ ]:
grid_df_HTR7_basal = pd.DataFrame(c)
grid_df_HTR7_basal.sort_values(by='all_points')
scale_factor = 4**2
grid_df_HTR7_basal['des'] = grid_df_HTR7_basal['all_points']/scale_factor
grid_df_HTR7_basal['fq'] = grid_df_HTR7_basal['db_points']/ grid_df_HTR7_basal['all_points']

In [ ]:
grid_df_HTR7_stim = pd.DataFrame(c)
grid_df_HTR7_stim.sort_values(by='all_points')
scale_factor = 4**2
grid_df_HTR7_stim['des'] = grid_df_HTR7_stim['all_points']/scale_factor
grid_df_HTR7_stim['fq'] = grid_df_HTR7_stim['db_points']/ grid_df_HTR7_stim['all_points']

In [ ]:
grid_df_HTR7_basal = pd.read_pickle('grid_df_HTR7_basal.pkl'
)
grid_df_HTR7_stim = pd.read_pickle('grid_df_HTR7_stim.pkl'
)

In [ ]:
import pandas as pd 

In [ ]:
grid_df_HTR7_basal = pd.read_pickle('grid_df_HTR7_basal.pkl'
)

In [ ]:
grid_df_HTR7_basal.groupby(['ele','cell','rep']).describe()

In [ ]:
grid_df_HTR7_stim_group = grid_df_HTR7_stim[grid_df_HTR7_stim.des<330].groupby(['all_points','ele']).mean().reset_index()
grid_df_HTR7_stim_group['fq'] = grid_df_HTR7_stim_group['db_points']/ grid_df_HTR7_stim_group['all_points']
grid_df_HTR7_stim_group.loc[grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='WT'].index,'smooth'] = grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='WT'].db_points.rolling(window=1).mean()
grid_df_HTR7_stim_group.loc[grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='MUT'].index,'smooth'] = grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='MUT'].db_points.rolling(window=1).mean()
grid_df_HTR7_stim_group.loc[grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='WT'].index,'smooth_fq'] = grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='WT'].fq.rolling(window=5,min_periods=1).mean()
grid_df_HTR7_stim_group.loc[grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='MUT'].index,'smooth_fq'] = grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.ele=='MUT'].fq.rolling(window=5,min_periods=1).mean()
# sns.lineplot(x='des',y='smooth_fq',data=grid_df_HTR7_stim_group,hue='ele',palette=['g','r'])
#plt.xlim([2,1500])
plt.xscale('log')

sns.scatterplot(x='des',y='fq',data=grid_df_HTR7_stim_group,hue='ele',alpha=1)

In [ ]:
grid_df_HTR7_basal_group = grid_df_HTR7_basal[grid_df_HTR7_basal.des<330].groupby(['all_points','ele']).mean().reset_index()
grid_df_HTR7_basal_group['fq'] = grid_df_HTR7_basal_group['db_points']/ grid_df_HTR7_basal_group['all_points']
grid_df_HTR7_basal_group.loc[grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='WT'].index,'smooth'] = grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='WT'].db_points.rolling(window=1).mean()
grid_df_HTR7_basal_group.loc[grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='MUT'].index,'smooth'] = grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='MUT'].db_points.rolling(window=1).mean()
grid_df_HTR7_basal_group.loc[grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='WT'].index,'smooth_fq'] = grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='WT'].fq.rolling(window=5,min_periods=1).mean()
grid_df_HTR7_basal_group.loc[grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='MUT'].index,'smooth_fq'] = grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.ele=='MUT'].fq.rolling(window=5,min_periods=1).mean()
# sns.lineplot(x='des',y='smooth_fq',data=grid_df_HTR7_basal_group,hue='ele',palette=['g','r'])
#plt.xlim([2,1500])
plt.xscale('log')

sns.scatterplot(x='des',y='fq',data=grid_df_HTR7_basal_group,hue='ele',alpha=1)

In [ ]:
grid_df_HTR7_basal

In [ ]:
bins_l = []
for j in flatten(grid_df_HTR7_basal[(grid_df_HTR7_basal.des.between(0,70))&(grid_df_HTR7_basal.ele=='WT')&(~pd.isna(grid_df_HTR7_basal.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,70),'cluster_sizes_full_vals_nm_mean':j,'ele':'WT'})
for j in flatten(grid_df_HTR7_basal[(grid_df_HTR7_basal.des.between(0,70))&(grid_df_HTR7_basal.ele=='MUT')&(~pd.isna(grid_df_HTR7_basal.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,70),'cluster_sizes_full_vals_nm_mean':j,'ele':'MUT'})

for j in flatten(grid_df_HTR7_basal[(grid_df_HTR7_basal.des.between(71,150))&(grid_df_HTR7_basal.ele=='WT')&(~pd.isna(grid_df_HTR7_basal.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(71,150),'cluster_sizes_full_vals_nm_mean':j,'ele':'WT'})
for j in flatten(grid_df_HTR7_basal[(grid_df_HTR7_basal.des.between(71,150))&(grid_df_HTR7_basal.ele=='MUT')&(~pd.isna(grid_df_HTR7_basal.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(71,150),'cluster_sizes_full_vals_nm_mean':j,'ele':'MUT'})
    
bins_df_basal = pd.DataFrame(bins_l)
bins_df_basal['state'] = 'basal'
bins_l = []
for j in flatten(grid_df_HTR7_stim[(grid_df_HTR7_stim.des.between(0,70))&(grid_df_HTR7_stim.ele=='WT')&(~pd.isna(grid_df_HTR7_stim.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,70),'cluster_sizes_full_vals_nm_mean':j,'ele':'WT'})
for j in flatten(grid_df_HTR7_stim[(grid_df_HTR7_stim.des.between(0,70))&(grid_df_HTR7_stim.ele=='MUT')&(~pd.isna(grid_df_HTR7_stim.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(0,70),'cluster_sizes_full_vals_nm_mean':j,'ele':'MUT'})

for j in flatten(grid_df_HTR7_stim[(grid_df_HTR7_stim.des.between(71,150))&(grid_df_HTR7_stim.ele=='WT')&(~pd.isna(grid_df_HTR7_stim.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(71,150),'cluster_sizes_full_vals_nm_mean':j,'ele':'WT'})
for j in flatten(grid_df_HTR7_stim[(grid_df_HTR7_stim.des.between(71,150))&(grid_df_HTR7_stim.ele=='MUT')&(~pd.isna(grid_df_HTR7_stim.cluster_sizes_full_mean))].cluster_sizes_full_mean):
    bins_l.append({'densitiy':'%d-%d'%(71,150),'cluster_sizes_full_vals_nm_mean':j,'ele':'MUT'})
bins_df_stim = pd.DataFrame(bins_l)
bins_df_stim['state'] = 'stim'
bins_df = pd.concat([bins_df_basal,bins_df_stim])

In [ ]:
bins_df[(bins_df.state=='basal')&(bins_df.ele=='WT')]

In [ ]:


fig = plt.figure(figsize = (4,2),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.01,'wspace':.2})
spec_basal = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.state=='basal'],hue='ele',hue_order=['WT','MUT'])
spec_stim = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.state!='basal'],hue='ele',hue_order=['WT','MUT'])

sns.stripplot(**spec_basal,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=['0-70','71-150'],ax=axs[0])
sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.state=='basal'].groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['WT','MUT'],
    join=False, dodge=.8 - .8 / 2,order=['0-70','71-150'],
    markers="_", scale=1.2, errorbar=None,palette=['k','k'],ax=axs[0])

sns.stripplot(**spec_stim,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=['0-70','71-150'],ax=axs[1])
sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.state!='basal'].groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['WT','MUT'],
    join=False, dodge=.8 - .8 / 2,order=['0-70','71-150'],
    markers="_", scale=1.2, errorbar=None,palette=['k','k'],ax=axs[1])
axs[0].set_yscale('log')
axs[1].set_yscale('log')
axs[1].set_ylim([0,500])
axs[0].set_ylim([0,500])
axs[0].legend([],frameon=False)
axs[1].legend([],frameon=False)
axs[1].set_ylabel('')
axs[1].set_yticklabels([''])
axs[1].set_xlabel('')
axs[0].set_title('$0$',fontsize=10)
axs[1].set_title('$100 nM$',fontsize=10)
# axs[0].set_xlabel('$\\rho\;(No.localizations/\\mu m^{2}}$)',fontsize=8)
# axs[0].set_ylabel('$Localizations\;per\;cluster$',fontsize=8)
axs[0].grid(zorder=0)
axs[1].grid(zorder=0)
#plt.savefig('Localizations_per_clsuter_HTR7.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig = plt.figure(figsize = (4,2),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.01,'wspace':.2})

sns.scatterplot(x='des',y='fq',data=grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.des<150],hue='ele',alpha=.3,
                s=10,hue_order=['WT','MUT'],palette=['#C03830','#B4B5B4'],ax=axs[0])
sns.scatterplot(x='des',y='smooth_fq',data=grid_df_HTR7_basal_group[grid_df_HTR7_basal_group.des<150],hue='ele',alpha=1,
                s=10,hue_order=['WT','MUT'],palette=['#C03830','#B4B5B4'],edgecolor='k',zorder=1,ax=axs[0])

sns.scatterplot(x='des',y='fq',data=grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.des<150],hue='ele',alpha=.3,
                s=10,hue_order=['WT','MUT'],palette=['#C03830','#B4B5B4'],ax=axs[1])
sns.scatterplot(x='des',y='smooth_fq',data=grid_df_HTR7_stim_group[grid_df_HTR7_stim_group.des<150],hue='ele',alpha=1,
                s=10,hue_order=['WT','MUT'],palette=['#C03830','#B4B5B4'],edgecolor='k',zorder=1,ax=axs[1])
axs[0].legend([],frameon=False)
axs[1].legend([],frameon=False)
axs[1].set_ylim([0,.6])
axs[0].set_ylim([0,.6])
axs[1].set_xlim([0,150])
axs[0].set_xlim([0,150])
axs[1].set_ylabel('')
axs[1].set_yticklabels([''])
axs[1].set_xlabel('')
#axs[0].set_xlabel('$\\rho\;(No.localizations/\\mu m^{2}}$)',fontsize=8)
#axs[0].set_ylabel('$\\frac{Localizations\;in\;clusters}{\Sigma Localizations}$',fontsize=12)
axs[0].set_title('$0$',fontsize=10)
axs[1].set_title('$100nM$',fontsize=10)
axs[0].grid(zorder=0)
axs[1].grid(zorder=0)
#plt.savefig('fq_smooth_fit_HTR7.svg',bbox_inches='tight',transparent=False)


In [ ]:
len(grid_df_HTR7_stim[(grid_df_HTR7_stim.des<150)&(grid_df_HTR7_stim.ele=='WT')].drop_duplicates(subset=['cell','rep','rec']))

In [ ]:
## Basal 
## WT 93 ROI, 15 Cells,2reps, MUT 86 ROIs,15 Cells,2 Reps
## STIM 
## WT 60 ROI, 13 Cells,2reps, MUT 57 ROIs,11 Cells,2 Reps


In [ ]:
from scipy import stats

col = "cluster_sizes_full_vals_nm_mean"

wt = bins_df.loc[
    (bins_df.state == "basal") & (bins_df.ele == "WT"),
    col
].dropna()

mut = bins_df.loc[
    (bins_df.state == "basal") & (bins_df.ele != "WT"),
    col
].dropna()

stats.ks_2samp(wt, mut)

In [ ]:
sns.boxplot(x='ele',y='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')])
plt.yscale('log')

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

df = bins_df.loc[
    bins_df["state"].eq("basal"),
    ["densitiy", "ele", "cluster_sizes_full_vals_nm_mean"]
].dropna().copy()

df = df[df["cluster_sizes_full_vals_nm_mean"] > 0]
df["log_cluster"] = np.log10(df["cluster_sizes_full_vals_nm_mean"])

df["densitiy"] = df["densitiy"].astype(str)
df["ele"] = df["ele"].astype(str)

model = smf.ols(
    "log_cluster ~ C(densitiy) * C(ele)",
    data=df
).fit()

print(model.summary())

In [ ]:
summary = (
    df.groupby(["densitiy", "ele"])
      .agg(
          n=("cluster_sizes_full_vals_nm_mean", "size"),
          median=("cluster_sizes_full_vals_nm_mean", "median"),
          mean=("cluster_sizes_full_vals_nm_mean", "mean"),
          geom_mean=("log_cluster", lambda x: 10 ** x.mean()),
          q95=("cluster_sizes_full_vals_nm_mean", lambda x: x.quantile(0.95)),
          q99=("cluster_sizes_full_vals_nm_mean", lambda x: x.quantile(0.99)),
      )
      .reset_index()
)

print(summary)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sub = df[df["densitiy"].eq("71-150")].copy()
wt = sub.loc[sub["ele"].eq("WT"), "cluster_sizes_full_vals_nm_mean"]
mut = sub.loc[sub["ele"].eq("MUT"), "cluster_sizes_full_vals_nm_mean"]

log_wt = np.log10(wt[wt > 0])
log_mut = np.log10(mut[mut > 0])

stats.probplot(log_wt, dist="norm", plot=plt)
plt.title("WT 71-150: Q-Q plot of log10 cluster size")
plt.tight_layout()
plt.show()

stats.probplot(log_mut, dist="norm", plot=plt)
plt.title("MUT 71-150: Q-Q plot of log10 cluster size")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# -----------------------------
# Settings
# -----------------------------
col = "cluster_sizes_full_vals_nm_mean"
density_col = "densitiy"
state_to_use = "basal"
high_density = "71-150"
ele_order = ["MUT", "WT"]

# -----------------------------
# Prepare high-density basal data only
# -----------------------------
df_high = bins_df.loc[
    bins_df["state"].eq(state_to_use) & bins_df[density_col].astype(str).eq(high_density),
    ["ele", col]
].dropna().copy()

df_high = df_high[df_high[col] > 0].copy()
df_high["log_cluster"] = np.log10(df_high[col])
df_high["ele"] = df_high["ele"].astype(str)

print("Counts:")
print(df_high["ele"].value_counts())

# -----------------------------
# Summary statistics
# -----------------------------
summary = []

for ele in ele_order:
    raw = df_high.loc[df_high["ele"].eq(ele), col].dropna()
    log_vals = np.log10(raw[raw > 0])

    shapiro_full = stats.shapiro(log_vals)

    if len(log_vals) > 500:
        log_sample = log_vals.sample(500, random_state=1)
    else:
        log_sample = log_vals

    shapiro_sample = stats.shapiro(log_sample)

    summary.append({
        "ele": ele,
        "n": len(raw),
        "median_raw": raw.median(),
        "arithmetic_mean_raw": raw.mean(),
        "geometric_mean_raw": 10 ** log_vals.mean(),
        "q90_raw": raw.quantile(0.90),
        "q95_raw": raw.quantile(0.95),
        "q99_raw": raw.quantile(0.99),
        "min_raw": raw.min(),
        "max_raw": raw.max(),
        "skew_log": stats.skew(log_vals, bias=False),
        "excess_kurtosis_log": stats.kurtosis(log_vals, bias=False),
        "shapiro_W_full": shapiro_full.statistic,
        "shapiro_p_full": shapiro_full.pvalue,
        "shapiro_W_sample500": shapiro_sample.statistic,
        "shapiro_p_sample500": shapiro_sample.pvalue,
    })

summary_df = pd.DataFrame(summary)
print("\nSummary:")
print(summary_df)

# -----------------------------
# Q-Q plots for log10 values
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for j, ele in enumerate(ele_order):
    ax = axes[j]

    vals = df_high.loc[
        df_high["ele"].eq(ele),
        "log_cluster"
    ].dropna()

    stats.probplot(vals, dist="norm", plot=ax)
    ax.set_title(f"{ele}, basal high density {high_density}\nn = {len(vals)}")
    ax.set_xlabel("Theoretical quantiles")
    ax.set_ylabel("Ordered log10(cluster size)")

plt.suptitle("High-density basal Q-Q plots: log10 cluster size", y=1.05)
plt.tight_layout()
plt.show()

# -----------------------------
# Direct WT vs MUT tests
# -----------------------------
wt = df_high.loc[df_high["ele"].eq("WT"), col].dropna()
mut = df_high.loc[df_high["ele"].eq("MUT"), col].dropna()

log_wt = np.log10(wt[wt > 0])
log_mut = np.log10(mut[mut > 0])

print("\nWelch t-test on log10 values, WT > MUT:")
print(stats.ttest_ind(log_wt, log_mut, equal_var=False, alternative="greater"))

print("\nMann-Whitney U, WT > MUT:")
print(stats.mannwhitneyu(wt, mut, alternative="greater"))

print("\nKS test, WT larger than MUT:")
print(stats.ks_2samp(wt, mut, alternative="less"))

print("\nKS two-sided:")
print(stats.ks_2samp(wt, mut, alternative="two-sided"))

# -----------------------------
# Tail-enrichment tests within high density
# -----------------------------
pooled = df_high[col]

print("\nTail enrichment tests, WT enriched above pooled threshold:")

for q in [0.90, 0.95, 0.99]:
    threshold = pooled.quantile(q)

    table = np.array([
        [(wt > threshold).sum(),  (wt <= threshold).sum()],
        [(mut > threshold).sum(), (mut <= threshold).sum()]
    ])

    oddsratio, p = stats.fisher_exact(table, alternative="greater")

    print(f"\nPooled {int(q*100)}th percentile threshold = {threshold:.2f}")
    print(table)
    print("WT high fraction:", (wt > threshold).mean())
    print("MUT high fraction:", (mut > threshold).mean())
    print("odds ratio:", oddsratio)
    print("Fisher p-value:", p)

# -----------------------------
# Permutation test for q99 difference
# -----------------------------
wt_vals = wt.values
mut_vals = mut.values

obs_q99_diff = np.quantile(wt_vals, 0.99) - np.quantile(mut_vals, 0.99)

all_vals = np.concatenate([wt_vals, mut_vals])
n_wt = len(wt_vals)

rng = np.random.default_rng(1)
n_perm = 10000
perm_q99 = np.empty(n_perm)

for i in range(n_perm):
    perm = rng.permutation(all_vals)
    perm_wt = perm[:n_wt]
    perm_mut = perm[n_wt:]
    perm_q99[i] = np.quantile(perm_wt, 0.99) - np.quantile(perm_mut, 0.99)

p_q99 = np.mean(perm_q99 >= obs_q99_diff)

print("\nPermutation test for q99 difference, WT - MUT:")
print("Observed q99 WT - MUT:", obs_q99_diff)
print("Permutation p-value:", p_q99)

# -----------------------------
# Permutation test for skew difference on log scale
# -----------------------------
obs_skew_diff = stats.skew(log_wt, bias=False) - stats.skew(log_mut, bias=False)

all_log = np.concatenate([log_wt.values, log_mut.values])
n_wt_log = len(log_wt)

perm_skew = np.empty(n_perm)

for i in range(n_perm):
    perm = rng.permutation(all_log)
    perm_wt = perm[:n_wt_log]
    perm_mut = perm[n_wt_log:]
    perm_skew[i] = stats.skew(perm_wt, bias=False) - stats.skew(perm_mut, bias=False)

p_skew = np.mean(perm_skew >= obs_skew_diff)

print("\nPermutation test for log-scale skewness difference, WT - MUT:")
print("Observed skew WT - MUT:", obs_skew_diff)
print("Permutation p-value:", p_skew)

In [ ]:
sns.boxplot(x='ele',y='des',data=grid_df_HTR7_basal_group)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

col = "cluster_sizes_full_vals_nm_mean"
density_col = "densitiy"

df = bins_df.loc[
    bins_df["state"].eq("basal"),
    [density_col, "ele", col]
].dropna().copy()

df = df[df[col] > 0].copy()
df["log_cluster"] = np.log10(df[col])

df[density_col] = df[density_col].astype(str)
df["ele"] = df["ele"].astype(str)

# Additive model: corrects WT/MUT comparison for density bin
model_additive = smf.ols(
    f"log_cluster ~ C(ele) + C({density_col})",
    data=df
).fit()

print("\nAdditive model: genotype adjusted for density bin")
print(model_additive.summary())

# Interaction model: asks whether WT effect differs between density bins
model_interaction = smf.ols(
    f"log_cluster ~ C(ele) * C({density_col})",
    data=df
).fit()

print("\nInteraction model: genotype x density")
print(model_interaction.summary())

# Compare additive vs interaction model
print("\nANOVA comparison: does interaction improve model?")
print(sm.stats.anova_lm(model_additive, model_interaction))

# Within-density WT vs MUT tests
print("\nWithin-density tests:")
for density, sub in df.groupby(density_col):
    wt_raw = sub.loc[sub["ele"].eq("WT"), col]
    mut_raw = sub.loc[sub["ele"].eq("MUT"), col]

    wt_log = np.log10(wt_raw)
    mut_log = np.log10(mut_raw)

    print(f"\nDensity: {density}")
    print("n WT:", len(wt_raw), "n MUT:", len(mut_raw))
    print("WT median:", wt_raw.median(), "MUT median:", mut_raw.median())
    print("WT geometric mean:", 10 ** wt_log.mean())
    print("MUT geometric mean:", 10 ** mut_log.mean())
    print("WT arithmetic mean:", wt_raw.mean())
    print("MUT arithmetic mean:", mut_raw.mean())

    print("Welch log t-test WT > MUT:")
    print(stats.ttest_ind(wt_log, mut_log, equal_var=False, alternative="greater"))

    print("Mann-Whitney WT > MUT:")
    print(stats.mannwhitneyu(wt_raw, mut_raw, alternative="greater"))

    print("KS WT larger:")
    print(stats.ks_2samp(wt_raw, mut_raw, alternative="less"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Settings
# -----------------------------
col = "cluster_sizes_full_vals_nm_mean"
density_col = "densitiy"
state_to_use = "basal"

density_order = ["0-70", "71-150"]
n_boot = 20000
random_seed = 1

# -----------------------------
# Prepare data
# -----------------------------
df = bins_df.loc[
    bins_df["state"].eq(state_to_use),
    [density_col, "ele", col]
].dropna().copy()

df = df[df[col] > 0].copy()
df[density_col] = df[density_col].astype(str)
df["ele"] = df["ele"].astype(str)
df["log_cluster"] = np.log10(df[col])

# -----------------------------
# Helper
# -----------------------------
def skew_log_from_raw(x):
    return pd.Series(np.log10(x)).skew()

def bootstrap_skew_diff(wt, mut, n_boot=20000, seed=1):
    rng = np.random.default_rng(seed)

    wt = np.asarray(wt)
    mut = np.asarray(mut)

    n_wt = len(wt)
    n_mut = len(mut)

    obs = skew_log_from_raw(wt) - skew_log_from_raw(mut)

    boot = np.empty(n_boot)

    for i in range(n_boot):
        wt_b = rng.choice(wt, size=n_wt, replace=True)
        mut_b = rng.choice(mut, size=n_mut, replace=True)
        boot[i] = skew_log_from_raw(wt_b) - skew_log_from_raw(mut_b)

    ci = np.percentile(boot, [2.5, 97.5])

    p_two = 2 * min(
        np.mean(boot <= 0),
        np.mean(boot >= 0)
    )
    p_two = min(p_two, 1.0)

    return obs, ci, p_two, boot

# -----------------------------
# Run bootstrap separately by density
# -----------------------------
results = []
boot_by_density = {}

for k, density in enumerate(density_order):
    sub = df[df[density_col].eq(density)]

    wt = sub.loc[sub["ele"].eq("WT"), col].to_numpy()
    mut = sub.loc[sub["ele"].eq("MUT"), col].to_numpy()

    obs, ci, p, boot = bootstrap_skew_diff(
        wt, mut, n_boot=n_boot, seed=random_seed + k
    )

    boot_by_density[density] = boot

    results.append({
        "density": density,
        "n_WT": len(wt),
        "n_MUT": len(mut),
        "WT_skew_log": skew_log_from_raw(wt),
        "MUT_skew_log": skew_log_from_raw(mut),
        "WT_minus_MUT_skew_log": obs,
        "ci95_low": ci[0],
        "ci95_high": ci[1],
        "bootstrap_p_two_sided": p
    })

results_df = pd.DataFrame(results)

print(results_df)

# -----------------------------
# Plot bootstrap skewness differences
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, density in zip(axes, density_order):
    boot = boot_by_density[density]
    row = results_df[results_df["density"].eq(density)].iloc[0]

    ax.hist(boot, bins=60)
    ax.axvline(0, linestyle="--")
    ax.axvline(row["WT_minus_MUT_skew_log"], linestyle="-")

    ax.set_title(
        f"Density {density}\n"
        f"obs={row['WT_minus_MUT_skew_log']:.2f}, "
        f"95% CI [{row['ci95_low']:.2f}, {row['ci95_high']:.2f}], "
        f"p={row['bootstrap_p_two_sided']:.3g}"
    )

    ax.set_xlabel("WT skewness - MUT skewness\non log10(cluster size)")
    ax.set_ylabel("Bootstrap count")

plt.suptitle(
    "WT tail-skewness effect is density-dependent",
    y=1.05,
    fontsize=14
)

plt.tight_layout()
plt.show()

# -----------------------------
# Optional: compact forest-style plot
# -----------------------------
plt.figure(figsize=(6, 3.5))

x = np.arange(len(results_df))
y = results_df["WT_minus_MUT_skew_log"].to_numpy()
yerr_low = y - results_df["ci95_low"].to_numpy()
yerr_high = results_df["ci95_high"].to_numpy() - y

plt.errorbar(
    x,
    y,
    yerr=[yerr_low, yerr_high],
    fmt="o",
    capsize=5
)

plt.axhline(0, linestyle="--")
plt.xticks(x, results_df["density"])
plt.ylabel("WT - MUT log-scale skewness")
plt.xlabel("Density bin")
plt.title("Bootstrap estimate of WT tail-heaviness effect")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# Publication-style forest plot
# matched to your violin aesthetic
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt

# Colors matching your example
wt_red = "#d62728"
mut_gray = "#b3b3b3"
black = "black"

# Use results from your bootstrap analysis
plot_df = results_df.copy()

# Order: low density on bottom, high density on top
plot_df["density_label"] = plot_df["density"].replace({
    "0-70": r"Low $\rho$" + "\n" + r"0–70",
    "71-150": r"High $\rho$" + "\n" + r"71–150"
})

plot_df = plot_df.set_index("density").loc[["0-70", "71-150"]].reset_index()

ypos = np.arange(len(plot_df))

x = plot_df["WT_minus_MUT_skew_log"].to_numpy()
xerr_low = x - plot_df["ci95_low"].to_numpy()
xerr_high = plot_df["ci95_high"].to_numpy() - x

fig, ax = plt.subplots(figsize=(3.0, 3.8))

# CI bars and points
for i, row in plot_df.iterrows():
    effect = row["WT_minus_MUT_skew_log"]
    ci_low = row["ci95_low"]
    ci_high = row["ci95_high"]
    p = row["bootstrap_p_two_sided"]

    color = wt_red if row["density"] == "71-150" else mut_gray

    ax.plot(
        [ci_low, ci_high],
        [i, i],
        color=color,
        linewidth=3,
        solid_capstyle="round"
    )

    ax.scatter(
        effect,
        i,
        s=95,
        color=color,
        edgecolor=black,
        linewidth=1.2,
        zorder=3
    )

    # p-value annotation
    if p < 0.001:
        p_text = r"$p < 0.001$"
    else:
        p_text = rf"$p = {p:.3f}$"

    ax.text(
        ci_high + 0.08,
        i,
        f"{effect:.2f}\n{p_text}",
        va="center",
        ha="left",
        fontsize=11
    )

# Zero line
ax.axvline(
    0,
    color=black,
    linestyle="--",
    linewidth=1.6,
    alpha=0.85
)

# Axis labels
ax.set_yticks(ypos)
ax.set_yticklabels(plot_df["density_label"], fontsize=13)

ax.set_xlabel(
    "WT − V451P log-scale skewness",
    fontsize=13
)

ax.set_title(
    "Tail-heaviness effect",
    fontsize=15,
    weight="bold",
    pad=12
)

# Directional annotation
y_arrow = -0.55
ax.annotate(
    "",
    xy=(ax.get_xlim()[1] * 0.75, y_arrow),
    xytext=(0, y_arrow),
    arrowprops=dict(arrowstyle="->", linewidth=1.8, color=black),
    annotation_clip=False
)

ax.text(
    ax.get_xlim()[1] * 0.38,
    y_arrow - 0.18,
    "WT more\nright-skewed",
    ha="center",
    va="top",
    fontsize=10
)

# Styling to match example
for spine in ax.spines.values():
    spine.set_linewidth(3)
    spine.set_color(black)

ax.tick_params(
    axis="both",
    which="major",
    width=3,
    length=7,
    labelsize=12,
    color=black
)

ax.tick_params(
    axis="both",
    which="minor",
    width=2,
    length=4,
    color=black
)

ax.set_ylim(-0.8, len(plot_df) - 0.35)

# Set x-limits with padding based on CI
xmin = min(plot_df["ci95_low"].min(), -0.25)
xmax = max(plot_df["ci95_high"].max(), 0.25)
pad = 0.25 * (xmax - xmin)
ax.set_xlim(xmin - pad, xmax + pad)

# Cleaner look
ax.grid(False)

plt.tight_layout()
plt.savefig('tail_heveyness.svg',bbox_inches='tight',transparent=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections.abc import Iterable

# -----------------------------
# Settings
# -----------------------------
col_list = "cluster_sizes_full_mean"
col_value = "cluster_size"
density_col = "des"

cutoffs = np.arange(40, 121, 5)
n_boot = 5000
random_seed = 1

# -----------------------------
# Simple robust flattener
# Handles scalars, lists, tuples, numpy arrays, nested lists
# -----------------------------
def flatten_any(x):
    if x is None:
        return []

    if isinstance(x, float) and np.isnan(x):
        return []

    if isinstance(x, (str, bytes)):
        return [x]

    if isinstance(x, np.ndarray):
        out = []
        for item in x.ravel():
            out.extend(flatten_any(item))
        return out

    if isinstance(x, Iterable):
        out = []
        for item in x:
            out.extend(flatten_any(item))
        return out

    return [x]

# -----------------------------
# Convert grid_df into long cluster-level dataframe
# -----------------------------
def make_long_cluster_df(grid_df, state_name):
    rows = []

    tmp = grid_df[["ele", density_col, col_list]].copy()
    tmp = tmp[tmp[col_list].notna()]

    for _, r in tmp.iterrows():
        ele = r["ele"]
        des = r[density_col]
        vals = flatten_any(r[col_list])

        for v in vals:
            try:
                v = float(v)
            except Exception:
                continue

            if np.isfinite(v) and v > 0:
                rows.append({
                    "state": state_name,
                    "ele": ele,
                    density_col: des,
                    col_value: v
                })

    return pd.DataFrame(rows)

long_basal = make_long_cluster_df(grid_df_HTR7_basal, "basal")

long_basal["ele"] = long_basal["ele"].astype(str)
long_basal[density_col] = pd.to_numeric(long_basal[density_col], errors="coerce")
long_basal = long_basal.dropna(subset=[density_col, col_value])

print("Long basal dataframe:")
print(long_basal.head())
print(long_basal["ele"].value_counts())
print(long_basal[density_col].describe())

# -----------------------------
# Helper functions
# -----------------------------
def skew_log_from_raw(x):
    x = np.asarray(x)
    x = x[x > 0]
    return pd.Series(np.log10(x)).skew()

def bootstrap_skew_diff(wt, mut, n_boot=5000, seed=1):
    rng = np.random.default_rng(seed)

    wt = np.asarray(wt)
    mut = np.asarray(mut)

    if len(wt) < 20 or len(mut) < 20:
        return np.nan, np.nan, np.nan, np.nan

    obs = skew_log_from_raw(wt) - skew_log_from_raw(mut)

    boot = np.empty(n_boot)

    for i in range(n_boot):
        wt_b = rng.choice(wt, size=len(wt), replace=True)
        mut_b = rng.choice(mut, size=len(mut), replace=True)
        boot[i] = skew_log_from_raw(wt_b) - skew_log_from_raw(mut_b)

    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    p_two = 2 * min(np.mean(boot <= 0), np.mean(boot >= 0))
    p_two = min(p_two, 1.0)

    return obs, ci_low, ci_high, p_two

# -----------------------------
# Cutoff scan
# -----------------------------
scan_results = []

for idx, cutoff in enumerate(cutoffs):
    for regime_name, mask in [
        ("low", long_basal[density_col] <= cutoff),
        ("high", long_basal[density_col] > cutoff),
    ]:
        sub = long_basal.loc[mask]

        wt = sub.loc[sub["ele"].eq("WT"), col_value].to_numpy()
        mut = sub.loc[sub["ele"].eq("MUT"), col_value].to_numpy()

        obs, ci_low, ci_high, p = bootstrap_skew_diff(
            wt,
            mut,
            n_boot=n_boot,
            seed=random_seed + idx
        )

        scan_results.append({
            "cutoff": cutoff,
            "regime": regime_name,
            "n_WT": len(wt),
            "n_MUT": len(mut),
            "WT_skew_log": skew_log_from_raw(wt) if len(wt) >= 3 else np.nan,
            "MUT_skew_log": skew_log_from_raw(mut) if len(mut) >= 3 else np.nan,
            "WT_minus_MUT_skew_log": obs,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "bootstrap_p": p
        })

scan_df = pd.DataFrame(scan_results)

print(scan_df)

# -----------------------------
# Plot robustness scan
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, regime_name in zip(axes, ["low", "high"]):
    sub = scan_df[scan_df["regime"].eq(regime_name)].dropna().copy()

    ax.plot(sub["cutoff"], sub["WT_minus_MUT_skew_log"], marker="o")
    ax.fill_between(
        sub["cutoff"],
        sub["ci95_low"],
        sub["ci95_high"],
        alpha=0.2
    )

    ax.axhline(0, linestyle="--")
    ax.axvline(70, linestyle=":", label="chosen cutoff = 70")

    ax.set_title(f"{regime_name} density regime")
    ax.set_xlabel("Density cutoff")
    ax.set_ylabel("WT - MUT log-scale skewness")
    ax.legend()

plt.suptitle(
    "Robustness of WT tail-skewness effect across density cutoffs",
    y=1.05
)

plt.tight_layout()
plt.show()

# -----------------------------
# Plot sample sizes
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, regime_name in zip(axes, ["low", "high"]):
    sub = scan_df[scan_df["regime"].eq(regime_name)]

    ax.plot(sub["cutoff"], sub["n_WT"], marker="o", label="WT")
    ax.plot(sub["cutoff"], sub["n_MUT"], marker="o", label="MUT")

    ax.axvline(70, linestyle=":")
    ax.set_title(f"{regime_name} density sample size")
    ax.set_xlabel("Density cutoff")
    ax.set_ylabel("n")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
def voro2cluster_v3(
    file,
    squre_size=50000,
    x1=0,
    y1=0,
    parent='APP',
    element='WT',
    cell=1,
    rectangle=1,
    replicate=1,
    save_path='/Users/aelazar/notebooks_for_paper/dSTORM/figures',
    rdgy_vmin=-6,
    rdgy_vmax=-1,
    add_colorbar=False
):
    s = pd.read_csv('csv/v2/%s.csv' % file)  # 5000 pixels = 1 micron

    x2 = x1 + squre_size
    y2 = y1 + squre_size

    sub_df = s[
        (s['uncertainty [nm]'] < 20) &
        (s.frame >= 200)
    ].copy()

    fig = plt.figure(figsize=(3, 9), constrained_layout=False)
    axs = fig.subplots(
        nrows=3,
        ncols=1,
        gridspec_kw={'hspace': .01, 'wspace': .1}
    )

    # ------------------------------------------------------------
    # Panel 1: raw localizations with rectangle
    # ------------------------------------------------------------
    sns.scatterplot(
        x='x [nm]',
        y='y [nm]',
        data=sub_df[
            (sub_df['x [nm]'].between(x1 - 5000, x2 + 5000)) &
            (sub_df['y [nm]'].between(y1 - 5000, y2 + 5000))
        ],
        s=.5,
        edgecolor=None,
        c='k',
        ax=axs[0],
        rasterized=True
    )

    axs[0].add_patch(
        Rectangle(
            (x1, y1),
            squre_size,
            squre_size,
            linewidth=1,
            edgecolor='#18827E',
            facecolor='none'
        )
    )

    axs[0].set_facecolor('xkcd:white')
    axs[0].tick_params(
        left=False,
        right=False,
        labelleft=False,
        labelbottom=False,
        bottom=False
    )
    axs[0].set_xlabel('')
    axs[0].set_ylabel('')

    # ------------------------------------------------------------
    # Crop data for analysis
    # ------------------------------------------------------------
    ss = sub_df[
        (sub_df['x [nm]'].between(x1, x2)) &
        (sub_df['y [nm]'].between(y1, y2))
    ].copy()

    ss['x'] = ss['x [nm]']
    ss['y'] = ss['y [nm]']
    ss['t'] = ss.frame.astype(int)
    ss['parent'] = parent
    ss['element'] = element
    ss['cell'] = cell
    ss['rectangle'] = rectangle
    ss['replicate'] = replicate
    ss['name'] = file

    df, x, thershold, _, cluster_probabilty = voronoi_thershold_v2(
        ss.loc[:, ['x', 'y']].to_numpy(),
        n=1,
        size_of_grid=4000,
        min_cluster_size=5
    )

    # Make sure vol_invers exists
    if 'vol_invers' not in df.columns:
        df['vol_invers'] = np.log10(1 / df['vol'])

    # ------------------------------------------------------------
    # Panel 2: RdGy filled Voronoi diagram by vol_invers
    # ------------------------------------------------------------
    vor_df = df.dropna(subset=['x', 'y', 'vol_invers']).copy()
    points = vor_df.loc[:, ['x', 'y']].to_numpy()

    vor = Voronoi(points)

    norm = plt.Normalize(vmin=rdgy_vmin, vmax=rdgy_vmax)
    mapper = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.RdGy_r)

    # Fill finite Voronoi cells
    for r, region_index in enumerate(vor.point_region):
        region = vor.regions[region_index]

        # skip infinite / empty regions
        if len(region) == 0 or -1 in region:
            continue

        area_value = vor_df.iloc[r]['vol_invers']
        polygon = [vor.vertices[i] for i in region]

        axs[1].fill(
            *zip(*polygon),
            color=mapper.to_rgba(area_value),
            edgecolor='none',
            linewidth=0,
            alpha=1
        )

    # Draw Voronoi borders on top
    voronoi_plot_2d(
        vor,
        show_points=False,
        show_vertices=False,
        point_size=1,
        line_width=.1,
        ax=axs[1],
        rasterized=False
    )

    axs[1].set_xlim([df.x.min(), df.x.max()])
    axs[1].set_ylim([df.y.min(), df.y.max()])
    axs[1].tick_params(
        left=False,
        right=False,
        labelleft=False,
        labelbottom=False,
        bottom=False
    )
    axs[1].set_xlabel('')
    axs[1].set_ylabel('')

    if add_colorbar:
        mapper.set_array([])
        axs[1].figure.colorbar(
            mapper,
            ax=axs[1],
            fraction=0.03,
            pad=0.01
        )

    # ------------------------------------------------------------
    # Panel 3: DBSCAN / cluster identity coloring
    # ------------------------------------------------------------
    clustered_df = df[
        (df.db != -1) &
        (~pd.isna(df.db))
    ].copy()

    noise_df = df[
        (df.db == -1) |
        (pd.isna(df.db))
    ].copy()

    samp = len(set(clustered_df.db))

    if samp > 0:
        cmap_test = matplotlib.colors.LinearSegmentedColormap.from_list(
            "",
            ["#F55536", "#0C8346", "#B8B8B8", "#8D5A97", "#274C77"],
            samp
        )

        sns.scatterplot(
            x='x',
            y='y',
            data=clustered_df,
            s=20,
            edgecolor=None,
            hue='db',
            hue_norm=(0, samp),
            palette=cmap_test,
            ax=axs[2],
            rasterized=True
        )

    sns.scatterplot(
        x='x',
        y='y',
        data=noise_df,
        s=1,
        edgecolor=None,
        c='k',
        alpha=.1,
        ax=axs[2],
        rasterized=True
    )

    axs[2].set_xlim([df.x.min(), df.x.max()])
    axs[2].set_ylim([df.y.min(), df.y.max()])
    axs[2].tick_params(
        left=False,
        right=False,
        labelleft=False,
        labelbottom=False,
        bottom=False
    )
    axs[2].set_xlabel('')
    axs[2].set_ylabel('')
    axs[2].legend([], frameon=False)

    # ------------------------------------------------------------
    # Save and return
    # ------------------------------------------------------------
    plt.savefig(
        '%s/%s_%d_%d_v3.png' % (save_path, file, cell, rectangle),
        bbox_inches='tight',
        transparent=True,
        dpi=600
        
    )
    plt.savefig(
        '%s/%s_%d_%d_v3.svg' % (save_path, file, cell, rectangle),
        bbox_inches='tight',
        transparent=True,
        
    )
    print('%s/%s_%d_%d_v3.png' % (save_path, file, cell, rectangle),)
    return ss.loc[:, [
        'x',
        'y',
        'parent',
        'element',
        'cell',
        'rectangle',
        'replicate',
        'name'
    ]]

In [ ]:
grid_df_HTR7_basal[grid_df_HTR7_basal.des.between(0,70)]

In [ ]:
def add_soft_density_background(
    ax,
    data,
    x='x [nm]',
    y='y [nm]',
    bins=350,
    sigma=8,
    alpha=0.28,
    zorder=0
):
    """
    Add a soft density background with no contour lines.
    Uses a smoothed 2D histogram instead of seaborn kdeplot.
    """

    import numpy as np
    from scipy.ndimage import gaussian_filter

    if len(data) < 50:
        return

    xvals = data[x].to_numpy()
    yvals = data[y].to_numpy()

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    H, xedges, yedges = np.histogram2d(
        xvals,
        yvals,
        bins=bins,
        range=[
            [xlim[0], xlim[1]],
            [ylim[0], ylim[1]]
        ]
    )

    H = gaussian_filter(H, sigma=sigma)

    if H.max() > 0:
        H = H / H.max()

    H = np.ma.masked_where(H <= 0.02, H)

    ax.imshow(
        H.T,
        origin='lower',
        extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
        cmap='Greys',
        alpha=alpha,
        interpolation='bilinear',
        aspect='auto',
        zorder=zorder
    )


def voro2cluster_v4(
    file,
    squre_size=50000,
    x1=0,
    y1=0,
    parent='APP',
    element='WT',
    cell=1,
    rectangle=1,
    replicate=1,
    save_path='/Users/aelazar/notebooks_for_paper/dSTORM/figures',
    rdgy_vmin=-6,
    rdgy_vmax=-1,
    add_colorbar=False,
    show_density_background=True,
    density_bins=350,
    density_sigma=8,
    density_alpha=0.28
):
    """
    Two-panel version:
    1. Raw localizations with smooth density background and crop rectangle.
    2. RdGy filled Voronoi diagram.

    Important SVG fix:
    - Does NOT use scipy.spatial.voronoi_plot_2d, because that draws infinite
      Voronoi rays extending outside the crop box.
    - Clips polygon fills and ridge lines to the square before writing SVG.
    - Saves SVG without bbox_inches='tight'.
    """

    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    from matplotlib.patches import Rectangle
    from scipy.spatial import Voronoi

    from shapely.geometry import Polygon, LineString, box
    from shapely.errors import GEOSException

    s = pd.read_csv('csv/v2/%s.csv' % file)

    x2 = x1 + squre_size
    y2 = y1 + squre_size

    crop_box = box(x1, y1, x2, y2)

    sub_df = s[
        (s['uncertainty [nm]'] < 20) &
        (s.frame >= 200)
    ].copy()

    raw_plot_df = sub_df[
        (sub_df['x [nm]'].between(x1 - 5000, x2 + 5000)) &
        (sub_df['y [nm]'].between(y1 - 5000, y2 + 5000))
    ].copy()

    fig = plt.figure(figsize=(3, 6), constrained_layout=False)

    axs = fig.subplots(
        nrows=2,
        ncols=1,
        gridspec_kw={'hspace': .01, 'wspace': .1}
    )

    axs[0].set_xlim([x1 - 5000, x2 + 5000])
    axs[0].set_ylim([y1 - 5000, y2 + 5000])

    if show_density_background:
        add_soft_density_background(
            axs[0],
            raw_plot_df,
            x='x [nm]',
            y='y [nm]',
            bins=density_bins,
            sigma=density_sigma,
            alpha=density_alpha,
            zorder=0
        )

    sns.scatterplot(
        x='x [nm]',
        y='y [nm]',
        data=raw_plot_df,
        s=.35,
        edgecolor=None,
        c='k',
        alpha=0.45,
        ax=axs[0],
        rasterized=True,
        zorder=10
    )

    axs[0].add_patch(
        Rectangle(
            (x1, y1),
            squre_size,
            squre_size,
            linewidth=1,
            edgecolor='#18827E',
            facecolor='none',
            zorder=50
        )
    )

    axs[0].set_facecolor('xkcd:white')
    axs[0].tick_params(
        left=False,
        right=False,
        labelleft=False,
        labelbottom=False,
        bottom=False
    )
    axs[0].set_xlabel('')
    axs[0].set_ylabel('')

    ss = sub_df[
        (sub_df['x [nm]'].between(x1, x2)) &
        (sub_df['y [nm]'].between(y1, y2))
    ].copy()

    ss['x'] = ss['x [nm]']
    ss['y'] = ss['y [nm]']
    ss['t'] = ss.frame.astype(int)
    ss['parent'] = parent
    ss['element'] = element
    ss['cell'] = cell
    ss['rectangle'] = rectangle
    ss['replicate'] = replicate
    ss['name'] = file

    df, x, thershold, _, cluster_probabilty = voronoi_thershold_v2(
        ss.loc[:, ['x', 'y']].to_numpy(),
        n=1,
        size_of_grid=4000,
        min_cluster_size=5
    )

    if 'vol_invers' not in df.columns:
        df['vol_invers'] = np.log10(1 / df['vol'])

    vor_df = df.dropna(subset=['x', 'y', 'vol_invers']).copy()
    points = vor_df.loc[:, ['x', 'y']].to_numpy()

    if len(points) < 4:
        print("Not enough points for Voronoi.")
        return ss.loc[:, [
            'x',
            'y',
            'parent',
            'element',
            'cell',
            'rectangle',
            'replicate',
            'name'
        ]]

    vor = Voronoi(points)

    norm = plt.Normalize(vmin=rdgy_vmin, vmax=rdgy_vmax)
    mapper = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.RdGy_r)

    axs[1].set_xlim([x1, x2])
    axs[1].set_ylim([y1, y2])

    # A real Matplotlib clipping rectangle, just in case.
    mpl_clip_rect = Rectangle(
        (x1, y1),
        squre_size,
        squre_size,
        transform=axs[1].transData
    )

    # ------------------------------------------------------------
    # Filled Voronoi polygons, clipped to the square before plotting.
    # This prevents huge off-artboard SVG geometry.
    # ------------------------------------------------------------
    for r, region_index in enumerate(vor.point_region):
        region = vor.regions[region_index]

        if len(region) == 0 or -1 in region:
            continue

        area_value = vor_df.iloc[r]['vol_invers']
        polygon_points = [vor.vertices[i] for i in region]

        try:
            poly = Polygon(polygon_points)

            if not poly.is_valid:
                poly = poly.buffer(0)

            clipped = poly.intersection(crop_box)

        except GEOSException:
            continue

        if clipped.is_empty:
            continue

        color = mapper.to_rgba(area_value)

        if clipped.geom_type == 'Polygon':
            geoms = [clipped]
        elif clipped.geom_type == 'MultiPolygon':
            geoms = list(clipped.geoms)
        else:
            continue

        for geom in geoms:
            if geom.is_empty:
                continue

            xs, ys = geom.exterior.xy

            patch = axs[1].fill(
                xs,
                ys,
                color=color,
                edgecolor='none',
                linewidth=0,
                alpha=1,
                zorder=1
            )

            for p in patch:
                p.set_clip_path(mpl_clip_rect)
                p.set_clip_on(True)

    # ------------------------------------------------------------
    # Draw only finite Voronoi ridge segments, clipped to the square.
    # This replaces voronoi_plot_2d and avoids long infinite blue lines.
    # ------------------------------------------------------------
    for ridge_vertices in vor.ridge_vertices:
        if len(ridge_vertices) != 2:
            continue

        v0, v1 = ridge_vertices

        if v0 < 0 or v1 < 0:
            continue

        p0 = vor.vertices[v0]
        p1 = vor.vertices[v1]

        try:
            line = LineString([p0, p1])
            clipped_line = line.intersection(crop_box)
        except GEOSException:
            continue

        if clipped_line.is_empty:
            continue

        if clipped_line.geom_type == 'LineString':
            lines = [clipped_line]
        elif clipped_line.geom_type == 'MultiLineString':
            lines = list(clipped_line.geoms)
        else:
            continue

        for one_line in lines:
            xs, ys = one_line.xy

            line_obj, = axs[1].plot(
                xs,
                ys,
                color='black',
                linewidth=0.1,
                zorder=5,
                clip_on=True
            )

            line_obj.set_clip_path(mpl_clip_rect)

    # Optional crop-border rectangle for the Voronoi panel.
    axs[1].add_patch(
        Rectangle(
            (x1, y1),
            squre_size,
            squre_size,
            linewidth=1,
            edgecolor='black',
            facecolor='none',
            zorder=100
        )
    )

    axs[1].tick_params(
        left=False,
        right=False,
        labelleft=False,
        labelbottom=False,
        bottom=False
    )
    axs[1].set_xlabel('')
    axs[1].set_ylabel('')
    axs[1].set_facecolor('xkcd:white')

    if add_colorbar:
        mapper.set_array([])
        axs[1].figure.colorbar(
            mapper,
            ax=axs[1],
            fraction=0.03,
            pad=0.01
        )

    for ax in axs:
        ax.set_aspect('equal', adjustable='box')
        ax.margins(0)

    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    os.makedirs(save_path, exist_ok=True)

    png_path = '%s/%s_%d_%d_v4.png' % (save_path, file, cell, rectangle)
    svg_path = '%s/%s_%d_%d_v4.svg' % (save_path, file, cell, rectangle)

    plt.savefig(
        png_path,
        bbox_inches=None,
        pad_inches=0,
        transparent=True,
        dpi=600
    )

    plt.savefig(
        svg_path,
        bbox_inches=None,
        pad_inches=0,
        transparent=True
    )

    plt.close(fig)

    print(png_path)
    print(svg_path)

    return ss.loc[:, [
        'x',
        'y',
        'parent',
        'element',
        'cell',
        'rectangle',
        'replicate',
        'name'
    ]]

In [ ]:
HTR7_B_1_2_1 = voro2cluster_v4('HTR7_Basal_1_2',squre_size=4000,x1=27000,y1=20000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=1)


In [ ]:
HTR7_B_1_2_1 = voro2cluster_v4('HTR7_Basal_1_2',squre_size=4000,x1=27000,y1=20000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=1)
HTR7_B_1_2_2 = voro2cluster_v4('HTR7_Basal_1_2',squre_size=4000,x1=26000,y1=24000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=1)
HTR7_B_1_2_3 = voro2cluster_v4('HTR7_Basal_1_2',squre_size=4000,x1=5000,y1=27000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=1)

HTR7_B_1_3_1 = voro2cluster_v4('HTR7_Basal_1_3',squre_size=4000,x1=25000,y1=28500,parent='HTR7',element='WT',cell=3,rectangle=1,replicate=1)
HTR7_B_1_3_2 = voro2cluster_v4('HTR7_Basal_1_3',squre_size=4000,x1=29000,y1=28500,parent='HTR7',element='WT',cell=3,rectangle=2,replicate=1)
HTR7_B_1_3_3 = voro2cluster_v4('HTR7_Basal_1_3',squre_size=4000,x1=33000,y1=26500,parent='HTR7',element='WT',cell=3,rectangle=3,replicate=1)

HTR7_B_1_4_1 = voro2cluster_v4('HTR7_Basal_1_4',squre_size=4000,x1=8000,y1=27000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=1)
HTR7_B_1_4_2 = voro2cluster_v4('HTR7_Basal_1_4',squre_size=4000,x1=31000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=1)
HTR7_B_1_4_3 = voro2cluster_v4('HTR7_Basal_1_4',squre_size=4000,x1=41000,y1=33500,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=1)

HTR7_B_1_5_1 = voro2cluster_v4('HTR7_Basal_1_5',squre_size=4000,x1=17000,y1=33500,parent='HTR7',element='WT',cell=5,rectangle=1,replicate=1)
HTR7_B_1_5_2 = voro2cluster_v4('HTR7_Basal_1_5',squre_size=4000,x1=27000,y1=14500,parent='HTR7',element='WT',cell=5,rectangle=2,replicate=1)
HTR7_B_1_5_3 = voro2cluster_v4('HTR7_Basal_1_5',squre_size=4000,x1=31000,y1=14500,parent='HTR7',element='WT',cell=5,rectangle=3,replicate=1)
HTR7_B_1_5_4 = voro2cluster_v4('HTR7_Basal_1_5',squre_size=4000,x1=41000,y1=3000,parent='HTR7',element='WT',cell=5,rectangle=4,replicate=1)

HTR7_B_1_6_1 = voro2cluster_v4('HTR7_Basal_1_6',squre_size=4000,x1=15000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=1)
HTR7_B_1_6_2 = voro2cluster_v4('HTR7_Basal_1_6',squre_size=4000,x1=19000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=1)
HTR7_B_1_6_3 = voro2cluster_v4('HTR7_Basal_1_6',squre_size=4000,x1=24000,y1=20000,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=1)

HTR7_B_1_7_1 = voro2cluster_v4('HTR7_Basal_1_7',squre_size=4000,x1=34000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=1,replicate=1)
HTR7_B_1_7_2 = voro2cluster_v4('HTR7_Basal_1_7',squre_size=4000,x1=38000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=2,replicate=1)
HTR7_B_1_7_3 = voro2cluster_v4('HTR7_Basal_1_7',squre_size=4000,x1=42000,y1=30000,parent='HTR7',element='WT',cell=7,rectangle=3,replicate=1)


HTR7_B_1_8_1 = voro2cluster_v4('HTR7_Basal_1_8',squre_size=4000,x1=35000,y1=30000,parent='HTR7',element='WT',cell=8,rectangle=1,replicate=1)
HTR7_B_1_8_2 = voro2cluster_v4('HTR7_Basal_1_8',squre_size=4000,x1=30000,y1=27000,parent='HTR7',element='WT',cell=8,rectangle=2,replicate=1)
HTR7_B_1_8_3 = voro2cluster_v4('HTR7_Basal_1_8',squre_size=4000,x1=26000,y1=27000,parent='HTR7',element='WT',cell=8,rectangle=3,replicate=1)
HTR7_B_1_8_4 = voro2cluster_v4('HTR7_Basal_1_8',squre_size=4000,x1=20000,y1=34000,parent='HTR7',element='WT',cell=8,rectangle=4,replicate=1)

HTR7_B_1_9_1 = voro2cluster_v4('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=31000,parent='HTR7',element='WT',cell=9,rectangle=1,replicate=1)
HTR7_B_1_9_2 = voro2cluster_v4('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=27000,parent='HTR7',element='WT',cell=9,rectangle=2,replicate=1)
HTR7_B_1_9_3 = voro2cluster_v4('HTR7_Basal_1_9',squre_size=4000,x1=26000,y1=23000,parent='HTR7',element='WT',cell=9,rectangle=3,replicate=1)
HTR7_B_1_9_4 = voro2cluster_v4('HTR7_Basal_1_9',squre_size=4000,x1=29000,y1=20000,parent='HTR7',element='WT',cell=9,rectangle=4,replicate=1)

HTR7_B_1_10_1 = voro2cluster_v4('HTR7_Basal_1_10',squre_size=4000,x1=17000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=1,replicate=1)
HTR7_B_1_10_2 = voro2cluster_v4('HTR7_Basal_1_10',squre_size=4000,x1=21000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=2,replicate=1)
HTR7_B_1_10_3 = voro2cluster_v4('HTR7_Basal_1_10',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='WT',cell=10,rectangle=3,replicate=1)
HTR7_B_1_10_4 = voro2cluster_v4('HTR7_Basal_1_10',squre_size=4000,x1=29000,y1=16500,parent='HTR7',element='WT',cell=10,rectangle=4,replicate=1)


HTR7_B_2_1_1 = voro2cluster_v4('HTR7_B_2_1',squre_size=4000,x1=27000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=1,replicate=2)
HTR7_B_2_1_2 = voro2cluster_v4('HTR7_B_2_1',squre_size=4000,x1=23000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=2,replicate=2)
HTR7_B_2_1_3 = voro2cluster_v4('HTR7_B_2_1',squre_size=4000,x1=19000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=3,replicate=2)
HTR7_B_2_1_4 = voro2cluster_v4('HTR7_B_2_1',squre_size=4000,x1=15000,y1=16500,parent='HTR7',element='WT',cell=1,rectangle=4,replicate=2)
HTR7_B_2_1_5 = voro2cluster_v4('HTR7_B_2_1',squre_size=4000,x1=18000,y1=20000,parent='HTR7',element='WT',cell=1,rectangle=5,replicate=2)

HTR7_B_2_2_1 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=26000,y1=10000,parent='HTR7',element='WT',cell=2,rectangle=1,replicate=2)
HTR7_B_2_2_2 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=26000,y1=14000,parent='HTR7',element='WT',cell=2,rectangle=2,replicate=2)
HTR7_B_2_2_3 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=26000,y1=18000,parent='HTR7',element='WT',cell=2,rectangle=3,replicate=2)
HTR7_B_2_2_4 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=26000,y1=22000,parent='HTR7',element='WT',cell=2,rectangle=4,replicate=2)
HTR7_B_2_2_5 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='WT',cell=2,rectangle=5,replicate=2)
HTR7_B_2_2_6 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='WT',cell=2,rectangle=6,replicate=2)
HTR7_B_2_2_7 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=40000,y1=10000,parent='HTR7',element='WT',cell=2,rectangle=7,replicate=2)
HTR7_B_2_2_8 = voro2cluster_v4('HTR7_B_2_2',squre_size=4000,x1=40000,y1=14000,parent='HTR7',element='WT',cell=2,rectangle=8,replicate=2)



HTR7_B_2_3_1 = voro2cluster_v4('HTR7_B_2_3',squre_size=4000,x1=30000,y1=14000,parent='HTR7',element='WT',cell=3,rectangle=1,replicate=2)
HTR7_B_2_3_2 = voro2cluster_v4('HTR7_B_2_3',squre_size=4000,x1=26000,y1=14000,parent='HTR7',element='WT',cell=3,rectangle=2,replicate=2)
HTR7_B_2_3_3 = voro2cluster_v4('HTR7_B_2_3',squre_size=4000,x1=26000,y1=18000,parent='HTR7',element='WT',cell=3,rectangle=3,replicate=2)
HTR7_B_2_3_4 = voro2cluster_v4('HTR7_B_2_3',squre_size=4000,x1=30000,y1=18000,parent='HTR7',element='WT',cell=3,rectangle=4,replicate=2)
HTR7_B_2_3_5 = voro2cluster_v4('HTR7_B_2_3',squre_size=4000,x1=20000,y1=20000,parent='HTR7',element='WT',cell=3,rectangle=5,replicate=2)


HTR7_B_2_4_1 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=18000,y1=21000,parent='HTR7',element='WT',cell=4,rectangle=1,replicate=2)
HTR7_B_2_4_2 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=18000,y1=25000,parent='HTR7',element='WT',cell=4,rectangle=2,replicate=2)
HTR7_B_2_4_3 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=18000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=3,replicate=2)
HTR7_B_2_4_4 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=19000,y1=37000,parent='HTR7',element='WT',cell=4,rectangle=4,replicate=2)
HTR7_B_2_4_5 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=24000,y1=37000,parent='HTR7',element='WT',cell=4,rectangle=5,replicate=2)
HTR7_B_2_4_6 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=24000,y1=33000,parent='HTR7',element='WT',cell=4,rectangle=6,replicate=2)
HTR7_B_2_4_7 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=35000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=7,replicate=2)
HTR7_B_2_4_8 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=31000,y1=13000,parent='HTR7',element='WT',cell=4,rectangle=8,replicate=2)
HTR7_B_2_4_9 = voro2cluster_v4('HTR7_B_2_4',squre_size=4000,x1=35000,y1=8000,parent='HTR7',element='WT',cell=4,rectangle=9,replicate=2)


HTR7_B_2_5_1 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=30000,y1=8000,parent='HTR7',element='WT',cell=5,rectangle=1,replicate=2)
HTR7_B_2_5_2 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=26000,y1=8000,parent='HTR7',element='WT',cell=5,rectangle=2,replicate=2)
HTR7_B_2_5_3 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=26000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=3,replicate=2)
HTR7_B_2_5_4 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=37000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=4,replicate=2)
HTR7_B_2_5_5 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=42000,y1=12000,parent='HTR7',element='WT',cell=5,rectangle=5,replicate=2)
HTR7_B_2_5_6 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=42000,y1=16000,parent='HTR7',element='WT',cell=5,rectangle=6,replicate=2)
HTR7_B_2_5_7 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=36000,y1=16000,parent='HTR7',element='WT',cell=5,rectangle=7,replicate=2)
HTR7_B_2_5_8 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=10000,y1=40000,parent='HTR7',element='WT',cell=5,rectangle=8,replicate=2)
HTR7_B_2_5_9 = voro2cluster_v4('HTR7_B_2_5',squre_size=4000,x1=2000,y1=30000,parent='HTR7',element='WT',cell=5,rectangle=9,replicate=2)


HTR7_B_2_6_1 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=2000,y1=10000,parent='HTR7',element='WT',cell=6,rectangle=1,replicate=2)
HTR7_B_2_6_2 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=2000,y1=14000,parent='HTR7',element='WT',cell=6,rectangle=2,replicate=2)
HTR7_B_2_6_3 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=8000,y1=14000,parent='HTR7',element='WT',cell=6,rectangle=3,replicate=2)
HTR7_B_2_6_4 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=15000,y1=24000,parent='HTR7',element='WT',cell=6,rectangle=4,replicate=2)
HTR7_B_2_6_5 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=19000,y1=24000,parent='HTR7',element='WT',cell=6,rectangle=5,replicate=2)
HTR7_B_2_6_6 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=25000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=6,replicate=2)
HTR7_B_2_6_7 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=30000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=7,replicate=2)
HTR7_B_2_6_8 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=35000,y1=30000,parent='HTR7',element='WT',cell=6,rectangle=8,replicate=2)
HTR7_B_2_6_9 = voro2cluster_v4('HTR7_B_2_6',squre_size=4000,x1=40000,y1=25000,parent='HTR7',element='WT',cell=6,rectangle=9,replicate=2)


HTR7_B_2_8_1 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=40500,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=1,replicate=2)
HTR7_B_2_8_2 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=38000,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=2,replicate=2)
HTR7_B_2_8_3 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=34000,y1=42000,parent='HTR7',element='WT',cell=8,rectangle=3,replicate=2)
HTR7_B_2_8_4 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=30000,y1=40000,parent='HTR7',element='WT',cell=8,rectangle=4,replicate=2)
HTR7_B_2_8_5 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=30000,y1=36000,parent='HTR7',element='WT',cell=8,rectangle=5,replicate=2)
HTR7_B_2_8_6 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=30000,y1=32000,parent='HTR7',element='WT',cell=8,rectangle=6,replicate=2)
HTR7_B_2_8_7 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=30000,y1=28000,parent='HTR7',element='WT',cell=8,rectangle=7,replicate=2)
HTR7_B_2_8_8 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=30000,y1=23000,parent='HTR7',element='WT',cell=8,rectangle=8,replicate=2)
HTR7_B_2_8_9 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=26000,y1=23000,parent='HTR7',element='WT',cell=8,rectangle=9,replicate=2)
HTR7_B_2_8_10 = voro2cluster_v4('HTR7_B_2_8',squre_size=4000,x1=600,y1=19000,parent='HTR7',element='WT',cell=8,rectangle=10,replicate=2)


HTR7_B_2_9_1 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=23000,y1=8000,parent='HTR7',element='WT',cell=9,rectangle=1,replicate=2)
HTR7_B_2_9_2 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=23000,y1=13000,parent='HTR7',element='WT',cell=9,rectangle=2,replicate=2)
HTR7_B_2_9_3 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=23000,y1=17000,parent='HTR7',element='WT',cell=9,rectangle=3,replicate=2)
HTR7_B_2_9_4 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=23000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=4,replicate=2)
HTR7_B_2_9_5 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=27000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=5,replicate=2)
HTR7_B_2_9_6 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=31000,y1=22000,parent='HTR7',element='WT',cell=9,rectangle=6,replicate=2)
HTR7_B_2_9_7 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=31000,y1=26000,parent='HTR7',element='WT',cell=9,rectangle=7,replicate=2)
HTR7_B_2_9_8 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=28000,y1=30000,parent='HTR7',element='WT',cell=9,rectangle=8,replicate=2)
HTR7_B_2_9_9 = voro2cluster_v4('HTR7_B_2_9',squre_size=4000,x1=24000,y1=30000,parent='HTR7',element='WT',cell=9,rectangle=9,replicate=2)


HTR7_B_2_10_1 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=26000,y1=30000,parent='HTR7',element='WT', cell=10,rectangle=1,replicate=2)
HTR7_B_2_10_2 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=26000,y1=34000,parent='HTR7',element='WT',cell=10,rectangle=2,replicate=2)
HTR7_B_2_10_3 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=24000,y1=38000,parent='HTR7',element='WT',cell=10,rectangle=3,replicate=2)
HTR7_B_2_10_4 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=27000,y1=20000,parent='HTR7',element='WT',cell=10,rectangle=4,replicate=2)
HTR7_B_2_10_5 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=27000,y1=16000,parent='HTR7',element='WT',cell=10,rectangle=5,replicate=2)
HTR7_B_2_10_6 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=22000,y1=18000,parent='HTR7',element='WT',cell=10,rectangle=6,replicate=2)
HTR7_B_2_10_7 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=12000,y1=28000,parent='HTR7',element='WT',cell=10,rectangle=7,replicate=2)
HTR7_B_2_10_8 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=12000,y1=33000,parent='HTR7',element='WT',cell=10,rectangle=8,replicate=2)
HTR7_B_2_10_9 = voro2cluster_v4('HTR7_B_2_10',squre_size=4000,x1=8000,y1=33000,parent='HTR7',element='WT',cell=10,rectangle=9,replicate=2)

HTR7_BM_1_2_1 = voro2cluster_v4('HTR7_MUT_Basal_1_2',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=1)
HTR7_BM_1_2_2 = voro2cluster_v4('HTR7_MUT_Basal_1_2',squre_size=4000,x1=34000,y1=23000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=1)

HTR7_BM_1_3_1 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=34000,y1=24000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=1)
HTR7_BM_1_3_2 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=30000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=1)
HTR7_BM_1_3_3 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=26000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=3,replicate=1)
HTR7_BM_1_3_4 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=22000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=4,replicate=1)
HTR7_BM_1_3_5 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=18000,y1=26000,parent='HTR7',element='MUT',cell=3,rectangle=5,replicate=1)
HTR7_BM_1_3_6 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=15000,y1=30000,parent='HTR7',element='MUT',cell=3,rectangle=6,replicate=1)
HTR7_BM_1_3_7 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=15000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=7,replicate=1)
HTR7_BM_1_3_8 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=5000,y1=15000,parent='HTR7',element='MUT',cell=3,rectangle=8,replicate=1)
HTR7_BM_1_3_9 = voro2cluster_v4('HTR7_MUT_Basal_1_3',squre_size=4000,x1=5000,y1=20000,parent='HTR7',element='MUT',cell=3,rectangle=9,replicate=1)


HTR7_BM_1_4_1 = voro2cluster_v4('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=34000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=1)
HTR7_BM_1_4_2 = voro2cluster_v4('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=30000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=1)
HTR7_BM_1_4_3 = voro2cluster_v4('HTR7_MUT_Basal_1_4',squre_size=4000,x1=20000,y1=26000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=1)
HTR7_BM_1_4_4 = voro2cluster_v4('HTR7_MUT_Basal_1_4',squre_size=4000,x1=19000,y1=22000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=1)
HTR7_BM_1_4_5 = voro2cluster_v4('HTR7_MUT_Basal_1_4',squre_size=4000,x1=17000,y1=29000,parent='HTR7',element='MUT',cell=4,rectangle=5,replicate=1)



HTR7_BM_1_5_1 = voro2cluster_v4('HTR7_MUT_Basal_1_5',squre_size=4000,x1=10000,y1=10000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=1)
HTR7_BM_1_5_2 = voro2cluster_v4('HTR7_MUT_Basal_1_5',squre_size=4000,x1=10000,y1=14000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=1)
HTR7_BM_1_5_3 = voro2cluster_v4('HTR7_MUT_Basal_1_5',squre_size=4000,x1=25000,y1=14000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=1)
HTR7_BM_1_5_4 = voro2cluster_v4('HTR7_MUT_Basal_1_5',squre_size=4000,x1=25000,y1=18000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=1)
HTR7_BM_1_5_5 = voro2cluster_v4('HTR7_MUT_Basal_1_5',squre_size=4000,x1=28000,y1=38000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=1)
HTR7_BM_1_5_6 = voro2cluster_v4('HTR7_MUT_Basal_1_5',squre_size=4000,x1=35000,y1=40000,parent='HTR7',element='MUT',cell=5,rectangle=6,replicate=1)


HTR7_BM_1_6_1 = voro2cluster_v4('HTR7_MUT_Basal_1_6',squre_size=4000,x1=17000,y1=26000,parent='HTR7',element='MUT',cell=6,rectangle=1,replicate=1)
HTR7_BM_1_6_2 = voro2cluster_v4('HTR7_MUT_Basal_1_6',squre_size=4000,x1=21000,y1=26000,parent='HTR7',element='MUT',cell=6,rectangle=2,replicate=1)

HTR7_BM_1_7_1 = voro2cluster_v4('HTR7_MUT_Basal_1_7',squre_size=4000,x1=21000,y1=26000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=1)
HTR7_BM_1_7_2 = voro2cluster_v4('HTR7_MUT_Basal_1_7',squre_size=4000,x1=21000,y1=30000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=1)


HTR7_BM_1_8_1 = voro2cluster_v4('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=30000,parent='HTR7',element='MUT',cell=8,rectangle=1,replicate=1)
HTR7_BM_1_8_2 = voro2cluster_v4('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=26000,parent='HTR7',element='MUT',cell=8,rectangle=2,replicate=1)
HTR7_BM_1_8_3 = voro2cluster_v4('HTR7_MUT_Basal_1_8',squre_size=4000,x1=25000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=3,replicate=1)
HTR7_BM_1_8_4 = voro2cluster_v4('HTR7_MUT_Basal_1_8',squre_size=4000,x1=20000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=4,replicate=1)
HTR7_BM_1_8_5 = voro2cluster_v4('HTR7_MUT_Basal_1_8',squre_size=4000,x1=40000,y1=18000,parent='HTR7',element='MUT',cell=8,rectangle=5,replicate=1)

HTR7_BM_1_9_1 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=30000,y1=12000,parent='HTR7',element='MUT',cell=9,rectangle=1,replicate=1)
HTR7_BM_1_9_2 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=12000,parent='HTR7',element='MUT',cell=9,rectangle=2,replicate=1)
HTR7_BM_1_9_3 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=16000,parent='HTR7',element='MUT',cell=9,rectangle=3,replicate=1)
HTR7_BM_1_9_4 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=27000,y1=18000,parent='HTR7',element='MUT',cell=9,rectangle=4,replicate=1)
HTR7_BM_1_9_5 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=27000,y1=25000,parent='HTR7',element='MUT',cell=9,rectangle=5,replicate=1)
HTR7_BM_1_9_6 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=26000,y1=29000,parent='HTR7',element='MUT',cell=9,rectangle=6,replicate=1)
HTR7_BM_1_9_7 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=22000,y1=29000,parent='HTR7',element='MUT',cell=9,rectangle=7,replicate=1)
HTR7_BM_1_9_8 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=18000,y1=30000,parent='HTR7',element='MUT', cell=9,rectangle=8,replicate=1)
HTR7_BM_1_9_9 = voro2cluster_v4('HTR7_MUT_Basal_1_9',squre_size=4000,x1=18000,y1=34000,parent='HTR7',element='MUT', cell=9,rectangle=9,replicate=1)


HTR7_BM_2_2_1 = voro2cluster_v4('HTR7Mut_B_2_2',squre_size=4000,x1=24000,y1=24000,parent='HTR7',element='MUT',cell=2,rectangle=1,replicate=2)
HTR7_BM_2_2_2 = voro2cluster_v4('HTR7Mut_B_2_2',squre_size=4000,x1=24000,y1=20000,parent='HTR7',element='MUT',cell=2,rectangle=2,replicate=2)
HTR7_BM_2_2_3 = voro2cluster_v4('HTR7Mut_B_2_2',squre_size=4000,x1=22000,y1=37000,parent='HTR7',element='MUT',cell=2,rectangle=3,replicate=2)


HTR7_BM_2_3_1 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=14000,y1=32000,parent='HTR7',element='MUT',cell=3,rectangle=1,replicate=2)
HTR7_BM_2_3_2 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=10000,y1=32000,parent='HTR7',element='MUT',cell=3,rectangle=2,replicate=2)
HTR7_BM_2_3_3 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=25000,y1=20000,parent='HTR7',element='MUT',cell=3,rectangle=3,replicate=2)
HTR7_BM_2_3_4 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=24000,y1=16000,parent='HTR7',element='MUT',cell=3,rectangle=4,replicate=2)
HTR7_BM_2_3_5 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=24000,y1=12000,parent='HTR7',element='MUT',cell=3,rectangle=5,replicate=2)
HTR7_BM_2_3_6 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=20000,y1=8000,parent='HTR7',element='MUT',cell=3,rectangle=6,replicate=2)
HTR7_BM_2_3_7 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=16000,y1=8000,parent='HTR7',element='MUT',cell=3,rectangle=7,replicate=2)
HTR7_BM_2_3_8 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=40000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=8,replicate=2)
HTR7_BM_2_3_9 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=45000,y1=35000,parent='HTR7',element='MUT',cell=3,rectangle=9,replicate=2)
HTR7_BM_2_3_10 = voro2cluster_v4('HTR7Mut_B_2_3',squre_size=4000,x1=43000,y1=39000,parent='HTR7',element='MUT',cell=3,rectangle=10,replicate=2)


HTR7_BM_2_4_1 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=25000,parent='HTR7',element='MUT',cell=4,rectangle=1,replicate=2)
HTR7_BM_2_4_2 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=29000,parent='HTR7',element='MUT',cell=4,rectangle=2,replicate=2)
HTR7_BM_2_4_3 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=20000,y1=33000,parent='HTR7',element='MUT',cell=4,rectangle=3,replicate=2)
HTR7_BM_2_4_4 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=16000,y1=33000,parent='HTR7',element='MUT',cell=4,rectangle=4,replicate=2)
HTR7_BM_2_4_5 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=16000,y1=27000,parent='HTR7',element='MUT',cell=4,rectangle=5,replicate=2)
HTR7_BM_2_4_6 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=3000,parent='HTR7',element='MUT', cell=4,rectangle=6,replicate=2)
HTR7_BM_2_4_7 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=7000,parent='HTR7',element='MUT', cell=4,rectangle=7,replicate=2)
HTR7_BM_2_4_8 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=3000,y1=11000,parent='HTR7',element='MUT',cell=4,rectangle=8,replicate=2)
HTR7_BM_2_4_9 = voro2cluster_v4('HTR7Mut_B_2_4',squre_size=4000,x1=36000,y1=4000,parent='HTR7',element='MUT',cell=4,rectangle=9,replicate=2)


HTR7_BM_2_5_1 = voro2cluster_v4('HTR7Mut_B_2_5',squre_size=4000,x1=25000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=1,replicate=2)
HTR7_BM_2_5_2 = voro2cluster_v4('HTR7Mut_B_2_5',squre_size=4000,x1=29000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=2,replicate=2)
HTR7_BM_2_5_3 = voro2cluster_v4('HTR7Mut_B_2_5',squre_size=4000,x1=33000,y1=23000,parent='HTR7',element='MUT',cell=5,rectangle=3,replicate=2)
HTR7_BM_2_5_4 = voro2cluster_v4('HTR7Mut_B_2_5',squre_size=4000,x1=33000,y1=27000,parent='HTR7',element='MUT',cell=5,rectangle=4,replicate=2)
HTR7_BM_2_5_5 = voro2cluster_v4('HTR7Mut_B_2_5',squre_size=4000,x1=25000,y1=17000,parent='HTR7',element='MUT',cell=5,rectangle=5,replicate=2)


HTR7_BM_2_7_1 = voro2cluster_v4('HTR7Mut_B_2_7',squre_size=4000,x1=20000,y1=17000,parent='HTR7',element='MUT',cell=7,rectangle=1,replicate=2)
HTR7_BM_2_7_2 = voro2cluster_v4('HTR7Mut_B_2_7',squre_size=4000,x1=16000,y1=17000,parent='HTR7',element='MUT',cell=7,rectangle=2,replicate=2)
HTR7_BM_2_7_3 = voro2cluster_v4('HTR7Mut_B_2_7',squre_size=4000,x1=20000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=3,replicate=2)
HTR7_BM_2_7_4 = voro2cluster_v4('HTR7Mut_B_2_7',squre_size=4000,x1=25000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=4,replicate=2)
HTR7_BM_2_7_5 = voro2cluster_v4('HTR7Mut_B_2_7',squre_size=4000,x1=30000,y1=22000,parent='HTR7',element='MUT',cell=7,rectangle=5,replicate=2)


HTR7_BM_2_8_1 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=27000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=1,replicate=2)
HTR7_BM_2_8_2 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=22000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=2,replicate=2)
HTR7_BM_2_8_3 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=17000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=3,replicate=2)
HTR7_BM_2_8_4 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=13000,y1=22000,parent='HTR7',element='MUT',cell=8,rectangle=4,replicate=2)
HTR7_BM_2_8_5 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=16000,y1=27000,parent='HTR7',element='MUT',cell=8,rectangle=5,replicate=2)
HTR7_BM_2_8_6 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=13000,y1=15000,parent='HTR7',element='MUT',  cell=8,rectangle=6,replicate=2)
HTR7_BM_2_8_7 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=19000,y1=15000,parent='HTR7',element='MUT',  cell=8,rectangle=7,replicate=2)
HTR7_BM_2_8_8 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=30000,y1=13000,parent='HTR7',element='MUT', cell=8,rectangle=8,replicate=2)
HTR7_BM_2_8_9 = voro2cluster_v4('HTR7Mut_B_2_8',squre_size=4000,x1=30000,y1=9000,parent='HTR7',element='MUT', cell=8,rectangle=9,replicate=2)


HTR7_BM_2_10_1 = voro2cluster_v4('HTR7Mut_B_2_10',squre_size=4000,x1=30000,y1=9000,parent='HTR7',element='MUT',cell=10,rectangle=1,replicate=2)
HTR7_BM_2_10_2 = voro2cluster_v4('HTR7Mut_B_2_10',squre_size=4000,x1=26000,y1=13000,parent='HTR7',element='MUT',cell=10,rectangle=2,replicate=2)
HTR7_BM_2_10_3 = voro2cluster_v4('HTR7Mut_B_2_10',squre_size=4000,x1=25000,y1=20000,parent='HTR7',element='MUT',cell=10,rectangle=3,replicate=2)
HTR7_BM_2_10_4 = voro2cluster_v4('HTR7Mut_B_2_10',squre_size=4000,x1=21000,y1=25000,parent='HTR7',element='MUT',cell=10,rectangle=4,replicate=2)
HTR7_BM_2_10_5 = voro2cluster_v4('HTR7Mut_B_2_10',squre_size=4000,x1=21000,y1=29000,parent='HTR7',element='MUT',cell=10,rectangle=5,replicate=2)
HTR7_BM_2_10_6 = voro2cluster_v4('HTR7Mut_B_2_10',squre_size=4000,x1=20000,y1=33000,parent='HTR7',element='MUT',cell=10,rectangle=6,replicate=2)


In [ ]:
import os
import glob

In [ ]:
images_main = []
for i in pd.Series(glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/figures/HTR7_Science/HTR7_MUT*.svg')).sample(2,random_state=42).to_list():
    images_main.append(i)
for i in pd.Series(glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/figures/HTR7_Science/HTR7Mut_*.svg')).sample(1,random_state=42).to_list():
    images_main.append(i)
for i in pd.Series(glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/figures/HTR7_Science/HTR7_B_*.svg')).sample(1,random_state=42).to_list():
    images_main.append(i)
for i in pd.Series(glob.glob('/Users/aelazar/notebooks_for_paper/dSTORM/figures/HTR7_Science/HTR7_Basal_*.svg')).sample(2,random_state=42).to_list():
    images_main.append(i)
    
images_main


In [ ]:

fig = plt.figure(figsize = (4,2),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=1,gridspec_kw = {'hspace':.01,'wspace':.2})
spec_basal = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.state=='basal'],hue='ele',hue_order=['WT','MUT'])
spec_stim = dict(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.state!='basal'],hue='ele',hue_order=['WT','MUT'])

sns.stripplot(**spec_basal,
    dodge=True, alpha=1, zorder=1, legend=False,s=3,palette=['#C03830','#B4B5B4'],order=['0-70','71-150'])
sns.pointplot(x='densitiy',y='cluster_sizes_full_vals_nm_mean',data=bins_df[bins_df.state=='basal'].groupby(['densitiy','ele']).mean().reset_index(),hue='ele',hue_order=['WT','MUT'],
    join=False, dodge=.8 - .8 / 2,order=['0-70','71-150'],
    markers="_", scale=1.2, errorbar=None,palette=['k','k'],ax=axs[0])



In [ ]:
# sns.ecdfplot(x='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy=='0-70')],hue='ele',hue_order=['WT','MUT'])
fig, ax = plt.subplots(figsize=(2,3))
sns.violinplot(y='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')],inner=None,
               hue='ele',hue_order=['WT','MUT'],log_scale=(False,True),fill=False,palette=['#C03830','#B4B5B4'])
sns.stripplot(y='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')],hue='ele',hue_order=['WT','MUT']
              ,log_scale=(False,True),dodge=True,s=2,palette=['#C03830','#B4B5B4'],zorder=1)
plt.ylim([0,900])
plt.legend([],frameon=False)
plt.savefig('Localizations_per_clsuter_HTR7_high_dens.svg',bbox_inches='tight',transparent=False)


In [ ]:
# sns.ecdfplot(x='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy=='0-70')],hue='ele',hue_order=['WT','MUT'])
fig, ax = plt.subplots(figsize=(2,3))
sns.violinplot(y='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')],inner=None,
               hue='ele',hue_order=['WT','MUT'],log_scale=(False,True),fill=False,palette=['#C03830','#B4B5B4'])
sns.stripplot(y='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')],hue='ele',hue_order=['WT','MUT']
              ,log_scale=(False,True),dodge=True,s=2,palette=['#C03830','#B4B5B4'],zorder=1)
plt.ylim([0,900])
plt.legend([],frameon=False)
plt.savefig('Localizations_per_clsuter_HTR7_high_dens.svg',bbox_inches='tight',transparent=False)


In [ ]:
fig, ax = plt.subplots(figsize=(2,3))
sns.violinplot(y='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy=='0-70')],inner=None,
               hue='ele',hue_order=['WT','MUT'],log_scale=(False,True),fill=False,palette=['#C03830','#B4B5B4'])
sns.stripplot(y='cluster_sizes_full_vals_nm_mean',data=bins_df[(bins_df.state=='basal')&(bins_df.densitiy=='0-70')],hue='ele',hue_order=['WT','MUT']
              ,log_scale=(False,True),dodge=True,s=2,palette=['#C03830','#B4B5B4'])
plt.ylim([0,900])
plt.legend([],frameon=False)
plt.savefig('Localizations_per_clsuter_HTR7_low_dens.svg',bbox_inches='tight',transparent=False)


In [ ]:
bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')].groupby(['ele']).describe()

In [ ]:
bins_df[(bins_df.state=='basal')&(bins_df.densitiy=='0-70')].groupby(['ele']).describe()

In [ ]:
stats.ttest_ind(bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')&(bins_df.ele=='WT')].cluster_sizes_full_vals_nm_mean,
                bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')&(bins_df.ele=='MUT')].cluster_sizes_full_vals_nm_mean)

In [ ]:
from scipy.stats import mannwhitneyu

mannwhitneyu(
bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')&(bins_df.ele=='WT')].cluster_sizes_full_vals_nm_mean,
                bins_df[(bins_df.state=='basal')&(bins_df.densitiy!='0-70')&(bins_df.ele=='MUT')].cluster_sizes_full_vals_nm_mean,
    alternative='two-sided'
)

In [ ]:
grid_df_HTR7_basal.all_points.hist()